# XE từ đầu · đối chứng CLIP và RegionCLIP
Đặt ARM = CLIP hoặc REGIONCLIP và dùng 1 GPU T4. Gắn output v150 hoặc một Dataset có chính sáu file NPY của v150. Mặc định không dùng Kaggle API. Bật Internet để cài bộ chấm.
Gắn thêm csa-scst-full-karpathy-cache-v1, csa-prompt-full-karpathy-v175, csa-semantic-banks-v130 và mscoco-2014. Giữ best.bin và gửi ZIP review.


In [ ]:
from pathlib import Path
import hashlib, io, json, math, os, subprocess, sys, tempfile, urllib.request, zipfile
import numpy as np

ARM = 'CLIP'   # Run separately with REGIONCLIP on the other notebook.
SECRET_NAME = 'codex'
MOUNTED_INPUTS = ''  # Optional exact folder containing the six v150 files.
ALLOW_API_FALLBACK = False
PATCH_ROOT = Path('/kaggle/input/datasets/ducanh3/csa-scst-full-karpathy-cache-v1')
PROMPT_ROOT = Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
BANK_ROOT = Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
COCO_JSON = Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
if ARM not in ('CLIP','REGIONCLIP'):
    raise ValueError('Select CLIP or REGIONCLIP')
for path in (PATCH_ROOT, PROMPT_ROOT, BANK_ROOT, COCO_JSON):
    if not path.exists():
        raise FileNotFoundError(path)
OUT = Path('/kaggle/working/regionclip_xe10k_'+ARM.lower())
OUT.mkdir(exist_ok=False)
WORK = Path(tempfile.mkdtemp(prefix='regionclip-xe10k-'))

def checked(path, expected):
    with path.open('rb') as stream:
        if hashlib.file_digest(stream,'sha256').hexdigest()!=expected:
            raise ValueError(f'Changed file: {path}')
    return path

def write_json(path, value):
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n',encoding='utf8')
    tmp.replace(path)

def read(path):
    return json.loads(path.read_bytes())

def download(url,path,pin):
    with urllib.request.urlopen(url,timeout=90) as response:
        path.write_bytes(response.read())
    return checked(path,pin)

def fetch_inputs():
    if MOUNTED_INPUTS:
        root=Path(MOUNTED_INPUTS)
        if not all((root/name).is_file() for name in PINS):
            raise FileNotFoundError(f'Expected all six v150 NPY files in {root}')
        return root
    matches=[]
    for path in sorted(Path('/kaggle/input').rglob('text_vectors.npy')):
        root=path.parent
        if all((root/name).is_file() for name in PINS) and hashlib.sha256(path.read_bytes()).hexdigest()==PINS[path.name]:
            matches.append(root)
    if len(matches)==1:
        return matches[0]
    if matches:
        raise ValueError('Set MOUNTED_INPUTS to the exact v150 directory: '+', '.join(map(str,matches)))
    if not ALLOW_API_FALLBACK:
        raise FileNotFoundError('Attach the v150 output or a Dataset containing its six NPY files. API fallback disabled.')
    from kaggle_secrets import UserSecretsClient
    cli=WORK/'cli'
    subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(cli),'kaggle==2.2.5'],
                   check=True,timeout=300)
    env=dict(os.environ,PYTHONPATH=str(cli))
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
        env.pop(key,None)
    env['KAGGLE_API_TOKEN']=UserSecretsClient().get_secret(SECRET_NAME).strip()
    dest=WORK/'download'
    dest.mkdir()
    subprocess.run([sys.executable,'-c','from kaggle.cli import main; main()',
                    'kernels','output','ducanh3/notebook7bc83c4c76/150','-p',str(dest),
                    '--file-pattern',r'(^|/)regionclip_xe10k_inputs/(boxes|coco_ids|region_vectors|text_id|text_vectors|valid)[.]npy$'],
                   env=env,check=True,timeout=900)
    path,=dest.rglob('coco_ids.npy')
    return path.parent


In [ ]:
MODULES={'csa_patch_model.py': '"""Paper-structure CSA encoder/parallel decoder; CLIP-patch + UPT extensions disclosed."""\nimport torch\nfrom torch import nn\nfrom bank_layers import CSACaptioner, Config\nfrom semantic_patch_model import PromptMemory\n\n\nclass CSAPatchCaptioner(nn.Module):\n    def __init__(self, vocab, nouns, verbs):\n        super().__init__()\n        self.core = CSACaptioner(Config(vocab=vocab), nouns, verbs)\n        self.object_prompt = PromptMemory(512 + 512 + 4)\n        self.relation_prompt = PromptMemory(512 + 512 + 512 + 8)\n\n    @property\n    def head(self):\n        return self.core.head\n\n    def streams(self, patches, p):\n        if patches.ndim != 3 or patches.shape[1:] != (196, 768):\n            raise ValueError(\'Expected196CLIPpatches, not ROI indices/features\')\n        valid = torch.ones(patches.shape[:2], dtype=torch.bool, device=patches.device)\n        vo, vr = self.core.encode(patches, valid)\n        obj = torch.cat((p[\'object_text\'], torch.bmm(p[\'object_weights\'], vo), p[\'object_box\']), -1)\n        rel = torch.cat((p[\'relation_text\'], torch.bmm(p[\'subject_weights\'], vo),\n                         torch.bmm(p[\'target_weights\'], vo), p[\'relation_geometry\']), -1)\n        return (self.object_prompt(vo, obj, p[\'object_weights\'], p[\'object_valid\']),\n                self.relation_prompt(vr, rel, p[\'subject_weights\'] + p[\'target_weights\'], p[\'relation_valid\']))\n\n    def encode(self, patches, p):\n        # Packaging for existing beam expansion only; decode restores both streams.\n        return torch.cat(self.streams(patches, p), dim=1)\n\n    def decode(self, memory, prefix):\n        if memory.ndim != 3 or memory.shape[1:] != (392, 512):\n            raise ValueError(\'Expected packaged pair of196x512memories\')\n        vo, vr = memory.split(196, dim=1)\n        valid = torch.ones(vo.shape[:2], dtype=torch.bool, device=vo.device)\n        return self.core.decode(prefix, vo, vr, valid)\n\n    def forward(self, patches, prefix, p):\n        return self.decode(self.encode(patches, p), prefix)\n', 'semantic_patch_model.py': '"""User-run bank/prompt patch memory; unchanged baseline causal decoder."""\nimport torch\nfrom torch import nn\nfrom patch_model import PatchCaptioner\nfrom bank_layers import Config,SemanticAggregation,Alignment\nfrom memory_fusion import fuse_memory\n\n\nclass BankMemory(nn.Module):\n    def __init__(self):\n        super().__init__();self.aggregate=SemanticAggregation(Config(vocab=10000,dropout=0))\n        self.align=Alignment();self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)\n        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)\n    def forward(self,visual,bank):\n        prototypes,_=self.aggregate(bank)\n        selected,_=self.align(visual,prototypes,torch.ones(visual.shape[:2],dtype=torch.bool,device=visual.device))\n        return visual+self.attention(visual,selected,selected,need_weights=False)[0]\n\n\nclass PromptMemory(nn.Module):\n    def __init__(self,input_dim):\n        super().__init__();self.slot=nn.Sequential(nn.Linear(input_dim,512),nn.ReLU(),nn.Linear(512,512))\n        self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)\n        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)\n    def forward(self,visual,slot_inputs,support,valid):\n        slots=self.slot(slot_inputs)\n        allowed=support.transpose(1,2).gt(0)&valid[:,None,:]\n        active=allowed.any(-1)\n        # Give inactive queries one finite key, then zero their entire residual.\n        safe=allowed.clone();safe[:,:,0]|=~active\n        mask=(~safe)[:,None].expand(-1,8,-1,-1).reshape(-1,visual.shape[1],slots.shape[1])\n        delta=self.attention(visual,slots,slots,attn_mask=mask,need_weights=False)[0]\n        return visual+delta.masked_fill(~active[...,None],0)\n\n\nclass SemanticPatchCaptioner(nn.Module):\n    def __init__(self,vocab,nouns,verbs,strength_gate=False):\n        super().__init__()\n        # Construct the baseline FIRST; never reinitialize its modules recursively.\n        self.base=PatchCaptioner(vocab)\n        self.register_buffer(\'noun_bank\',nouns);self.register_buffer(\'verb_bank\',verbs)\n        self.object_bank=BankMemory();self.relation_bank=BankMemory()\n        self.object_prompt=PromptMemory(512+512+4)\n        self.relation_prompt=PromptMemory(512+512+512+8)\n        self.gate=nn.Linear(1024,512);nn.init.zeros_(self.gate.weight);nn.init.zeros_(self.gate.bias)\n        self.strength_gate=nn.Linear(1024,1) if strength_gate else None\n        if self.strength_gate is not None:\n            nn.init.zeros_(self.strength_gate.weight);nn.init.zeros_(self.strength_gate.bias)\n    @property\n    def project(self):return self.base.project\n    @property\n    def head(self):return self.base.head\n    def memories(self,patches,p,visual=None):\n        if visual is None:visual=self.base.encode(patches)\n        mo=self.object_bank(visual,self.noun_bank);mr=self.relation_bank(mo,self.verb_bank)\n        obj=torch.cat((p[\'object_text\'],torch.bmm(p[\'object_weights\'],mo),p[\'object_box\']),-1)\n        rel=torch.cat((p[\'relation_text\'],torch.bmm(p[\'subject_weights\'],mo),torch.bmm(p[\'target_weights\'],mo),p[\'relation_geometry\']),-1)\n        vo=self.object_prompt(mo,obj,p[\'object_weights\'],p[\'object_valid\'])\n        vr=self.relation_prompt(mr,rel,p[\'subject_weights\']+p[\'target_weights\'],p[\'relation_valid\'])\n        return vo,vr\n    def encode(self,patches,p):\n        if self.strength_gate is not None:return self.encode_with_gate(patches,p)[0]\n        vo,vr=self.memories(patches,p);gate=self.gate(torch.cat((vo,vr),-1)).sigmoid()\n        return gate*vo+(1-gate)*vr\n    def encode_with_gate(self,patches,p):\n        if self.strength_gate is None:raise ValueError(\'Strength-gated model required\')\n        visual=self.base.encode(patches);vo,vr=self.memories(patches,p,visual)\n        branch=self.gate(torch.cat((vo,vr),-1)).sigmoid()\n        delta=branch*(vo-visual)+(1-branch)*(vr-visual)\n        strength=self.strength_gate(torch.cat((visual,delta),-1)).sigmoid()\n        return fuse_memory(visual,vo,vr,branch,strength),strength\n    def decode(self,memory,prefix):return self.base.decode(memory,prefix)\n    def forward(self,patches,prefix,p):return self.decode(self.encode(patches,p),prefix)\n', 'patch_model.py': '"""User-run neural baseline: one patch memory, standard causal decoder."""\nimport math\nimport torch\nfrom torch import nn\n\nclass PatchCaptioner(nn.Module):\n    def __init__(self,vocab,dim=512,heads=8,layers=4,hidden=2048,dropout=.1,maximum=64):\n        super().__init__();self.dim=dim\n        self.project=nn.Sequential(nn.Linear(768,dim),nn.LayerNorm(dim))\n        self.embedding=nn.Embedding(vocab,dim,padding_idx=0)\n        p=torch.arange(maximum,dtype=torch.float32)[:,None];scale=torch.exp(torch.arange(0,dim,2)*(-math.log(10000)/dim))\n        pos=torch.zeros(maximum,dim);pos[:,0::2]=torch.sin(p*scale);pos[:,1::2]=torch.cos(p*scale)\n        self.register_buffer(\'positions\',pos,persistent=True);self.dropout=nn.Dropout(dropout)\n        layer=nn.TransformerDecoderLayer(dim,heads,hidden,dropout,batch_first=True,norm_first=False)\n        self.decoder=nn.TransformerDecoder(layer,layers,norm=nn.LayerNorm(dim));self.head=nn.Linear(dim,vocab)\n        for module in self.modules():\n            if isinstance(module,nn.Linear):\n                nn.init.xavier_uniform_(module.weight)\n                if module.bias is not None:nn.init.zeros_(module.bias)\n            elif isinstance(module,nn.Embedding):nn.init.normal_(module.weight,std=.02);module.weight.data[0].zero_()\n            elif isinstance(module,nn.MultiheadAttention):nn.init.xavier_uniform_(module.in_proj_weight);nn.init.zeros_(module.in_proj_bias)\n\n    def encode(self,patches):\n        if patches.ndim!=3 or patches.shape[1:]!=(196,768):raise ValueError(\'Wrong patch shape\')\n        return self.project(patches)\n\n    def decode(self,memory,prefix):\n        if prefix.ndim!=2 or len(prefix)!=len(memory):raise ValueError(\'Wrong memory/prefix shape\')\n        if prefix.shape[1]>len(self.positions) or (prefix==0).all(1).any():raise ValueError(\'Empty or too long prefix\')\n        text=self.dropout(self.embedding(prefix)*math.sqrt(self.dim)+self.positions[:prefix.shape[1]])\n        causal=torch.ones(prefix.shape[1],prefix.shape[1],dtype=torch.bool,device=prefix.device).triu(1)\n        output=self.decoder(text,memory,tgt_mask=causal,tgt_key_padding_mask=prefix==0)\n        return self.head(output)\n\n    def forward(self,patches,prefix):return self.decode(self.encode(patches),prefix)\n\n    @torch.no_grad()\n    def greedy(self,patches,maximum=20):\n        ids=torch.ones(len(patches),1,dtype=torch.long,device=patches.device);done=torch.zeros(len(patches),dtype=torch.bool,device=patches.device)\n        for _ in range(maximum):\n            word=self(patches,ids)[:,-1].argmax(-1);word=word.masked_fill(done,0)\n            ids=torch.cat((ids,word[:,None]),1);done|=word==2\n            if done.all():break\n        return ids\n', 'memory_fusion.py': '"""Shared neural/NumPy fusion algebra and saved gate diagnostics, no Torch import."""\nimport math\nimport numpy as np\n\n\ndef fuse_memory(visual,objects,relations,branch,strength):\n    delta=branch*(objects-visual)+(1-branch)*(relations-visual)\n    return visual+strength*delta\n\n\ndef update_gate_stats(state,values):\n    a=np.asarray(values,dtype=np.float64).reshape(-1)\n    if not len(a) or not np.isfinite(a).all() or (a<0).any() or (a>1).any():raise ValueError(\'Invalid semantic strength\')\n    state=state or dict(count=0,total=0.,minimum=1.,maximum=0.,below_point1=0,above_point9=0)\n    return dict(count=state[\'count\']+len(a),total=state[\'total\']+float(a.sum()),minimum=min(state[\'minimum\'],float(a.min())),maximum=max(state[\'maximum\'],float(a.max())),\n        below_point1=state[\'below_point1\']+int((a<.1).sum()),above_point9=state[\'above_point9\']+int((a>.9).sum()))\n\n\ndef merge_gate_stats(states):\n    if not states or any(s[\'count\']<=0 or not math.isfinite(s[\'total\']) for s in states):raise ValueError(\'Missing semantic gate diagnostics\')\n    count=sum(s[\'count\'] for s in states)\n    return dict(patches=count,mean=sum(s[\'total\'] for s in states)/count,minimum=min(s[\'minimum\'] for s in states),maximum=max(s[\'maximum\'] for s in states),\n        fraction_below_point1=sum(s[\'below_point1\'] for s in states)/count,fraction_above_point9=sum(s[\'above_point9\'] for s in states)/count,\n        interpretation=\'Learned residual strength, not a calibrated confidence or grounding accuracy.\')\n', 'semantic_data.py': '"""Pure NumPy fixed-bank/prompt175 to patch-grid preparation; no models."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import load_stage,TRAIN_IMAGES,VALID_IMAGES\n\nHERE=Path(__file__).resolve().parent\nARRAYS=dict(object_text_id=((8,),np.int16),object_box=((8,4),np.float32),object_weights=((8,196),np.float32),object_valid=((8,),bool),\n    relation_text_id=((3,),np.int16),subject_weights=((3,196),np.float32),target_weights=((3,196),np.float32),relation_geometry=((3,8),np.float32),relation_valid=((3,),bool))\n\ndef box_weights(value):\n    b=np.asarray(value,np.float64)\n    if b.shape!=(4,) or not np.isfinite(b).all() or (b<0).any() or (b>1).any() or (b[2:]<=b[:2]).any():raise ValueError(\'Invalid normalized xyxy box\')\n    x0,y0,x1,y1=b*14;axis=np.arange(14,dtype=np.float64)\n    wx=np.maximum(0,np.minimum(axis+1,x1)-np.maximum(axis,x0));wy=np.maximum(0,np.minimum(axis+1,y1)-np.maximum(axis,y0));weights=(wy[:,None]*wx[None,:]).reshape(196)\n    if weights.sum()<=0:raise ValueError(\'Box has no patch support\')\n    return (weights/weights.sum()).astype(np.float32)\n\ndef record_arrays(row,texts):\n    expected={\'coco_id\',\'filename\',\'group\',\'roi_count\',\'source_mode\',\'objects\',\'relations\',\'counts\'}\n    if set(row)!=expected or len(row[\'objects\'])>8 or len(row[\'relations\'])>3:raise ValueError(\'Unexpected prompt fields/caps\')\n    out={k:np.zeros(shape,dtype=dtype) for k,(shape,dtype) in ARRAYS.items()};lookup={t:i for i,t in enumerate(texts)};objects={}\n    for slot,obj in enumerate(row[\'objects\']):\n        instance=obj[\'instance_id\']\n        if not instance or instance in objects or obj[\'text\'] not in lookup:raise ValueError(\'Invalid/duplicate object instance/text\')\n        if not 0<=obj[\'roi\']<row[\'roi_count\']:raise ValueError(\'Invalid original ROI provenance\')\n        # Pool the SAME matched YOLO rectangle used by legacy ROI features, not the ROI index.\n        w=box_weights(obj[\'roi_bbox\']);box_weights(obj[\'bbox\'])\n        objects[instance]=w;out[\'object_weights\'][slot]=w;out[\'object_box\'][slot]=obj[\'bbox\'];out[\'object_text_id\'][slot]=lookup[obj[\'text\']];out[\'object_valid\'][slot]=True\n    for slot,rel in enumerate(row[\'relations\']):\n        s,o=rel[\'subject_instance\'],rel[\'object_instance\']\n        if s==o or s not in objects or o not in objects or rel[\'text\'] not in lookup:raise ValueError(\'Missing/collapsed relation endpoints/text\')\n        g=np.asarray(rel[\'geometry\'],np.float32)\n        if g.shape!=(8,) or not np.isfinite(g).all():raise ValueError(\'Invalid relation geometry\')\n        out[\'subject_weights\'][slot]=objects[s];out[\'target_weights\'][slot]=objects[o];out[\'relation_text_id\'][slot]=lookup[rel[\'text\']];out[\'relation_geometry\'][slot]=g;out[\'relation_valid\'][slot]=True\n    return out\n\ndef checked_root(mount,receipt_name,expected):\n    found=[p.parent for p in Path(mount).rglob(receipt_name) if sha(p)==expected[receipt_name]]\n    if len(found)!=1:raise ValueError(\'Need one exact semantic cache root: \'+receipt_name)\n    root=found[0]\n    for n,h in expected.items():\n        if Path(n).name!=n or not (root/n).is_file() or sha(root/n)!=h:raise ValueError(\'Missing/changed semantic payload \'+n)\n    return root\n\ndef fixed_banks(mount):\n    package=read(HERE/\'SAVE_BANK_V130_PACKAGE.json\');root=checked_root(mount,\'cache_receipt.json\',package[\'files\']);bank=read(HERE/\'bank_words_corrected.json\');manifest=read(root/\'cache_manifest.json\')\n    if sha(HERE/\'bank_words_corrected.json\')!=\'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac\' or bank[\'source\']!=\'TRAIN only, train+restval\' or manifest[\'source_bank_sha256\']!=sha(HERE/\'bank_words_corrected.json\'):raise ValueError(\'Bank supervision/provenance changed\')\n    arrays={}\n    with np.load(root/\'semantic_banks.npz\',allow_pickle=False) as f:\n        for kind in (\'noun\',\'verb\'):\n            words=[x[\'word\'] for x in bank[kind]];v=f[kind+\'_vectors\']\n            if f[kind+\'_words\'].tolist()!=words or v.shape!=(len(words),512) or v.dtype!=np.float32 or not np.isfinite(v).all() or not np.allclose(np.linalg.norm(v,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Bad bank vector order/space\')\n            arrays[kind]=v.copy()\n    return arrays,package,manifest\n\ndef checked_inputs(cfg):\n    inv=read(HERE/\'FULL_PROMPT_FRESH_V175_RESULT.json\')[\'transport_inventory\'];root=checked_root(cfg[\'prompt_root\'],\'cache_receipt.json\',inv[\'cache_files\']);cohort=read(root/\'cohort.json\');texts=read(root/\'prompt_texts.json\');vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False);prompt_manifest=read(root/\'cache_manifest.json\')\n    if len(texts)!=410 or texts!=sorted(set(texts)) or vectors.shape!=(410,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Wrong frozen175 text vectors\')\n    banks,bank_package,bank_manifest=fixed_banks(cfg[\'bank_root\'])\n    if (prompt_manifest[\'model\'],prompt_manifest[\'revision\'])!=(bank_manifest[\'model\'],bank_manifest[\'revision\']):raise ValueError(\'Bank/prompt CLIP text spaces differ\')\n    if cohort[\'test\'] or len(cohort[\'train\'])!=TRAIN_IMAGES or len(cohort[\'val\'])!=VALID_IMAGES or len(set(cohort[\'train\']+cohort[\'val\']))!=TRAIN_IMAGES+VALID_IMAGES:raise ValueError(\'Prompt source scope differs\')\n    return inv,root,cohort,texts,vectors,prompt_manifest,banks,bank_package\n\n\ndef stage_semantics(cfg,deadline):\n    started=time.monotonic();stage=Path(cfg[\'stage\']);out=Path(cfg[\'output\']);data,_,meta=load_stage(stage,verify=False);count=TRAIN_IMAGES+VALID_IMAGES\n    if len(data[\'ids\'])!=count or meta.get(\'semantic\'):raise ValueError(\'Expected new full patch stage\')\n    inv,root,cohort,texts,vectors,prompt_manifest,banks,bank_package=checked_inputs(cfg)\n    if set(cohort[\'train\'])!=set(map(int,data[\'ids\'][:TRAIN_IMAGES])) or set(cohort[\'val\'])!=set(map(int,data[\'ids\'][TRAIN_IMAGES:])):raise ValueError(\'Prompt/full patch split differs\')\n    arrays={k:np.lib.format.open_memmap(stage/(\'sem_\'+k+\'.npy\'),mode=\'w+\',dtype=dtype,shape=(count,)+shape) for k,(shape,dtype) in ARRAYS.items()}\n    for value in arrays.values():value[:]=0\n    index={int(i):j for j,i in enumerate(data[\'ids\'])};seen=set();totals={g:Counter() for g in (\'train\',\'val\')}\n    try:\n        with (root/\'prompt_links.jsonl\').open(encoding=\'utf-8\') as stream:\n            for line in stream:\n                if time.monotonic()>deadline:raise TimeoutError(\'Semantic stage cap; not complete training input\')\n                row=json.loads(line);i=row[\'coco_id\'];j=index.get(i)\n                if type(i)!=int or i in seen or j is None or row[\'group\']!=(\'train\' if j<TRAIN_IMAGES else \'val\'):raise ValueError(\'Foreign/repeated/leaked prompt image\')\n                encoded=record_arrays(row,texts)\n                for key,a in encoded.items():arrays[key][j]=a\n                seen.add(i);totals[row[\'group\']].update(images=1,objects=len(row[\'objects\']),relations=len(row[\'relations\']),with_object=bool(row[\'objects\']),with_relation=bool(row[\'relations\']))\n                if len(seen)%10000==0:print(\'Semantic patch inputs\',len(seen),\'/\',count,flush=True)\n        if seen!=set(index):raise ValueError(\'Missing semantic coverage\')\n        for value in arrays.values():value.flush()\n    finally:\n        for value in arrays.values():value._mmap.close()\n        for value in data.values():value._mmap.close()\n    for kind,v in banks.items():np.save(stage/(\'sem_\'+kind+\'_bank.npy\'),v)\n    np.save(stage/\'sem_text_vectors.npy\',vectors)\n    names=[\'sem_\'+n+\'.npy\' for n in ARRAYS]+[\'sem_noun_bank.npy\',\'sem_verb_bank.npy\',\'sem_text_vectors.npy\']\n    seals={n:sha(stage/n) for n in names}\n    if time.monotonic()>deadline:raise TimeoutError(\'Semantic sealing cap\')\n    meta[\'files_sha256\'].update(seals);meta[\'semantic\']=dict(method=\'bank_prompt_patch_v1\',prompt_cache=inv[\'cache_files\'],bank_cache=bank_package[\'files\'],text_model=prompt_manifest[\'model\'],text_revision=prompt_manifest[\'revision\'],geometry=\'14x14 fractional area on matched ROI box; Resize224x224, no center crop\',counts={g:dict(c) for g,c in totals.items()},seconds=time.monotonic()-started,TRAIN_VALID_only=True,test_used=False,files_sha256=seals)\n    save(stage/\'manifest.json\',meta);save(out/\'stage_summary.json\',meta);save(out/\'semantic_stage.json\',meta[\'semantic\'])\n    print(\'Semantic stage COMPLETE; no CLIP/UPT/model execution.\',flush=True)\n\nclass SemanticContext:\n    def __init__(self,stage):\n        self.arrays={k:np.load(Path(stage)/(\'sem_\'+k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in ARRAYS}\n        self.text=np.load(Path(stage)/\'sem_text_vectors.npy\',mmap_mode=\'r\',allow_pickle=False)\n    def images(self,indices,device):\n        import torch\n        p={k:np.array(v[indices],copy=True) for k,v in self.arrays.items() if k not in (\'object_text_id\',\'relation_text_id\')}\n        for role in (\'object\',\'relation\'):\n            p[role+\'_text\']=self.text[self.arrays[role+\'_text_id\'][indices]]*p[role+\'_valid\'][...,None]\n        return {k:torch.from_numpy(np.ascontiguousarray(v)).to(device) for k,v in p.items()}\n    def events(self,data,events,device):return self.images(data[\'caption_image\'][events],device)\n', 'full_data.py': '"""Full metadata, schedules and lazy raw H5 reader; no neural imports."""\nfrom collections import OrderedDict,Counter\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport unicodedata\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom data import vocabulary\nfrom patch_data import DATA_SHA,COUNTS,check_features\n\nTRAIN_IMAGES=113287;TRAIN_EVENTS=566747;VALID_IMAGES=5000;VALID_EVENTS=25010\nSTEPS=(TRAIN_EVENTS+31)//32;EPOCHS=10\n\ndef encode_tokens(tokens,lookup):\n    if not isinstance(tokens,list) or not all(isinstance(t,str) and t for t in tokens):raise ValueError(\'Invalid caption tokens\')\n    x=[lookup.get(unicodedata.normalize(\'NFKC\',t).lower(),3) for t in tokens[:52]]\n    return [1]+x+[2]+[0]*(52-len(x))\n\ndef learning_rate(step):\n    if type(step)!=int or not 1<=step<=EPOCHS*STEPS:raise ValueError(\'Invalid update\')\n    if step<=STEPS:return 8e-6+(1e-4-8e-6)*(step-1)/(STEPS-1)\n    ratio=(step-STEPS)/(9*STEPS)\n    return 1e-5+.5*(1e-4-1e-5)*(1+math.cos(math.pi*ratio))\n\ndef order_for_epoch(data,epoch):\n    if type(epoch)!=int or not 1<=epoch<=10:raise ValueError(\'Invalid epoch\')\n    image=np.asarray(data[\'caption_image\'][:TRAIN_EVENTS])\n    if (np.diff(image)<0).any() or len(np.unique(image))!=TRAIN_IMAGES:raise ValueError(\'TRAIN caption mapping differs\')\n    if image.min()!=0 or image.max()!=TRAIN_IMAGES-1:raise ValueError(\'Heldout/foreign TRAIN image index\')\n    return np.random.default_rng(42+epoch).permutation(TRAIN_EVENTS)\n\ndef validate_state(s):\n    epoch=s[\'epoch\'];cursor=s[\'batch_cursor\'];phase=s[\'phase\']\n    if type(epoch)!=int or type(cursor)!=int or not 1<=epoch<=11 or not 0<=cursor<=STEPS or phase not in (\'train\',\'validate\'):raise ValueError(\'Invalid cursor\')\n    if epoch==11 and (cursor or phase!=\'train\'):raise ValueError(\'Invalid completed cursor\')\n    if phase==\'validate\' and cursor!=STEPS:raise ValueError(\'Premature validation\')\n    if s[\'updates\']!=(epoch-1)*STEPS+cursor or s[\'caption_events\']!=(epoch-1)*TRAIN_EVENTS+min(cursor*32,TRAIN_EVENTS):raise ValueError(\'Exposure mismatch\')\n    if [r[\'epoch\'] for r in s[\'history\']]!=list(range(1,epoch)):raise ValueError(\'Incomplete history\')\n\ndef prepare(cfg):\n    import h5py\n    started=time.monotonic();root=Path(cfg[\'stage\']);root.mkdir(parents=True,exist_ok=False);out=Path(cfg[\'output\'])\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO JSON\')\n    ds=read(cfg[\'dataset_json\']);all_rows=ds[\'images\']\n    if dict(Counter(r[\'split\'] for r in all_rows))!=COUNTS:raise ValueError(\'Wrong Karpathy splits\')\n    mapping={int(r[\'cocoid\']):r for r in all_rows}\n    if len(mapping)!=len(all_rows):raise ValueError(\'Duplicate COCO ID\')\n    train=[r for r in all_rows if r[\'split\'] in (\'train\',\'restval\')];val=[r for r in all_rows if r[\'split\']==\'val\'];rows=train+val\n    wanted={int(r[\'cocoid\']):j for j,r in enumerate(rows)}\n    index=np.full((len(rows),2),-1,np.int32);seen=set();profiles=[];sources=[]\n    for number,path in enumerate(sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\'))):\n        with h5py.File(path,\'r\') as f:\n            text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n            if not f.attrs.get(\'complete\',False) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete/ambiguous H5\')\n            meta=json.loads(text(f.attrs[\'metadata_json\']));gold=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'feature_profile\']\n            profile={k:meta.get(k) for k in gold}\n            if profile!=gold:raise ValueError(\'Cache extraction profile differs from smoke192\')\n            ids=f[\'imgids\'][:];names=f[\'filename\'][:];features=f[\'features\']\n            if features.shape!=(len(ids),197,768) or features.dtype!=np.dtype(\'float16\') or len(names)!=len(ids):raise ValueError(\'Raw H5 shape differs\')\n            for i,(coco,name) in enumerate(zip(ids,names)):\n                coco=int(coco)\n                if coco in seen or coco not in mapping or text(name)!=mapping[coco][\'filename\']:raise ValueError(\'H5 ID/filename mismatch\')\n                seen.add(coco)\n                if coco in wanted:index[wanted[coco]]=(number,i)\n            sources.append(dict(path=str(path.resolve()),bytes=path.stat().st_size,shape=list(features.shape),dtype=str(features.dtype),chunks=features.chunks,compression=features.compression))\n        print(\'Full H5 metadata:\',number+1,\'mapped\',int((index[:,0]>=0).sum()),\'/\',len(rows),flush=True)\n    if seen!=set(mapping) or (index<0).any() or len(train)!=TRAIN_IMAGES or len(val)!=VALID_IMAGES:raise ValueError(\'Missing/foreign raw cache coverage\')\n    words=vocabulary(ds);lookup={w:i for i,w in enumerate(words)}\n    sequences=[];images=[];refs=[]\n    for j,r in enumerate(rows):\n        for sentence in r[\'sentences\']:sequences.append(encode_tokens(sentence[\'tokens\'],lookup));images.append(j)\n        if j>=TRAIN_IMAGES:refs.append(dict(coco_id=int(r[\'cocoid\']),references=[s[\'raw\'] for s in r[\'sentences\']]))\n        if (j+1)%20000==0:print(\'Caption metadata:\',j+1,\'/\',len(rows),flush=True)\n    arrays=dict(sequences=np.asarray(sequences,np.int32),caption_image=np.asarray(images,np.int32),ids=np.asarray([int(r[\'cocoid\']) for r in rows],np.int64),locations=index)\n    if arrays[\'sequences\'].shape!=(TRAIN_EVENTS+VALID_EVENTS,54) or int((arrays[\'caption_image\']<TRAIN_IMAGES).sum())!=TRAIN_EVENTS:raise ValueError(\'Wrong full caption count\')\n    for n,v in arrays.items():np.save(root/(n+\'.npy\'),v)\n    save(root/\'vocabulary.json\',words);save(root/\'validation_references.json\',refs)\n    if sha(root/\'vocabulary.json\')!=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'vocabulary_sha256\']:raise ValueError(\'TRAIN vocabulary differs from smoke\')\n    manifest=dict(status=\'FULL_PATCH_METADATA_COMPLETE\',schema=\'raw_h5_patch_stream_v1\',dataset_sha256=DATA_SHA,sources=sources,feature_profile=gold,\n        train_images=TRAIN_IMAGES,train_caption_events=TRAIN_EVENTS,valid_images=VALID_IMAGES,valid_caption_events=VALID_EVENTS,\n        files_sha256={p.name:sha(p) for p in root.iterdir() if p.is_file()},test_features_used=False,feature_arrays_staged=False,seconds=time.monotonic()-started)\n    save(root/\'manifest.json\',manifest);save(out/\'stage_summary.json\',manifest)\n\ndef load_stage(root,verify=True):\n    root=Path(root);m=read(root/\'manifest.json\')\n    if m[\'status\']!=\'FULL_PATCH_METADATA_COMPLETE\' or m[\'train_caption_events\']!=TRAIN_EVENTS or m[\'test_features_used\']:raise ValueError(\'Wrong stage scope\')\n    if verify:\n        for n,h in m[\'files_sha256\'].items():\n            if Path(n).name!=n or sha(root/n)!=h:raise ValueError(\'Changed stage file \'+n)\n    data={k:np.load(root/(k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in (\'sequences\',\'caption_image\',\'ids\',\'locations\')}\n    return data,read(root/\'vocabulary.json\'),m\n\nclass FeatureReader:\n    def __init__(self,root,capacity=256):\n        self.data,_,self.meta=load_stage(root,verify=False);self.handles={};self.cache=OrderedDict();self.capacity=capacity\n        self.golden=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'raw_selected_row_sha256\']\n        self.local=None;self.local_rows=None\n        if self.meta.get(\'feature_storage\')==\'local_fp16_memmap_v2\':\n            spec=self.meta[\'local_features\']\n            if spec[\'file\']!=\'local_patches.npy\' or spec[\'rows\']!=\'local_rows.npy\':raise ValueError(\'Unexpected local payload names\')\n            self.local=np.load(Path(root)/spec[\'file\'],mmap_mode=\'r\',allow_pickle=False)\n            self.local_rows=np.load(Path(root)/spec[\'rows\'],mmap_mode=\'r\',allow_pickle=False)\n            if self.local.shape!=(len(self.data[\'ids\']),196,768) or self.local.dtype!=np.float16 or self.local_rows.shape!=(len(self.data[\'ids\']),):raise ValueError(\'Invalid local patch mapping/shape\')\n    def get(self,image):\n        image=int(image)\n        if image in self.cache:self.cache.move_to_end(image);return self.cache[image]\n        if self.local is not None:\n            row=int(self.local_rows[image])\n            if not 0<=row<len(self.local):raise ValueError(\'Invalid local row\')\n            value=self.local[row]\n            if not np.isfinite(value).all():raise ValueError(\'Nonfinite local feature\')\n            self.cache[image]=value\n            if len(self.cache)>self.capacity:self.cache.popitem(last=False)\n            return value\n        import h5py\n        file,row=map(int,self.data[\'locations\'][image]);source=self.meta[\'sources\'][file]\n        if file not in self.handles:\n            path=Path(source[\'path\'])\n            if path.stat().st_size!=source[\'bytes\']:raise ValueError(\'Raw file size changed\')\n            self.handles[file]=h5py.File(path,\'r\',rdcc_nbytes=2*1024**2)\n        raw=self.handles[file][\'features\'][row];coco=str(int(self.data[\'ids\'][image]))\n        if coco in self.golden and hashlib.sha256(np.ascontiguousarray(raw).tobytes()).hexdigest()!=self.golden[coco]:raise ValueError(\'Smoke raw row replay differs \'+coco)\n        value=check_features(raw[None])[0];self.cache[image]=value\n        if len(self.cache)>self.capacity:self.cache.popitem(last=False)\n        return value\n    def close(self):\n        self.cache.clear()\n        for f in self.handles.values():f.close()\n        self.handles.clear()\n        if self.local is not None:self.local._mmap.close();self.local=None\n        if self.local_rows is not None:self.local_rows._mmap.close();self.local_rows=None\n\nclass CaptionBatches:\n    def __init__(self,root,events,rank=None,cursor=0,batch_size=32):\n        self.root=str(root);self.events=np.asarray(events,np.int64);self.rank=rank;self.cursor=cursor;self.batch_size=batch_size;self.reader=None\n    def __len__(self):return (len(self.events)+self.batch_size-1)//self.batch_size-self.cursor\n    def __getstate__(self):return dict(self.__dict__,reader=None)\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=FeatureReader(self.root)\n        start=(item+self.cursor)*self.batch_size;ev=self.events[start:start+self.batch_size]\n        if self.rank is not None:ev=ev[self.rank*16:(self.rank+1)*16]\n        if not len(ev):raise ValueError(\'Empty distributed caption batch\')\n        d=self.reader.data;image=d[\'caption_image\'][ev];x=np.stack([self.reader.get(i) for i in image]).astype(np.float32)\n        seq=np.asarray(d[\'sequences\'][ev],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n        return x,seq,ev.copy()\n\nclass ImageBatches:\n    def __init__(self,root,images,batch_size=8):self.root=str(root);self.images=np.asarray(images,np.int64);self.batch_size=batch_size;self.reader=None\n    def __len__(self):return (len(self.images)+self.batch_size-1)//self.batch_size\n    def __getstate__(self):return dict(self.__dict__,reader=None)\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=FeatureReader(self.root)\n        indices=self.images[item*self.batch_size:(item+1)*self.batch_size]\n        return np.stack([self.reader.get(i) for i in indices]).astype(np.float32),indices.copy()\n\ndef loader(dataset,workers=2):\n    from torch.utils.data import DataLoader\n    return DataLoader(dataset,batch_size=None,num_workers=workers,pin_memory=True,multiprocessing_context=\'spawn\' if workers else None,prefetch_factor=2 if workers else None)\n', 'patch_data.py': '"""TRAIN-only raw-patch smoke stage; importing never imports a neural library."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom data import vocabulary,encode\nfrom real_bank_data import sha,save,read\n\nDATA_SHA=\'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d\'\nCOUNTS=dict(train=82783,restval=30504,val=5000,test=5000)\n\ndef selected_rows(dataset,count=256):\n    from collections import Counter\n    rows=dataset[\'images\']\n    if dict(Counter(r[\'split\'] for r in rows))!=COUNTS:raise ValueError(\'Wrong Karpathy splits\')\n    ids=[int(r[\'cocoid\']) for r in rows]\n    if len(set(ids))!=len(ids):raise ValueError(\'Duplicate COCO ID\')\n    pool=[r for r in rows if r[\'split\'] in (\'train\',\'restval\')]\n    if not 1<=count<=len(pool):raise ValueError(\'Invalid TRAIN smoke size\')\n    return sorted(pool,key=lambda r:(hashlib.sha256(f"patch-baseline/42/{r[\'cocoid\']}".encode()).hexdigest(),int(r[\'cocoid\'])))[:count]\n\ndef check_features(raw):\n    if raw.ndim!=3 or raw.shape[1:]!=(197,768) or not np.isfinite(raw).all():raise ValueError(\'Expected finite raw [B,197,768]\')\n    if raw.dtype.kind!=\'f\':raise ValueError(\'Expected floating raw features\')\n    return raw[:,1:,:].astype(np.float16)\n\ndef batch(data,events):\n    events=np.asarray(events,dtype=np.int64)\n    if events.ndim!=1 or not len(events) or np.any(events<0) or np.any(events>=len(data[\'sequences\'])):raise ValueError(\'Invalid caption event\')\n    seq=np.asarray(data[\'sequences\'][events],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n    return np.asarray(data[\'patches\'][data[\'caption_image\'][events]],np.float32),seq\n\ndef prepare(cfg):\n    import h5py\n    started=time.monotonic();out=Path(cfg[\'output\']);stage=Path(cfg[\'stage\']);stage.mkdir(parents=True,exist_ok=False)\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO JSON\')\n    dataset=read(cfg[\'dataset_json\']);rows=selected_rows(dataset);words=vocabulary(dataset)\n    if len(words)!=10000:raise ValueError(\'Wrong TRAIN vocabulary size\')\n    wanted={int(r[\'cocoid\']):j for j,r in enumerate(rows)};found=set();profile=None;shards=[]\n    patches=np.empty((len(rows),196,768),np.float16);row_hashes={}\n    paths=sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\'))\n    if not paths:raise FileNotFoundError(\'Add raw visual-cache Dataset; ROI134 cannot replace raw patches\')\n    for path in paths:\n        if time.monotonic()-started>cfg[\'prepare_seconds\']:raise TimeoutError(\'Raw-cache scan/staging cap exceeded\')\n        with h5py.File(path,\'r\') as f:\n            text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n            if not bool(f.attrs.get(\'complete\',False)) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete or ambiguous cache IDs\')\n            meta=json.loads(text(f.attrs[\'metadata_json\']))\n            expected=dict(model_id=\'openai/clip-vit-base-patch16\',output=\'vision_model.last_hidden_state\',preprocessing=\'bilinear\',image_size=[224,224],antialias=True,dataset_json_sha256=DATA_SHA)\n            if any(meta.get(k)!=v for k,v in expected.items()):raise ValueError(\'Raw feature profile differs\')\n            np.testing.assert_allclose(meta[\'mean\'],[.48145466,.4578275,.40821073],rtol=0,atol=1e-8)\n            np.testing.assert_allclose(meta[\'std\'],[.26862954,.26130258,.27577711],rtol=0,atol=1e-8)\n            current={k:meta.get(k) for k in (*expected,\'model_revision\',\'mean\',\'std\',\'compute_dtype\',\'storage_dtype\',\'autocast\')}\n            if profile is not None and current!=profile:raise ValueError(\'Mixed raw extraction metadata\')\n            profile=current;ids=f[\'imgids\'][:];features=f[\'features\'];names=f[\'filename\']\n            if features.shape!=(len(ids),197,768) or features.dtype.kind!=\'f\' or features.dtype.itemsize not in (2,4) or len(names)!=len(ids):raise ValueError(\'Raw cache shape/dtype differs\')\n            positions=[i for i,v in enumerate(ids) if int(v) in wanted]\n            if positions:\n                raw=features[positions];values=check_features(raw)\n                for pos,full,value in zip(positions,raw,values):\n                    coco=int(ids[pos]);j=wanted[coco]\n                    if coco in found or text(names[pos])!=rows[j][\'filename\']:raise ValueError(\'Duplicate/mismatched selected raw ID\')\n                    found.add(coco);patches[j]=value;row_hashes[str(coco)]=hashlib.sha256(np.ascontiguousarray(full).tobytes()).hexdigest()\n            shards.append(dict(name=path.name,bytes=path.stat().st_size,shape=list(features.shape),selected_rows=len(positions)))\n        print(\'Raw patch scan:\',path.name,\'TRAIN extracted\',len(found),\'/\',len(rows),flush=True)\n    if found!=set(wanted):raise ValueError(\'Missing selected TRAIN features\')\n    sequences=[];caption_image=[]\n    for j,r in enumerate(rows):\n        for sentence in r[\'sentences\']:\n            sequences.append(encode(sentence[\'tokens\'],words));caption_image.append(j)\n    arrays=dict(patches=patches,sequences=np.asarray(sequences,np.int32),caption_image=np.asarray(caption_image,np.int32),coco_ids=np.asarray([int(r[\'cocoid\']) for r in rows],np.int64))\n    for k,v in arrays.items():np.save(stage/(k+\'.npy\'),v)\n    save(stage/\'vocabulary.json\',words)\n    manifest=dict(status=\'PATCH_TRAIN_SMOKE_STAGE_COMPLETE\',images=len(rows),caption_events=len(sequences),split=\'TRAIN_only\',test_used=False,validation_used=False,\n        dataset_sha256=DATA_SHA,feature_profile=profile,shards=shards,raw_selected_row_sha256=row_hashes,\n        files_sha256={p.name:sha(p) for p in stage.iterdir() if p.is_file()},seconds=time.monotonic()-started,\n        limits=\'Inherited producer/revision/preprocessing; no full H5 payload hash or new CLIP extraction. Only selected TRAIN feature rows read.\')\n    save(stage/\'manifest.json\',manifest);save(out/\'stage_summary.json\',manifest)\n\ndef load_stage(root):\n    root=Path(root);m=read(root/\'manifest.json\')\n    if m[\'status\']!=\'PATCH_TRAIN_SMOKE_STAGE_COMPLETE\' or m[\'images\']!=256 or m[\'test_used\'] or m[\'validation_used\']:raise ValueError(\'Wrong stage scope\')\n    for n,h in m[\'files_sha256\'].items():\n        if Path(n).name!=n or sha(root/n)!=h:raise ValueError(\'Changed stage \'+n)\n    data={k:np.load(root/(k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in (\'patches\',\'sequences\',\'caption_image\',\'coco_ids\')}\n    return data,read(root/\'vocabulary.json\'),m\n', 'patch_beam.py': '"""User-run batched beam search; no model import at module load."""\ndef beam_search(model,memory,width=5,maximum=20):\n    import torch\n    if model.training or not 1<=width<=model.head.out_features-2:raise ValueError(\'Beam requires eval model/valid width\')\n    b,n,d=memory.shape;vocab=model.head.out_features;device=memory.device\n    seq=torch.ones((b,width,1),dtype=torch.long,device=device);score=torch.full((b,width),-torch.inf,device=device);score[:,0]=0\n    done=torch.zeros((b,width),dtype=torch.bool,device=device);expanded=memory[:,None].expand(-1,width,-1,-1).reshape(b*width,n,d)\n    for _ in range(maximum):\n        logits=model.decode(expanded,seq.reshape(b*width,-1))[:,-1].float();logp=logits.log_softmax(-1).view(b,width,vocab)\n        logp[:,:,:2]=-torch.inf;logp=logp.masked_fill(done[:,:,None],-torch.inf);logp[:,:,0]=torch.where(done,torch.zeros_like(score),logp[:,:,0])\n        score,flat=(score[:,:,None]+logp).flatten(1).topk(width,dim=1)\n        parent=flat//vocab;token=flat%vocab\n        seq=torch.cat((seq.gather(1,parent[:,:,None].expand(-1,-1,seq.shape[-1])),token[:,:,None]),-1)\n        done=done.gather(1,parent)|(token==2)\n        if done.all():break\n    return seq,score\n\ndef check_paths(model,memory,seq,scores):\n    import torch\n    from torch.nn import functional as F\n    b,k,length=seq.shape;vocab=model.head.out_features\n    if not torch.isfinite(scores).all() or (scores[:,:-1]<scores[:,1:]).any() or not (seq[:,:,0]==1).all():raise ValueError(\'Invalid beam paths/scores\')\n    target=seq[:,:,1:];before_eos=(target==2).cumsum(-1)-(target==2).long();mask=before_eos==0\n    if ((target<2)&mask).any() or ((target!=0)&~mask).any():raise ValueError(\'Invalid BOS/PAD/EOS path\')\n    mem=memory[:,None].expand(-1,k,-1,-1).reshape(b*k,*memory.shape[1:]);prefix=seq[:,:,:-1].reshape(b*k,-1)\n    logits=model.decode(mem,prefix).float();logp=F.log_softmax(logits,-1);logp[:,:,:2]=-torch.inf\n    # Exclude post-EOS PAD without multiplying negative infinity by zero.\n    per=logp.gather(-1,target.reshape(b*k,-1,1)).squeeze(-1).masked_fill(~mask.reshape(b*k,-1),0).sum(-1).view(b,k)\n    torch.testing.assert_close(scores,per,rtol=1e-4,atol=2e-4)\n\ndef text_from_ids(ids,words):\n    out=[]\n    for word in ids[1:]:\n        word=int(word)\n        if word==2:break\n        if word>1:out.append(words[word])\n    return \' \'.join(out)\n', 'bank_layers.py': '"""CSA equations/conventions from IMPLEMENTATION_PROTOCOL; self-written, no C0."""\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass Config:\n    vocab: int\n    dim: int = 512\n    heads: int = 8\n    hidden: int = 2048\n    prototypes: int = 150\n    layers: int = 3\n    dropout: float = .2\n    raw_dim: int = 768\n\n\ndef zero_pad(x, valid):\n    return x.masked_fill(~valid[..., None], 0)\n\n\nclass FFN(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(c.dim, c.hidden), nn.ReLU(), nn.Dropout(c.dropout),\n                                 nn.Linear(c.hidden, c.dim), nn.Dropout(c.dropout))\n        self.norm = nn.LayerNorm(c.dim, eps=1e-5)\n\n    def forward(self, x):\n        return self.norm(x + self.net(x))\n\n\nclass SemanticAggregation(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.wa = nn.Linear(c.dim, c.prototypes, bias=False)\n        self.mlp = nn.Sequential(nn.Linear(c.prototypes, c.prototypes), nn.ReLU(),\n                                 nn.Linear(c.prototypes, c.prototypes))\n        self.wb = nn.Linear(c.dim, c.dim, bias=False)\n\n    def forward(self, bank):\n        weights = self.mlp(self.wa(bank)).float().softmax(dim=0)\n        return weights.T.to(bank.dtype) @ self.wb(bank), weights\n\n\nclass Alignment(nn.Module):\n    def forward(self, visual, prototypes, valid):\n        if prototypes.ndim == 2:\n            prototypes = prototypes.unsqueeze(0).expand(visual.shape[0], -1, -1)\n        cosine = torch.bmm(F.normalize(prototypes.float(), dim=-1, eps=1e-6),\n                           F.normalize(visual.float(), dim=-1, eps=1e-6).transpose(1, 2))\n        weights = (cosine / .1).masked_fill(~valid[:, None, :], -torch.inf).softmax(dim=-1)\n        relevance = (weights * cosine).sum(dim=-1)\n        selected = relevance.softmax(dim=-1)[..., None] * prototypes\n        return selected.to(visual.dtype), weights\n\n\nclass Fusion(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.align = Alignment()\n        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.cross_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.norm_s, self.norm_c = nn.LayerNorm(c.dim), nn.LayerNorm(c.dim)\n        self.ff_s, self.ff_c = FFN(c), FFN(c)\n        self.drop = nn.Dropout(c.dropout)\n        self.gate = nn.Linear(2 * c.dim, 1, bias=False)\n\n    def forward(self, visual, prototypes, valid):\n        selected, _ = self.align(visual, prototypes, valid)\n        context = self.self_attn(visual, visual, visual, key_padding_mask=~valid, need_weights=False)[0]\n        own = zero_pad(self.norm_s(visual + self.drop(context)), valid)\n        own = zero_pad(self.ff_s(own), valid)\n        context = self.cross_attn(own, selected, selected, need_weights=False)[0]\n        cross = zero_pad(self.norm_c(own + self.drop(context)), valid)\n        cross = zero_pad(self.ff_c(cross), valid)\n        gate = self.gate(torch.cat((own, cross), dim=-1)).sigmoid()\n        return zero_pad(gate * own + (1 - gate) * cross, valid), selected\n\n\nclass DecoderLayer(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.object_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.relation_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.norm = nn.LayerNorm(c.dim)\n        self.drop = nn.Dropout(c.dropout)\n        self.gate_o, self.gate_r = nn.Linear(2 * c.dim, c.dim), nn.Linear(2 * c.dim, c.dim)\n        self.ffn = FFN(c)\n\n    def forward(self, x, previous_word, vo, vr, text_valid, region_valid):\n        causal = torch.ones(x.shape[1], x.shape[1], device=x.device, dtype=torch.bool).triu(1)\n        h = self.self_attn(x, x, x, attn_mask=causal, key_padding_mask=~text_valid, need_weights=False)[0]\n        h = zero_pad(self.norm(x + self.drop(h)), text_valid)\n        co = self.object_attn(h, vo, vo, key_padding_mask=~region_valid, need_weights=False)[0]\n        cr = self.relation_attn(h, vr, vr, key_padding_mask=~region_valid, need_weights=False)[0]\n        go = self.gate_o(torch.cat((co, previous_word), dim=-1)).sigmoid()\n        gr = self.gate_r(torch.cat((cr, previous_word), dim=-1)).sigmoid()\n        return zero_pad(self.ffn(go * co + gr * cr), text_valid)\n\n\nclass CSACaptioner(nn.Module):\n    def __init__(self, c, nouns, verbs):\n        super().__init__()\n        if c.dim % 2 or c.dim % c.heads or c.vocab < 4 or c.layers < 1 or c.prototypes < 1:\n            raise ValueError(\'Invalid model dimensions\')\n        for bank in (nouns, verbs):\n            if bank.ndim != 2 or bank.shape[1] != c.dim or not len(bank) or not torch.isfinite(bank).all():\n                raise ValueError(\'Empty, nonfinite or incompatible semantic bank\')\n        self.c = c\n        self.register_buffer(\'nouns\', nouns.detach().float().clone())\n        self.register_buffer(\'verbs\', verbs.detach().float().clone())\n        self.project = nn.Linear(c.raw_dim, c.dim)\n        self.noun_agg, self.verb_agg = SemanticAggregation(c), SemanticAggregation(c)\n        self.object1, self.object2, self.relation = Fusion(c), Fusion(c), Fusion(c)\n        self.embedding = nn.Embedding(c.vocab, c.dim, padding_idx=0)\n        self.decoder = nn.ModuleList([DecoderLayer(c) for _ in range(c.layers)])\n        self.head = nn.Linear(c.dim, c.vocab)\n        self.apply(self._init)\n        for module in self.modules():\n            if isinstance(module, nn.MultiheadAttention):\n                nn.init.xavier_uniform_(module.in_proj_weight)\n                nn.init.zeros_(module.in_proj_bias)\n        nn.init.normal_(self.embedding.weight, std=c.dim ** -.5)\n        with torch.no_grad():\n            self.embedding.weight[0].zero_()\n\n    @staticmethod\n    def _init(module):\n        if isinstance(module, nn.Linear):\n            nn.init.xavier_uniform_(module.weight)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.LayerNorm):\n            nn.init.ones_(module.weight)\n            nn.init.zeros_(module.bias)\n\n    def encode(self, regions, valid):\n        if valid.dtype != torch.bool or regions.ndim != 3 or valid.shape != regions.shape[:2] or not valid.any(dim=1).all():\n            raise ValueError(\'Every image requires at least one valid region\')\n        if not torch.isfinite(regions).all():\n            raise ValueError(\'Nonfinite visual feature\')\n        v = zero_pad(self.project(regions), valid)\n        nouns, _ = self.noun_agg(self.nouns)\n        verbs, _ = self.verb_agg(self.verbs)\n        v, selected = self.object1(v, nouns, valid)\n        vo, _ = self.object2(v, selected, valid)\n        vr, _ = self.relation(vo, verbs, valid)\n        return vo, vr\n\n    def decode(self, ids, vo, vr, valid):\n        if ids.ndim != 2 or ids.shape[1] == 0 or not (ids[:, 0] == 1).all():\n            raise ValueError(\'Prefix must start with BOS, with nonempty time dimension\')\n        raw = self.embedding(ids)\n        position = torch.arange(ids.shape[1], device=ids.device, dtype=torch.float32)[:, None]\n        frequency = torch.exp(torch.arange(0, self.c.dim, 2, device=ids.device).float() * (-math.log(10000) / self.c.dim))\n        pe = torch.empty(ids.shape[1], self.c.dim, device=ids.device)\n        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * frequency), torch.cos(position * frequency)\n        text_valid = ids != 0\n        x = zero_pad(raw * math.sqrt(self.c.dim) + pe.to(raw.dtype), text_valid)\n        for layer in self.decoder:\n            x = layer(x, raw, vo, vr, text_valid, valid)\n        return self.head(x)\n\n    def forward(self, regions, valid, ids):\n        return self.decode(ids, *self.encode(regions, valid), valid)\n\n\ndef xe_loss(logits, target):\n    mask = target != 0\n    if not mask.any():\n        raise ValueError(\'XE batch has no target tokens\')\n    nll = F.cross_entropy(logits.flatten(0, 1), target.flatten(), reduction=\'none\', ignore_index=0).view_as(target)\n    return (nll * mask).sum() / mask.sum()\n\n\n@torch.no_grad()\ndef beam_search(model, vo, vr, valid, beams=5, maximum=20):\n    """Per-image beams; no hidden length normalization, BOS/PAD cannot be emitted."""\n    if model.training or maximum < 1 or not 1 <= beams <= model.c.vocab - 2:\n        raise ValueError(\'Beam search requires eval mode and valid width/length\')\n    results = []\n    for i in range(len(vo)):\n        seq = torch.ones(1, 1, dtype=torch.long, device=vo.device)\n        scores = torch.zeros(1, device=vo.device)\n        finished = torch.zeros(1, device=vo.device, dtype=torch.bool)\n        for _ in range(maximum):\n            count = len(seq)\n            logits = model.decode(seq, vo[i:i+1].expand(count, -1, -1),\n                                  vr[i:i+1].expand(count, -1, -1), valid[i:i+1].expand(count, -1))[:, -1]\n            logp = logits.float().log_softmax(-1)\n            logp[:, :2] = -torch.inf\n            logp[finished] = -torch.inf\n            logp[finished, 0] = 0\n            best, flat = (scores[:, None] + logp).flatten().topk(beams)\n            parent, token = flat // model.c.vocab, flat % model.c.vocab\n            seq = torch.cat((seq[parent], token[:, None]), dim=1)\n            finished = finished[parent] | (token == 2)\n            scores = best\n            if finished.all():\n                break\n        results.append((seq, scores))\n    return results\n\n\ndef beam_mean_loss(model, regions, valid, sequences, rewards, maximum=20):\n    """Gradient rescore; rewards [B,K] are externally computed CIDEr-D and detached."""\n    b, k, length = sequences.shape\n    if rewards.shape != (b, k) or length < 2 or length > maximum + 1:\n        raise ValueError(\'Invalid beam/reward shape\')\n    flat = sequences.reshape(b * k, length)\n    logits = model(regions.repeat_interleave(k, 0), valid.repeat_interleave(k, 0), flat[:, :-1])\n    target = flat[:, 1:]\n    # Include first EOS, exclude every later position (even malformed non-PAD tokens).\n    eos_before = (target == 2).cumsum(dim=1) - (target == 2).long()\n    mask = (target != 0) & (eos_before == 0)\n    logp = logits.float().log_softmax(-1).gather(-1, target[..., None]).squeeze(-1)\n    score = (logp * mask).sum(-1).reshape(b, k) / maximum\n    advantage = rewards.detach() - rewards.detach().mean(dim=1, keepdim=True)\n    return -(advantage * score).mean()\n', 'real_bank_data.py': '"""Safe real-input contracts. No neural imports or execution."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nHERE = Path(__file__).resolve().parent\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024*1024), b\'\'): h.update(block)\n    return h.hexdigest()\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef save(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False)+\'\\n\', encoding=\'utf-8\')\n\ndef validate_arrays(a, images, vocab):\n    n = len(images); total = len(a[\'features\'])\n    shapes = dict(features=(total,768), boxes=(total,4), scores=(total,), offsets=(n+1,), ids=(n,))\n    types = dict(features=np.float16, boxes=np.float32, scores=np.float32, offsets=np.int64, ids=np.int64)\n    if set(a) != set(shapes): raise ValueError(\'Region members differ\')\n    for key in shapes:\n        if a[key].shape != shapes[key] or a[key].dtype != types[key] or not np.isfinite(a[key]).all():\n            raise ValueError(\'Invalid region array: \'+key)\n    offsets = a[\'offsets\']\n    if offsets[0] != 0 or offsets[-1] != total or not ((np.diff(offsets)>=1)&(np.diff(offsets)<=150)).all():\n        raise ValueError(\'Invalid offsets\')\n    for i, m in enumerate(images):\n        if int(a[\'ids\'][i]) != m[\'coco_id\']: raise ValueError(\'Image order differs\')\n        b = a[\'boxes\'][offsets[i]:offsets[i+1]]\n        if len(b)!=len(m[\'region_kinds\']) or m[\'width\']<=0 or m[\'height\']<=0: raise ValueError(\'Region metadata differs\')\n        if not ((b[:,0]>=0)&(b[:,1]>=0)&(b[:,2]<=m[\'width\'])&(b[:,3]<=m[\'height\'])&(b[:,2]>b[:,0])&(b[:,3]>b[:,1])).all():\n            raise ValueError(\'Invalid boxes\')\n        if any(k not in (\'predicted_roi\',\'full_frame_fallback\') for k in m[\'region_kinds\']): raise ValueError(\'Unknown ROI kind\')\n        for seq in m[\'sequences\']:\n            if len(seq)!=54 or seq[0]!=1 or seq.count(2)!=1 or any(type(x)!=int or x<0 or x>=len(vocab) for x in seq):\n                raise ValueError(\'Invalid target sequence\')\n            end=seq.index(2)\n            if 0 in seq[:end] or any(seq[end+1:]): raise ValueError(\'Invalid EOS/PAD sequence\')\n\ndef load_prepared(root):\n    root=Path(root); p=read(HERE/\'REAL_BANK_INPUT_PACKAGE.json\')\n    if not root.is_dir():\n        raise FileNotFoundError(\'Add Input csa-real-v126-lemma-fixed or correct PREPARED_ROOT; directory not mounted: \'+str(root))\n    if not (root/\'corrected_assets_receipt.json\').is_file():\n        candidates=list(root.rglob(\'corrected_assets_receipt.json\'))\n        if len(candidates)!=1:\n            raise ValueError(f\'Expected one corrected_assets_receipt.json under {root}; found {len(candidates)}. Add the combined csa-real-v126-lemma-fixed Dataset, not only the semantic-bank Dataset.\')\n        root=candidates[0].parent\n    receipt=read(root/\'corrected_assets_receipt.json\')\n    for field, expected in [(\'original_files_sha256\',p[\'original_files_sha256\']),(\'corrected_overlay_sha256\',p[\'overlay_sha256\'])]:\n        if receipt.get(field)!=expected: raise ValueError(\'Input receipt differs: \'+field)\n    if receipt.get(\'selected_bank_file\')!=p[\'selected_bank_file\'] or receipt.get(\'original_kernel_version\')!=p[\'original_kernel_version\']:\n        raise ValueError(\'Selected bank/version differs\')\n    for name, expected in {**p[\'original_files_sha256\'],**p[\'overlay_sha256\']}.items():\n        if not (root/name).is_file() or sha(root/name)!=expected: raise ValueError(\'Missing/changed input: \'+name)\n    with np.load(root/\'regions.npz\', allow_pickle=False) as f: arrays={k:f[k] for k in f.files}\n    metadata=read(root/\'pilot_metadata.json\'); vocab=read(root/\'vocabulary.json\'); bank=read(root/p[\'selected_bank_file\'])\n    images=metadata[\'images\']\n    if len(images)!=320 or sum(x[\'group\']==\'train\' for x in images)!=256 or sum(x[\'group\']==\'val\' for x in images)!=64:\n        raise ValueError(\'Pilot groups differ\')\n    validate_arrays(arrays,images,vocab)\n    if len(bank[\'noun\'])!=5269 or len(bank[\'verb\'])!=1594 or len(vocab)!=10000: raise ValueError(\'Bank/vocabulary size differs\')\n    for group in (\'train\',\'val\'):\n        if [images[i][\'coco_id\'] for i in p[group+\'_indices\']]!=p[group+\'_ids\']: raise ValueError(\'Smoke selection differs\')\n    return root,p,arrays,images,vocab,bank\n\ndef batch(arrays, images, indices):\n    lengths=[int(arrays[\'offsets\'][i+1]-arrays[\'offsets\'][i]) for i in indices]\n    x=np.zeros((len(indices),max(lengths),768),np.float32); valid=np.zeros(x.shape[:2],bool)\n    seq=np.array([images[i][\'sequences\'][0] for i in indices],np.int64)\n    seq=seq[:,:int((seq!=0).sum(1).max())]\n    for j,i in enumerate(indices):\n        lo,hi=arrays[\'offsets\'][i:i+2]; x[j,:lengths[j]]=arrays[\'features\'][lo:hi]; valid[j,:lengths[j]]=True\n    return x,valid,seq\n', 'data.py': '"""Pure NumPy/data contracts; importing this file never imports a model."""\nfrom collections import Counter\nimport re\nimport unicodedata\n\nimport numpy as np\n\nPAD, BOS, EOS, UNK = range(4)\nSPECIALS = [\'<pad>\', \'<bos>\', \'<eos>\', \'<unk>\']\n\n\ndef roi_pool(patches, objects, width, height, cap=150):\n    """Pixel xyxy -> fractional-area-weighted raw patch features, no class inputs."""\n    patches = np.asarray(patches, dtype=np.float32)\n    if patches.ndim != 3 or patches.shape[0] != patches.shape[1]:\n        raise ValueError(\'Expected square patch grid [G,G,C], without CLS\')\n    if not np.isfinite(patches).all() or not np.isfinite([width, height]).all():\n        raise ValueError(\'Nonfinite patch or dimensions\')\n    if width <= 0 or height <= 0 or cap < 1 or not isinstance(objects, list):\n        raise ValueError(\'Invalid dimensions, cap, or detector record\')\n    if not objects:\n        return patches.mean(axis=(0, 1), keepdims=False)[None], np.array([[0, 0, width, height]], np.float32), [\'full_frame_fallback\']\n    checked = []\n    for index, obj in enumerate(objects):\n        box = np.asarray(obj[\'bbox\'], dtype=np.float32)\n        score = float(obj[\'conf\'])\n        if box.shape != (4,) or not np.isfinite(box).all() or not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Malformed detection\')\n        box = np.clip(box, [0, 0, 0, 0], [width, height, width, height]).astype(np.float32)\n        if box[2] <= box[0] or box[3] <= box[1]:\n            raise ValueError(\'Nonpositive detection area; do not turn corruption into empty detection\')\n        checked.append((-score, index, box))\n    selected = sorted(checked, key=lambda item: item[:2])[:cap]\n    grid = patches.shape[0]\n    axis = np.arange(grid, dtype=np.float32)\n    pooled, boxes = [], []\n    for _, _, box in selected:\n        x0, y0, x1, y1 = box / np.array([width, height, width, height]) * grid\n        wx = np.maximum(0, np.minimum(axis + 1, x1) - np.maximum(axis, x0))\n        wy = np.maximum(0, np.minimum(axis + 1, y1) - np.maximum(axis, y0))\n        weights = (wy[:, None] * wx[None, :]).astype(np.float32)\n        pooled.append(np.einsum(\'ij,ijc->c\', weights, patches) / weights.sum())\n        boxes.append(box)\n    return np.stack(pooled), np.stack(boxes), [\'predicted_roi\'] * len(pooled)\n\n\ndef train_captions(dataset):\n    for image in dataset[\'images\']:\n        if image[\'split\'] in (\'train\', \'restval\'):\n            for sentence in image[\'sentences\']:\n                tokens = sentence[\'tokens\']\n                if not isinstance(tokens, list) or not all(isinstance(t, str) and t for t in tokens):\n                    raise ValueError(\'Malformed caption tokens\')\n                yield [unicodedata.normalize(\'NFKC\', t).lower() for t in tokens]\n\n\ndef vocabulary(dataset, limit=10000):\n    if limit < 4:\n        raise ValueError(\'Vocabulary limit includes four specials\')\n    counts = Counter(t for caption in train_captions(dataset) for t in caption if t not in SPECIALS)\n    return SPECIALS + sorted(counts, key=lambda word: (-counts[word], word))[:limit - 4]\n\n\ndef encode(tokens, words, maximum=54):\n    if maximum < 2:\n        raise ValueError(\'Sequence must retain BOS/EOS\')\n    lookup = {word: i for i, word in enumerate(words)}\n    content = [lookup.get(unicodedata.normalize(\'NFKC\', t).lower(), UNK) for t in tokens[:maximum - 2]]\n    return [BOS] + content + [EOS] + [PAD] * (maximum - len(content) - 2)\n\n\ndef bank_words(tagged_lemmas, minimum=5):\n    """Input must come from pinned POS model applied to TRAIN captions only."""\n    counts = {\'noun\': Counter(), \'verb\': Counter()}\n    for lemma, pos in tagged_lemmas:\n        word = unicodedata.normalize(\'NFKC\', lemma).lower()\n        kind = \'noun\' if pos in (\'NOUN\', \'PROPN\') else \'verb\' if pos == \'VERB\' else None\n        if kind and re.fullmatch(\'[a-z]+\', word):\n            counts[kind][word] += 1\n    return {kind: [(w, c[w]) for w in sorted(c, key=lambda w: (-c[w], w)) if c[w] >= minimum]\n            for kind, c in counts.items()}\n', 'saved_quality_data.py': '"""Saved text contracts only; no models or metric dependencies."""\nimport hashlib\nimport json\nimport math\n\ndef check_rows(rows,expected):\n    if len(rows)!=len(expected) or len({x[\'coco_id\'] for x in rows})!=len(rows) or {x[\'coco_id\'] for x in rows}!=set(expected):\n        raise ValueError(\'Missing/duplicate/foreign caption IDs\')\n    if any(not isinstance(x[\'caption\'],str) for x in rows):raise ValueError(\'Non-text caption\')\n\ndef common_refs(references,bundle):\n    all_refs={x[\'coco_id\']:x[\'references\'] for x in references}\n    ids=bundle[\'cohort_ids\']\n    if len(ids)!=len(set(ids)) or len(ids)!=4000:raise ValueError(\'Expected4000unique IDs\')\n    if not set(ids)<=set(all_refs):raise ValueError(\'Old4000 not contained in new5000\')\n    selected=[dict(coco_id=i,references=all_refs[i][:5]) for i in sorted(ids)]\n    if any(len(x[\'references\'])!=5 for x in selected):raise ValueError(\'Missing reference\')\n    digest=hashlib.sha256(json.dumps(selected,ensure_ascii=False,separators=(\',\',\':\')).encode()).hexdigest()\n    if digest!=bundle[\'reference_sha256\']:raise ValueError(\'Old/new reference identities differ\')\n    order=bundle.get(\'legacy_order\',sorted(ids))\n    if len(order)!=len(ids) or set(order)!=set(ids):raise ValueError(\'Wrong legacy input order\')\n    by_id={x[\'coco_id\']:x for x in selected}\n    return [by_id[i] for i in order]\n\ndef chair_counts(chair,rows):\n    sentences=chair[\'sentences\'];ids={x[\'coco_id\'] for x in rows};mapping={x[\'coco_id\']:x[\'caption\'] for x in rows}\n    if len(sentences)!=len(ids) or {x[\'image_id\'] for x in sentences}!=ids:raise ValueError(\'CHAIR ID coverage\')\n    if any(x[\'caption\']!=mapping[x[\'image_id\']] for x in sentences):raise ValueError(\'CHAIR captions differ\')\n    counts=dict(images=len(ids),hallucinated_captions=sum(bool(x[\'mscoco_hallucinated_words\']) for x in sentences),\n        hallucinated_mentions=sum(len(x[\'mscoco_hallucinated_words\']) for x in sentences),\n        object_mentions=sum(len(x[\'mscoco_generated_words\']) for x in sentences),\n        recalled_unique=sum(len(set(x[\'mscoco_generated_words\'])&set(x[\'mscoco_gt_words\'])) for x in sentences),\n        reference_unique=sum(len(set(x[\'mscoco_gt_words\'])) for x in sentences))\n    metric=dict(CHAIRs=counts[\'hallucinated_captions\']/len(ids),CHAIRi=counts[\'hallucinated_mentions\']/max(1,counts[\'object_mentions\']),\n                Recall=counts[\'recalled_unique\']/max(1,counts[\'reference_unique\']))\n    if any(not math.isfinite(v) or abs(v-chair[\'overall_metrics\'][k])>1e-12 for k,v in metric.items()):raise ValueError(\'CHAIR count aggregation differs\')\n    return metric,counts\n\ndef align_cider_scores(references,values,rows):\n    if len(references)!=len(values):raise ValueError(\'CIDEr score/reference length mismatch\')\n    indexed={r[\'coco_id\']:float(v) for r,v in zip(references,values)}\n    if len(indexed)!=len(references) or {r[\'coco_id\'] for r in rows}!=set(indexed):raise ValueError(\'CIDEr score IDs differ\')\n    return [indexed[r[\'coco_id\']] for r in rows]\n', 'SAVE_BANK_V130_PACKAGE.json': '{\n  "kernel_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/130",\n  "files": {\n    "semantic_banks.npz": "e77608563fb23622d52d27ada06f49e6e578ef573c9fe41861878c36357fa698",\n    "cache_manifest.json": "1fe9b24391bc580acf13cf488224e92a187389d680fa40293d0f22ce15b1d8ea",\n    "cache_receipt.json": "a50d8a43ab4eb0f69f1bf91644054c4a6b30da65cb54194f804815793868de24",\n    "smoke_summary.json": "d59ffc2496f71ac5c65c995ab19d60affa28865c58c71eb624bb9257ebaefc90",\n    "notebook.log": "2c0bb4ffab63bad3993a379797d05583472bdffa7ee2c99c4df5feaf1d7d20ca",\n    "source_receipt.json": "5e26da5acd1d597441e0497763ef3c0afcf0e19e90731e9a2ad8717afc75c537"\n  },\n  "source_review_zip_sha256": "7b14d35af654b630f6a5addf9943dbe5910074c9f772e7d8ee38026558f15f9e"\n}\n', 'bank_words_corrected.json': '{\n  "source": "TRAIN only, train+restval",\n  "pos_library": "spacy3.8.11",\n  "model": "en_core_web_sm3.8.0",\n  "model_wheel_sha256": "1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85",\n  "minimum_count": 5,\n  "noun": [\n    {\n      "word": "man",\n      "count": 80610\n    },\n    {\n      "word": "people",\n      "count": 40378\n    },\n    {\n      "word": "woman",\n      "count": 37596\n    },\n    {\n      "word": "table",\n      "count": 30258\n    },\n    {\n      "word": "street",\n      "count": 29416\n    },\n    {\n      "word": "person",\n      "count": 24076\n    },\n    {\n      "word": "top",\n      "count": 20976\n    },\n    {\n      "word": "group",\n      "count": 20905\n    },\n    {\n      "word": "dog",\n      "count": 20179\n    },\n    {\n      "word": "plate",\n      "count": 20152\n    },\n    {\n      "word": "field",\n      "count": 19953\n    },\n    {\n      "word": "tennis",\n      "count": 18993\n    },\n    {\n      "word": "train",\n      "count": 18646\n    },\n    {\n      "word": "front",\n      "count": 18125\n    },\n    {\n      "word": "cat",\n      "count": 17683\n    },\n    {\n      "word": "room",\n      "count": 17623\n    },\n    {\n      "word": "sign",\n      "count": 16891\n    },\n    {\n      "word": "water",\n      "count": 16104\n    },\n    {\n      "word": "baseball",\n      "count": 15168\n    },\n    {\n      "word": "building",\n      "count": 14800\n    },\n    {\n      "word": "bus",\n      "count": 14128\n    },\n    {\n      "word": "bathroom",\n      "count": 13771\n    },\n    {\n      "word": "tree",\n      "count": 13606\n    },\n    {\n      "word": "food",\n      "count": 13410\n    },\n    {\n      "word": "pizza",\n      "count": 13089\n    },\n    {\n      "word": "kitchen",\n      "count": 12832\n    },\n    {\n      "word": "grass",\n      "count": 12545\n    },\n    {\n      "word": "horse",\n      "count": 12518\n    },\n    {\n      "word": "side",\n      "count": 12347\n    },\n    {\n      "word": "bed",\n      "count": 12193\n    },\n    {\n      "word": "giraffe",\n      "count": 12046\n    },\n    {\n      "word": "boy",\n      "count": 11743\n    },\n    {\n      "word": "player",\n      "count": 11734\n    },\n    {\n      "word": "snow",\n      "count": 11337\n    },\n    {\n      "word": "ball",\n      "count": 11220\n    },\n    {\n      "word": "car",\n      "count": 11159\n    },\n    {\n      "word": "beach",\n      "count": 10777\n    },\n    {\n      "word": "couple",\n      "count": 10690\n    },\n    {\n      "word": "elephant",\n      "count": 10614\n    },\n    {\n      "word": "clock",\n      "count": 10602\n    },\n    {\n      "word": "toilet",\n      "count": 10534\n    },\n    {\n      "word": "girl",\n      "count": 10522\n    },\n    {\n      "word": "bear",\n      "count": 10180\n    },\n    {\n      "word": "road",\n      "count": 9954\n    },\n    {\n      "word": "city",\n      "count": 9929\n    },\n    {\n      "word": "game",\n      "count": 9846\n    },\n    {\n      "word": "picture",\n      "count": 9815\n    },\n    {\n      "word": "umbrella",\n      "count": 9638\n    },\n    {\n      "word": "bench",\n      "count": 9462\n    },\n    {\n      "word": "laptop",\n      "count": 9364\n    },\n    {\n      "word": "motorcycle",\n      "count": 9315\n    },\n    {\n      "word": "skateboard",\n      "count": 9250\n    },\n    {\n      "word": "computer",\n      "count": 9009\n    },\n    {\n      "word": "child",\n      "count": 8995\n    },\n    {\n      "word": "zebra",\n      "count": 8900\n    },\n    {\n      "word": "phone",\n      "count": 8783\n    },\n    {\n      "word": "cake",\n      "count": 8481\n    },\n    {\n      "word": "area",\n      "count": 8456\n    },\n    {\n      "word": "board",\n      "count": 8439\n    },\n    {\n      "word": "kite",\n      "count": 8418\n    },\n    {\n      "word": "window",\n      "count": 8403\n    },\n    {\n      "word": "glass",\n      "count": 8291\n    },\n    {\n      "word": "truck",\n      "count": 8148\n    },\n    {\n      "word": "boat",\n      "count": 8138\n    },\n    {\n      "word": "chair",\n      "count": 8079\n    },\n    {\n      "word": "wall",\n      "count": 8035\n    },\n    {\n      "word": "frisbee",\n      "count": 7977\n    },\n    {\n      "word": "sink",\n      "count": 7888\n    },\n    {\n      "word": "bird",\n      "count": 7745\n    },\n    {\n      "word": "ski",\n      "count": 7452\n    },\n    {\n      "word": "track",\n      "count": 7346\n    },\n    {\n      "word": "air",\n      "count": 7282\n    },\n    {\n      "word": "desk",\n      "count": 7189\n    },\n    {\n      "word": "lot",\n      "count": 6899\n    },\n    {\n      "word": "hand",\n      "count": 6865\n    },\n    {\n      "word": "living",\n      "count": 6710\n    },\n    {\n      "word": "park",\n      "count": 6691\n    },\n    {\n      "word": "light",\n      "count": 6490\n    },\n    {\n      "word": "cow",\n      "count": 6440\n    },\n    {\n      "word": "banana",\n      "count": 6433\n    },\n    {\n      "word": "sky",\n      "count": 6422\n    },\n    {\n      "word": "bowl",\n      "count": 6266\n    },\n    {\n      "word": "court",\n      "count": 6158\n    },\n    {\n      "word": "surfboard",\n      "count": 6106\n    },\n    {\n      "word": "fire",\n      "count": 6062\n    },\n    {\n      "word": "wave",\n      "count": 6051\n    },\n    {\n      "word": "couch",\n      "count": 5847\n    },\n    {\n      "word": "flower",\n      "count": 5796\n    },\n    {\n      "word": "photo",\n      "count": 5720\n    },\n    {\n      "word": "airplane",\n      "count": 5676\n    },\n    {\n      "word": "bunch",\n      "count": 5612\n    },\n    {\n      "word": "sandwich",\n      "count": 5587\n    },\n    {\n      "word": "stop",\n      "count": 5552\n    },\n    {\n      "word": "background",\n      "count": 5526\n    },\n    {\n      "word": "plane",\n      "count": 5429\n    },\n    {\n      "word": "sheep",\n      "count": 5324\n    },\n    {\n      "word": "bat",\n      "count": 5299\n    },\n    {\n      "word": "shirt",\n      "count": 5286\n    },\n    {\n      "word": "view",\n      "count": 5203\n    },\n    {\n      "word": "ocean",\n      "count": 5154\n    },\n    {\n      "word": "mirror",\n      "count": 5152\n    },\n    {\n      "word": "cell",\n      "count": 5096\n    },\n    {\n      "word": "floor",\n      "count": 5072\n    },\n    {\n      "word": "traffic",\n      "count": 5016\n    },\n    {\n      "word": "fence",\n      "count": 4916\n    },\n    {\n      "word": "counter",\n      "count": 4878\n    },\n    {\n      "word": "bike",\n      "count": 4857\n    },\n    {\n      "word": "racket",\n      "count": 4839\n    },\n    {\n      "word": "teddy",\n      "count": 4825\n    },\n    {\n      "word": "sidewalk",\n      "count": 4698\n    },\n    {\n      "word": "animal",\n      "count": 4616\n    },\n    {\n      "word": "piece",\n      "count": 4600\n    },\n    {\n      "word": "vegetable",\n      "count": 4581\n    },\n    {\n      "word": "fruit",\n      "count": 4471\n    },\n    {\n      "word": "tower",\n      "count": 4460\n    },\n    {\n      "word": "baby",\n      "count": 4406\n    },\n    {\n      "word": "vase",\n      "count": 4393\n    },\n    {\n      "word": "parking",\n      "count": 4392\n    },\n    {\n      "word": "mountain",\n      "count": 4360\n    },\n    {\n      "word": "ground",\n      "count": 4229\n    },\n    {\n      "word": "image",\n      "count": 4089\n    },\n    {\n      "word": "hill",\n      "count": 4036\n    },\n    {\n      "word": "day",\n      "count": 4019\n    },\n    {\n      "word": "tie",\n      "count": 3971\n    },\n    {\n      "word": "pole",\n      "count": 3886\n    },\n    {\n      "word": "head",\n      "count": 3854\n    },\n    {\n      "word": "slope",\n      "count": 3853\n    },\n    {\n      "word": "station",\n      "count": 3848\n    },\n    {\n      "word": "dirt",\n      "count": 3810\n    },\n    {\n      "word": "trick",\n      "count": 3797\n    },\n    {\n      "word": "skier",\n      "count": 3763\n    },\n    {\n      "word": "middle",\n      "count": 3732\n    },\n    {\n      "word": "hat",\n      "count": 3603\n    },\n    {\n      "word": "suit",\n      "count": 3594\n    },\n    {\n      "word": "close",\n      "count": 3561\n    },\n    {\n      "word": "box",\n      "count": 3557\n    },\n    {\n      "word": "back",\n      "count": 3550\n    },\n    {\n      "word": "wine",\n      "count": 3516\n    },\n    {\n      "word": "guy",\n      "count": 3418\n    },\n    {\n      "word": "orange",\n      "count": 3416\n    },\n    {\n      "word": "luggage",\n      "count": 3407\n    },\n    {\n      "word": "door",\n      "count": 3406\n    },\n    {\n      "word": "wood",\n      "count": 3394\n    },\n    {\n      "word": "refrigerator",\n      "count": 3346\n    },\n    {\n      "word": "hydrant",\n      "count": 3315\n    },\n    {\n      "word": "wii",\n      "count": 3271\n    },\n    {\n      "word": "broccoli",\n      "count": 3254\n    },\n    {\n      "word": "kid",\n      "count": 3251\n    },\n    {\n      "word": "book",\n      "count": 3249\n    },\n    {\n      "word": "camera",\n      "count": 3218\n    },\n    {\n      "word": "display",\n      "count": 3136\n    },\n    {\n      "word": "herd",\n      "count": 3095\n    },\n    {\n      "word": "keyboard",\n      "count": 3070\n    },\n    {\n      "word": "pair",\n      "count": 3070\n    },\n    {\n      "word": "slice",\n      "count": 3045\n    },\n    {\n      "word": "passenger",\n      "count": 3044\n    },\n    {\n      "word": "lady",\n      "count": 2997\n    },\n    {\n      "word": "corner",\n      "count": 2986\n    },\n    {\n      "word": "bicycle",\n      "count": 2973\n    },\n    {\n      "word": "cup",\n      "count": 2955\n    },\n    {\n      "word": "paper",\n      "count": 2922\n    },\n    {\n      "word": "skate",\n      "count": 2921\n    },\n    {\n      "word": "donut",\n      "count": 2910\n    },\n    {\n      "word": "surfer",\n      "count": 2863\n    },\n    {\n      "word": "television",\n      "count": 2855\n    },\n    {\n      "word": "stove",\n      "count": 2827\n    },\n    {\n      "word": "crowd",\n      "count": 2769\n    },\n    {\n      "word": "bottle",\n      "count": 2764\n    },\n    {\n      "word": "house",\n      "count": 2754\n    },\n    {\n      "word": "soccer",\n      "count": 2752\n    },\n    {\n      "word": "airport",\n      "count": 2738\n    },\n    {\n      "word": "bag",\n      "count": 2737\n    },\n    {\n      "word": "tv",\n      "count": 2736\n    },\n    {\n      "word": "suitcase",\n      "count": 2708\n    },\n    {\n      "word": "video",\n      "count": 2708\n    },\n    {\n      "word": "apple",\n      "count": 2640\n    },\n    {\n      "word": "body",\n      "count": 2578\n    },\n    {\n      "word": "rock",\n      "count": 2563\n    },\n    {\n      "word": "snowboard",\n      "count": 2535\n    },\n    {\n      "word": "brick",\n      "count": 2509\n    },\n    {\n      "word": "jet",\n      "count": 2502\n    },\n    {\n      "word": "item",\n      "count": 2459\n    },\n    {\n      "word": "coffee",\n      "count": 2417\n    },\n    {\n      "word": "river",\n      "count": 2409\n    },\n    {\n      "word": "cabinet",\n      "count": 2369\n    },\n    {\n      "word": "meat",\n      "count": 2362\n    },\n    {\n      "word": "standing",\n      "count": 2361\n    },\n    {\n      "word": "night",\n      "count": 2358\n    },\n    {\n      "word": "skateboarder",\n      "count": 2357\n    },\n    {\n      "word": "bedroom",\n      "count": 2354\n    },\n    {\n      "word": "tray",\n      "count": 2353\n    },\n    {\n      "word": "restaurant",\n      "count": 2348\n    },\n    {\n      "word": "surf",\n      "count": 2333\n    },\n    {\n      "word": "cheese",\n      "count": 2332\n    },\n    {\n      "word": "face",\n      "count": 2323\n    },\n    {\n      "word": "racquet",\n      "count": 2317\n    },\n    {\n      "word": "line",\n      "count": 2310\n    },\n    {\n      "word": "home",\n      "count": 2278\n    },\n    {\n      "word": "runway",\n      "count": 2256\n    },\n    {\n      "word": "metal",\n      "count": 2232\n    },\n    {\n      "word": "decker",\n      "count": 2225\n    },\n    {\n      "word": "ramp",\n      "count": 2204\n    },\n    {\n      "word": "toy",\n      "count": 2155\n    },\n    {\n      "word": "adult",\n      "count": 2118\n    },\n    {\n      "word": "carrot",\n      "count": 2093\n    },\n    {\n      "word": "screen",\n      "count": 2089\n    },\n    {\n      "word": "row",\n      "count": 2045\n    },\n    {\n      "word": "intersection",\n      "count": 2043\n    },\n    {\n      "word": "mouth",\n      "count": 2039\n    },\n    {\n      "word": "shower",\n      "count": 2036\n    },\n    {\n      "word": "shelf",\n      "count": 2019\n    },\n    {\n      "word": "doughnut",\n      "count": 2003\n    },\n    {\n      "word": "plant",\n      "count": 1996\n    },\n    {\n      "word": "store",\n      "count": 1996\n    },\n    {\n      "word": "dish",\n      "count": 1995\n    },\n    {\n      "word": "drink",\n      "count": 1992\n    },\n    {\n      "word": "monitor",\n      "count": 1982\n    },\n    {\n      "word": "number",\n      "count": 1957\n    },\n    {\n      "word": "batter",\n      "count": 1925\n    },\n    {\n      "word": "knife",\n      "count": 1906\n    },\n    {\n      "word": "bridge",\n      "count": 1900\n    },\n    {\n      "word": "jacket",\n      "count": 1851\n    },\n    {\n      "word": "scissor",\n      "count": 1845\n    },\n    {\n      "word": "vehicle",\n      "count": 1831\n    },\n    {\n      "word": "zoo",\n      "count": 1823\n    },\n    {\n      "word": "pile",\n      "count": 1800\n    },\n    {\n      "word": "seat",\n      "count": 1800\n    },\n    {\n      "word": "way",\n      "count": 1796\n    },\n    {\n      "word": "enclosure",\n      "count": 1783\n    },\n    {\n      "word": "pan",\n      "count": 1778\n    },\n    {\n      "word": "controller",\n      "count": 1756\n    },\n    {\n      "word": "sand",\n      "count": 1746\n    },\n    {\n      "word": "mouse",\n      "count": 1743\n    },\n    {\n      "word": "meter",\n      "count": 1739\n    },\n    {\n      "word": "microwave",\n      "count": 1723\n    },\n    {\n      "word": "meal",\n      "count": 1710\n    },\n    {\n      "word": "lake",\n      "count": 1663\n    },\n    {\n      "word": "bread",\n      "count": 1652\n    },\n    {\n      "word": "forest",\n      "count": 1645\n    },\n    {\n      "word": "tooth",\n      "count": 1628\n    },\n    {\n      "word": "photograph",\n      "count": 1603\n    },\n    {\n      "word": "stand",\n      "count": 1583\n    },\n    {\n      "word": "furniture",\n      "count": 1572\n    },\n    {\n      "word": "branch",\n      "count": 1571\n    },\n    {\n      "word": "oven",\n      "count": 1565\n    },\n    {\n      "word": "hair",\n      "count": 1561\n    },\n    {\n      "word": "tub",\n      "count": 1561\n    },\n    {\n      "word": "type",\n      "count": 1551\n    },\n    {\n      "word": "blanket",\n      "count": 1543\n    },\n    {\n      "word": "snowboarder",\n      "count": 1500\n    },\n    {\n      "word": "uniform",\n      "count": 1485\n    },\n    {\n      "word": "chocolate",\n      "count": 1464\n    },\n    {\n      "word": "stone",\n      "count": 1454\n    },\n    {\n      "word": "pillow",\n      "count": 1452\n    },\n    {\n      "word": "basket",\n      "count": 1432\n    },\n    {\n      "word": "rail",\n      "count": 1431\n    },\n    {\n      "word": "fork",\n      "count": 1430\n    },\n    {\n      "word": "statue",\n      "count": 1430\n    },\n    {\n      "word": "scene",\n      "count": 1420\n    },\n    {\n      "word": "dining",\n      "count": 1395\n    },\n    {\n      "word": "foot",\n      "count": 1394\n    },\n    {\n      "word": "birthday",\n      "count": 1383\n    },\n    {\n      "word": "fry",\n      "count": 1372\n    },\n    {\n      "word": "pot",\n      "count": 1372\n    },\n    {\n      "word": "helmet",\n      "count": 1357\n    },\n    {\n      "word": "office",\n      "count": 1354\n    },\n    {\n      "word": "arm",\n      "count": 1332\n    },\n    {\n      "word": "salad",\n      "count": 1328\n    },\n    {\n      "word": "cart",\n      "count": 1322\n    },\n    {\n      "word": "lamp",\n      "count": 1322\n    },\n    {\n      "word": "market",\n      "count": 1322\n    },\n    {\n      "word": "leaf",\n      "count": 1309\n    },\n    {\n      "word": "candle",\n      "count": 1306\n    },\n    {\n      "word": "rain",\n      "count": 1302\n    },\n    {\n      "word": "remote",\n      "count": 1301\n    },\n    {\n      "word": "case",\n      "count": 1283\n    },\n    {\n      "word": "cellphone",\n      "count": 1267\n    },\n    {\n      "word": "edge",\n      "count": 1262\n    },\n    {\n      "word": "shoe",\n      "count": 1262\n    },\n    {\n      "word": "dress",\n      "count": 1261\n    },\n    {\n      "word": "sun",\n      "count": 1253\n    },\n    {\n      "word": "set",\n      "count": 1241\n    },\n    {\n      "word": "half",\n      "count": 1230\n    },\n    {\n      "word": "container",\n      "count": 1225\n    },\n    {\n      "word": "towel",\n      "count": 1217\n    },\n    {\n      "word": "time",\n      "count": 1215\n    },\n    {\n      "word": "platform",\n      "count": 1213\n    },\n    {\n      "word": "catcher",\n      "count": 1208\n    },\n    {\n      "word": "shot",\n      "count": 1200\n    },\n    {\n      "word": "tomato",\n      "count": 1187\n    },\n    {\n      "word": "yard",\n      "count": 1179\n    },\n    {\n      "word": "bath",\n      "count": 1174\n    },\n    {\n      "word": "swing",\n      "count": 1168\n    },\n    {\n      "word": "toothbrush",\n      "count": 1158\n    },\n    {\n      "word": "tile",\n      "count": 1156\n    },\n    {\n      "word": "bush",\n      "count": 1146\n    },\n    {\n      "word": "country",\n      "count": 1142\n    },\n    {\n      "word": "motor",\n      "count": 1118\n    },\n    {\n      "word": "topping",\n      "count": 1117\n    },\n    {\n      "word": "pitcher",\n      "count": 1100\n    },\n    {\n      "word": "path",\n      "count": 1099\n    },\n    {\n      "word": "leg",\n      "count": 1093\n    },\n    {\n      "word": "family",\n      "count": 1092\n    },\n    {\n      "word": "hotel",\n      "count": 1089\n    },\n    {\n      "word": "engine",\n      "count": 1080\n    },\n    {\n      "word": "surface",\n      "count": 1079\n    },\n    {\n      "word": "post",\n      "count": 1078\n    },\n    {\n      "word": "police",\n      "count": 1073\n    },\n    {\n      "word": "beer",\n      "count": 1072\n    },\n    {\n      "word": "base",\n      "count": 1071\n    },\n    {\n      "word": "shore",\n      "count": 1069\n    },\n    {\n      "word": "control",\n      "count": 1063\n    },\n    {\n      "word": "lap",\n      "count": 1058\n    },\n    {\n      "word": "shop",\n      "count": 1058\n    },\n    {\n      "word": "pen",\n      "count": 1053\n    },\n    {\n      "word": "dinner",\n      "count": 1051\n    },\n    {\n      "word": "walk",\n      "count": 1051\n    },\n    {\n      "word": "variety",\n      "count": 1046\n    },\n    {\n      "word": "appliance",\n      "count": 1042\n    },\n    {\n      "word": "flag",\n      "count": 1035\n    },\n    {\n      "word": "town",\n      "count": 1032\n    },\n    {\n      "word": "bar",\n      "count": 1031\n    },\n    {\n      "word": "wire",\n      "count": 1025\n    },\n    {\n      "word": "skiing",\n      "count": 1023\n    },\n    {\n      "word": "potato",\n      "count": 1007\n    },\n    {\n      "word": "pitch",\n      "count": 1006\n    },\n    {\n      "word": "sauce",\n      "count": 1005\n    },\n    {\n      "word": "other",\n      "count": 999\n    },\n    {\n      "word": "carriage",\n      "count": 991\n    },\n    {\n      "word": "trunk",\n      "count": 990\n    },\n    {\n      "word": "place",\n      "count": 987\n    },\n    {\n      "word": "chicken",\n      "count": 983\n    },\n    {\n      "word": "vas",\n      "count": 974\n    },\n    {\n      "word": "end",\n      "count": 968\n    },\n    {\n      "word": "kind",\n      "count": 967\n    },\n    {\n      "word": "distance",\n      "count": 962\n    },\n    {\n      "word": "cutting",\n      "count": 954\n    },\n    {\n      "word": "tarmac",\n      "count": 951\n    },\n    {\n      "word": "pastry",\n      "count": 950\n    },\n    {\n      "word": "neck",\n      "count": 949\n    },\n    {\n      "word": "silver",\n      "count": 949\n    },\n    {\n      "word": "dessert",\n      "count": 938\n    },\n    {\n      "word": "gear",\n      "count": 934\n    },\n    {\n      "word": "graffiti",\n      "count": 932\n    },\n    {\n      "word": "hotdog",\n      "count": 932\n    },\n    {\n      "word": "rice",\n      "count": 932\n    },\n    {\n      "word": "male",\n      "count": 929\n    },\n    {\n      "word": "curb",\n      "count": 920\n    },\n    {\n      "word": "nintendo",\n      "count": 919\n    },\n    {\n      "word": "egg",\n      "count": 917\n    },\n    {\n      "word": "color",\n      "count": 904\n    },\n    {\n      "word": "sofa",\n      "count": 900\n    },\n    {\n      "word": "eye",\n      "count": 899\n    },\n    {\n      "word": "dock",\n      "count": 898\n    },\n    {\n      "word": "cattle",\n      "count": 895\n    },\n    {\n      "word": "pasture",\n      "count": 891\n    },\n    {\n      "word": "bathtub",\n      "count": 885\n    },\n    {\n      "word": "railroad",\n      "count": 881\n    },\n    {\n      "word": "brush",\n      "count": 876\n    },\n    {\n      "word": "school",\n      "count": 876\n    },\n    {\n      "word": "curtain",\n      "count": 872\n    },\n    {\n      "word": "rider",\n      "count": 871\n    },\n    {\n      "word": "jump",\n      "count": 869\n    },\n    {\n      "word": "onion",\n      "count": 869\n    },\n    {\n      "word": "rack",\n      "count": 861\n    },\n    {\n      "word": "steel",\n      "count": 856\n    },\n    {\n      "word": "match",\n      "count": 852\n    },\n    {\n      "word": "fireplace",\n      "count": 842\n    },\n    {\n      "word": "team",\n      "count": 840\n    },\n    {\n      "word": "style",\n      "count": 825\n    },\n    {\n      "word": "cross",\n      "count": 821\n    },\n    {\n      "word": "fridge",\n      "count": 821\n    },\n    {\n      "word": "spoon",\n      "count": 819\n    },\n    {\n      "word": "playing",\n      "count": 815\n    },\n    {\n      "word": "sheet",\n      "count": 810\n    },\n    {\n      "word": "space",\n      "count": 802\n    },\n    {\n      "word": "equipment",\n      "count": 798\n    },\n    {\n      "word": "object",\n      "count": 792\n    },\n    {\n      "word": "church",\n      "count": 786\n    },\n    {\n      "word": "glove",\n      "count": 784\n    },\n    {\n      "word": "backpack",\n      "count": 778\n    },\n    {\n      "word": "center",\n      "count": 774\n    },\n    {\n      "word": "bun",\n      "count": 771\n    },\n    {\n      "word": "stall",\n      "count": 756\n    },\n    {\n      "word": "clothe",\n      "count": 754\n    },\n    {\n      "word": "green",\n      "count": 748\n    },\n    {\n      "word": "trail",\n      "count": 747\n    },\n    {\n      "word": "pedestrian",\n      "count": 744\n    },\n    {\n      "word": "direction",\n      "count": 737\n    },\n    {\n      "word": "hay",\n      "count": 733\n    },\n    {\n      "word": "white",\n      "count": 724\n    },\n    {\n      "word": "reflection",\n      "count": 721\n    },\n    {\n      "word": "cream",\n      "count": 718\n    },\n    {\n      "word": "business",\n      "count": 716\n    },\n    {\n      "word": "mother",\n      "count": 712\n    },\n    {\n      "word": "thing",\n      "count": 711\n    },\n    {\n      "word": "friend",\n      "count": 707\n    },\n    {\n      "word": "wheel",\n      "count": 705\n    },\n    {\n      "word": "christmas",\n      "count": 704\n    },\n    {\n      "word": "short",\n      "count": 704\n    },\n    {\n      "word": "cement",\n      "count": 702\n    },\n    {\n      "word": "breakfast",\n      "count": 696\n    },\n    {\n      "word": "step",\n      "count": 691\n    },\n    {\n      "word": "painting",\n      "count": 689\n    },\n    {\n      "word": "subway",\n      "count": 688\n    },\n    {\n      "word": "stair",\n      "count": 687\n    },\n    {\n      "word": "soup",\n      "count": 682\n    },\n    {\n      "word": "blender",\n      "count": 681\n    },\n    {\n      "word": "device",\n      "count": 679\n    },\n    {\n      "word": "wedding",\n      "count": 674\n    },\n    {\n      "word": "one",\n      "count": 673\n    },\n    {\n      "word": "van",\n      "count": 673\n    },\n    {\n      "word": "boarder",\n      "count": 670\n    },\n    {\n      "word": "highway",\n      "count": 669\n    },\n    {\n      "word": "cloud",\n      "count": 664\n    },\n    {\n      "word": "pool",\n      "count": 664\n    },\n    {\n      "word": "rug",\n      "count": 664\n    },\n    {\n      "word": "umpire",\n      "count": 662\n    },\n    {\n      "word": "officer",\n      "count": 659\n    },\n    {\n      "word": "ear",\n      "count": 648\n    },\n    {\n      "word": "desert",\n      "count": 639\n    },\n    {\n      "word": "garden",\n      "count": 639\n    },\n    {\n      "word": "sunglass",\n      "count": 639\n    },\n    {\n      "word": "assortment",\n      "count": 636\n    },\n    {\n      "word": "scooter",\n      "count": 634\n    },\n    {\n      "word": "coat",\n      "count": 631\n    },\n    {\n      "word": "gate",\n      "count": 628\n    },\n    {\n      "word": "restroom",\n      "count": 627\n    },\n    {\n      "word": "duck",\n      "count": 619\n    },\n    {\n      "word": "hillside",\n      "count": 617\n    },\n    {\n      "word": "ledge",\n      "count": 616\n    },\n    {\n      "word": "desktop",\n      "count": 615\n    },\n    {\n      "word": "can",\n      "count": 612\n    },\n    {\n      "word": "work",\n      "count": 610\n    },\n    {\n      "word": "machine",\n      "count": 607\n    },\n    {\n      "word": "pie",\n      "count": 607\n    },\n    {\n      "word": "goat",\n      "count": 606\n    },\n    {\n      "word": "lawn",\n      "count": 606\n    },\n    {\n      "word": "cage",\n      "count": 605\n    },\n    {\n      "word": "flock",\n      "count": 599\n    },\n    {\n      "word": "toddler",\n      "count": 598\n    },\n    {\n      "word": "trash",\n      "count": 597\n    },\n    {\n      "word": "setting",\n      "count": 595\n    },\n    {\n      "word": "ceiling",\n      "count": 594\n    },\n    {\n      "word": "riding",\n      "count": 593\n    },\n    {\n      "word": "ice",\n      "count": 592\n    },\n    {\n      "word": "ship",\n      "count": 586\n    },\n    {\n      "word": "shade",\n      "count": 582\n    },\n    {\n      "word": "event",\n      "count": 580\n    },\n    {\n      "word": "roll",\n      "count": 574\n    },\n    {\n      "word": "fish",\n      "count": 573\n    },\n    {\n      "word": "signal",\n      "count": 573\n    },\n    {\n      "word": "veggie",\n      "count": 568\n    },\n    {\n      "word": "pepper",\n      "count": 567\n    },\n    {\n      "word": "race",\n      "count": 567\n    },\n    {\n      "word": "party",\n      "count": 560\n    },\n    {\n      "word": "construction",\n      "count": 556\n    },\n    {\n      "word": "kitten",\n      "count": 553\n    },\n    {\n      "word": "picnic",\n      "count": 553\n    },\n    {\n      "word": "log",\n      "count": 551\n    },\n    {\n      "word": "roof",\n      "count": 549\n    },\n    {\n      "word": "stick",\n      "count": 548\n    },\n    {\n      "word": "trailer",\n      "count": 547\n    },\n    {\n      "word": "cloth",\n      "count": 545\n    },\n    {\n      "word": "sale",\n      "count": 543\n    },\n    {\n      "word": "hole",\n      "count": 542\n    },\n    {\n      "word": "lunch",\n      "count": 541\n    },\n    {\n      "word": "decoration",\n      "count": 537\n    },\n    {\n      "word": "part",\n      "count": 531\n    },\n    {\n      "word": "sail",\n      "count": 528\n    },\n    {\n      "word": "pond",\n      "count": 523\n    },\n    {\n      "word": "stack",\n      "count": 522\n    },\n    {\n      "word": "beverage",\n      "count": 517\n    },\n    {\n      "word": "outfit",\n      "count": 517\n    },\n    {\n      "word": "bite",\n      "count": 512\n    },\n    {\n      "word": "pasta",\n      "count": 510\n    },\n    {\n      "word": "turn",\n      "count": 509\n    },\n    {\n      "word": "sunset",\n      "count": 508\n    },\n    {\n      "word": "cupcake",\n      "count": 506\n    },\n    {\n      "word": "leash",\n      "count": 505\n    },\n    {\n      "word": "chef",\n      "count": 503\n    },\n    {\n      "word": "chip",\n      "count": 503\n    },\n    {\n      "word": "walkway",\n      "count": 502\n    },\n    {\n      "word": "costume",\n      "count": 501\n    },\n    {\n      "word": "cover",\n      "count": 499\n    },\n    {\n      "word": "doll",\n      "count": 499\n    },\n    {\n      "word": "tent",\n      "count": 499\n    },\n    {\n      "word": "transit",\n      "count": 499\n    },\n    {\n      "word": "commuter",\n      "count": 498\n    },\n    {\n      "word": "leather",\n      "count": 498\n    },\n    {\n      "word": "sea",\n      "count": 497\n    },\n    {\n      "word": "foreground",\n      "count": 495\n    },\n    {\n      "word": "worker",\n      "count": 495\n    },\n    {\n      "word": "fighter",\n      "count": 493\n    },\n    {\n      "word": "grazing",\n      "count": 493\n    },\n    {\n      "word": "structure",\n      "count": 489\n    },\n    {\n      "word": "action",\n      "count": 487\n    },\n    {\n      "word": "pant",\n      "count": 485\n    },\n    {\n      "word": "show",\n      "count": 485\n    },\n    {\n      "word": "land",\n      "count": 484\n    },\n    {\n      "word": "clothing",\n      "count": 480\n    },\n    {\n      "word": "pipe",\n      "count": 480\n    },\n    {\n      "word": "doorway",\n      "count": 478\n    },\n    {\n      "word": "palm",\n      "count": 477\n    },\n    {\n      "word": "bean",\n      "count": 476\n    },\n    {\n      "word": "plastic",\n      "count": 475\n    },\n    {\n      "word": "net",\n      "count": 472\n    },\n    {\n      "word": "wing",\n      "count": 472\n    },\n    {\n      "word": "mushroom",\n      "count": 471\n    },\n    {\n      "word": "closeup",\n      "count": 470\n    },\n    {\n      "word": "watch",\n      "count": 468\n    },\n    {\n      "word": "model",\n      "count": 466\n    },\n    {\n      "word": "ride",\n      "count": 465\n    },\n    {\n      "word": "cone",\n      "count": 463\n    },\n    {\n      "word": "red",\n      "count": 461\n    },\n    {\n      "word": "pepperoni",\n      "count": 460\n    },\n    {\n      "word": "tour",\n      "count": 459\n    },\n    {\n      "word": "fan",\n      "count": 457\n    },\n    {\n      "word": "sausage",\n      "count": 457\n    },\n    {\n      "word": "cap",\n      "count": 456\n    },\n    {\n      "word": "apartment",\n      "count": 455\n    },\n    {\n      "word": "smoke",\n      "count": 455\n    },\n    {\n      "word": "cut",\n      "count": 454\n    },\n    {\n      "word": "mug",\n      "count": 453\n    },\n    {\n      "word": "harbor",\n      "count": 452\n    },\n    {\n      "word": "pier",\n      "count": 452\n    },\n    {\n      "word": "sport",\n      "count": 452\n    },\n    {\n      "word": "deck",\n      "count": 450\n    },\n    {\n      "word": "paw",\n      "count": 449\n    },\n    {\n      "word": "nose",\n      "count": 448\n    },\n    {\n      "word": "island",\n      "count": 446\n    },\n    {\n      "word": "winter",\n      "count": 443\n    },\n    {\n      "word": "wetsuit",\n      "count": 440\n    },\n    {\n      "word": "boarding",\n      "count": 439\n    },\n    {\n      "word": "grill",\n      "count": 439\n    },\n    {\n      "word": "look",\n      "count": 438\n    },\n    {\n      "word": "sticker",\n      "count": 437\n    },\n    {\n      "word": "soda",\n      "count": 436\n    },\n    {\n      "word": "garage",\n      "count": 432\n    },\n    {\n      "word": "tan",\n      "count": 432\n    },\n    {\n      "word": "bottom",\n      "count": 431\n    },\n    {\n      "word": "bull",\n      "count": 429\n    },\n    {\n      "word": "farm",\n      "count": 429\n    },\n    {\n      "word": "steam",\n      "count": 429\n    },\n    {\n      "word": "vanity",\n      "count": 428\n    },\n    {\n      "word": "design",\n      "count": 426\n    },\n    {\n      "word": "lift",\n      "count": 424\n    },\n    {\n      "word": "noodle",\n      "count": 423\n    },\n    {\n      "word": "crossing",\n      "count": 422\n    },\n    {\n      "word": "landing",\n      "count": 420\n    },\n    {\n      "word": "disc",\n      "count": 419\n    },\n    {\n      "word": "lid",\n      "count": 416\n    },\n    {\n      "word": "shoulder",\n      "count": 416\n    },\n    {\n      "word": "size",\n      "count": 415\n    },\n    {\n      "word": "bow",\n      "count": 412\n    },\n    {\n      "word": "sort",\n      "count": 412\n    },\n    {\n      "word": "word",\n      "count": 412\n    },\n    {\n      "word": "pickle",\n      "count": 411\n    },\n    {\n      "word": "platter",\n      "count": 410\n    },\n    {\n      "word": "strawberry",\n      "count": 410\n    },\n    {\n      "word": "tea",\n      "count": 410\n    },\n    {\n      "word": "purse",\n      "count": 409\n    },\n    {\n      "word": "stunt",\n      "count": 409\n    },\n    {\n      "word": "gold",\n      "count": 406\n    },\n    {\n      "word": "pavement",\n      "count": 404\n    },\n    {\n      "word": "patio",\n      "count": 400\n    },\n    {\n      "word": "rest",\n      "count": 400\n    },\n    {\n      "word": "urinal",\n      "count": 398\n    },\n    {\n      "word": "sculpture",\n      "count": 397\n    },\n    {\n      "word": "dresser",\n      "count": 396\n    },\n    {\n      "word": "carpet",\n      "count": 395\n    },\n    {\n      "word": "bacon",\n      "count": 394\n    },\n    {\n      "word": "flight",\n      "count": 394\n    },\n    {\n      "word": "railing",\n      "count": 391\n    },\n    {\n      "word": "juice",\n      "count": 389\n    },\n    {\n      "word": "seagull",\n      "count": 388\n    },\n    {\n      "word": "chain",\n      "count": 385\n    },\n    {\n      "word": "rose",\n      "count": 385\n    },\n    {\n      "word": "frame",\n      "count": 383\n    },\n    {\n      "word": "name",\n      "count": 383\n    },\n    {\n      "word": "sweater",\n      "count": 382\n    },\n    {\n      "word": "lamb",\n      "count": 381\n    },\n    {\n      "word": "barn",\n      "count": 379\n    },\n    {\n      "word": "milk",\n      "count": 377\n    },\n    {\n      "word": "rope",\n      "count": 376\n    },\n    {\n      "word": "stuff",\n      "count": 375\n    },\n    {\n      "word": "jar",\n      "count": 373\n    },\n    {\n      "word": "produce",\n      "count": 373\n    },\n    {\n      "word": "course",\n      "count": 372\n    },\n    {\n      "word": "up",\n      "count": 372\n    },\n    {\n      "word": "bucket",\n      "count": 369\n    },\n    {\n      "word": "parade",\n      "count": 369\n    },\n    {\n      "word": "string",\n      "count": 369\n    },\n    {\n      "word": "mustard",\n      "count": 368\n    },\n    {\n      "word": "plain",\n      "count": 367\n    },\n    {\n      "word": "shape",\n      "count": 367\n    },\n    {\n      "word": "driver",\n      "count": 366\n    },\n    {\n      "word": "bride",\n      "count": 365\n    },\n    {\n      "word": "gas",\n      "count": 365\n    },\n    {\n      "word": "mound",\n      "count": 365\n    },\n    {\n      "word": "terminal",\n      "count": 365\n    },\n    {\n      "word": "mitt",\n      "count": 363\n    },\n    {\n      "word": "holder",\n      "count": 361\n    },\n    {\n      "word": "tank",\n      "count": 361\n    },\n    {\n      "word": "ketchup",\n      "count": 359\n    },\n    {\n      "word": "airliner",\n      "count": 357\n    },\n    {\n      "word": "lemon",\n      "count": 355\n    },\n    {\n      "word": "collection",\n      "count": 354\n    },\n    {\n      "word": "guitar",\n      "count": 353\n    },\n    {\n      "word": "art",\n      "count": 349\n    },\n    {\n      "word": "handle",\n      "count": 348\n    },\n    {\n      "word": "wagon",\n      "count": 348\n    },\n    {\n      "word": "spectator",\n      "count": 347\n    },\n    {\n      "word": "paddle",\n      "count": 345\n    },\n    {\n      "word": "utensil",\n      "count": 344\n    },\n    {\n      "word": "wind",\n      "count": 342\n    },\n    {\n      "word": "aircraft",\n      "count": 339\n    },\n    {\n      "word": "block",\n      "count": 339\n    },\n    {\n      "word": "cooking",\n      "count": 339\n    },\n    {\n      "word": "roadway",\n      "count": 339\n    },\n    {\n      "word": "sill",\n      "count": 339\n    },\n    {\n      "word": "smile",\n      "count": 339\n    },\n    {\n      "word": "tail",\n      "count": 339\n    },\n    {\n      "word": "telephone",\n      "count": 339\n    },\n    {\n      "word": "crosswalk",\n      "count": 338\n    },\n    {\n      "word": "napkin",\n      "count": 338\n    },\n    {\n      "word": "supply",\n      "count": 336\n    },\n    {\n      "word": "lettuce",\n      "count": 335\n    },\n    {\n      "word": "stadium",\n      "count": 335\n    },\n    {\n      "word": "jean",\n      "count": 334\n    },\n    {\n      "word": "motorcyclist",\n      "count": 334\n    },\n    {\n      "word": "puppy",\n      "count": 334\n    },\n    {\n      "word": "formation",\n      "count": 333\n    },\n    {\n      "word": "fun",\n      "count": 333\n    },\n    {\n      "word": "power",\n      "count": 333\n    },\n    {\n      "word": "tow",\n      "count": 333\n    },\n    {\n      "word": "pickup",\n      "count": 332\n    },\n    {\n      "word": "sailboat",\n      "count": 330\n    },\n    {\n      "word": "female",\n      "count": 329\n    },\n    {\n      "word": "t",\n      "count": 329\n    },\n    {\n      "word": "vest",\n      "count": 326\n    },\n    {\n      "word": "electronic",\n      "count": 325\n    },\n    {\n      "word": "grey",\n      "count": 324\n    },\n    {\n      "word": "inside",\n      "count": 323\n    },\n    {\n      "word": "shirtless",\n      "count": 323\n    },\n    {\n      "word": "bank",\n      "count": 321\n    },\n    {\n      "word": "jetliner",\n      "count": 321\n    },\n    {\n      "word": "museum",\n      "count": 321\n    },\n    {\n      "word": "play",\n      "count": 319\n    },\n    {\n      "word": "selfie",\n      "count": 319\n    },\n    {\n      "word": "stool",\n      "count": 319\n    },\n    {\n      "word": "paint",\n      "count": 318\n    },\n    {\n      "word": "speed",\n      "count": 317\n    },\n    {\n      "word": "toast",\n      "count": 317\n    },\n    {\n      "word": "position",\n      "count": 316\n    },\n    {\n      "word": "pack",\n      "count": 313\n    },\n    {\n      "word": "freezer",\n      "count": 312\n    },\n    {\n      "word": "motion",\n      "count": 311\n    },\n    {\n      "word": "propeller",\n      "count": 311\n    },\n    {\n      "word": "patch",\n      "count": 310\n    },\n    {\n      "word": "toaster",\n      "count": 310\n    },\n    {\n      "word": "groom",\n      "count": 308\n    },\n    {\n      "word": "landscape",\n      "count": 308\n    },\n    {\n      "word": "run",\n      "count": 308\n    },\n    {\n      "word": "vintage",\n      "count": 308\n    },\n    {\n      "word": "olive",\n      "count": 307\n    },\n    {\n      "word": "microphone",\n      "count": 306\n    },\n    {\n      "word": "sprinkle",\n      "count": 306\n    },\n    {\n      "word": "tool",\n      "count": 306\n    },\n    {\n      "word": "garbage",\n      "count": 305\n    },\n    {\n      "word": "lane",\n      "count": 305\n    },\n    {\n      "word": "stream",\n      "count": 305\n    },\n    {\n      "word": "tongue",\n      "count": 305\n    },\n    {\n      "word": "drawer",\n      "count": 303\n    },\n    {\n      "word": "grassy",\n      "count": 302\n    },\n    {\n      "word": "ingredient",\n      "count": 301\n    },\n    {\n      "word": "shadow",\n      "count": 301\n    },\n    {\n      "word": "drive",\n      "count": 300\n    },\n    {\n      "word": "snack",\n      "count": 299\n    },\n    {\n      "word": "square",\n      "count": 298\n    },\n    {\n      "word": "arrangement",\n      "count": 296\n    },\n    {\n      "word": "cupboard",\n      "count": 296\n    },\n    {\n      "word": "wild",\n      "count": 296\n    },\n    {\n      "word": "advertisement",\n      "count": 295\n    },\n    {\n      "word": "shopping",\n      "count": 294\n    },\n    {\n      "word": "tourist",\n      "count": 294\n    },\n    {\n      "word": "biker",\n      "count": 293\n    },\n    {\n      "word": "cycle",\n      "count": 293\n    },\n    {\n      "word": "life",\n      "count": 292\n    },\n    {\n      "word": "skirt",\n      "count": 292\n    },\n    {\n      "word": "finger",\n      "count": 291\n    },\n    {\n      "word": "big",\n      "count": 289\n    },\n    {\n      "word": "headphone",\n      "count": 289\n    },\n    {\n      "word": "ring",\n      "count": 289\n    },\n    {\n      "word": "marble",\n      "count": 287\n    },\n    {\n      "word": "tag",\n      "count": 287\n    },\n    {\n      "word": "balloon",\n      "count": 286\n    },\n    {\n      "word": "competition",\n      "count": 286\n    },\n    {\n      "word": "individual",\n      "count": 284\n    },\n    {\n      "word": "dark",\n      "count": 283\n    },\n    {\n      "word": "entrance",\n      "count": 283\n    },\n    {\n      "word": "newspaper",\n      "count": 283\n    },\n    {\n      "word": "horseback",\n      "count": 282\n    },\n    {\n      "word": "habitat",\n      "count": 281\n    },\n    {\n      "word": "sitting",\n      "count": 279\n    },\n    {\n      "word": "card",\n      "count": 276\n    },\n    {\n      "word": "cargo",\n      "count": 276\n    },\n    {\n      "word": "london",\n      "count": 275\n    },\n    {\n      "word": "range",\n      "count": 275\n    },\n    {\n      "word": "gentleman",\n      "count": 274\n    },\n    {\n      "word": "motorbike",\n      "count": 274\n    },\n    {\n      "word": "neon",\n      "count": 274\n    },\n    {\n      "word": "stage",\n      "count": 274\n    },\n    {\n      "word": "blue",\n      "count": 273\n    },\n    {\n      "word": "spot",\n      "count": 271\n    },\n    {\n      "word": "calf",\n      "count": 270\n    },\n    {\n      "word": "drinking",\n      "count": 270\n    },\n    {\n      "word": "straw",\n      "count": 270\n    },\n    {\n      "word": "swimming",\n      "count": 270\n    },\n    {\n      "word": "burger",\n      "count": 269\n    },\n    {\n      "word": "fly",\n      "count": 269\n    },\n    {\n      "word": "horn",\n      "count": 269\n    },\n    {\n      "word": "pigeon",\n      "count": 269\n    },\n    {\n      "word": "muffin",\n      "count": 268\n    },\n    {\n      "word": "ben",\n      "count": 266\n    },\n    {\n      "word": "beard",\n      "count": 265\n    },\n    {\n      "word": "poster",\n      "count": 265\n    },\n    {\n      "word": "basketball",\n      "count": 264\n    },\n    {\n      "word": "stoplight",\n      "count": 264\n    },\n    {\n      "word": "bookshelf",\n      "count": 263\n    },\n    {\n      "word": "cooky",\n      "count": 262\n    },\n    {\n      "word": "figure",\n      "count": 262\n    },\n    {\n      "word": "owner",\n      "count": 262\n    },\n    {\n      "word": "beef",\n      "count": 260\n    },\n    {\n      "word": "condiment",\n      "count": 260\n    },\n    {\n      "word": "fountain",\n      "count": 260\n    },\n    {\n      "word": "serve",\n      "count": 260\n    },\n    {\n      "word": "heart",\n      "count": 258\n    },\n    {\n      "word": "hamburger",\n      "count": 256\n    },\n    {\n      "word": "rainbow",\n      "count": 255\n    },\n    {\n      "word": "concrete",\n      "count": 254\n    },\n    {\n      "word": "foil",\n      "count": 254\n    },\n    {\n      "word": "coach",\n      "count": 253\n    },\n    {\n      "word": "tire",\n      "count": 253\n    },\n    {\n      "word": "candy",\n      "count": 252\n    },\n    {\n      "word": "clay",\n      "count": 252\n    },\n    {\n      "word": "speaker",\n      "count": 252\n    },\n    {\n      "word": "trolley",\n      "count": 252\n    },\n    {\n      "word": "boot",\n      "count": 250\n    },\n    {\n      "word": "story",\n      "count": 250\n    },\n    {\n      "word": "teenager",\n      "count": 250\n    },\n    {\n      "word": "bunk",\n      "count": 249\n    },\n    {\n      "word": "hood",\n      "count": 249\n    },\n    {\n      "word": "pose",\n      "count": 249\n    },\n    {\n      "word": "butter",\n      "count": 248\n    },\n    {\n      "word": "soldier",\n      "count": 248\n    },\n    {\n      "word": "liquid",\n      "count": 247\n    },\n    {\n      "word": "corn",\n      "count": 246\n    },\n    {\n      "word": "grape",\n      "count": 246\n    },\n    {\n      "word": "ham",\n      "count": 245\n    },\n    {\n      "word": "brown",\n      "count": 243\n    },\n    {\n      "word": "travel",\n      "count": 243\n    },\n    {\n      "word": "bell",\n      "count": 242\n    },\n    {\n      "word": "comforter",\n      "count": 242\n    },\n    {\n      "word": "crust",\n      "count": 242\n    },\n    {\n      "word": "league",\n      "count": 242\n    },\n    {\n      "word": "cop",\n      "count": 240\n    },\n    {\n      "word": "figurine",\n      "count": 240\n    },\n    {\n      "word": "magazine",\n      "count": 240\n    },\n    {\n      "word": "pad",\n      "count": 240\n    },\n    {\n      "word": "accessory",\n      "count": 239\n    },\n    {\n      "word": "meadow",\n      "count": 239\n    },\n    {\n      "word": "outside",\n      "count": 239\n    },\n    {\n      "word": "section",\n      "count": 239\n    },\n    {\n      "word": "booth",\n      "count": 238\n    },\n    {\n      "word": "knee",\n      "count": 238\n    },\n    {\n      "word": "hose",\n      "count": 237\n    },\n    {\n      "word": "mat",\n      "count": 237\n    },\n    {\n      "word": "canoe",\n      "count": 236\n    },\n    {\n      "word": "series",\n      "count": 236\n    },\n    {\n      "word": "system",\n      "count": 236\n    },\n    {\n      "word": "attire",\n      "count": 235\n    },\n    {\n      "word": "point",\n      "count": 235\n    },\n    {\n      "word": "exhibit",\n      "count": 234\n    },\n    {\n      "word": "cafe",\n      "count": 233\n    },\n    {\n      "word": "iron",\n      "count": 233\n    },\n    {\n      "word": "mask",\n      "count": 233\n    },\n    {\n      "word": "someone",\n      "count": 232\n    },\n    {\n      "word": "strip",\n      "count": 232\n    },\n    {\n      "word": "taxi",\n      "count": 232\n    },\n    {\n      "word": "feeder",\n      "count": 231\n    },\n    {\n      "word": "student",\n      "count": 231\n    },\n    {\n      "word": "move",\n      "count": 230\n    },\n    {\n      "word": "button",\n      "count": 229\n    },\n    {\n      "word": "railway",\n      "count": 229\n    },\n    {\n      "word": "surfing",\n      "count": 228\n    },\n    {\n      "word": "hallway",\n      "count": 227\n    },\n    {\n      "word": "parrot",\n      "count": 227\n    },\n    {\n      "word": "steak",\n      "count": 227\n    },\n    {\n      "word": "cardboard",\n      "count": 226\n    },\n    {\n      "word": "countertop",\n      "count": 226\n    },\n    {\n      "word": "pancake",\n      "count": 226\n    },\n    {\n      "word": "tractor",\n      "count": 226\n    },\n    {\n      "word": "crate",\n      "count": 225\n    },\n    {\n      "word": "library",\n      "count": 225\n    },\n    {\n      "word": "cowboy",\n      "count": 223\n    },\n    {\n      "word": "dishwasher",\n      "count": 223\n    },\n    {\n      "word": "jockey",\n      "count": 223\n    },\n    {\n      "word": "apron",\n      "count": 222\n    },\n    {\n      "word": "graze",\n      "count": 222\n    },\n    {\n      "word": "hospital",\n      "count": 221\n    },\n    {\n      "word": "collar",\n      "count": 220\n    },\n    {\n      "word": "magnet",\n      "count": 220\n    },\n    {\n      "word": "snowy",\n      "count": 220\n    },\n    {\n      "word": "marina",\n      "count": 219\n    },\n    {\n      "word": "neighborhood",\n      "count": 219\n    },\n    {\n      "word": "batting",\n      "count": 218\n    },\n    {\n      "word": "bin",\n      "count": 218\n    },\n    {\n      "word": "father",\n      "count": 218\n    },\n    {\n      "word": "onlooker",\n      "count": 218\n    },\n    {\n      "word": "bouquet",\n      "count": 217\n    },\n    {\n      "word": "dryer",\n      "count": 217\n    },\n    {\n      "word": "grocery",\n      "count": 217\n    },\n    {\n      "word": "unmade",\n      "count": 217\n    },\n    {\n      "word": "watering",\n      "count": 217\n    },\n    {\n      "word": "arrow",\n      "count": 216\n    },\n    {\n      "word": "chest",\n      "count": 216\n    },\n    {\n      "word": "cliff",\n      "count": 215\n    },\n    {\n      "word": "countryside",\n      "count": 214\n    },\n    {\n      "word": "driveway",\n      "count": 214\n    },\n    {\n      "word": "steeple",\n      "count": 214\n    },\n    {\n      "word": "tunnel",\n      "count": 214\n    },\n    {\n      "word": "farmer",\n      "count": 213\n    },\n    {\n      "word": "flooring",\n      "count": 213\n    },\n    {\n      "word": "mattress",\n      "count": 213\n    },\n    {\n      "word": "walking",\n      "count": 213\n    },\n    {\n      "word": "audience",\n      "count": 212\n    },\n    {\n      "word": "berry",\n      "count": 212\n    },\n    {\n      "word": "cigarette",\n      "count": 212\n    },\n    {\n      "word": "buffet",\n      "count": 211\n    },\n    {\n      "word": "bicyclist",\n      "count": 210\n    },\n    {\n      "word": "santa",\n      "count": 210\n    },\n    {\n      "word": "stare",\n      "count": 210\n    },\n    {\n      "word": "piano",\n      "count": 209\n    },\n    {\n      "word": "sub",\n      "count": 209\n    },\n    {\n      "word": "bakery",\n      "count": 208\n    },\n    {\n      "word": "lighthouse",\n      "count": 207\n    },\n    {\n      "word": "note",\n      "count": 207\n    },\n    {\n      "word": "sunlight",\n      "count": 206\n    },\n    {\n      "word": "tusk",\n      "count": 206\n    },\n    {\n      "word": "circle",\n      "count": 205\n    },\n    {\n      "word": "downtown",\n      "count": 205\n    },\n    {\n      "word": "ribbon",\n      "count": 205\n    },\n    {\n      "word": "content",\n      "count": 204\n    },\n    {\n      "word": "entertainment",\n      "count": 204\n    },\n    {\n      "word": "vendor",\n      "count": 204\n    },\n    {\n      "word": "dusk",\n      "count": 203\n    },\n    {\n      "word": "nut",\n      "count": 203\n    },\n    {\n      "word": "pony",\n      "count": 202\n    },\n    {\n      "word": "resort",\n      "count": 201\n    },\n    {\n      "word": "tablet",\n      "count": 201\n    },\n    {\n      "word": "baggage",\n      "count": 200\n    },\n    {\n      "word": "cook",\n      "count": 200\n    },\n    {\n      "word": "para",\n      "count": 199\n    },\n    {\n      "word": "backyard",\n      "count": 198\n    },\n    {\n      "word": "canopy",\n      "count": 198\n    },\n    {\n      "word": "dozen",\n      "count": 198\n    },\n    {\n      "word": "fixture",\n      "count": 198\n    },\n    {\n      "word": "ladder",\n      "count": 197\n    },\n    {\n      "word": "sugar",\n      "count": 197\n    },\n    {\n      "word": "balcony",\n      "count": 196\n    },\n    {\n      "word": "jersey",\n      "count": 196\n    },\n    {\n      "word": "obstacle",\n      "count": 196\n    },\n    {\n      "word": "progress",\n      "count": 196\n    },\n    {\n      "word": "stripe",\n      "count": 196\n    },\n    {\n      "word": "turkey",\n      "count": 196\n    },\n    {\n      "word": "writing",\n      "count": 196\n    },\n    {\n      "word": "bay",\n      "count": 195\n    },\n    {\n      "word": "cushion",\n      "count": 195\n    },\n    {\n      "word": "loading",\n      "count": 195\n    },\n    {\n      "word": "interior",\n      "count": 193\n    },\n    {\n      "word": "level",\n      "count": 193\n    },\n    {\n      "word": "mud",\n      "count": 193\n    },\n    {\n      "word": "son",\n      "count": 193\n    },\n    {\n      "word": "storage",\n      "count": 193\n    },\n    {\n      "word": "belt",\n      "count": 192\n    },\n    {\n      "word": "focus",\n      "count": 192\n    },\n    {\n      "word": "layer",\n      "count": 192\n    },\n    {\n      "word": "scarf",\n      "count": 192\n    },\n    {\n      "word": "crane",\n      "count": 191\n    },\n    {\n      "word": "fishing",\n      "count": 191\n    },\n    {\n      "word": "amount",\n      "count": 190\n    },\n    {\n      "word": "collage",\n      "count": 190\n    },\n    {\n      "word": "evening",\n      "count": 189\n    },\n    {\n      "word": "kitty",\n      "count": 188\n    },\n    {\n      "word": "spinach",\n      "count": 188\n    },\n    {\n      "word": "decor",\n      "count": 187\n    },\n    {\n      "word": "language",\n      "count": 187\n    },\n    {\n      "word": "lighting",\n      "count": 187\n    },\n    {\n      "word": "cabin",\n      "count": 186\n    },\n    {\n      "word": "football",\n      "count": 186\n    },\n    {\n      "word": "porch",\n      "count": 186\n    },\n    {\n      "word": "skating",\n      "count": 186\n    },\n    {\n      "word": "star",\n      "count": 186\n    },\n    {\n      "word": "gathering",\n      "count": 185\n    },\n    {\n      "word": "golf",\n      "count": 185\n    },\n    {\n      "word": "wait",\n      "count": 185\n    },\n    {\n      "word": "weather",\n      "count": 185\n    },\n    {\n      "word": "lay",\n      "count": 184\n    },\n    {\n      "word": "lounge",\n      "count": 184\n    },\n    {\n      "word": "bagel",\n      "count": 183\n    },\n    {\n      "word": "cub",\n      "count": 182\n    },\n    {\n      "word": "wilderness",\n      "count": 182\n    },\n    {\n      "word": "bikini",\n      "count": 181\n    },\n    {\n      "word": "cookie",\n      "count": 181\n    },\n    {\n      "word": "monkey",\n      "count": 181\n    },\n    {\n      "word": "posing",\n      "count": 181\n    },\n    {\n      "word": "right",\n      "count": 181\n    },\n    {\n      "word": "castle",\n      "count": 179\n    },\n    {\n      "word": "delivery",\n      "count": 179\n    },\n    {\n      "word": "dispenser",\n      "count": 179\n    },\n    {\n      "word": "freight",\n      "count": 179\n    },\n    {\n      "word": "panda",\n      "count": 179\n    },\n    {\n      "word": "treat",\n      "count": 179\n    },\n    {\n      "word": "goggle",\n      "count": 178\n    },\n    {\n      "word": "service",\n      "count": 178\n    },\n    {\n      "word": "alley",\n      "count": 177\n    },\n    {\n      "word": "policeman",\n      "count": 177\n    },\n    {\n      "word": "beak",\n      "count": 176\n    },\n    {\n      "word": "clutter",\n      "count": 176\n    },\n    {\n      "word": "peanut",\n      "count": 176\n    },\n    {\n      "word": "good",\n      "count": 175\n    },\n    {\n      "word": "pattern",\n      "count": 175\n    },\n    {\n      "word": "skiis",\n      "count": 175\n    },\n    {\n      "word": "product",\n      "count": 174\n    },\n    {\n      "word": "closet",\n      "count": 173\n    },\n    {\n      "word": "mans",\n      "count": 173\n    },\n    {\n      "word": "thumb",\n      "count": 173\n    },\n    {\n      "word": "fabric",\n      "count": 172\n    },\n    {\n      "word": "faucet",\n      "count": 172\n    },\n    {\n      "word": "indoor",\n      "count": 172\n    },\n    {\n      "word": "safety",\n      "count": 172\n    },\n    {\n      "word": "classroom",\n      "count": 171\n    },\n    {\n      "word": "courtyard",\n      "count": 171\n    },\n    {\n      "word": "army",\n      "count": 169\n    },\n    {\n      "word": "black",\n      "count": 169\n    },\n    {\n      "word": "break",\n      "count": 169\n    },\n    {\n      "word": "pin",\n      "count": 169\n    },\n    {\n      "word": "barrel",\n      "count": 168\n    },\n    {\n      "word": "basin",\n      "count": 168\n    },\n    {\n      "word": "daytime",\n      "count": 168\n    },\n    {\n      "word": "package",\n      "count": 168\n    },\n    {\n      "word": "jungle",\n      "count": 167\n    },\n    {\n      "word": "plaza",\n      "count": 167\n    },\n    {\n      "word": "silverware",\n      "count": 167\n    },\n    {\n      "word": "vegetation",\n      "count": 167\n    },\n    {\n      "word": "cereal",\n      "count": 166\n    },\n    {\n      "word": "customer",\n      "count": 166\n    },\n    {\n      "word": "dump",\n      "count": 166\n    },\n    {\n      "word": "nap",\n      "count": 166\n    },\n    {\n      "word": "port",\n      "count": 166\n    },\n    {\n      "word": "swan",\n      "count": 166\n    },\n    {\n      "word": "cherry",\n      "count": 164\n    },\n    {\n      "word": "sailing",\n      "count": 164\n    },\n    {\n      "word": "shoreline",\n      "count": 164\n    },\n    {\n      "word": "tin",\n      "count": 164\n    },\n    {\n      "word": "guard",\n      "count": 163\n    },\n    {\n      "word": "bit",\n      "count": 162\n    },\n    {\n      "word": "notebook",\n      "count": 162\n    },\n    {\n      "word": "print",\n      "count": 162\n    },\n    {\n      "word": "war",\n      "count": 162\n    },\n    {\n      "word": "key",\n      "count": 161\n    },\n    {\n      "word": "ram",\n      "count": 160\n    },\n    {\n      "word": "skiier",\n      "count": 160\n    },\n    {\n      "word": "arena",\n      "count": 159\n    },\n    {\n      "word": "barrier",\n      "count": 159\n    },\n    {\n      "word": "sneaker",\n      "count": 159\n    },\n    {\n      "word": "talk",\n      "count": 159\n    },\n    {\n      "word": "dot",\n      "count": 158\n    },\n    {\n      "word": "holding",\n      "count": 158\n    },\n    {\n      "word": "message",\n      "count": 158\n    },\n    {\n      "word": "skyline",\n      "count": 158\n    },\n    {\n      "word": "cab",\n      "count": 157\n    },\n    {\n      "word": "craft",\n      "count": 157\n    },\n    {\n      "word": "music",\n      "count": 157\n    },\n    {\n      "word": "process",\n      "count": 157\n    },\n    {\n      "word": "village",\n      "count": 157\n    },\n    {\n      "word": "waterway",\n      "count": 157\n    },\n    {\n      "word": "material",\n      "count": 156\n    },\n    {\n      "word": "yellow",\n      "count": 156\n    },\n    {\n      "word": "moped",\n      "count": 155\n    },\n    {\n      "word": "soap",\n      "count": 155\n    },\n    {\n      "word": "spray",\n      "count": 155\n    },\n    {\n      "word": "wrapper",\n      "count": 154\n    },\n    {\n      "word": "conversation",\n      "count": 153\n    },\n    {\n      "word": "granite",\n      "count": 153\n    },\n    {\n      "word": "character",\n      "count": 152\n    },\n    {\n      "word": "flying",\n      "count": 152\n    },\n    {\n      "word": "graz",\n      "count": 152\n    },\n    {\n      "word": "photographer",\n      "count": 152\n    },\n    {\n      "word": "softball",\n      "count": 152\n    },\n    {\n      "word": "valley",\n      "count": 152\n    },\n    {\n      "word": "cord",\n      "count": 151\n    },\n    {\n      "word": "wit",\n      "count": 151\n    },\n    {\n      "word": "pick",\n      "count": 150\n    },\n    {\n      "word": "portrait",\n      "count": 150\n    },\n    {\n      "word": "cyclist",\n      "count": 149\n    },\n    {\n      "word": "lime",\n      "count": 149\n    },\n    {\n      "word": "link",\n      "count": 149\n    },\n    {\n      "word": "mall",\n      "count": 149\n    },\n    {\n      "word": "advertising",\n      "count": 148\n    },\n    {\n      "word": "peel",\n      "count": 148\n    },\n    {\n      "word": "at",\n      "count": 147\n    },\n    {\n      "word": "monument",\n      "count": 147\n    },\n    {\n      "word": "raft",\n      "count": 147\n    },\n    {\n      "word": "site",\n      "count": 147\n    },\n    {\n      "word": "stance",\n      "count": 147\n    },\n    {\n      "word": "terrain",\n      "count": 147\n    },\n    {\n      "word": "toothpaste",\n      "count": 147\n    },\n    {\n      "word": "utility",\n      "count": 147\n    },\n    {\n      "word": "pear",\n      "count": 146\n    },\n    {\n      "word": "deer",\n      "count": 145\n    },\n    {\n      "word": "skater",\n      "count": 145\n    },\n    {\n      "word": "trough",\n      "count": 145\n    },\n    {\n      "word": "canal",\n      "count": 144\n    },\n    {\n      "word": "freeway",\n      "count": 144\n    },\n    {\n      "word": "parachute",\n      "count": 144\n    },\n    {\n      "word": "shrimp",\n      "count": 144\n    },\n    {\n      "word": "cartoon",\n      "count": 143\n    },\n    {\n      "word": "helicopter",\n      "count": 143\n    },\n    {\n      "word": "factory",\n      "count": 142\n    },\n    {\n      "word": "movie",\n      "count": 142\n    },\n    {\n      "word": "buggy",\n      "count": 141\n    },\n    {\n      "word": "den",\n      "count": 141\n    },\n    {\n      "word": "planter",\n      "count": 141\n    },\n    {\n      "word": "tape",\n      "count": 141\n    },\n    {\n      "word": "boogie",\n      "count": 140\n    },\n    {\n      "word": "drawing",\n      "count": 140\n    },\n    {\n      "word": "greenery",\n      "count": 140\n    },\n    {\n      "word": "letter",\n      "count": 140\n    },\n    {\n      "word": "limb",\n      "count": 140\n    },\n    {\n      "word": "open",\n      "count": 140\n    },\n    {\n      "word": "pointing",\n      "count": 140\n    },\n    {\n      "word": "shrub",\n      "count": 140\n    },\n    {\n      "word": "conference",\n      "count": 139\n    },\n    {\n      "word": "console",\n      "count": 139\n    },\n    {\n      "word": "curve",\n      "count": 139\n    },\n    {\n      "word": "diamond",\n      "count": 139\n    },\n    {\n      "word": "pug",\n      "count": 139\n    },\n    {\n      "word": "transportation",\n      "count": 139\n    },\n    {\n      "word": "crib",\n      "count": 138\n    },\n    {\n      "word": "nightstand",\n      "count": 138\n    },\n    {\n      "word": "ostrich",\n      "count": 138\n    },\n    {\n      "word": "bookcase",\n      "count": 137\n    },\n    {\n      "word": "hangar",\n      "count": 137\n    },\n    {\n      "word": "left",\n      "count": 137\n    },\n    {\n      "word": "order",\n      "count": 137\n    },\n    {\n      "word": "quilt",\n      "count": 137\n    },\n    {\n      "word": "cluster",\n      "count": 136\n    },\n    {\n      "word": "feeding",\n      "count": 136\n    },\n    {\n      "word": "overpass",\n      "count": 136\n    },\n    {\n      "word": "owl",\n      "count": 136\n    },\n    {\n      "word": "plaid",\n      "count": 136\n    },\n    {\n      "word": "tablecloth",\n      "count": 136\n    },\n    {\n      "word": "banner",\n      "count": 135\n    },\n    {\n      "word": "cucumber",\n      "count": 135\n    },\n    {\n      "word": "daughter",\n      "count": 135\n    },\n    {\n      "word": "fashion",\n      "count": 135\n    },\n    {\n      "word": "pillar",\n      "count": 134\n    },\n    {\n      "word": "backdrop",\n      "count": 133\n    },\n    {\n      "word": "hall",\n      "count": 133\n    },\n    {\n      "word": "angle",\n      "count": 132\n    },\n    {\n      "word": "disk",\n      "count": 132\n    },\n    {\n      "word": "logo",\n      "count": 132\n    },\n    {\n      "word": "meeting",\n      "count": 132\n    },\n    {\n      "word": "pocket",\n      "count": 132\n    },\n    {\n      "word": "puddle",\n      "count": 131\n    },\n    {\n      "word": "balance",\n      "count": 130\n    },\n    {\n      "word": "gravel",\n      "count": 130\n    },\n    {\n      "word": "mom",\n      "count": 130\n    },\n    {\n      "word": "opponent",\n      "count": 130\n    },\n    {\n      "word": "runner",\n      "count": 130\n    },\n    {\n      "word": "tube",\n      "count": 130\n    },\n    {\n      "word": "dough",\n      "count": 129\n    },\n    {\n      "word": "round",\n      "count": 129\n    },\n    {\n      "word": "st",\n      "count": 129\n    },\n    {\n      "word": "circus",\n      "count": 128\n    },\n    {\n      "word": "parent",\n      "count": 128\n    },\n    {\n      "word": "silhouette",\n      "count": 128\n    },\n    {\n      "word": "stroller",\n      "count": 128\n    },\n    {\n      "word": "trio",\n      "count": 128\n    },\n    {\n      "word": "unit",\n      "count": 128\n    },\n    {\n      "word": "depot",\n      "count": 127\n    },\n    {\n      "word": "kettle",\n      "count": 127\n    },\n    {\n      "word": "pathway",\n      "count": 127\n    },\n    {\n      "word": "printer",\n      "count": 127\n    },\n    {\n      "word": "suite",\n      "count": 127\n    },\n    {\n      "word": "cracker",\n      "count": 126\n    },\n    {\n      "word": "world",\n      "count": 126\n    },\n    {\n      "word": "youth",\n      "count": 126\n    },\n    {\n      "word": "zebras",\n      "count": 126\n    },\n    {\n      "word": "cheesy",\n      "count": 125\n    },\n    {\n      "word": "heard",\n      "count": 125\n    },\n    {\n      "word": "windowsill",\n      "count": 125\n    },\n    {\n      "word": "savannah",\n      "count": 124\n    },\n    {\n      "word": "airline",\n      "count": 123\n    },\n    {\n      "word": "athlete",\n      "count": 123\n    },\n    {\n      "word": "billboard",\n      "count": 123\n    },\n    {\n      "word": "instrument",\n      "count": 123\n    },\n    {\n      "word": "pass",\n      "count": 123\n    },\n    {\n      "word": "pineapple",\n      "count": 123\n    },\n    {\n      "word": "cubicle",\n      "count": 122\n    },\n    {\n      "word": "donkey",\n      "count": 122\n    },\n    {\n      "word": "festival",\n      "count": 122\n    },\n    {\n      "word": "flip",\n      "count": 122\n    },\n    {\n      "word": "gravy",\n      "count": 122\n    },\n    {\n      "word": "polka",\n      "count": 122\n    },\n    {\n      "word": "robe",\n      "count": 122\n    },\n    {\n      "word": "stomach",\n      "count": 122\n    },\n    {\n      "word": "blow",\n      "count": 121\n    },\n    {\n      "word": "china",\n      "count": 121\n    },\n    {\n      "word": "dip",\n      "count": 121\n    },\n    {\n      "word": "lobby",\n      "count": 121\n    },\n    {\n      "word": "metro",\n      "count": 121\n    },\n    {\n      "word": "waffle",\n      "count": 121\n    },\n    {\n      "word": "warehouse",\n      "count": 121\n    },\n    {\n      "word": "band",\n      "count": 120\n    },\n    {\n      "word": "compartment",\n      "count": 120\n    },\n    {\n      "word": "foliage",\n      "count": 120\n    },\n    {\n      "word": "calico",\n      "count": 119\n    },\n    {\n      "word": "carousel",\n      "count": 119\n    },\n    {\n      "word": "fur",\n      "count": 119\n    },\n    {\n      "word": "tissue",\n      "count": 119\n    },\n    {\n      "word": "cauliflower",\n      "count": 118\n    },\n    {\n      "word": "cruise",\n      "count": 118\n    },\n    {\n      "word": "diner",\n      "count": 118\n    },\n    {\n      "word": "ferry",\n      "count": 118\n    },\n    {\n      "word": "headboard",\n      "count": 118\n    },\n    {\n      "word": "load",\n      "count": 118\n    },\n    {\n      "word": "shuttle",\n      "count": 118\n    },\n    {\n      "word": "slide",\n      "count": 118\n    },\n    {\n      "word": "wicker",\n      "count": 118\n    },\n    {\n      "word": "airlines",\n      "count": 117\n    },\n    {\n      "word": "chili",\n      "count": 117\n    },\n    {\n      "word": "marker",\n      "count": 117\n    },\n    {\n      "word": "staircase",\n      "count": 117\n    },\n    {\n      "word": "boardwalk",\n      "count": 116\n    },\n    {\n      "word": "chopstick",\n      "count": 116\n    },\n    {\n      "word": "loaf",\n      "count": 116\n    },\n    {\n      "word": "panel",\n      "count": 116\n    },\n    {\n      "word": "skyscraper",\n      "count": 116\n    },\n    {\n      "word": "teen",\n      "count": 116\n    },\n    {\n      "word": "asphalt",\n      "count": 115\n    },\n    {\n      "word": "bedspread",\n      "count": 115\n    },\n    {\n      "word": "force",\n      "count": 115\n    },\n    {\n      "word": "mixture",\n      "count": 115\n    },\n    {\n      "word": "practice",\n      "count": 115\n    },\n    {\n      "word": "blind",\n      "count": 114\n    },\n    {\n      "word": "class",\n      "count": 114\n    },\n    {\n      "word": "hit",\n      "count": 114\n    },\n    {\n      "word": "kit",\n      "count": 114\n    },\n    {\n      "word": "location",\n      "count": 114\n    },\n    {\n      "word": "podium",\n      "count": 114\n    },\n    {\n      "word": "washer",\n      "count": 114\n    },\n    {\n      "word": "weed",\n      "count": 114\n    },\n    {\n      "word": "coke",\n      "count": 113\n    },\n    {\n      "word": "menu",\n      "count": 113\n    },\n    {\n      "word": "radio",\n      "count": 113\n    },\n    {\n      "word": "roadside",\n      "count": 113\n    },\n    {\n      "word": "selection",\n      "count": 113\n    },\n    {\n      "word": "suv",\n      "count": 113\n    },\n    {\n      "word": "warning",\n      "count": 113\n    },\n    {\n      "word": "blueberry",\n      "count": 112\n    },\n    {\n      "word": "hind",\n      "count": 112\n    },\n    {\n      "word": "macaroni",\n      "count": 112\n    },\n    {\n      "word": "mote",\n      "count": 112\n    },\n    {\n      "word": "antelope",\n      "count": 111\n    },\n    {\n      "word": "feather",\n      "count": 111\n    },\n    {\n      "word": "firetruck",\n      "count": 111\n    },\n    {\n      "word": "fog",\n      "count": 111\n    },\n    {\n      "word": "seating",\n      "count": 111\n    },\n    {\n      "word": "skillet",\n      "count": 111\n    },\n    {\n      "word": "stem",\n      "count": 111\n    },\n    {\n      "word": "wake",\n      "count": 111\n    },\n    {\n      "word": "zone",\n      "count": 111\n    },\n    {\n      "word": "ad",\n      "count": 110\n    },\n    {\n      "word": "double",\n      "count": 110\n    },\n    {\n      "word": "new",\n      "count": 110\n    },\n    {\n      "word": "repair",\n      "count": 110\n    },\n    {\n      "word": "storm",\n      "count": 110\n    },\n    {\n      "word": "cola",\n      "count": 109\n    },\n    {\n      "word": "environment",\n      "count": 109\n    },\n    {\n      "word": "infant",\n      "count": 109\n    },\n    {\n      "word": "maroon",\n      "count": 109\n    },\n    {\n      "word": "portion",\n      "count": 109\n    },\n    {\n      "word": "signage",\n      "count": 109\n    },\n    {\n      "word": "sock",\n      "count": 109\n    },\n    {\n      "word": "member",\n      "count": 108\n    },\n    {\n      "word": "parasol",\n      "count": 108\n    },\n    {\n      "word": "self",\n      "count": 108\n    },\n    {\n      "word": "tuxedo",\n      "count": 108\n    },\n    {\n      "word": "wildlife",\n      "count": 108\n    },\n    {\n      "word": "guest",\n      "count": 107\n    },\n    {\n      "word": "hut",\n      "count": 107\n    },\n    {\n      "word": "love",\n      "count": 107\n    },\n    {\n      "word": "midair",\n      "count": 107\n    },\n    {\n      "word": "pine",\n      "count": 107\n    },\n    {\n      "word": "sit",\n      "count": 107\n    },\n    {\n      "word": "artwork",\n      "count": 106\n    },\n    {\n      "word": "bunny",\n      "count": 106\n    },\n    {\n      "word": "grandfather",\n      "count": 106\n    },\n    {\n      "word": "gun",\n      "count": 106\n    },\n    {\n      "word": "heel",\n      "count": 106\n    },\n    {\n      "word": "hitter",\n      "count": 106\n    },\n    {\n      "word": "kayak",\n      "count": 106\n    },\n    {\n      "word": "pea",\n      "count": 106\n    },\n    {\n      "word": "price",\n      "count": 106\n    },\n    {\n      "word": "alcohol",\n      "count": 105\n    },\n    {\n      "word": "cable",\n      "count": 105\n    },\n    {\n      "word": "chandelier",\n      "count": 105\n    },\n    {\n      "word": "commode",\n      "count": 105\n    },\n    {\n      "word": "fall",\n      "count": 105\n    },\n    {\n      "word": "goal",\n      "count": 105\n    },\n    {\n      "word": "necktie",\n      "count": 105\n    },\n    {\n      "word": "plan",\n      "count": 105\n    },\n    {\n      "word": "wheelchair",\n      "count": 105\n    },\n    {\n      "word": "hook",\n      "count": 104\n    },\n    {\n      "word": "lodge",\n      "count": 104\n    },\n    {\n      "word": "map",\n      "count": 104\n    },\n    {\n      "word": "seed",\n      "count": 104\n    },\n    {\n      "word": "eating",\n      "count": 103\n    },\n    {\n      "word": "jumping",\n      "count": 103\n    },\n    {\n      "word": "pet",\n      "count": 103\n    },\n    {\n      "word": "spatula",\n      "count": 103\n    },\n    {\n      "word": "stump",\n      "count": 103\n    },\n    {\n      "word": "tiger",\n      "count": 103\n    },\n    {\n      "word": "array",\n      "count": 102\n    },\n    {\n      "word": "coast",\n      "count": 102\n    },\n    {\n      "word": "grassland",\n      "count": 102\n    },\n    {\n      "word": "harness",\n      "count": 102\n    },\n    {\n      "word": "linen",\n      "count": 102\n    },\n    {\n      "word": "saucer",\n      "count": 102\n    },\n    {\n      "word": "shrubbery",\n      "count": 102\n    },\n    {\n      "word": "tee",\n      "count": 102\n    },\n    {\n      "word": "clearing",\n      "count": 101\n    },\n    {\n      "word": "herb",\n      "count": 101\n    },\n    {\n      "word": "icing",\n      "count": 101\n    },\n    {\n      "word": "motel",\n      "count": 101\n    },\n    {\n      "word": "roller",\n      "count": 101\n    },\n    {\n      "word": "sushi",\n      "count": 101\n    },\n    {\n      "word": "automobile",\n      "count": 100\n    },\n    {\n      "word": "bamboo",\n      "count": 100\n    },\n    {\n      "word": "catch",\n      "count": 100\n    },\n    {\n      "word": "gaming",\n      "count": 100\n    },\n    {\n      "word": "gourmet",\n      "count": 100\n    },\n    {\n      "word": "page",\n      "count": 100\n    },\n    {\n      "word": "porcelain",\n      "count": 100\n    },\n    {\n      "word": "traveler",\n      "count": 100\n    },\n    {\n      "word": "volley",\n      "count": 100\n    },\n    {\n      "word": "bend",\n      "count": 99\n    },\n    {\n      "word": "burner",\n      "count": 99\n    },\n    {\n      "word": "cutter",\n      "count": 99\n    },\n    {\n      "word": "float",\n      "count": 99\n    },\n    {\n      "word": "nest",\n      "count": 99\n    },\n    {\n      "word": "pink",\n      "count": 99\n    },\n    {\n      "word": "wrap",\n      "count": 99\n    },\n    {\n      "word": "alarm",\n      "count": 98\n    },\n    {\n      "word": "bundle",\n      "count": 98\n    },\n    {\n      "word": "mural",\n      "count": 98\n    },\n    {\n      "word": "penguin",\n      "count": 98\n    },\n    {\n      "word": "pig",\n      "count": 98\n    },\n    {\n      "word": "pork",\n      "count": 98\n    },\n    {\n      "word": "rabbit",\n      "count": 98\n    },\n    {\n      "word": "rubber",\n      "count": 98\n    },\n    {\n      "word": "toiletry",\n      "count": 98\n    },\n    {\n      "word": "united",\n      "count": 98\n    },\n    {\n      "word": "yogurt",\n      "count": 98\n    },\n    {\n      "word": "company",\n      "count": 97\n    },\n    {\n      "word": "oil",\n      "count": 97\n    },\n    {\n      "word": "windshield",\n      "count": 97\n    },\n    {\n      "word": "avenue",\n      "count": 96\n    },\n    {\n      "word": "butterfly",\n      "count": 96\n    },\n    {\n      "word": "cabbage",\n      "count": 96\n    },\n    {\n      "word": "caption",\n      "count": 96\n    },\n    {\n      "word": "rim",\n      "count": 96\n    },\n    {\n      "word": "holiday",\n      "count": 95\n    },\n    {\n      "word": "pencil",\n      "count": 95\n    },\n    {\n      "word": "saddle",\n      "count": 95\n    },\n    {\n      "word": "use",\n      "count": 95\n    },\n    {\n      "word": "briefcase",\n      "count": 94\n    },\n    {\n      "word": "care",\n      "count": 94\n    },\n    {\n      "word": "maker",\n      "count": 94\n    },\n    {\n      "word": "navy",\n      "count": 94\n    },\n    {\n      "word": "opening",\n      "count": 94\n    },\n    {\n      "word": "streetlight",\n      "count": 94\n    },\n    {\n      "word": "theme",\n      "count": 94\n    },\n    {\n      "word": "tram",\n      "count": 94\n    },\n    {\n      "word": "blood",\n      "count": 93\n    },\n    {\n      "word": "cobblestone",\n      "count": 93\n    },\n    {\n      "word": "eagle",\n      "count": 93\n    },\n    {\n      "word": "furnishing",\n      "count": 93\n    },\n    {\n      "word": "hanger",\n      "count": 93\n    },\n    {\n      "word": "junk",\n      "count": 93\n    },\n    {\n      "word": "playground",\n      "count": 93\n    },\n    {\n      "word": "sunflower",\n      "count": 93\n    },\n    {\n      "word": "unripe",\n      "count": 93\n    },\n    {\n      "word": "wear",\n      "count": 93\n    },\n    {\n      "word": "accent",\n      "count": 92\n    },\n    {\n      "word": "cathedral",\n      "count": 92\n    },\n    {\n      "word": "celery",\n      "count": 92\n    },\n    {\n      "word": "frosting",\n      "count": 92\n    },\n    {\n      "word": "pottery",\n      "count": 92\n    },\n    {\n      "word": "steer",\n      "count": 92\n    },\n    {\n      "word": "arch",\n      "count": 91\n    },\n    {\n      "word": "ave",\n      "count": 91\n    },\n    {\n      "word": "bidet",\n      "count": 91\n    },\n    {\n      "word": "biscuit",\n      "count": 91\n    },\n    {\n      "word": "bowling",\n      "count": 91\n    },\n    {\n      "word": "coin",\n      "count": 91\n    },\n    {\n      "word": "hipster",\n      "count": 91\n    },\n    {\n      "word": "hoodie",\n      "count": 91\n    },\n    {\n      "word": "money",\n      "count": 91\n    },\n    {\n      "word": "racer",\n      "count": 91\n    },\n    {\n      "word": "afternoon",\n      "count": 90\n    },\n    {\n      "word": "carrier",\n      "count": 90\n    },\n    {\n      "word": "crest",\n      "count": 90\n    },\n    {\n      "word": "dugout",\n      "count": 90\n    },\n    {\n      "word": "handbag",\n      "count": 90\n    },\n    {\n      "word": "pumpkin",\n      "count": 90\n    },\n    {\n      "word": "smartphone",\n      "count": 90\n    },\n    {\n      "word": "sweatshirt",\n      "count": 90\n    },\n    {\n      "word": "mid",\n      "count": 89\n    },\n    {\n      "word": "ottoman",\n      "count": 89\n    },\n    {\n      "word": "plenty",\n      "count": 89\n    },\n    {\n      "word": "route",\n      "count": 89\n    },\n    {\n      "word": "shell",\n      "count": 89\n    },\n    {\n      "word": "slab",\n      "count": 89\n    },\n    {\n      "word": "syrup",\n      "count": 89\n    },\n    {\n      "word": "transport",\n      "count": 89\n    },\n    {\n      "word": "york",\n      "count": 89\n    },\n    {\n      "word": "aluminum",\n      "count": 88\n    },\n    {\n      "word": "bathing",\n      "count": 88\n    },\n    {\n      "word": "club",\n      "count": 88\n    },\n    {\n      "word": "delta",\n      "count": 88\n    },\n    {\n      "word": "gym",\n      "count": 88\n    },\n    {\n      "word": "kites",\n      "count": 88\n    },\n    {\n      "word": "lego",\n      "count": 88\n    },\n    {\n      "word": "mixer",\n      "count": 88\n    },\n    {\n      "word": "setup",\n      "count": 88\n    },\n    {\n      "word": "stalk",\n      "count": 88\n    },\n    {\n      "word": "washing",\n      "count": 88\n    },\n    {\n      "word": "employee",\n      "count": 87\n    },\n    {\n      "word": "grizzly",\n      "count": 87\n    },\n    {\n      "word": "news",\n      "count": 87\n    },\n    {\n      "word": "perch",\n      "count": 87\n    },\n    {\n      "word": "pirate",\n      "count": 87\n    },\n    {\n      "word": "symbol",\n      "count": 87\n    },\n    {\n      "word": "trip",\n      "count": 87\n    },\n    {\n      "word": "waterfall",\n      "count": 87\n    },\n    {\n      "word": "biplane",\n      "count": 86\n    },\n    {\n      "word": "claim",\n      "count": 86\n    },\n    {\n      "word": "jeep",\n      "count": 86\n    },\n    {\n      "word": "kneeling",\n      "count": 86\n    },\n    {\n      "word": "pedestal",\n      "count": 86\n    },\n    {\n      "word": "spice",\n      "count": 86\n    },\n    {\n      "word": "file",\n      "count": 85\n    },\n    {\n      "word": "gift",\n      "count": 85\n    },\n    {\n      "word": "matching",\n      "count": 85\n    },\n    {\n      "word": "pilot",\n      "count": 85\n    },\n    {\n      "word": "rise",\n      "count": 85\n    },\n    {\n      "word": "shed",\n      "count": 85\n    },\n    {\n      "word": "text",\n      "count": 85\n    },\n    {\n      "word": "call",\n      "count": 84\n    },\n    {\n      "word": "chrome",\n      "count": 84\n    },\n    {\n      "word": "clown",\n      "count": 84\n    },\n    {\n      "word": "scale",\n      "count": 84\n    },\n    {\n      "word": "spread",\n      "count": 84\n    },\n    {\n      "word": "throw",\n      "count": 84\n    },\n    {\n      "word": "wallpaper",\n      "count": 84\n    },\n    {\n      "word": "wedge",\n      "count": 84\n    },\n    {\n      "word": "bleacher",\n      "count": 83\n    },\n    {\n      "word": "coleslaw",\n      "count": 83\n    },\n    {\n      "word": "lion",\n      "count": 83\n    },\n    {\n      "word": "lip",\n      "count": 83\n    },\n    {\n      "word": "state",\n      "count": 83\n    },\n    {\n      "word": "frisbe",\n      "count": 82\n    },\n    {\n      "word": "goose",\n      "count": 82\n    },\n    {\n      "word": "meatball",\n      "count": 82\n    },\n    {\n      "word": "champagne",\n      "count": 81\n    },\n    {\n      "word": "dashboard",\n      "count": 81\n    },\n    {\n      "word": "king",\n      "count": 81\n    },\n    {\n      "word": "laying",\n      "count": 81\n    },\n    {\n      "word": "midst",\n      "count": 81\n    },\n    {\n      "word": "o",\n      "count": 81\n    },\n    {\n      "word": "peace",\n      "count": 81\n    },\n    {\n      "word": "raspberry",\n      "count": 81\n    },\n    {\n      "word": "roast",\n      "count": 81\n    },\n    {\n      "word": "stretch",\n      "count": 81\n    },\n    {\n      "word": "tip",\n      "count": 81\n    },\n    {\n      "word": "watermelon",\n      "count": 81\n    },\n    {\n      "word": "wooden",\n      "count": 81\n    },\n    {\n      "word": "cd",\n      "count": 80\n    },\n    {\n      "word": "combination",\n      "count": 80\n    },\n    {\n      "word": "gazelle",\n      "count": 80\n    },\n    {\n      "word": "mustache",\n      "count": 80\n    },\n    {\n      "word": "reading",\n      "count": 80\n    },\n    {\n      "word": "us",\n      "count": 80\n    },\n    {\n      "word": "blade",\n      "count": 79\n    },\n    {\n      "word": "cheesecake",\n      "count": 79\n    },\n    {\n      "word": "creek",\n      "count": 79\n    },\n    {\n      "word": "crew",\n      "count": 79\n    },\n    {\n      "word": "crown",\n      "count": 79\n    },\n    {\n      "word": "exit",\n      "count": 79\n    },\n    {\n      "word": "jam",\n      "count": 79\n    },\n    {\n      "word": "lab",\n      "count": 79\n    },\n    {\n      "word": "pit",\n      "count": 79\n    },\n    {\n      "word": "studio",\n      "count": 79\n    },\n    {\n      "word": "caution",\n      "count": 78\n    },\n    {\n      "word": "column",\n      "count": 78\n    },\n    {\n      "word": "crouch",\n      "count": 78\n    },\n    {\n      "word": "mannequin",\n      "count": 78\n    },\n    {\n      "word": "mantle",\n      "count": 78\n    },\n    {\n      "word": "mess",\n      "count": 78\n    },\n    {\n      "word": "celebration",\n      "count": 77\n    },\n    {\n      "word": "coca",\n      "count": 77\n    },\n    {\n      "word": "cooler",\n      "count": 77\n    },\n    {\n      "word": "liquor",\n      "count": 77\n    },\n    {\n      "word": "ornament",\n      "count": 77\n    },\n    {\n      "word": "relish",\n      "count": 77\n    },\n    {\n      "word": "tortilla",\n      "count": 77\n    },\n    {\n      "word": "trainer",\n      "count": 77\n    },\n    {\n      "word": "washington",\n      "count": 77\n    },\n    {\n      "word": "airfield",\n      "count": 76\n    },\n    {\n      "word": "bbq",\n      "count": 76\n    },\n    {\n      "word": "camper",\n      "count": 76\n    },\n    {\n      "word": "cane",\n      "count": 76\n    },\n    {\n      "word": "dinning",\n      "count": 76\n    },\n    {\n      "word": "emergency",\n      "count": 76\n    },\n    {\n      "word": "iphone",\n      "count": 76\n    },\n    {\n      "word": "mac",\n      "count": 76\n    },\n    {\n      "word": "oval",\n      "count": 76\n    },\n    {\n      "word": "rear",\n      "count": 76\n    },\n    {\n      "word": "spire",\n      "count": 76\n    },\n    {\n      "word": "tarp",\n      "count": 76\n    },\n    {\n      "word": "yarn",\n      "count": 76\n    },\n    {\n      "word": "cabinetry",\n      "count": 75\n    },\n    {\n      "word": "convention",\n      "count": 75\n    },\n    {\n      "word": "information",\n      "count": 75\n    },\n    {\n      "word": "men",\n      "count": 75\n    },\n    {\n      "word": "outfield",\n      "count": 75\n    },\n    {\n      "word": "polo",\n      "count": 75\n    },\n    {\n      "word": "powder",\n      "count": 75\n    },\n    {\n      "word": "preparation",\n      "count": 75\n    },\n    {\n      "word": "activity",\n      "count": 74\n    },\n    {\n      "word": "bullet",\n      "count": 74\n    },\n    {\n      "word": "dad",\n      "count": 74\n    },\n    {\n      "word": "grain",\n      "count": 74\n    },\n    {\n      "word": "human",\n      "count": 74\n    },\n    {\n      "word": "label",\n      "count": 74\n    },\n    {\n      "word": "pajama",\n      "count": 74\n    },\n    {\n      "word": "peach",\n      "count": 74\n    },\n    {\n      "word": "presentation",\n      "count": 74\n    },\n    {\n      "word": "president",\n      "count": 74\n    },\n    {\n      "word": "referee",\n      "count": 74\n    },\n    {\n      "word": "rodeo",\n      "count": 74\n    },\n    {\n      "word": "skatepark",\n      "count": 74\n    },\n    {\n      "word": "trim",\n      "count": 74\n    },\n    {\n      "word": "volleyball",\n      "count": 74\n    },\n    {\n      "word": "wheeler",\n      "count": 74\n    },\n    {\n      "word": "casserole",\n      "count": 73\n    },\n    {\n      "word": "destination",\n      "count": 73\n    },\n    {\n      "word": "lean",\n      "count": 73\n    },\n    {\n      "word": "numeral",\n      "count": 73\n    },\n    {\n      "word": "on",\n      "count": 73\n    },\n    {\n      "word": "spaghetti",\n      "count": 73\n    },\n    {\n      "word": "takeoff",\n      "count": 73\n    },\n    {\n      "word": "tulip",\n      "count": 73\n    },\n    {\n      "word": "wax",\n      "count": 73\n    },\n    {\n      "word": "year",\n      "count": 73\n    },\n    {\n      "word": "brownie",\n      "count": 72\n    },\n    {\n      "word": "businessman",\n      "count": 72\n    },\n    {\n      "word": "college",\n      "count": 72\n    },\n    {\n      "word": "gesture",\n      "count": 72\n    },\n    {\n      "word": "obama",\n      "count": 72\n    },\n    {\n      "word": "purple",\n      "count": 72\n    },\n    {\n      "word": "sewing",\n      "count": 72\n    },\n    {\n      "word": "summer",\n      "count": 72\n    },\n    {\n      "word": "surfs",\n      "count": 72\n    },\n    {\n      "word": "wash",\n      "count": 72\n    },\n    {\n      "word": "basil",\n      "count": 71\n    },\n    {\n      "word": "brocolli",\n      "count": 71\n    },\n    {\n      "word": "buffalo",\n      "count": 71\n    },\n    {\n      "word": "department",\n      "count": 71\n    },\n    {\n      "word": "dragon",\n      "count": 71\n    },\n    {\n      "word": "needle",\n      "count": 71\n    },\n    {\n      "word": "wallet",\n      "count": 71\n    },\n    {\n      "word": "blurry",\n      "count": 70\n    },\n    {\n      "word": "bushel",\n      "count": 70\n    },\n    {\n      "word": "conveyor",\n      "count": 70\n    },\n    {\n      "word": "corral",\n      "count": 70\n    },\n    {\n      "word": "covering",\n      "count": 70\n    },\n    {\n      "word": "moon",\n      "count": 70\n    },\n    {\n      "word": "present",\n      "count": 70\n    },\n    {\n      "word": "project",\n      "count": 70\n    },\n    {\n      "word": "sprout",\n      "count": 70\n    },\n    {\n      "word": "storefront",\n      "count": 70\n    },\n    {\n      "word": "tone",\n      "count": 70\n    },\n    {\n      "word": "twig",\n      "count": 70\n    },\n    {\n      "word": "wool",\n      "count": 70\n    },\n    {\n      "word": "breast",\n      "count": 69\n    },\n    {\n      "word": "carton",\n      "count": 69\n    },\n    {\n      "word": "dipping",\n      "count": 69\n    },\n    {\n      "word": "honey",\n      "count": 69\n    },\n    {\n      "word": "kneel",\n      "count": 69\n    },\n    {\n      "word": "liner",\n      "count": 69\n    },\n    {\n      "word": "plaque",\n      "count": 69\n    },\n    {\n      "word": "skull",\n      "count": 69\n    },\n    {\n      "word": "stew",\n      "count": 69\n    },\n    {\n      "word": "sword",\n      "count": 69\n    },\n    {\n      "word": "visitor",\n      "count": 69\n    },\n    {\n      "word": "american",\n      "count": 68\n    },\n    {\n      "word": "avocado",\n      "count": 68\n    },\n    {\n      "word": "form",\n      "count": 68\n    },\n    {\n      "word": "glide",\n      "count": 68\n    },\n    {\n      "word": "leap",\n      "count": 68\n    },\n    {\n      "word": "limit",\n      "count": 68\n    },\n    {\n      "word": "mail",\n      "count": 68\n    },\n    {\n      "word": "makeup",\n      "count": 68\n    },\n    {\n      "word": "near",\n      "count": 68\n    },\n    {\n      "word": "pelican",\n      "count": 68\n    },\n    {\n      "word": "projector",\n      "count": 68\n    },\n    {\n      "word": "racing",\n      "count": 68\n    },\n    {\n      "word": "skill",\n      "count": 68\n    },\n    {\n      "word": "architecture",\n      "count": 67\n    },\n    {\n      "word": "bubble",\n      "count": 67\n    },\n    {\n      "word": "expression",\n      "count": 67\n    },\n    {\n      "word": "mix",\n      "count": 67\n    },\n    {\n      "word": "multi",\n      "count": 67\n    },\n    {\n      "word": "pump",\n      "count": 67\n    },\n    {\n      "word": "serving",\n      "count": 67\n    },\n    {\n      "word": "tangerine",\n      "count": 67\n    },\n    {\n      "word": "tug",\n      "count": 67\n    },\n    {\n      "word": "carnival",\n      "count": 66\n    },\n    {\n      "word": "cleaning",\n      "count": 66\n    },\n    {\n      "word": "dvd",\n      "count": 66\n    },\n    {\n      "word": "hedge",\n      "count": 66\n    },\n    {\n      "word": "sled",\n      "count": 66\n    },\n    {\n      "word": "stocking",\n      "count": 66\n    },\n    {\n      "word": "tide",\n      "count": 66\n    },\n    {\n      "word": "today",\n      "count": 66\n    },\n    {\n      "word": "beam",\n      "count": 65\n    },\n    {\n      "word": "belly",\n      "count": 65\n    },\n    {\n      "word": "border",\n      "count": 65\n    },\n    {\n      "word": "district",\n      "count": 65\n    },\n    {\n      "word": "peacock",\n      "count": 65\n    },\n    {\n      "word": "sepia",\n      "count": 65\n    },\n    {\n      "word": "splash",\n      "count": 65\n    },\n    {\n      "word": "steering",\n      "count": 65\n    },\n    {\n      "word": "tier",\n      "count": 65\n    },\n    {\n      "word": "training",\n      "count": 65\n    },\n    {\n      "word": "vine",\n      "count": 65\n    },\n    {\n      "word": "cafeteria",\n      "count": 64\n    },\n    {\n      "word": "dressing",\n      "count": 64\n    },\n    {\n      "word": "escalator",\n      "count": 64\n    },\n    {\n      "word": "hour",\n      "count": 64\n    },\n    {\n      "word": "public",\n      "count": 64\n    },\n    {\n      "word": "sandal",\n      "count": 64\n    },\n    {\n      "word": "boulder",\n      "count": 63\n    },\n    {\n      "word": "cemetery",\n      "count": 63\n    },\n    {\n      "word": "chop",\n      "count": 63\n    },\n    {\n      "word": "gadget",\n      "count": 63\n    },\n    {\n      "word": "morning",\n      "count": 63\n    },\n    {\n      "word": "mountainside",\n      "count": 63\n    },\n    {\n      "word": "queen",\n      "count": 63\n    },\n    {\n      "word": "sailor",\n      "count": 63\n    },\n    {\n      "word": "turquoise",\n      "count": 63\n    },\n    {\n      "word": "dairy",\n      "count": 62\n    },\n    {\n      "word": "debris",\n      "count": 62\n    },\n    {\n      "word": "deli",\n      "count": 62\n    },\n    {\n      "word": "dorm",\n      "count": 62\n    },\n    {\n      "word": "english",\n      "count": 62\n    },\n    {\n      "word": "facility",\n      "count": 62\n    },\n    {\n      "word": "fireman",\n      "count": 62\n    },\n    {\n      "word": "kiwi",\n      "count": 62\n    },\n    {\n      "word": "out",\n      "count": 62\n    },\n    {\n      "word": "slaw",\n      "count": 62\n    },\n    {\n      "word": "while",\n      "count": 62\n    },\n    {\n      "word": "wok",\n      "count": 62\n    },\n    {\n      "word": "age",\n      "count": 61\n    },\n    {\n      "word": "baking",\n      "count": 61\n    },\n    {\n      "word": "belonging",\n      "count": 61\n    },\n    {\n      "word": "brand",\n      "count": 61\n    },\n    {\n      "word": "earth",\n      "count": 61\n    },\n    {\n      "word": "flamingo",\n      "count": 61\n    },\n    {\n      "word": "flavor",\n      "count": 61\n    },\n    {\n      "word": "instruction",\n      "count": 61\n    },\n    {\n      "word": "record",\n      "count": 61\n    },\n    {\n      "word": "rod",\n      "count": 61\n    },\n    {\n      "word": "sleep",\n      "count": 61\n    },\n    {\n      "word": "split",\n      "count": 61\n    },\n    {\n      "word": "swim",\n      "count": 61\n    },\n    {\n      "word": "underwear",\n      "count": 61\n    },\n    {\n      "word": "antique",\n      "count": 60\n    },\n    {\n      "word": "archway",\n      "count": 60\n    },\n    {\n      "word": "asparagus",\n      "count": 60\n    },\n    {\n      "word": "attempt",\n      "count": 60\n    },\n    {\n      "word": "barbecue",\n      "count": 60\n    },\n    {\n      "word": "claw",\n      "count": 60\n    },\n    {\n      "word": "crab",\n      "count": 60\n    },\n    {\n      "word": "daisy",\n      "count": 60\n    },\n    {\n      "word": "feature",\n      "count": 60\n    },\n    {\n      "word": "hawk",\n      "count": 60\n    },\n    {\n      "word": "job",\n      "count": 60\n    },\n    {\n      "word": "jug",\n      "count": 60\n    },\n    {\n      "word": "knack",\n      "count": 60\n    },\n    {\n      "word": "medicine",\n      "count": 60\n    },\n    {\n      "word": "quarter",\n      "count": 60\n    },\n    {\n      "word": "smoothie",\n      "count": 60\n    },\n    {\n      "word": "spring",\n      "count": 60\n    },\n    {\n      "word": "waiting",\n      "count": 60\n    },\n    {\n      "word": "bill",\n      "count": 59\n    },\n    {\n      "word": "comb",\n      "count": 59\n    },\n    {\n      "word": "crystal",\n      "count": 59\n    },\n    {\n      "word": "hanging",\n      "count": 59\n    },\n    {\n      "word": "livestock",\n      "count": 59\n    },\n    {\n      "word": "security",\n      "count": 59\n    },\n    {\n      "word": "stairway",\n      "count": 59\n    },\n    {\n      "word": "yacht",\n      "count": 59\n    },\n    {\n      "word": "airways",\n      "count": 58\n    },\n    {\n      "word": "attention",\n      "count": 58\n    },\n    {\n      "word": "bronze",\n      "count": 58\n    },\n    {\n      "word": "dust",\n      "count": 58\n    },\n    {\n      "word": "grouping",\n      "count": 58\n    },\n    {\n      "word": "hundred",\n      "count": 58\n    },\n    {\n      "word": "monster",\n      "count": 58\n    },\n    {\n      "word": "oak",\n      "count": 58\n    },\n    {\n      "word": "showing",\n      "count": 58\n    },\n    {\n      "word": "strap",\n      "count": 58\n    },\n    {\n      "word": "ticket",\n      "count": 58\n    },\n    {\n      "word": "ambulance",\n      "count": 57\n    },\n    {\n      "word": "awning",\n      "count": 57\n    },\n    {\n      "word": "dance",\n      "count": 57\n    },\n    {\n      "word": "entry",\n      "count": 57\n    },\n    {\n      "word": "lens",\n      "count": 57\n    },\n    {\n      "word": "overhead",\n      "count": 57\n    },\n    {\n      "word": "paperwork",\n      "count": 57\n    },\n    {\n      "word": "rhino",\n      "count": 57\n    },\n    {\n      "word": "robot",\n      "count": 57\n    },\n    {\n      "word": "shelter",\n      "count": 57\n    },\n    {\n      "word": "sporting",\n      "count": 57\n    },\n    {\n      "word": "trey",\n      "count": 57\n    },\n    {\n      "word": "washroom",\n      "count": 57\n    },\n    {\n      "word": "appetizer",\n      "count": 56\n    },\n    {\n      "word": "calculator",\n      "count": 56\n    },\n    {\n      "word": "camouflage",\n      "count": 56\n    },\n    {\n      "word": "custom",\n      "count": 56\n    },\n    {\n      "word": "daylight",\n      "count": 56\n    },\n    {\n      "word": "headlight",\n      "count": 56\n    },\n    {\n      "word": "pic",\n      "count": 56\n    },\n    {\n      "word": "approach",\n      "count": 55\n    },\n    {\n      "word": "autumn",\n      "count": 55\n    },\n    {\n      "word": "canada",\n      "count": 55\n    },\n    {\n      "word": "dome",\n      "count": 55\n    },\n    {\n      "word": "feed",\n      "count": 55\n    },\n    {\n      "word": "frizbee",\n      "count": 55\n    },\n    {\n      "word": "glider",\n      "count": 55\n    },\n    {\n      "word": "locomotive",\n      "count": 55\n    },\n    {\n      "word": "mesh",\n      "count": 55\n    },\n    {\n      "word": "patron",\n      "count": 55\n    },\n    {\n      "word": "peak",\n      "count": 55\n    },\n    {\n      "word": "peer",\n      "count": 55\n    },\n    {\n      "word": "plank",\n      "count": 55\n    },\n    {\n      "word": "rowboat",\n      "count": 55\n    },\n    {\n      "word": "savanna",\n      "count": 55\n    },\n    {\n      "word": "snowsuit",\n      "count": 55\n    },\n    {\n      "word": "tanker",\n      "count": 55\n    },\n    {\n      "word": "amusement",\n      "count": 54\n    },\n    {\n      "word": "bug",\n      "count": 54\n    },\n    {\n      "word": "ceremony",\n      "count": 54\n    },\n    {\n      "word": "drape",\n      "count": 54\n    },\n    {\n      "word": "hip",\n      "count": 54\n    },\n    {\n      "word": "oar",\n      "count": 54\n    },\n    {\n      "word": "octopus",\n      "count": 54\n    },\n    {\n      "word": "outdoor",\n      "count": 54\n    },\n    {\n      "word": "retro",\n      "count": 54\n    },\n    {\n      "word": "scenery",\n      "count": 54\n    },\n    {\n      "word": "seafood",\n      "count": 54\n    },\n    {\n      "word": "squirrel",\n      "count": 54\n    },\n    {\n      "word": "twilight",\n      "count": 54\n    },\n    {\n      "word": "backhand",\n      "count": 53\n    },\n    {\n      "word": "frog",\n      "count": 53\n    },\n    {\n      "word": "grapefruit",\n      "count": 53\n    },\n    {\n      "word": "handler",\n      "count": 53\n    },\n    {\n      "word": "horizon",\n      "count": 53\n    },\n    {\n      "word": "incline",\n      "count": 53\n    },\n    {\n      "word": "looking",\n      "count": 53\n    },\n    {\n      "word": "melon",\n      "count": 53\n    },\n    {\n      "word": "necklace",\n      "count": 53\n    },\n    {\n      "word": "paste",\n      "count": 53\n    },\n    {\n      "word": "plantain",\n      "count": 53\n    },\n    {\n      "word": "plow",\n      "count": 53\n    },\n    {\n      "word": "protest",\n      "count": 53\n    },\n    {\n      "word": "tofu",\n      "count": 53\n    },\n    {\n      "word": "ton",\n      "count": 53\n    },\n    {\n      "word": "ware",\n      "count": 53\n    },\n    {\n      "word": "wig",\n      "count": 53\n    },\n    {\n      "word": "aisle",\n      "count": 52\n    },\n    {\n      "word": "bud",\n      "count": 52\n    },\n    {\n      "word": "caboose",\n      "count": 52\n    },\n    {\n      "word": "chin",\n      "count": 52\n    },\n    {\n      "word": "cole",\n      "count": 52\n    },\n    {\n      "word": "finish",\n      "count": 52\n    },\n    {\n      "word": "hover",\n      "count": 52\n    },\n    {\n      "word": "ipod",\n      "count": 52\n    },\n    {\n      "word": "lapse",\n      "count": 52\n    },\n    {\n      "word": "mascot",\n      "count": 52\n    },\n    {\n      "word": "pc",\n      "count": 52\n    },\n    {\n      "word": "personnel",\n      "count": 52\n    },\n    {\n      "word": "radiator",\n      "count": 52\n    },\n    {\n      "word": "resting",\n      "count": 52\n    },\n    {\n      "word": "sidecar",\n      "count": 52\n    },\n    {\n      "word": "stable",\n      "count": 52\n    },\n    {\n      "word": "waterfront",\n      "count": 52\n    },\n    {\n      "word": "accident",\n      "count": 51\n    },\n    {\n      "word": "airstrip",\n      "count": 51\n    },\n    {\n      "word": "alpine",\n      "count": 51\n    },\n    {\n      "word": "battery",\n      "count": 51\n    },\n    {\n      "word": "chew",\n      "count": 51\n    },\n    {\n      "word": "diaper",\n      "count": 51\n    },\n    {\n      "word": "foal",\n      "count": 51\n    },\n    {\n      "word": "hammer",\n      "count": 51\n    },\n    {\n      "word": "heater",\n      "count": 51\n    },\n    {\n      "word": "length",\n      "count": 51\n    },\n    {\n      "word": "lettering",\n      "count": 51\n    },\n    {\n      "word": "processor",\n      "count": 51\n    },\n    {\n      "word": "prop",\n      "count": 51\n    },\n    {\n      "word": "sunshine",\n      "count": 51\n    },\n    {\n      "word": "tournament",\n      "count": 51\n    },\n    {\n      "word": "trophy",\n      "count": 51\n    },\n    {\n      "word": "angel",\n      "count": 50\n    },\n    {\n      "word": "baseman",\n      "count": 50\n    },\n    {\n      "word": "bone",\n      "count": 50\n    },\n    {\n      "word": "buss",\n      "count": 50\n    },\n    {\n      "word": "charger",\n      "count": 50\n    },\n    {\n      "word": "cityscape",\n      "count": 50\n    },\n    {\n      "word": "floret",\n      "count": 50\n    },\n    {\n      "word": "goalie",\n      "count": 50\n    },\n    {\n      "word": "kick",\n      "count": 50\n    },\n    {\n      "word": "lie",\n      "count": 50\n    },\n    {\n      "word": "marking",\n      "count": 50\n    },\n    {\n      "word": "moment",\n      "count": 50\n    },\n    {\n      "word": "nail",\n      "count": 50\n    },\n    {\n      "word": "omelet",\n      "count": 50\n    },\n    {\n      "word": "pay",\n      "count": 50\n    },\n    {\n      "word": "pita",\n      "count": 50\n    },\n    {\n      "word": "radish",\n      "count": 50\n    },\n    {\n      "word": "sight",\n      "count": 50\n    },\n    {\n      "word": "smoking",\n      "count": 50\n    },\n    {\n      "word": "swimmer",\n      "count": 50\n    },\n    {\n      "word": "tabletop",\n      "count": 50\n    },\n    {\n      "word": "bananas",\n      "count": 49\n    },\n    {\n      "word": "beanie",\n      "count": 49\n    },\n    {\n      "word": "bulldog",\n      "count": 49\n    },\n    {\n      "word": "bumper",\n      "count": 49\n    },\n    {\n      "word": "camel",\n      "count": 49\n    },\n    {\n      "word": "camping",\n      "count": 49\n    },\n    {\n      "word": "drain",\n      "count": 49\n    },\n    {\n      "word": "expanse",\n      "count": 49\n    },\n    {\n      "word": "fist",\n      "count": 49\n    },\n    {\n      "word": "gown",\n      "count": 49\n    },\n    {\n      "word": "handlebar",\n      "count": 49\n    },\n    {\n      "word": "machinery",\n      "count": 49\n    },\n    {\n      "word": "old",\n      "count": 49\n    },\n    {\n      "word": "raquet",\n      "count": 49\n    },\n    {\n      "word": "recliner",\n      "count": 49\n    },\n    {\n      "word": "salon",\n      "count": 49\n    },\n    {\n      "word": "skin",\n      "count": 49\n    },\n    {\n      "word": "squat",\n      "count": 49\n    },\n    {\n      "word": "workstation",\n      "count": 49\n    },\n    {\n      "word": "barber",\n      "count": 48\n    },\n    {\n      "word": "barge",\n      "count": 48\n    },\n    {\n      "word": "being",\n      "count": 48\n    },\n    {\n      "word": "bridle",\n      "count": 48\n    },\n    {\n      "word": "camp",\n      "count": 48\n    },\n    {\n      "word": "check",\n      "count": 48\n    },\n    {\n      "word": "diving",\n      "count": 48\n    },\n    {\n      "word": "goer",\n      "count": 48\n    },\n    {\n      "word": "grove",\n      "count": 48\n    },\n    {\n      "word": "height",\n      "count": 48\n    },\n    {\n      "word": "help",\n      "count": 48\n    },\n    {\n      "word": "pop",\n      "count": 48\n    },\n    {\n      "word": "rein",\n      "count": 48\n    },\n    {\n      "word": "root",\n      "count": 48\n    },\n    {\n      "word": "salmon",\n      "count": 48\n    },\n    {\n      "word": "speech",\n      "count": 48\n    },\n    {\n      "word": "surrounding",\n      "count": 48\n    },\n    {\n      "word": "touch",\n      "count": 48\n    },\n    {\n      "word": "vane",\n      "count": 48\n    },\n    {\n      "word": "vanilla",\n      "count": 48\n    },\n    {\n      "word": "vessel",\n      "count": 48\n    },\n    {\n      "word": "waist",\n      "count": 48\n    },\n    {\n      "word": "almond",\n      "count": 47\n    },\n    {\n      "word": "analog",\n      "count": 47\n    },\n    {\n      "word": "bib",\n      "count": 47\n    },\n    {\n      "word": "clip",\n      "count": 47\n    },\n    {\n      "word": "conductor",\n      "count": 47\n    },\n    {\n      "word": "detail",\n      "count": 47\n    },\n    {\n      "word": "fleet",\n      "count": 47\n    },\n    {\n      "word": "gang",\n      "count": 47\n    },\n    {\n      "word": "garb",\n      "count": 47\n    },\n    {\n      "word": "gull",\n      "count": 47\n    },\n    {\n      "word": "hang",\n      "count": 47\n    },\n    {\n      "word": "motorboat",\n      "count": 47\n    },\n    {\n      "word": "n",\n      "count": 47\n    },\n    {\n      "word": "renovation",\n      "count": 47\n    },\n    {\n      "word": "rv",\n      "count": 47\n    },\n    {\n      "word": "salt",\n      "count": 47\n    },\n    {\n      "word": "shack",\n      "count": 47\n    },\n    {\n      "word": "showroom",\n      "count": 47\n    },\n    {\n      "word": "snowboarding",\n      "count": 47\n    },\n    {\n      "word": "stir",\n      "count": 47\n    },\n    {\n      "word": "taco",\n      "count": 47\n    },\n    {\n      "word": "theater",\n      "count": 47\n    },\n    {\n      "word": "umbrellas",\n      "count": 47\n    },\n    {\n      "word": "windmill",\n      "count": 47\n    },\n    {\n      "word": "amtrak",\n      "count": 46\n    },\n    {\n      "word": "backside",\n      "count": 46\n    },\n    {\n      "word": "bandana",\n      "count": 46\n    },\n    {\n      "word": "burrito",\n      "count": 46\n    },\n    {\n      "word": "cast",\n      "count": 46\n    },\n    {\n      "word": "cave",\n      "count": 46\n    },\n    {\n      "word": "chunk",\n      "count": 46\n    },\n    {\n      "word": "crop",\n      "count": 46\n    },\n    {\n      "word": "drum",\n      "count": 46\n    },\n    {\n      "word": "garlic",\n      "count": 46\n    },\n    {\n      "word": "goblet",\n      "count": 46\n    },\n    {\n      "word": "halloween",\n      "count": 46\n    },\n    {\n      "word": "krispy",\n      "count": 46\n    },\n    {\n      "word": "lace",\n      "count": 46\n    },\n    {\n      "word": "lone",\n      "count": 46\n    },\n    {\n      "word": "marketplace",\n      "count": 46\n    },\n    {\n      "word": "nature",\n      "count": 46\n    },\n    {\n      "word": "propellor",\n      "count": 46\n    },\n    {\n      "word": "ranch",\n      "count": 46\n    },\n    {\n      "word": "refridgerator",\n      "count": 46\n    },\n    {\n      "word": "residence",\n      "count": 46\n    },\n    {\n      "word": "rooster",\n      "count": 46\n    },\n    {\n      "word": "sandwhich",\n      "count": 46\n    },\n    {\n      "word": "server",\n      "count": 46\n    },\n    {\n      "word": "slipper",\n      "count": 46\n    },\n    {\n      "word": "substance",\n      "count": 46\n    },\n    {\n      "word": "teammate",\n      "count": 46\n    },\n    {\n      "word": "zucchini",\n      "count": 46\n    },\n    {\n      "word": "baker",\n      "count": 45\n    },\n    {\n      "word": "banquet",\n      "count": 45\n    },\n    {\n      "word": "bead",\n      "count": 45\n    },\n    {\n      "word": "blazer",\n      "count": 45\n    },\n    {\n      "word": "boxing",\n      "count": 45\n    },\n    {\n      "word": "butt",\n      "count": 45\n    },\n    {\n      "word": "cactus",\n      "count": 45\n    },\n    {\n      "word": "captivity",\n      "count": 45\n    },\n    {\n      "word": "driving",\n      "count": 45\n    },\n    {\n      "word": "fedex",\n      "count": 45\n    },\n    {\n      "word": "ginger",\n      "count": 45\n    },\n    {\n      "word": "hoop",\n      "count": 45\n    },\n    {\n      "word": "household",\n      "count": 45\n    },\n    {\n      "word": "maneuver",\n      "count": 45\n    },\n    {\n      "word": "oxen",\n      "count": 45\n    },\n    {\n      "word": "teapot",\n      "count": 45\n    },\n    {\n      "word": "trashcan",\n      "count": 45\n    },\n    {\n      "word": "viewer",\n      "count": 45\n    },\n    {\n      "word": "west",\n      "count": 45\n    },\n    {\n      "word": "barricade",\n      "count": 44\n    },\n    {\n      "word": "channel",\n      "count": 44\n    },\n    {\n      "word": "competitor",\n      "count": 44\n    },\n    {\n      "word": "concert",\n      "count": 44\n    },\n    {\n      "word": "consumption",\n      "count": 44\n    },\n    {\n      "word": "contest",\n      "count": 44\n    },\n    {\n      "word": "drop",\n      "count": 44\n    },\n    {\n      "word": "fair",\n      "count": 44\n    },\n    {\n      "word": "flowery",\n      "count": 44\n    },\n    {\n      "word": "hammock",\n      "count": 44\n    },\n    {\n      "word": "reception",\n      "count": 44\n    },\n    {\n      "word": "rig",\n      "count": 44\n    },\n    {\n      "word": "seasoning",\n      "count": 44\n    },\n    {\n      "word": "see",\n      "count": 44\n    },\n    {\n      "word": "siting",\n      "count": 44\n    },\n    {\n      "word": "starbucks",\n      "count": 44\n    },\n    {\n      "word": "thread",\n      "count": 44\n    },\n    {\n      "word": "turtle",\n      "count": 44\n    },\n    {\n      "word": "waste",\n      "count": 44\n    },\n    {\n      "word": "assembly",\n      "count": 43\n    },\n    {\n      "word": "batch",\n      "count": 43\n    },\n    {\n      "word": "british",\n      "count": 43\n    },\n    {\n      "word": "clocktower",\n      "count": 43\n    },\n    {\n      "word": "coaster",\n      "count": 43\n    },\n    {\n      "word": "contraption",\n      "count": 43\n    },\n    {\n      "word": "firefighter",\n      "count": 43\n    },\n    {\n      "word": "forehead",\n      "count": 43\n    },\n    {\n      "word": "fryer",\n      "count": 43\n    },\n    {\n      "word": "kreme",\n      "count": 43\n    },\n    {\n      "word": "macbook",\n      "count": 43\n    },\n    {\n      "word": "monk",\n      "count": 43\n    },\n    {\n      "word": "mozzarella",\n      "count": 43\n    },\n    {\n      "word": "pepsi",\n      "count": 43\n    },\n    {\n      "word": "plat",\n      "count": 43\n    },\n    {\n      "word": "rust",\n      "count": 43\n    },\n    {\n      "word": "s",\n      "count": 43\n    },\n    {\n      "word": "shepherd",\n      "count": 43\n    },\n    {\n      "word": "sleigh",\n      "count": 43\n    },\n    {\n      "word": "snowman",\n      "count": 43\n    },\n    {\n      "word": "supermarket",\n      "count": 43\n    },\n    {\n      "word": "support",\n      "count": 43\n    },\n    {\n      "word": "tong",\n      "count": 43\n    },\n    {\n      "word": "urn",\n      "count": 43\n    },\n    {\n      "word": "companion",\n      "count": 42\n    },\n    {\n      "word": "croissant",\n      "count": 42\n    },\n    {\n      "word": "crow",\n      "count": 42\n    },\n    {\n      "word": "doctor",\n      "count": 42\n    },\n    {\n      "word": "flame",\n      "count": 42\n    },\n    {\n      "word": "garnish",\n      "count": 42\n    },\n    {\n      "word": "grate",\n      "count": 42\n    },\n    {\n      "word": "mango",\n      "count": 42\n    },\n    {\n      "word": "replica",\n      "count": 42\n    },\n    {\n      "word": "san",\n      "count": 42\n    },\n    {\n      "word": "states",\n      "count": 42\n    },\n    {\n      "word": "stereo",\n      "count": 42\n    },\n    {\n      "word": "triangle",\n      "count": 42\n    },\n    {\n      "word": "walker",\n      "count": 42\n    },\n    {\n      "word": "atv",\n      "count": 41\n    },\n    {\n      "word": "broadway",\n      "count": 41\n    },\n    {\n      "word": "era",\n      "count": 41\n    },\n    {\n      "word": "folding",\n      "count": 41\n    },\n    {\n      "word": "function",\n      "count": 41\n    },\n    {\n      "word": "haircut",\n      "count": 41\n    },\n    {\n      "word": "hug",\n      "count": 41\n    },\n    {\n      "word": "instructor",\n      "count": 41\n    },\n    {\n      "word": "maintenance",\n      "count": 41\n    },\n    {\n      "word": "master",\n      "count": 41\n    },\n    {\n      "word": "patty",\n      "count": 41\n    },\n    {\n      "word": "poodle",\n      "count": 41\n    },\n    {\n      "word": "sofas",\n      "count": 41\n    },\n    {\n      "word": "styrofoam",\n      "count": 41\n    },\n    {\n      "word": "wheelie",\n      "count": 41\n    },\n    {\n      "word": "wrist",\n      "count": 41\n    },\n    {\n      "word": "asia",\n      "count": 40\n    },\n    {\n      "word": "boston",\n      "count": 40\n    },\n    {\n      "word": "brass",\n      "count": 40\n    },\n    {\n      "word": "canvas",\n      "count": 40\n    },\n    {\n      "word": "coal",\n      "count": 40\n    },\n    {\n      "word": "england",\n      "count": 40\n    },\n    {\n      "word": "jack",\n      "count": 40\n    },\n    {\n      "word": "loft",\n      "count": 40\n    },\n    {\n      "word": "median",\n      "count": 40\n    },\n    {\n      "word": "mobile",\n      "count": 40\n    },\n    {\n      "word": "nook",\n      "count": 40\n    },\n    {\n      "word": "ox",\n      "count": 40\n    },\n    {\n      "word": "paddock",\n      "count": 40\n    },\n    {\n      "word": "rally",\n      "count": 40\n    },\n    {\n      "word": "rd",\n      "count": 40\n    },\n    {\n      "word": "rooftop",\n      "count": 40\n    },\n    {\n      "word": "sing",\n      "count": 40\n    },\n    {\n      "word": "streamer",\n      "count": 40\n    },\n    {\n      "word": "tot",\n      "count": 40\n    },\n    {\n      "word": "wineglass",\n      "count": 40\n    },\n    {\n      "word": "award",\n      "count": 39\n    },\n    {\n      "word": "beauty",\n      "count": 39\n    },\n    {\n      "word": "bundt",\n      "count": 39\n    },\n    {\n      "word": "california",\n      "count": 39\n    },\n    {\n      "word": "chicago",\n      "count": 39\n    },\n    {\n      "word": "chick",\n      "count": 39\n    },\n    {\n      "word": "chihuahua",\n      "count": 39\n    },\n    {\n      "word": "citrus",\n      "count": 39\n    },\n    {\n      "word": "coconut",\n      "count": 39\n    },\n    {\n      "word": "contact",\n      "count": 39\n    },\n    {\n      "word": "dollar",\n      "count": 39\n    },\n    {\n      "word": "dunkin",\n      "count": 39\n    },\n    {\n      "word": "fencing",\n      "count": 39\n    },\n    {\n      "word": "handrail",\n      "count": 39\n    },\n    {\n      "word": "highchair",\n      "count": 39\n    },\n    {\n      "word": "ofa",\n      "count": 39\n    },\n    {\n      "word": "pane",\n      "count": 39\n    },\n    {\n      "word": "region",\n      "count": 39\n    },\n    {\n      "word": "rocky",\n      "count": 39\n    },\n    {\n      "word": "semi",\n      "count": 39\n    },\n    {\n      "word": "sideline",\n      "count": 39\n    },\n    {\n      "word": "sleeve",\n      "count": 39\n    },\n    {\n      "word": "teacher",\n      "count": 39\n    },\n    {\n      "word": "toilette",\n      "count": 39\n    },\n    {\n      "word": "yak",\n      "count": 39\n    },\n    {\n      "word": "attendant",\n      "count": 38\n    },\n    {\n      "word": "backseat",\n      "count": 38\n    },\n    {\n      "word": "centerpiece",\n      "count": 38\n    },\n    {\n      "word": "complex",\n      "count": 38\n    },\n    {\n      "word": "curl",\n      "count": 38\n    },\n    {\n      "word": "dude",\n      "count": 38\n    },\n    {\n      "word": "establishment",\n      "count": 38\n    },\n    {\n      "word": "eyeglass",\n      "count": 38\n    },\n    {\n      "word": "glaze",\n      "count": 38\n    },\n    {\n      "word": "graduation",\n      "count": 38\n    },\n    {\n      "word": "guide",\n      "count": 38\n    },\n    {\n      "word": "hurdle",\n      "count": 38\n    },\n    {\n      "word": "int",\n      "count": 38\n    },\n    {\n      "word": "lobster",\n      "count": 38\n    },\n    {\n      "word": "luxury",\n      "count": 38\n    },\n    {\n      "word": "need",\n      "count": 38\n    },\n    {\n      "word": "paneling",\n      "count": 38\n    },\n    {\n      "word": "pantry",\n      "count": 38\n    },\n    {\n      "word": "perspective",\n      "count": 38\n    },\n    {\n      "word": "quaint",\n      "count": 38\n    },\n    {\n      "word": "recipe",\n      "count": 38\n    },\n    {\n      "word": "rink",\n      "count": 38\n    },\n    {\n      "word": "shaker",\n      "count": 38\n    },\n    {\n      "word": "shark",\n      "count": 38\n    },\n    {\n      "word": "swimsuit",\n      "count": 38\n    },\n    {\n      "word": "thomas",\n      "count": 38\n    },\n    {\n      "word": "attachment",\n      "count": 37\n    },\n    {\n      "word": "carving",\n      "count": 37\n    },\n    {\n      "word": "claus",\n      "count": 37\n    },\n    {\n      "word": "combo",\n      "count": 37\n    },\n    {\n      "word": "community",\n      "count": 37\n    },\n    {\n      "word": "demonstration",\n      "count": 37\n    },\n    {\n      "word": "folk",\n      "count": 37\n    },\n    {\n      "word": "hockey",\n      "count": 37\n    },\n    {\n      "word": "jelly",\n      "count": 37\n    },\n    {\n      "word": "kiss",\n      "count": 37\n    },\n    {\n      "word": "lead",\n      "count": 37\n    },\n    {\n      "word": "mane",\n      "count": 37\n    },\n    {\n      "word": "minute",\n      "count": 37\n    },\n    {\n      "word": "nokia",\n      "count": 37\n    },\n    {\n      "word": "official",\n      "count": 37\n    },\n    {\n      "word": "packet",\n      "count": 37\n    },\n    {\n      "word": "palace",\n      "count": 37\n    },\n    {\n      "word": "parasail",\n      "count": 37\n    },\n    {\n      "word": "prey",\n      "count": 37\n    },\n    {\n      "word": "rescue",\n      "count": 37\n    },\n    {\n      "word": "shear",\n      "count": 37\n    },\n    {\n      "word": "sweet",\n      "count": 37\n    },\n    {\n      "word": "windsurfer",\n      "count": 37\n    },\n    {\n      "word": "a",\n      "count": 36\n    },\n    {\n      "word": "bale",\n      "count": 36\n    },\n    {\n      "word": "blackberry",\n      "count": 36\n    },\n    {\n      "word": "blossom",\n      "count": 36\n    },\n    {\n      "word": "caramel",\n      "count": 36\n    },\n    {\n      "word": "creature",\n      "count": 36\n    },\n    {\n      "word": "cube",\n      "count": 36\n    },\n    {\n      "word": "film",\n      "count": 36\n    },\n    {\n      "word": "frisby",\n      "count": 36\n    },\n    {\n      "word": "headband",\n      "count": 36\n    },\n    {\n      "word": "lifeguard",\n      "count": 36\n    },\n    {\n      "word": "lock",\n      "count": 36\n    },\n    {\n      "word": "mill",\n      "count": 36\n    },\n    {\n      "word": "nurse",\n      "count": 36\n    },\n    {\n      "word": "pong",\n      "count": 36\n    },\n    {\n      "word": "rapid",\n      "count": 36\n    },\n    {\n      "word": "slat",\n      "count": 36\n    },\n    {\n      "word": "standup",\n      "count": 36\n    },\n    {\n      "word": "start",\n      "count": 36\n    },\n    {\n      "word": "tattoo",\n      "count": 36\n    },\n    {\n      "word": "virgin",\n      "count": 36\n    },\n    {\n      "word": "wheat",\n      "count": 36\n    },\n    {\n      "word": "wildflower",\n      "count": 36\n    },\n    {\n      "word": "alleyway",\n      "count": 35\n    },\n    {\n      "word": "change",\n      "count": 35\n    },\n    {\n      "word": "collie",\n      "count": 35\n    },\n    {\n      "word": "discussion",\n      "count": 35\n    },\n    {\n      "word": "harley",\n      "count": 35\n    },\n    {\n      "word": "japan",\n      "count": 35\n    },\n    {\n      "word": "juicer",\n      "count": 35\n    },\n    {\n      "word": "knob",\n      "count": 35\n    },\n    {\n      "word": "laugh",\n      "count": 35\n    },\n    {\n      "word": "little",\n      "count": 35\n    },\n    {\n      "word": "mark",\n      "count": 35\n    },\n    {\n      "word": "outhouse",\n      "count": 35\n    },\n    {\n      "word": "pallet",\n      "count": 35\n    },\n    {\n      "word": "patrol",\n      "count": 35\n    },\n    {\n      "word": "ping",\n      "count": 35\n    },\n    {\n      "word": "plunger",\n      "count": 35\n    },\n    {\n      "word": "raincoat",\n      "count": 35\n    },\n    {\n      "word": "ready",\n      "count": 35\n    },\n    {\n      "word": "salsa",\n      "count": 35\n    },\n    {\n      "word": "sauerkraut",\n      "count": 35\n    },\n    {\n      "word": "selling",\n      "count": 35\n    },\n    {\n      "word": "speeding",\n      "count": 35\n    },\n    {\n      "word": "trees",\n      "count": 35\n    },\n    {\n      "word": "tripod",\n      "count": 35\n    },\n    {\n      "word": "tshirt",\n      "count": 35\n    },\n    {\n      "word": "vent",\n      "count": 35\n    },\n    {\n      "word": "wreath",\n      "count": 35\n    },\n    {\n      "word": "auto",\n      "count": 34\n    },\n    {\n      "word": "blur",\n      "count": 34\n    },\n    {\n      "word": "cheeseburger",\n      "count": 34\n    },\n    {\n      "word": "cockpit",\n      "count": 34\n    },\n    {\n      "word": "dumpster",\n      "count": 34\n    },\n    {\n      "word": "flood",\n      "count": 34\n    },\n    {\n      "word": "ford",\n      "count": 34\n    },\n    {\n      "word": "four",\n      "count": 34\n    },\n    {\n      "word": "housing",\n      "count": 34\n    },\n    {\n      "word": "leafy",\n      "count": 34\n    },\n    {\n      "word": "lite",\n      "count": 34\n    },\n    {\n      "word": "litter",\n      "count": 34\n    },\n    {\n      "word": "lizard",\n      "count": 34\n    },\n    {\n      "word": "main",\n      "count": 34\n    },\n    {\n      "word": "mama",\n      "count": 34\n    },\n    {\n      "word": "mcdonald",\n      "count": 34\n    },\n    {\n      "word": "overall",\n      "count": 34\n    },\n    {\n      "word": "performance",\n      "count": 34\n    },\n    {\n      "word": "placemat",\n      "count": 34\n    },\n    {\n      "word": "pretzel",\n      "count": 34\n    },\n    {\n      "word": "projection",\n      "count": 34\n    },\n    {\n      "word": "quiche",\n      "count": 34\n    },\n    {\n      "word": "remain",\n      "count": 34\n    },\n    {\n      "word": "rugby",\n      "count": 34\n    },\n    {\n      "word": "staff",\n      "count": 34\n    },\n    {\n      "word": "sunrise",\n      "count": 34\n    },\n    {\n      "word": "toe",\n      "count": 34\n    },\n    {\n      "word": "tricycle",\n      "count": 34\n    },\n    {\n      "word": "twin",\n      "count": 34\n    },\n    {\n      "word": "visor",\n      "count": 34\n    },\n    {\n      "word": "workspace",\n      "count": 34\n    },\n    {\n      "word": "article",\n      "count": 33\n    },\n    {\n      "word": "basis",\n      "count": 33\n    },\n    {\n      "word": "bison",\n      "count": 33\n    },\n    {\n      "word": "boar",\n      "count": 33\n    },\n    {\n      "word": "cottage",\n      "count": 33\n    },\n    {\n      "word": "dune",\n      "count": 33\n    },\n    {\n      "word": "elevator",\n      "count": 33\n    },\n    {\n      "word": "europe",\n      "count": 33\n    },\n    {\n      "word": "flatbread",\n      "count": 33\n    },\n    {\n      "word": "handing",\n      "count": 33\n    },\n    {\n      "word": "hiker",\n      "count": 33\n    },\n    {\n      "word": "hold",\n      "count": 33\n    },\n    {\n      "word": "india",\n      "count": 33\n    },\n    {\n      "word": "jewelry",\n      "count": 33\n    },\n    {\n      "word": "lantern",\n      "count": 33\n    },\n    {\n      "word": "mason",\n      "count": 33\n    },\n    {\n      "word": "mickey",\n      "count": 33\n    },\n    {\n      "word": "moss",\n      "count": 33\n    },\n    {\n      "word": "parasailer",\n      "count": 33\n    },\n    {\n      "word": "parlor",\n      "count": 33\n    },\n    {\n      "word": "parsley",\n      "count": 33\n    },\n    {\n      "word": "program",\n      "count": 33\n    },\n    {\n      "word": "rearview",\n      "count": 33\n    },\n    {\n      "word": "sample",\n      "count": 33\n    },\n    {\n      "word": "scrub",\n      "count": 33\n    },\n    {\n      "word": "shutter",\n      "count": 33\n    },\n    {\n      "word": "union",\n      "count": 33\n    },\n    {\n      "word": "youngster",\n      "count": 33\n    },\n    {\n      "word": "ankle",\n      "count": 32\n    },\n    {\n      "word": "auditorium",\n      "count": 32\n    },\n    {\n      "word": "beet",\n      "count": 32\n    },\n    {\n      "word": "bento",\n      "count": 32\n    },\n    {\n      "word": "brushing",\n      "count": 32\n    },\n    {\n      "word": "butcher",\n      "count": 32\n    },\n    {\n      "word": "cel",\n      "count": 32\n    },\n    {\n      "word": "century",\n      "count": 32\n    },\n    {\n      "word": "cheek",\n      "count": 32\n    },\n    {\n      "word": "cocktail",\n      "count": 32\n    },\n    {\n      "word": "davidson",\n      "count": 32\n    },\n    {\n      "word": "eat",\n      "count": 32\n    },\n    {\n      "word": "entree",\n      "count": 32\n    },\n    {\n      "word": "exercise",\n      "count": 32\n    },\n    {\n      "word": "flash",\n      "count": 32\n    },\n    {\n      "word": "foam",\n      "count": 32\n    },\n    {\n      "word": "futon",\n      "count": 32\n    },\n    {\n      "word": "hamster",\n      "count": 32\n    },\n    {\n      "word": "judge",\n      "count": 32\n    },\n    {\n      "word": "l",\n      "count": 32\n    },\n    {\n      "word": "mailbox",\n      "count": 32\n    },\n    {\n      "word": "mini",\n      "count": 32\n    },\n    {\n      "word": "motorist",\n      "count": 32\n    },\n    {\n      "word": "nighttime",\n      "count": 32\n    },\n    {\n      "word": "outlet",\n      "count": 32\n    },\n    {\n      "word": "parked",\n      "count": 32\n    },\n    {\n      "word": "pitching",\n      "count": 32\n    },\n    {\n      "word": "plum",\n      "count": 32\n    },\n    {\n      "word": "plush",\n      "count": 32\n    },\n    {\n      "word": "potty",\n      "count": 32\n    },\n    {\n      "word": "pudding",\n      "count": 32\n    },\n    {\n      "word": "samsung",\n      "count": 32\n    },\n    {\n      "word": "scoop",\n      "count": 32\n    },\n    {\n      "word": "seal",\n      "count": 32\n    },\n    {\n      "word": "stovetop",\n      "count": 32\n    },\n    {\n      "word": "underside",\n      "count": 32\n    },\n    {\n      "word": "whip",\n      "count": 32\n    },\n    {\n      "word": "amenity",\n      "count": 31\n    },\n    {\n      "word": "bee",\n      "count": 31\n    },\n    {\n      "word": "binder",\n      "count": 31\n    },\n    {\n      "word": "bloom",\n      "count": 31\n    },\n    {\n      "word": "capitol",\n      "count": 31\n    },\n    {\n      "word": "cinnamon",\n      "count": 31\n    },\n    {\n      "word": "clydesdale",\n      "count": 31\n    },\n    {\n      "word": "condition",\n      "count": 31\n    },\n    {\n      "word": "coupe",\n      "count": 31\n    },\n    {\n      "word": "credit",\n      "count": 31\n    },\n    {\n      "word": "crochet",\n      "count": 31\n    },\n    {\n      "word": "crumb",\n      "count": 31\n    },\n    {\n      "word": "dalmatian",\n      "count": 31\n    },\n    {\n      "word": "date",\n      "count": 31\n    },\n    {\n      "word": "eatery",\n      "count": 31\n    },\n    {\n      "word": "facade",\n      "count": 31\n    },\n    {\n      "word": "flour",\n      "count": 31\n    },\n    {\n      "word": "gazebo",\n      "count": 31\n    },\n    {\n      "word": "kiteboard",\n      "count": 31\n    },\n    {\n      "word": "lamppost",\n      "count": 31\n    },\n    {\n      "word": "license",\n      "count": 31\n    },\n    {\n      "word": "livingroom",\n      "count": 31\n    },\n    {\n      "word": "make",\n      "count": 31\n    },\n    {\n      "word": "marshmallow",\n      "count": 31\n    },\n    {\n      "word": "neath",\n      "count": 31\n    },\n    {\n      "word": "overview",\n      "count": 31\n    },\n    {\n      "word": "pajamas",\n      "count": 31\n    },\n    {\n      "word": "passanger",\n      "count": 31\n    },\n    {\n      "word": "pew",\n      "count": 31\n    },\n    {\n      "word": "skeleton",\n      "count": 31\n    },\n    {\n      "word": "slop",\n      "count": 31\n    },\n    {\n      "word": "stroll",\n      "count": 31\n    },\n    {\n      "word": "submarine",\n      "count": 31\n    },\n    {\n      "word": "switch",\n      "count": 31\n    },\n    {\n      "word": "texting",\n      "count": 31\n    },\n    {\n      "word": "velvet",\n      "count": 31\n    },\n    {\n      "word": "waiter",\n      "count": 31\n    },\n    {\n      "word": "aquarium",\n      "count": 30\n    },\n    {\n      "word": "attraction",\n      "count": 30\n    },\n    {\n      "word": "barefoot",\n      "count": 30\n    },\n    {\n      "word": "bedside",\n      "count": 30\n    },\n    {\n      "word": "broth",\n      "count": 30\n    },\n    {\n      "word": "dash",\n      "count": 30\n    },\n    {\n      "word": "granola",\n      "count": 30\n    },\n    {\n      "word": "hairdryer",\n      "count": 30\n    },\n    {\n      "word": "hash",\n      "count": 30\n    },\n    {\n      "word": "icebox",\n      "count": 30\n    },\n    {\n      "word": "kitchenette",\n      "count": 30\n    },\n    {\n      "word": "knight",\n      "count": 30\n    },\n    {\n      "word": "mantel",\n      "count": 30\n    },\n    {\n      "word": "medal",\n      "count": 30\n    },\n    {\n      "word": "orchid",\n      "count": 30\n    },\n    {\n      "word": "packaging",\n      "count": 30\n    },\n    {\n      "word": "penny",\n      "count": 30\n    },\n    {\n      "word": "performer",\n      "count": 30\n    },\n    {\n      "word": "petal",\n      "count": 30\n    },\n    {\n      "word": "protester",\n      "count": 30\n    },\n    {\n      "word": "razor",\n      "count": 30\n    },\n    {\n      "word": "return",\n      "count": 30\n    },\n    {\n      "word": "scout",\n      "count": 30\n    },\n    {\n      "word": "sound",\n      "count": 30\n    },\n    {\n      "word": "stain",\n      "count": 30\n    },\n    {\n      "word": "tasting",\n      "count": 30\n    },\n    {\n      "word": "university",\n      "count": 30\n    },\n    {\n      "word": "wildebeest",\n      "count": 30\n    },\n    {\n      "word": "yield",\n      "count": 30\n    },\n    {\n      "word": "armchair",\n      "count": 29\n    },\n    {\n      "word": "badminton",\n      "count": 29\n    },\n    {\n      "word": "buliding",\n      "count": 29\n    },\n    {\n      "word": "chalkboard",\n      "count": 29\n    },\n    {\n      "word": "charter",\n      "count": 29\n    },\n    {\n      "word": "colt",\n      "count": 29\n    },\n    {\n      "word": "cooker",\n      "count": 29\n    },\n    {\n      "word": "cot",\n      "count": 29\n    },\n    {\n      "word": "diet",\n      "count": 29\n    },\n    {\n      "word": "dinosaur",\n      "count": 29\n    },\n    {\n      "word": "eiffel",\n      "count": 29\n    },\n    {\n      "word": "firework",\n      "count": 29\n    },\n    {\n      "word": "france",\n      "count": 29\n    },\n    {\n      "word": "galley",\n      "count": 29\n    },\n    {\n      "word": "glassware",\n      "count": 29\n    },\n    {\n      "word": "golden",\n      "count": 29\n    },\n    {\n      "word": "halfpipe",\n      "count": 29\n    },\n    {\n      "word": "headset",\n      "count": 29\n    },\n    {\n      "word": "hollywood",\n      "count": 29\n    },\n    {\n      "word": "ipad",\n      "count": 29\n    },\n    {\n      "word": "lavender",\n      "count": 29\n    },\n    {\n      "word": "law",\n      "count": 29\n    },\n    {\n      "word": "maple",\n      "count": 29\n    },\n    {\n      "word": "mint",\n      "count": 29\n    },\n    {\n      "word": "motorcyle",\n      "count": 29\n    },\n    {\n      "word": "nursing",\n      "count": 29\n    },\n    {\n      "word": "partition",\n      "count": 29\n    },\n    {\n      "word": "peek",\n      "count": 29\n    },\n    {\n      "word": "profile",\n      "count": 29\n    },\n    {\n      "word": "pub",\n      "count": 29\n    },\n    {\n      "word": "rat",\n      "count": 29\n    },\n    {\n      "word": "ridge",\n      "count": 29\n    },\n    {\n      "word": "ruler",\n      "count": 29\n    },\n    {\n      "word": "sigh",\n      "count": 29\n    },\n    {\n      "word": "snake",\n      "count": 29\n    },\n    {\n      "word": "sox",\n      "count": 29\n    },\n    {\n      "word": "taking",\n      "count": 29\n    },\n    {\n      "word": "toll",\n      "count": 29\n    },\n    {\n      "word": "version",\n      "count": 29\n    },\n    {\n      "word": "watching",\n      "count": 29\n    },\n    {\n      "word": "alert",\n      "count": 28\n    },\n    {\n      "word": "altitude",\n      "count": 28\n    },\n    {\n      "word": "basement",\n      "count": 28\n    },\n    {\n      "word": "bump",\n      "count": 28\n    },\n    {\n      "word": "clause",\n      "count": 28\n    },\n    {\n      "word": "concept",\n      "count": 28\n    },\n    {\n      "word": "copper",\n      "count": 28\n    },\n    {\n      "word": "ditch",\n      "count": 28\n    },\n    {\n      "word": "doily",\n      "count": 28\n    },\n    {\n      "word": "dolphin",\n      "count": 28\n    },\n    {\n      "word": "duffel",\n      "count": 28\n    },\n    {\n      "word": "elbow",\n      "count": 28\n    },\n    {\n      "word": "forehand",\n      "count": 28\n    },\n    {\n      "word": "francisco",\n      "count": 28\n    },\n    {\n      "word": "heron",\n      "count": 28\n    },\n    {\n      "word": "hilltop",\n      "count": 28\n    },\n    {\n      "word": "ivy",\n      "count": 28\n    },\n    {\n      "word": "jacuzzi",\n      "count": 28\n    },\n    {\n      "word": "knot",\n      "count": 28\n    },\n    {\n      "word": "lots",\n      "count": 28\n    },\n    {\n      "word": "lunge",\n      "count": 28\n    },\n    {\n      "word": "medium",\n      "count": 28\n    },\n    {\n      "word": "north",\n      "count": 28\n    },\n    {\n      "word": "off",\n      "count": 28\n    },\n    {\n      "word": "option",\n      "count": 28\n    },\n    {\n      "word": "patient",\n      "count": 28\n    },\n    {\n      "word": "potter",\n      "count": 28\n    },\n    {\n      "word": "reign",\n      "count": 28\n    },\n    {\n      "word": "rocket",\n      "count": 28\n    },\n    {\n      "word": "surfboarder",\n      "count": 28\n    },\n    {\n      "word": "tater",\n      "count": 28\n    },\n    {\n      "word": "tech",\n      "count": 28\n    },\n    {\n      "word": "temple",\n      "count": 28\n    },\n    {\n      "word": "walnut",\n      "count": 28\n    },\n    {\n      "word": "web",\n      "count": 28\n    },\n    {\n      "word": "wife",\n      "count": 28\n    },\n    {\n      "word": "women",\n      "count": 28\n    },\n    {\n      "word": "working",\n      "count": 28\n    },\n    {\n      "word": "wrapping",\n      "count": 28\n    },\n    {\n      "word": "abundance",\n      "count": 27\n    },\n    {\n      "word": "africa",\n      "count": 27\n    },\n    {\n      "word": "artist",\n      "count": 27\n    },\n    {\n      "word": "badge",\n      "count": 27\n    },\n    {\n      "word": "bark",\n      "count": 27\n    },\n    {\n      "word": "blvd",\n      "count": 27\n    },\n    {\n      "word": "bulb",\n      "count": 27\n    },\n    {\n      "word": "chalk",\n      "count": 27\n    },\n    {\n      "word": "coral",\n      "count": 27\n    },\n    {\n      "word": "cutout",\n      "count": 27\n    },\n    {\n      "word": "dial",\n      "count": 27\n    },\n    {\n      "word": "e",\n      "count": 27\n    },\n    {\n      "word": "effect",\n      "count": 27\n    },\n    {\n      "word": "folder",\n      "count": 27\n    },\n    {\n      "word": "fuel",\n      "count": 27\n    },\n    {\n      "word": "goatee",\n      "count": 27\n    },\n    {\n      "word": "hello",\n      "count": 27\n    },\n    {\n      "word": "hero",\n      "count": 27\n    },\n    {\n      "word": "interstate",\n      "count": 27\n    },\n    {\n      "word": "isle",\n      "count": 27\n    },\n    {\n      "word": "jesus",\n      "count": 27\n    },\n    {\n      "word": "jockeys",\n      "count": 27\n    },\n    {\n      "word": "lapel",\n      "count": 27\n    },\n    {\n      "word": "manual",\n      "count": 27\n    },\n    {\n      "word": "notepad",\n      "count": 27\n    },\n    {\n      "word": "orchard",\n      "count": 27\n    },\n    {\n      "word": "parakeet",\n      "count": 27\n    },\n    {\n      "word": "plug",\n      "count": 27\n    },\n    {\n      "word": "pouch",\n      "count": 27\n    },\n    {\n      "word": "raisin",\n      "count": 27\n    },\n    {\n      "word": "ray",\n      "count": 27\n    },\n    {\n      "word": "read",\n      "count": 27\n    },\n    {\n      "word": "recording",\n      "count": 27\n    },\n    {\n      "word": "shaving",\n      "count": 27\n    },\n    {\n      "word": "skylight",\n      "count": 27\n    },\n    {\n      "word": "soil",\n      "count": 27\n    },\n    {\n      "word": "source",\n      "count": 27\n    },\n    {\n      "word": "statute",\n      "count": 27\n    },\n    {\n      "word": "taste",\n      "count": 27\n    },\n    {\n      "word": "technology",\n      "count": 27\n    },\n    {\n      "word": "tinfoil",\n      "count": 27\n    },\n    {\n      "word": "toliet",\n      "count": 27\n    },\n    {\n      "word": "usa",\n      "count": 27\n    },\n    {\n      "word": "vw",\n      "count": 27\n    },\n    {\n      "word": "whit",\n      "count": 27\n    },\n    {\n      "word": "aqua",\n      "count": 26\n    },\n    {\n      "word": "bandanna",\n      "count": 26\n    },\n    {\n      "word": "barbeque",\n      "count": 26\n    },\n    {\n      "word": "bomber",\n      "count": 26\n    },\n    {\n      "word": "brace",\n      "count": 26\n    },\n    {\n      "word": "by",\n      "count": 26\n    },\n    {\n      "word": "component",\n      "count": 26\n    },\n    {\n      "word": "dancing",\n      "count": 26\n    },\n    {\n      "word": "emo",\n      "count": 26\n    },\n    {\n      "word": "exposure",\n      "count": 26\n    },\n    {\n      "word": "glow",\n      "count": 26\n    },\n    {\n      "word": "handstand",\n      "count": 26\n    },\n    {\n      "word": "hound",\n      "count": 26\n    },\n    {\n      "word": "kiosk",\n      "count": 26\n    },\n    {\n      "word": "mass",\n      "count": 26\n    },\n    {\n      "word": "memorabilia",\n      "count": 26\n    },\n    {\n      "word": "photography",\n      "count": 26\n    },\n    {\n      "word": "rib",\n      "count": 26\n    },\n    {\n      "word": "riverbank",\n      "count": 26\n    },\n    {\n      "word": "safari",\n      "count": 26\n    },\n    {\n      "word": "shake",\n      "count": 26\n    },\n    {\n      "word": "skateboarding",\n      "count": 26\n    },\n    {\n      "word": "skewer",\n      "count": 26\n    },\n    {\n      "word": "slalom",\n      "count": 26\n    },\n    {\n      "word": "slicing",\n      "count": 26\n    },\n    {\n      "word": "venue",\n      "count": 26\n    },\n    {\n      "word": "w",\n      "count": 26\n    },\n    {\n      "word": "wet",\n      "count": 26\n    },\n    {\n      "word": "winery",\n      "count": 26\n    },\n    {\n      "word": "zombie",\n      "count": 26\n    },\n    {\n      "word": "av",\n      "count": 25\n    },\n    {\n      "word": "bedding",\n      "count": 25\n    },\n    {\n      "word": "bmw",\n      "count": 25\n    },\n    {\n      "word": "boulevard",\n      "count": 25\n    },\n    {\n      "word": "canyon",\n      "count": 25\n    },\n    {\n      "word": "choice",\n      "count": 25\n    },\n    {\n      "word": "chute",\n      "count": 25\n    },\n    {\n      "word": "climb",\n      "count": 25\n    },\n    {\n      "word": "cob",\n      "count": 25\n    },\n    {\n      "word": "cotton",\n      "count": 25\n    },\n    {\n      "word": "dancer",\n      "count": 25\n    },\n    {\n      "word": "description",\n      "count": 25\n    },\n    {\n      "word": "divider",\n      "count": 25\n    },\n    {\n      "word": "docking",\n      "count": 25\n    },\n    {\n      "word": "exterior",\n      "count": 25\n    },\n    {\n      "word": "filling",\n      "count": 25\n    },\n    {\n      "word": "gauge",\n      "count": 25\n    },\n    {\n      "word": "gorilla",\n      "count": 25\n    },\n    {\n      "word": "graphic",\n      "count": 25\n    },\n    {\n      "word": "husband",\n      "count": 25\n    },\n    {\n      "word": "lasagna",\n      "count": 25\n    },\n    {\n      "word": "leftover",\n      "count": 25\n    },\n    {\n      "word": "lipstick",\n      "count": 25\n    },\n    {\n      "word": "loveseat",\n      "count": 25\n    },\n    {\n      "word": "moose",\n      "count": 25\n    },\n    {\n      "word": "muscle",\n      "count": 25\n    },\n    {\n      "word": "paris",\n      "count": 25\n    },\n    {\n      "word": "pedal",\n      "count": 25\n    },\n    {\n      "word": "poll",\n      "count": 25\n    },\n    {\n      "word": "pooh",\n      "count": 25\n    },\n    {\n      "word": "purchase",\n      "count": 25\n    },\n    {\n      "word": "shopper",\n      "count": 25\n    },\n    {\n      "word": "sliver",\n      "count": 25\n    },\n    {\n      "word": "stairwell",\n      "count": 25\n    },\n    {\n      "word": "suspender",\n      "count": 25\n    },\n    {\n      "word": "task",\n      "count": 25\n    },\n    {\n      "word": "toothpick",\n      "count": 25\n    },\n    {\n      "word": "weight",\n      "count": 25\n    },\n    {\n      "word": "womans",\n      "count": 25\n    },\n    {\n      "word": "workshop",\n      "count": 25\n    },\n    {\n      "word": "aeroplane",\n      "count": 24\n    },\n    {\n      "word": "america",\n      "count": 24\n    },\n    {\n      "word": "apparatus",\n      "count": 24\n    },\n    {\n      "word": "bake",\n      "count": 24\n    },\n    {\n      "word": "beginning",\n      "count": 24\n    },\n    {\n      "word": "booze",\n      "count": 24\n    },\n    {\n      "word": "bowtie",\n      "count": 24\n    },\n    {\n      "word": "brake",\n      "count": 24\n    },\n    {\n      "word": "broom",\n      "count": 24\n    },\n    {\n      "word": "brother",\n      "count": 24\n    },\n    {\n      "word": "buoy",\n      "count": 24\n    },\n    {\n      "word": "cardinal",\n      "count": 24\n    },\n    {\n      "word": "clump",\n      "count": 24\n    },\n    {\n      "word": "cold",\n      "count": 24\n    },\n    {\n      "word": "cuisine",\n      "count": 24\n    },\n    {\n      "word": "down",\n      "count": 24\n    },\n    {\n      "word": "duckling",\n      "count": 24\n    },\n    {\n      "word": "feild",\n      "count": 24\n    },\n    {\n      "word": "globe",\n      "count": 24\n    },\n    {\n      "word": "graduate",\n      "count": 24\n    },\n    {\n      "word": "jail",\n      "count": 24\n    },\n    {\n      "word": "keeper",\n      "count": 24\n    },\n    {\n      "word": "lecture",\n      "count": 24\n    },\n    {\n      "word": "lit",\n      "count": 24\n    },\n    {\n      "word": "making",\n      "count": 24\n    },\n    {\n      "word": "mean",\n      "count": 24\n    },\n    {\n      "word": "merry",\n      "count": 24\n    },\n    {\n      "word": "mount",\n      "count": 24\n    },\n    {\n      "word": "multitude",\n      "count": 24\n    },\n    {\n      "word": "musician",\n      "count": 24\n    },\n    {\n      "word": "pavilion",\n      "count": 24\n    },\n    {\n      "word": "prairie",\n      "count": 24\n    },\n    {\n      "word": "rag",\n      "count": 24\n    },\n    {\n      "word": "rectangle",\n      "count": 24\n    },\n    {\n      "word": "sack",\n      "count": 24\n    },\n    {\n      "word": "seaweed",\n      "count": 24\n    },\n    {\n      "word": "shipping",\n      "count": 24\n    },\n    {\n      "word": "shoot",\n      "count": 24\n    },\n    {\n      "word": "singing",\n      "count": 24\n    },\n    {\n      "word": "someones",\n      "count": 24\n    },\n    {\n      "word": "speedboat",\n      "count": 24\n    },\n    {\n      "word": "stork",\n      "count": 24\n    },\n    {\n      "word": "streetcar",\n      "count": 24\n    },\n    {\n      "word": "sweat",\n      "count": 24\n    },\n    {\n      "word": "tale",\n      "count": 24\n    },\n    {\n      "word": "theatre",\n      "count": 24\n    },\n    {\n      "word": "third",\n      "count": 24\n    },\n    {\n      "word": "tight",\n      "count": 24\n    },\n    {\n      "word": "wrench",\n      "count": 24\n    },\n    {\n      "word": "aid",\n      "count": 23\n    },\n    {\n      "word": "antenna",\n      "count": 23\n    },\n    {\n      "word": "battle",\n      "count": 23\n    },\n    {\n      "word": "beagle",\n      "count": 23\n    },\n    {\n      "word": "bracelet",\n      "count": 23\n    },\n    {\n      "word": "breed",\n      "count": 23\n    },\n    {\n      "word": "brussel",\n      "count": 23\n    },\n    {\n      "word": "cape",\n      "count": 23\n    },\n    {\n      "word": "children",\n      "count": 23\n    },\n    {\n      "word": "chimney",\n      "count": 23\n    },\n    {\n      "word": "cider",\n      "count": 23\n    },\n    {\n      "word": "contrail",\n      "count": 23\n    },\n    {\n      "word": "cuckoo",\n      "count": 23\n    },\n    {\n      "word": "daffodil",\n      "count": 23\n    },\n    {\n      "word": "earphone",\n      "count": 23\n    },\n    {\n      "word": "easel",\n      "count": 23\n    },\n    {\n      "word": "equestrian",\n      "count": 23\n    },\n    {\n      "word": "fatigue",\n      "count": 23\n    },\n    {\n      "word": "flyer",\n      "count": 23\n    },\n    {\n      "word": "fox",\n      "count": 23\n    },\n    {\n      "word": "full",\n      "count": 23\n    },\n    {\n      "word": "gymnasium",\n      "count": 23\n    },\n    {\n      "word": "joke",\n      "count": 23\n    },\n    {\n      "word": "kale",\n      "count": 23\n    },\n    {\n      "word": "khaki",\n      "count": 23\n    },\n    {\n      "word": "leafs",\n      "count": 23\n    },\n    {\n      "word": "lesson",\n      "count": 23\n    },\n    {\n      "word": "lincoln",\n      "count": 23\n    },\n    {\n      "word": "locker",\n      "count": 23\n    },\n    {\n      "word": "memorial",\n      "count": 23\n    },\n    {\n      "word": "motif",\n      "count": 23\n    },\n    {\n      "word": "pacifier",\n      "count": 23\n    },\n    {\n      "word": "plethora",\n      "count": 23\n    },\n    {\n      "word": "pound",\n      "count": 23\n    },\n    {\n      "word": "privacy",\n      "count": 23\n    },\n    {\n      "word": "retriever",\n      "count": 23\n    },\n    {\n      "word": "scheme",\n      "count": 23\n    },\n    {\n      "word": "sesame",\n      "count": 23\n    },\n    {\n      "word": "silo",\n      "count": 23\n    },\n    {\n      "word": "spool",\n      "count": 23\n    },\n    {\n      "word": "study",\n      "count": 23\n    },\n    {\n      "word": "terrier",\n      "count": 23\n    },\n    {\n      "word": "trade",\n      "count": 23\n    },\n    {\n      "word": "tuna",\n      "count": 23\n    },\n    {\n      "word": "act",\n      "count": 22\n    },\n    {\n      "word": "ass",\n      "count": 22\n    },\n    {\n      "word": "blinder",\n      "count": 22\n    },\n    {\n      "word": "bra",\n      "count": 22\n    },\n    {\n      "word": "campus",\n      "count": 22\n    },\n    {\n      "word": "checker",\n      "count": 22\n    },\n    {\n      "word": "checkerboard",\n      "count": 22\n    },\n    {\n      "word": "chinese",\n      "count": 22\n    },\n    {\n      "word": "citizen",\n      "count": 22\n    },\n    {\n      "word": "coastline",\n      "count": 22\n    },\n    {\n      "word": "crack",\n      "count": 22\n    },\n    {\n      "word": "crepe",\n      "count": 22\n    },\n    {\n      "word": "damage",\n      "count": 22\n    },\n    {\n      "word": "dell",\n      "count": 22\n    },\n    {\n      "word": "detour",\n      "count": 22\n    },\n    {\n      "word": "document",\n      "count": 22\n    },\n    {\n      "word": "downhill",\n      "count": 22\n    },\n    {\n      "word": "dummy",\n      "count": 22\n    },\n    {\n      "word": "enthusiast",\n      "count": 22\n    },\n    {\n      "word": "envelope",\n      "count": 22\n    },\n    {\n      "word": "ewe",\n      "count": 22\n    },\n    {\n      "word": "flop",\n      "count": 22\n    },\n    {\n      "word": "flow",\n      "count": 22\n    },\n    {\n      "word": "giraffes",\n      "count": 22\n    },\n    {\n      "word": "google",\n      "count": 22\n    },\n    {\n      "word": "gutter",\n      "count": 22\n    },\n    {\n      "word": "happy",\n      "count": 22\n    },\n    {\n      "word": "ink",\n      "count": 22\n    },\n    {\n      "word": "labrador",\n      "count": 22\n    },\n    {\n      "word": "ladle",\n      "count": 22\n    },\n    {\n      "word": "landscaping",\n      "count": 22\n    },\n    {\n      "word": "leopard",\n      "count": 22\n    },\n    {\n      "word": "llama",\n      "count": 22\n    },\n    {\n      "word": "marine",\n      "count": 22\n    },\n    {\n      "word": "mate",\n      "count": 22\n    },\n    {\n      "word": "mercedes",\n      "count": 22\n    },\n    {\n      "word": "military",\n      "count": 22\n    },\n    {\n      "word": "minivan",\n      "count": 22\n    },\n    {\n      "word": "montage",\n      "count": 22\n    },\n    {\n      "word": "motocross",\n      "count": 22\n    },\n    {\n      "word": "movement",\n      "count": 22\n    },\n    {\n      "word": "mule",\n      "count": 22\n    },\n    {\n      "word": "novelty",\n      "count": 22\n    },\n    {\n      "word": "occasion",\n      "count": 22\n    },\n    {\n      "word": "pace",\n      "count": 22\n    },\n    {\n      "word": "parallel",\n      "count": 22\n    },\n    {\n      "word": "pause",\n      "count": 22\n    },\n    {\n      "word": "picket",\n      "count": 22\n    },\n    {\n      "word": "pizzeria",\n      "count": 22\n    },\n    {\n      "word": "plier",\n      "count": 22\n    },\n    {\n      "word": "plumbing",\n      "count": 22\n    },\n    {\n      "word": "princess",\n      "count": 22\n    },\n    {\n      "word": "punch",\n      "count": 22\n    },\n    {\n      "word": "puppet",\n      "count": 22\n    },\n    {\n      "word": "push",\n      "count": 22\n    },\n    {\n      "word": "pyramid",\n      "count": 22\n    },\n    {\n      "word": "rackett",\n      "count": 22\n    },\n    {\n      "word": "royal",\n      "count": 22\n    },\n    {\n      "word": "seashore",\n      "count": 22\n    },\n    {\n      "word": "shovel",\n      "count": 22\n    },\n    {\n      "word": "signpost",\n      "count": 22\n    },\n    {\n      "word": "skiie",\n      "count": 22\n    },\n    {\n      "word": "small",\n      "count": 22\n    },\n    {\n      "word": "soar",\n      "count": 22\n    },\n    {\n      "word": "spider",\n      "count": 22\n    },\n    {\n      "word": "squash",\n      "count": 22\n    },\n    {\n      "word": "stock",\n      "count": 22\n    },\n    {\n      "word": "tack",\n      "count": 22\n    },\n    {\n      "word": "texas",\n      "count": 22\n    },\n    {\n      "word": "victory",\n      "count": 22\n    },\n    {\n      "word": "alcove",\n      "count": 21\n    },\n    {\n      "word": "atm",\n      "count": 21\n    },\n    {\n      "word": "australia",\n      "count": 21\n    },\n    {\n      "word": "b",\n      "count": 21\n    },\n    {\n      "word": "barb",\n      "count": 21\n    },\n    {\n      "word": "bikinis",\n      "count": 21\n    },\n    {\n      "word": "birdhouse",\n      "count": 21\n    },\n    {\n      "word": "blouse",\n      "count": 21\n    },\n    {\n      "word": "boxer",\n      "count": 21\n    },\n    {\n      "word": "bulding",\n      "count": 21\n    },\n    {\n      "word": "carnation",\n      "count": 21\n    },\n    {\n      "word": "chairlift",\n      "count": 21\n    },\n    {\n      "word": "creme",\n      "count": 21\n    },\n    {\n      "word": "draft",\n      "count": 21\n    },\n    {\n      "word": "emblem",\n      "count": 21\n    },\n    {\n      "word": "engineer",\n      "count": 21\n    },\n    {\n      "word": "evergreen",\n      "count": 21\n    },\n    {\n      "word": "ex",\n      "count": 21\n    },\n    {\n      "word": "exhaust",\n      "count": 21\n    },\n    {\n      "word": "fedora",\n      "count": 21\n    },\n    {\n      "word": "fixing",\n      "count": 21\n    },\n    {\n      "word": "gnar",\n      "count": 21\n    },\n    {\n      "word": "grab",\n      "count": 21\n    },\n    {\n      "word": "gum",\n      "count": 21\n    },\n    {\n      "word": "heat",\n      "count": 21\n    },\n    {\n      "word": "host",\n      "count": 21\n    },\n    {\n      "word": "illustration",\n      "count": 21\n    },\n    {\n      "word": "jetway",\n      "count": 21\n    },\n    {\n      "word": "knitting",\n      "count": 21\n    },\n    {\n      "word": "lava",\n      "count": 21\n    },\n    {\n      "word": "liberty",\n      "count": 21\n    },\n    {\n      "word": "lotion",\n      "count": 21\n    },\n    {\n      "word": "marsh",\n      "count": 21\n    },\n    {\n      "word": "mile",\n      "count": 21\n    },\n    {\n      "word": "no",\n      "count": 21\n    },\n    {\n      "word": "novel",\n      "count": 21\n    },\n    {\n      "word": "panorama",\n      "count": 21\n    },\n    {\n      "word": "partner",\n      "count": 21\n    },\n    {\n      "word": "past",\n      "count": 21\n    },\n    {\n      "word": "pill",\n      "count": 21\n    },\n    {\n      "word": "pull",\n      "count": 21\n    },\n    {\n      "word": "puzzle",\n      "count": 21\n    },\n    {\n      "word": "receipt",\n      "count": 21\n    },\n    {\n      "word": "recline",\n      "count": 21\n    },\n    {\n      "word": "remodel",\n      "count": 21\n    },\n    {\n      "word": "role",\n      "count": 21\n    },\n    {\n      "word": "rolling",\n      "count": 21\n    },\n    {\n      "word": "rubble",\n      "count": 21\n    },\n    {\n      "word": "scape",\n      "count": 21\n    },\n    {\n      "word": "song",\n      "count": 21\n    },\n    {\n      "word": "sparkler",\n      "count": 21\n    },\n    {\n      "word": "tabby",\n      "count": 21\n    },\n    {\n      "word": "teh",\n      "count": 21\n    },\n    {\n      "word": "thermometer",\n      "count": 21\n    },\n    {\n      "word": "thre",\n      "count": 21\n    },\n    {\n      "word": "trench",\n      "count": 21\n    },\n    {\n      "word": "triangular",\n      "count": 21\n    },\n    {\n      "word": "turban",\n      "count": 21\n    },\n    {\n      "word": "user",\n      "count": 21\n    },\n    {\n      "word": "whine",\n      "count": 21\n    },\n    {\n      "word": "wiimote",\n      "count": 21\n    },\n    {\n      "word": "winnie",\n      "count": 21\n    },\n    {\n      "word": "witha",\n      "count": 21\n    },\n    {\n      "word": "alligator",\n      "count": 20\n    },\n    {\n      "word": "apparel",\n      "count": 20\n    },\n    {\n      "word": "backsplash",\n      "count": 20\n    },\n    {\n      "word": "bears",\n      "count": 20\n    },\n    {\n      "word": "brunette",\n      "count": 20\n    },\n    {\n      "word": "bust",\n      "count": 20\n    },\n    {\n      "word": "cameraman",\n      "count": 20\n    },\n    {\n      "word": "carpeting",\n      "count": 20\n    },\n    {\n      "word": "cellar",\n      "count": 20\n    },\n    {\n      "word": "chase",\n      "count": 20\n    },\n    {\n      "word": "chopper",\n      "count": 20\n    },\n    {\n      "word": "colorful",\n      "count": 20\n    },\n    {\n      "word": "conveyer",\n      "count": 20\n    },\n    {\n      "word": "d",\n      "count": 20\n    },\n    {\n      "word": "danger",\n      "count": 20\n    },\n    {\n      "word": "dawn",\n      "count": 20\n    },\n    {\n      "word": "dye",\n      "count": 20\n    },\n    {\n      "word": "eachother",\n      "count": 20\n    },\n    {\n      "word": "earring",\n      "count": 20\n    },\n    {\n      "word": "eggplant",\n      "count": 20\n    },\n    {\n      "word": "fed",\n      "count": 20\n    },\n    {\n      "word": "fern",\n      "count": 20\n    },\n    {\n      "word": "forklift",\n      "count": 20\n    },\n    {\n      "word": "gap",\n      "count": 20\n    },\n    {\n      "word": "hairbrush",\n      "count": 20\n    },\n    {\n      "word": "herder",\n      "count": 20\n    },\n    {\n      "word": "hiking",\n      "count": 20\n    },\n    {\n      "word": "hog",\n      "count": 20\n    },\n    {\n      "word": "icon",\n      "count": 20\n    },\n    {\n      "word": "insect",\n      "count": 20\n    },\n    {\n      "word": "it",\n      "count": 20\n    },\n    {\n      "word": "jumper",\n      "count": 20\n    },\n    {\n      "word": "lagoon",\n      "count": 20\n    },\n    {\n      "word": "lavatory",\n      "count": 20\n    },\n    {\n      "word": "lemonade",\n      "count": 20\n    },\n    {\n      "word": "lingerie",\n      "count": 20\n    },\n    {\n      "word": "lying",\n      "count": 20\n    },\n    {\n      "word": "merchandise",\n      "count": 20\n    },\n    {\n      "word": "mop",\n      "count": 20\n    },\n    {\n      "word": "oatmeal",\n      "count": 20\n    },\n    {\n      "word": "offspring",\n      "count": 20\n    },\n    {\n      "word": "organ",\n      "count": 20\n    },\n    {\n      "word": "pail",\n      "count": 20\n    },\n    {\n      "word": "parchment",\n      "count": 20\n    },\n    {\n      "word": "parka",\n      "count": 20\n    },\n    {\n      "word": "parmesan",\n      "count": 20\n    },\n    {\n      "word": "pesto",\n      "count": 20\n    },\n    {\n      "word": "pontoon",\n      "count": 20\n    },\n    {\n      "word": "priest",\n      "count": 20\n    },\n    {\n      "word": "prize",\n      "count": 20\n    },\n    {\n      "word": "reader",\n      "count": 20\n    },\n    {\n      "word": "rendering",\n      "count": 20\n    },\n    {\n      "word": "rub",\n      "count": 20\n    },\n    {\n      "word": "scrap",\n      "count": 20\n    },\n    {\n      "word": "sequence",\n      "count": 20\n    },\n    {\n      "word": "sister",\n      "count": 20\n    },\n    {\n      "word": "snowmobile",\n      "count": 20\n    },\n    {\n      "word": "snowstorm",\n      "count": 20\n    },\n    {\n      "word": "spear",\n      "count": 20\n    },\n    {\n      "word": "spout",\n      "count": 20\n    },\n    {\n      "word": "swamp",\n      "count": 20\n    },\n    {\n      "word": "target",\n      "count": 20\n    },\n    {\n      "word": "thicket",\n      "count": 20\n    },\n    {\n      "word": "tiling",\n      "count": 20\n    },\n    {\n      "word": "turf",\n      "count": 20\n    },\n    {\n      "word": "turnip",\n      "count": 20\n    },\n    {\n      "word": "vacation",\n      "count": 20\n    },\n    {\n      "word": "win",\n      "count": 20\n    },\n    {\n      "word": "autograph",\n      "count": 19\n    },\n    {\n      "word": "bi",\n      "count": 19\n    },\n    {\n      "word": "bikes",\n      "count": 19\n    },\n    {\n      "word": "calzone",\n      "count": 19\n    },\n    {\n      "word": "canister",\n      "count": 19\n    },\n    {\n      "word": "capital",\n      "count": 19\n    },\n    {\n      "word": "central",\n      "count": 19\n    },\n    {\n      "word": "conditioner",\n      "count": 19\n    },\n    {\n      "word": "convenience",\n      "count": 19\n    },\n    {\n      "word": "corridor",\n      "count": 19\n    },\n    {\n      "word": "creation",\n      "count": 19\n    },\n    {\n      "word": "cycling",\n      "count": 19\n    },\n    {\n      "word": "decal",\n      "count": 19\n    },\n    {\n      "word": "east",\n      "count": 19\n    },\n    {\n      "word": "embankment",\n      "count": 19\n    },\n    {\n      "word": "energy",\n      "count": 19\n    },\n    {\n      "word": "fellow",\n      "count": 19\n    },\n    {\n      "word": "filing",\n      "count": 19\n    },\n    {\n      "word": "gallery",\n      "count": 19\n    },\n    {\n      "word": "giant",\n      "count": 19\n    },\n    {\n      "word": "government",\n      "count": 19\n    },\n    {\n      "word": "grip",\n      "count": 19\n    },\n    {\n      "word": "growth",\n      "count": 19\n    },\n    {\n      "word": "guacamole",\n      "count": 19\n    },\n    {\n      "word": "halter",\n      "count": 19\n    },\n    {\n      "word": "hatch",\n      "count": 19\n    },\n    {\n      "word": "hate",\n      "count": 19\n    },\n    {\n      "word": "hoagie",\n      "count": 19\n    },\n    {\n      "word": "husky",\n      "count": 19\n    },\n    {\n      "word": "implement",\n      "count": 19\n    },\n    {\n      "word": "indicator",\n      "count": 19\n    },\n    {\n      "word": "infield",\n      "count": 19\n    },\n    {\n      "word": "internet",\n      "count": 19\n    },\n    {\n      "word": "kids",\n      "count": 19\n    },\n    {\n      "word": "kilt",\n      "count": 19\n    },\n    {\n      "word": "lily",\n      "count": 19\n    },\n    {\n      "word": "ma",\n      "count": 19\n    },\n    {\n      "word": "mayonnaise",\n      "count": 19\n    },\n    {\n      "word": "merchant",\n      "count": 19\n    },\n    {\n      "word": "mist",\n      "count": 19\n    },\n    {\n      "word": "mohawk",\n      "count": 19\n    },\n    {\n      "word": "national",\n      "count": 19\n    },\n    {\n      "word": "olympics",\n      "count": 19\n    },\n    {\n      "word": "pod",\n      "count": 19\n    },\n    {\n      "word": "ponytail",\n      "count": 19\n    },\n    {\n      "word": "press",\n      "count": 19\n    },\n    {\n      "word": "procession",\n      "count": 19\n    },\n    {\n      "word": "reach",\n      "count": 19\n    },\n    {\n      "word": "rifle",\n      "count": 19\n    },\n    {\n      "word": "roundabout",\n      "count": 19\n    },\n    {\n      "word": "shift",\n      "count": 19\n    },\n    {\n      "word": "sightseeing",\n      "count": 19\n    },\n    {\n      "word": "sip",\n      "count": 19\n    },\n    {\n      "word": "slopes",\n      "count": 19\n    },\n    {\n      "word": "sony",\n      "count": 19\n    },\n    {\n      "word": "stamp",\n      "count": 19\n    },\n    {\n      "word": "streak",\n      "count": 19\n    },\n    {\n      "word": "summit",\n      "count": 19\n    },\n    {\n      "word": "tokyo",\n      "count": 19\n    },\n    {\n      "word": "trek",\n      "count": 19\n    },\n    {\n      "word": "try",\n      "count": 19\n    },\n    {\n      "word": "uk",\n      "count": 19\n    },\n    {\n      "word": "videogame",\n      "count": 19\n    },\n    {\n      "word": "width",\n      "count": 19\n    },\n    {\n      "word": "yoga",\n      "count": 19\n    },\n    {\n      "word": "access",\n      "count": 18\n    },\n    {\n      "word": "actor",\n      "count": 18\n    },\n    {\n      "word": "airshow",\n      "count": 18\n    },\n    {\n      "word": "alaska",\n      "count": 18\n    },\n    {\n      "word": "alot",\n      "count": 18\n    },\n    {\n      "word": "angeles",\n      "count": 18\n    },\n    {\n      "word": "batman",\n      "count": 18\n    },\n    {\n      "word": "birth",\n      "count": 18\n    },\n    {\n      "word": "bystander",\n      "count": 18\n    },\n    {\n      "word": "cappuccino",\n      "count": 18\n    },\n    {\n      "word": "charm",\n      "count": 18\n    },\n    {\n      "word": "cock",\n      "count": 18\n    },\n    {\n      "word": "congratulation",\n      "count": 18\n    },\n    {\n      "word": "cookware",\n      "count": 18\n    },\n    {\n      "word": "cork",\n      "count": 18\n    },\n    {\n      "word": "county",\n      "count": 18\n    },\n    {\n      "word": "dollhouse",\n      "count": 18\n    },\n    {\n      "word": "donation",\n      "count": 18\n    },\n    {\n      "word": "doubledecker",\n      "count": 18\n    },\n    {\n      "word": "efficiency",\n      "count": 18\n    },\n    {\n      "word": "eggs",\n      "count": 18\n    },\n    {\n      "word": "electric",\n      "count": 18\n    },\n    {\n      "word": "example",\n      "count": 18\n    },\n    {\n      "word": "fairy",\n      "count": 18\n    },\n    {\n      "word": "fisherman",\n      "count": 18\n    },\n    {\n      "word": "flatscreen",\n      "count": 18\n    },\n    {\n      "word": "flier",\n      "count": 18\n    },\n    {\n      "word": "garment",\n      "count": 18\n    },\n    {\n      "word": "get",\n      "count": 18\n    },\n    {\n      "word": "ghost",\n      "count": 18\n    },\n    {\n      "word": "goody",\n      "count": 18\n    },\n    {\n      "word": "goofy",\n      "count": 18\n    },\n    {\n      "word": "harry",\n      "count": 18\n    },\n    {\n      "word": "hiding",\n      "count": 18\n    },\n    {\n      "word": "honda",\n      "count": 18\n    },\n    {\n      "word": "hummingbird",\n      "count": 18\n    },\n    {\n      "word": "hump",\n      "count": 18\n    },\n    {\n      "word": "hygiene",\n      "count": 18\n    },\n    {\n      "word": "jalapenos",\n      "count": 18\n    },\n    {\n      "word": "kiwis",\n      "count": 18\n    },\n    {\n      "word": "lunchbox",\n      "count": 18\n    },\n    {\n      "word": "manner",\n      "count": 18\n    },\n    {\n      "word": "march",\n      "count": 18\n    },\n    {\n      "word": "mario",\n      "count": 18\n    },\n    {\n      "word": "mast",\n      "count": 18\n    },\n    {\n      "word": "monorail",\n      "count": 18\n    },\n    {\n      "word": "netting",\n      "count": 18\n    },\n    {\n      "word": "nick",\n      "count": 18\n    },\n    {\n      "word": "nike",\n      "count": 18\n    },\n    {\n      "word": "nun",\n      "count": 18\n    },\n    {\n      "word": "omelette",\n      "count": 18\n    },\n    {\n      "word": "pamphlet",\n      "count": 18\n    },\n    {\n      "word": "picker",\n      "count": 18\n    },\n    {\n      "word": "polaroid",\n      "count": 18\n    },\n    {\n      "word": "politician",\n      "count": 18\n    },\n    {\n      "word": "postcard",\n      "count": 18\n    },\n    {\n      "word": "preserve",\n      "count": 18\n    },\n    {\n      "word": "pro",\n      "count": 18\n    },\n    {\n      "word": "protection",\n      "count": 18\n    },\n    {\n      "word": "racetrack",\n      "count": 18\n    },\n    {\n      "word": "react",\n      "count": 18\n    },\n    {\n      "word": "ruin",\n      "count": 18\n    },\n    {\n      "word": "satellite",\n      "count": 18\n    },\n    {\n      "word": "seaplane",\n      "count": 18\n    },\n    {\n      "word": "session",\n      "count": 18\n    },\n    {\n      "word": "shepard",\n      "count": 18\n    },\n    {\n      "word": "sibling",\n      "count": 18\n    },\n    {\n      "word": "snapshot",\n      "count": 18\n    },\n    {\n      "word": "snout",\n      "count": 18\n    },\n    {\n      "word": "specialty",\n      "count": 18\n    },\n    {\n      "word": "sponge",\n      "count": 18\n    },\n    {\n      "word": "strike",\n      "count": 18\n    },\n    {\n      "word": "tar",\n      "count": 18\n    },\n    {\n      "word": "teacup",\n      "count": 18\n    },\n    {\n      "word": "temperature",\n      "count": 18\n    },\n    {\n      "word": "terrace",\n      "count": 18\n    },\n    {\n      "word": "tiara",\n      "count": 18\n    },\n    {\n      "word": "timer",\n      "count": 18\n    },\n    {\n      "word": "underpass",\n      "count": 18\n    },\n    {\n      "word": "unison",\n      "count": 18\n    },\n    {\n      "word": "v",\n      "count": 18\n    },\n    {\n      "word": "vegtable",\n      "count": 18\n    },\n    {\n      "word": "vulture",\n      "count": 18\n    },\n    {\n      "word": "windows",\n      "count": 18\n    },\n    {\n      "word": "address",\n      "count": 17\n    },\n    {\n      "word": "apples",\n      "count": 17\n    },\n    {\n      "word": "armor",\n      "count": 17\n    },\n    {\n      "word": "bartender",\n      "count": 17\n    },\n    {\n      "word": "beside",\n      "count": 17\n    },\n    {\n      "word": "bodysuit",\n      "count": 17\n    },\n    {\n      "word": "booty",\n      "count": 17\n    },\n    {\n      "word": "bot",\n      "count": 17\n    },\n    {\n      "word": "boxcar",\n      "count": 17\n    },\n    {\n      "word": "camo",\n      "count": 17\n    },\n    {\n      "word": "captain",\n      "count": 17\n    },\n    {\n      "word": "cent",\n      "count": 17\n    },\n    {\n      "word": "chess",\n      "count": 17\n    },\n    {\n      "word": "climate",\n      "count": 17\n    },\n    {\n      "word": "cookbook",\n      "count": 17\n    },\n    {\n      "word": "darkness",\n      "count": 17\n    },\n    {\n      "word": "denim",\n      "count": 17\n    },\n    {\n      "word": "dew",\n      "count": 17\n    },\n    {\n      "word": "dogs",\n      "count": 17\n    },\n    {\n      "word": "dove",\n      "count": 17\n    },\n    {\n      "word": "espresso",\n      "count": 17\n    },\n    {\n      "word": "filth",\n      "count": 17\n    },\n    {\n      "word": "fir",\n      "count": 17\n    },\n    {\n      "word": "fit",\n      "count": 17\n    },\n    {\n      "word": "forefront",\n      "count": 17\n    },\n    {\n      "word": "freez",\n      "count": 17\n    },\n    {\n      "word": "glare",\n      "count": 17\n    },\n    {\n      "word": "glue",\n      "count": 17\n    },\n    {\n      "word": "god",\n      "count": 17\n    },\n    {\n      "word": "gps",\n      "count": 17\n    },\n    {\n      "word": "grafitti",\n      "count": 17\n    },\n    {\n      "word": "graveyard",\n      "count": 17\n    },\n    {\n      "word": "hardware",\n      "count": 17\n    },\n    {\n      "word": "hazard",\n      "count": 17\n    },\n    {\n      "word": "herding",\n      "count": 17\n    },\n    {\n      "word": "inch",\n      "count": 17\n    },\n    {\n      "word": "interest",\n      "count": 17\n    },\n    {\n      "word": "iwth",\n      "count": 17\n    },\n    {\n      "word": "kangaroo",\n      "count": 17\n    },\n    {\n      "word": "kimono",\n      "count": 17\n    },\n    {\n      "word": "laundry",\n      "count": 17\n    },\n    {\n      "word": "loop",\n      "count": 17\n    },\n    {\n      "word": "los",\n      "count": 17\n    },\n    {\n      "word": "muzzle",\n      "count": 17\n    },\n    {\n      "word": "nasa",\n      "count": 17\n    },\n    {\n      "word": "ollie",\n      "count": 17\n    },\n    {\n      "word": "pacific",\n      "count": 17\n    },\n    {\n      "word": "peripheral",\n      "count": 17\n    },\n    {\n      "word": "register",\n      "count": 17\n    },\n    {\n      "word": "reporter",\n      "count": 17\n    },\n    {\n      "word": "sanctuary",\n      "count": 17\n    },\n    {\n      "word": "saying",\n      "count": 17\n    },\n    {\n      "word": "season",\n      "count": 17\n    },\n    {\n      "word": "slack",\n      "count": 17\n    },\n    {\n      "word": "slider",\n      "count": 17\n    },\n    {\n      "word": "southwest",\n      "count": 17\n    },\n    {\n      "word": "stopping",\n      "count": 17\n    },\n    {\n      "word": "strainer",\n      "count": 17\n    },\n    {\n      "word": "sunbather",\n      "count": 17\n    },\n    {\n      "word": "teddybear",\n      "count": 17\n    },\n    {\n      "word": "tote",\n      "count": 17\n    },\n    {\n      "word": "trinket",\n      "count": 17\n    },\n    {\n      "word": "tuck",\n      "count": 17\n    },\n    {\n      "word": "turbine",\n      "count": 17\n    },\n    {\n      "word": "un",\n      "count": 17\n    },\n    {\n      "word": "vespa",\n      "count": 17\n    },\n    {\n      "word": "violin",\n      "count": 17\n    },\n    {\n      "word": "wintery",\n      "count": 17\n    },\n    {\n      "word": "wolf",\n      "count": 17\n    },\n    {\n      "word": "adventure",\n      "count": 16\n    },\n    {\n      "word": "ale",\n      "count": 16\n    },\n    {\n      "word": "an",\n      "count": 16\n    },\n    {\n      "word": "anchor",\n      "count": 16\n    },\n    {\n      "word": "armoire",\n      "count": 16\n    },\n    {\n      "word": "bail",\n      "count": 16\n    },\n    {\n      "word": "banister",\n      "count": 16\n    },\n    {\n      "word": "bathrobe",\n      "count": 16\n    },\n    {\n      "word": "bookstore",\n      "count": 16\n    },\n    {\n      "word": "brochure",\n      "count": 16\n    },\n    {\n      "word": "campaign",\n      "count": 16\n    },\n    {\n      "word": "cantaloupe",\n      "count": 16\n    },\n    {\n      "word": "carrying",\n      "count": 16\n    },\n    {\n      "word": "cash",\n      "count": 16\n    },\n    {\n      "word": "cgi",\n      "count": 16\n    },\n    {\n      "word": "charcoal",\n      "count": 16\n    },\n    {\n      "word": "chubby",\n      "count": 16\n    },\n    {\n      "word": "cigar",\n      "count": 16\n    },\n    {\n      "word": "climbing",\n      "count": 16\n    },\n    {\n      "word": "confetti",\n      "count": 16\n    },\n    {\n      "word": "connection",\n      "count": 16\n    },\n    {\n      "word": "converse",\n      "count": 16\n    },\n    {\n      "word": "copy",\n      "count": 16\n    },\n    {\n      "word": "current",\n      "count": 16\n    },\n    {\n      "word": "cylinder",\n      "count": 16\n    },\n    {\n      "word": "dc",\n      "count": 16\n    },\n    {\n      "word": "diesel",\n      "count": 16\n    },\n    {\n      "word": "dirtbike",\n      "count": 16\n    },\n    {\n      "word": "disney",\n      "count": 16\n    },\n    {\n      "word": "dj",\n      "count": 16\n    },\n    {\n      "word": "dodgers",\n      "count": 16\n    },\n    {\n      "word": "donuts",\n      "count": 16\n    },\n    {\n      "word": "duty",\n      "count": 16\n    },\n    {\n      "word": "enforcement",\n      "count": 16\n    },\n    {\n      "word": "exhibition",\n      "count": 16\n    },\n    {\n      "word": "feast",\n      "count": 16\n    },\n    {\n      "word": "fight",\n      "count": 16\n    },\n    {\n      "word": "fighting",\n      "count": 16\n    },\n    {\n      "word": "flea",\n      "count": 16\n    },\n    {\n      "word": "foodstuff",\n      "count": 16\n    },\n    {\n      "word": "fudge",\n      "count": 16\n    },\n    {\n      "word": "gallon",\n      "count": 16\n    },\n    {\n      "word": "giraffee",\n      "count": 16\n    },\n    {\n      "word": "gnome",\n      "count": 16\n    },\n    {\n      "word": "gray",\n      "count": 16\n    },\n    {\n      "word": "greeting",\n      "count": 16\n    },\n    {\n      "word": "grin",\n      "count": 16\n    },\n    {\n      "word": "grind",\n      "count": 16\n    },\n    {\n      "word": "handful",\n      "count": 16\n    },\n    {\n      "word": "haul",\n      "count": 16\n    },\n    {\n      "word": "health",\n      "count": 16\n    },\n    {\n      "word": "hoof",\n      "count": 16\n    },\n    {\n      "word": "ii",\n      "count": 16\n    },\n    {\n      "word": "inclosure",\n      "count": 16\n    },\n    {\n      "word": "junction",\n      "count": 16\n    },\n    {\n      "word": "lights",\n      "count": 16\n    },\n    {\n      "word": "lumber",\n      "count": 16\n    },\n    {\n      "word": "marinara",\n      "count": 16\n    },\n    {\n      "word": "miss",\n      "count": 16\n    },\n    {\n      "word": "mountaintop",\n      "count": 16\n    },\n    {\n      "word": "mousepad",\n      "count": 16\n    },\n    {\n      "word": "moving",\n      "count": 16\n    },\n    {\n      "word": "outfielder",\n      "count": 16\n    },\n    {\n      "word": "outskirt",\n      "count": 16\n    },\n    {\n      "word": "overhang",\n      "count": 16\n    },\n    {\n      "word": "passport",\n      "count": 16\n    },\n    {\n      "word": "pearl",\n      "count": 16\n    },\n    {\n      "word": "period",\n      "count": 16\n    },\n    {\n      "word": "plot",\n      "count": 16\n    },\n    {\n      "word": "pm",\n      "count": 16\n    },\n    {\n      "word": "puff",\n      "count": 16\n    },\n    {\n      "word": "roasting",\n      "count": 16\n    },\n    {\n      "word": "sex",\n      "count": 16\n    },\n    {\n      "word": "shelve",\n      "count": 16\n    },\n    {\n      "word": "spa",\n      "count": 16\n    },\n    {\n      "word": "sparrow",\n      "count": 16\n    },\n    {\n      "word": "specie",\n      "count": 16\n    },\n    {\n      "word": "squadron",\n      "count": 16\n    },\n    {\n      "word": "starting",\n      "count": 16\n    },\n    {\n      "word": "stationary",\n      "count": 16\n    },\n    {\n      "word": "stucco",\n      "count": 16\n    },\n    {\n      "word": "surround",\n      "count": 16\n    },\n    {\n      "word": "tattooed",\n      "count": 16\n    },\n    {\n      "word": "taxis",\n      "count": 16\n    },\n    {\n      "word": "there",\n      "count": 16\n    },\n    {\n      "word": "trimming",\n      "count": 16\n    },\n    {\n      "word": "typewriter",\n      "count": 16\n    },\n    {\n      "word": "u",\n      "count": 16\n    },\n    {\n      "word": "ump",\n      "count": 16\n    },\n    {\n      "word": "upright",\n      "count": 16\n    },\n    {\n      "word": "valve",\n      "count": 16\n    },\n    {\n      "word": "volkswagon",\n      "count": 16\n    },\n    {\n      "word": "waitress",\n      "count": 16\n    },\n    {\n      "word": "wars",\n      "count": 16\n    },\n    {\n      "word": "waterside",\n      "count": 16\n    },\n    {\n      "word": "well",\n      "count": 16\n    },\n    {\n      "word": "wiener",\n      "count": 16\n    },\n    {\n      "word": "windsurfing",\n      "count": 16\n    },\n    {\n      "word": "yamaha",\n      "count": 16\n    },\n    {\n      "word": "aa",\n      "count": 15\n    },\n    {\n      "word": "am",\n      "count": 15\n    },\n    {\n      "word": "ana",\n      "count": 15\n    },\n    {\n      "word": "bang",\n      "count": 15\n    },\n    {\n      "word": "beautiful",\n      "count": 15\n    },\n    {\n      "word": "birds",\n      "count": 15\n    },\n    {\n      "word": "boater",\n      "count": 15\n    },\n    {\n      "word": "breeze",\n      "count": 15\n    },\n    {\n      "word": "ceramic",\n      "count": 15\n    },\n    {\n      "word": "chevrolet",\n      "count": 15\n    },\n    {\n      "word": "crash",\n      "count": 15\n    },\n    {\n      "word": "crispy",\n      "count": 15\n    },\n    {\n      "word": "crossroad",\n      "count": 15\n    },\n    {\n      "word": "curry",\n      "count": 15\n    },\n    {\n      "word": "dandelion",\n      "count": 15\n    },\n    {\n      "word": "dealership",\n      "count": 15\n    },\n    {\n      "word": "derby",\n      "count": 15\n    },\n    {\n      "word": "disrepair",\n      "count": 15\n    },\n    {\n      "word": "drift",\n      "count": 15\n    },\n    {\n      "word": "duct",\n      "count": 15\n    },\n    {\n      "word": "dumpling",\n      "count": 15\n    },\n    {\n      "word": "element",\n      "count": 15\n    },\n    {\n      "word": "estate",\n      "count": 15\n    },\n    {\n      "word": "express",\n      "count": 15\n    },\n    {\n      "word": "extinguisher",\n      "count": 15\n    },\n    {\n      "word": "flashlight",\n      "count": 15\n    },\n    {\n      "word": "floss",\n      "count": 15\n    },\n    {\n      "word": "font",\n      "count": 15\n    },\n    {\n      "word": "freeze",\n      "count": 15\n    },\n    {\n      "word": "french",\n      "count": 15\n    },\n    {\n      "word": "girafee",\n      "count": 15\n    },\n    {\n      "word": "griddle",\n      "count": 15\n    },\n    {\n      "word": "him",\n      "count": 15\n    },\n    {\n      "word": "hydrogen",\n      "count": 15\n    },\n    {\n      "word": "imac",\n      "count": 15\n    },\n    {\n      "word": "inn",\n      "count": 15\n    },\n    {\n      "word": "john",\n      "count": 15\n    },\n    {\n      "word": "laughing",\n      "count": 15\n    },\n    {\n      "word": "len",\n      "count": 15\n    },\n    {\n      "word": "limo",\n      "count": 15\n    },\n    {\n      "word": "m",\n      "count": 15\n    },\n    {\n      "word": "mayo",\n      "count": 15\n    },\n    {\n      "word": "meatloaf",\n      "count": 15\n    },\n    {\n      "word": "mit",\n      "count": 15\n    },\n    {\n      "word": "mitten",\n      "count": 15\n    },\n    {\n      "word": "mustang",\n      "count": 15\n    },\n    {\n      "word": "nunchuck",\n      "count": 15\n    },\n    {\n      "word": "nursery",\n      "count": 15\n    },\n    {\n      "word": "origami",\n      "count": 15\n    },\n    {\n      "word": "patchwork",\n      "count": 15\n    },\n    {\n      "word": "penn",\n      "count": 15\n    },\n    {\n      "word": "popsicle",\n      "count": 15\n    },\n    {\n      "word": "proximity",\n      "count": 15\n    },\n    {\n      "word": "punk",\n      "count": 15\n    },\n    {\n      "word": "purpose",\n      "count": 15\n    },\n    {\n      "word": "quality",\n      "count": 15\n    },\n    {\n      "word": "quesadilla",\n      "count": 15\n    },\n    {\n      "word": "recorder",\n      "count": 15\n    },\n    {\n      "word": "reed",\n      "count": 15\n    },\n    {\n      "word": "rhinoceros",\n      "count": 15\n    },\n    {\n      "word": "rodent",\n      "count": 15\n    },\n    {\n      "word": "rustic",\n      "count": 15\n    },\n    {\n      "word": "sedan",\n      "count": 15\n    },\n    {\n      "word": "sheepdog",\n      "count": 15\n    },\n    {\n      "word": "shelving",\n      "count": 15\n    },\n    {\n      "word": "shit",\n      "count": 15\n    },\n    {\n      "word": "sling",\n      "count": 15\n    },\n    {\n      "word": "sniff",\n      "count": 15\n    },\n    {\n      "word": "spotlight",\n      "count": 15\n    },\n    {\n      "word": "steal",\n      "count": 15\n    },\n    {\n      "word": "surboard",\n      "count": 15\n    },\n    {\n      "word": "swirl",\n      "count": 15\n    },\n    {\n      "word": "take",\n      "count": 15\n    },\n    {\n      "word": "territory",\n      "count": 15\n    },\n    {\n      "word": "thames",\n      "count": 15\n    },\n    {\n      "word": "thee",\n      "count": 15\n    },\n    {\n      "word": "topper",\n      "count": 15\n    },\n    {\n      "word": "torso",\n      "count": 15\n    },\n    {\n      "word": "toyota",\n      "count": 15\n    },\n    {\n      "word": "tugboat",\n      "count": 15\n    },\n    {\n      "word": "tux",\n      "count": 15\n    },\n    {\n      "word": "ultimate",\n      "count": 15\n    },\n    {\n      "word": "usb",\n      "count": 15\n    },\n    {\n      "word": "whale",\n      "count": 15\n    },\n    {\n      "word": "album",\n      "count": 14\n    },\n    {\n      "word": "alter",\n      "count": 14\n    },\n    {\n      "word": "anticipation",\n      "count": 14\n    },\n    {\n      "word": "arcade",\n      "count": 14\n    },\n    {\n      "word": "artichoke",\n      "count": 14\n    },\n    {\n      "word": "baguette",\n      "count": 14\n    },\n    {\n      "word": "bandage",\n      "count": 14\n    },\n    {\n      "word": "baseline",\n      "count": 14\n    },\n    {\n      "word": "birdcage",\n      "count": 14\n    },\n    {\n      "word": "boiling",\n      "count": 14\n    },\n    {\n      "word": "broccolli",\n      "count": 14\n    },\n    {\n      "word": "buddha",\n      "count": 14\n    },\n    {\n      "word": "bunt",\n      "count": 14\n    },\n    {\n      "word": "c",\n      "count": 14\n    },\n    {\n      "word": "calendar",\n      "count": 14\n    },\n    {\n      "word": "cay",\n      "count": 14\n    },\n    {\n      "word": "cleaner",\n      "count": 14\n    },\n    {\n      "word": "clementine",\n      "count": 14\n    },\n    {\n      "word": "coo",\n      "count": 14\n    },\n    {\n      "word": "custard",\n      "count": 14\n    },\n    {\n      "word": "dachshund",\n      "count": 14\n    },\n    {\n      "word": "dinette",\n      "count": 14\n    },\n    {\n      "word": "doggie",\n      "count": 14\n    },\n    {\n      "word": "dread",\n      "count": 14\n    },\n    {\n      "word": "egret",\n      "count": 14\n    },\n    {\n      "word": "empire",\n      "count": 14\n    },\n    {\n      "word": "escape",\n      "count": 14\n    },\n    {\n      "word": "feel",\n      "count": 14\n    },\n    {\n      "word": "filter",\n      "count": 14\n    },\n    {\n      "word": "flatbed",\n      "count": 14\n    },\n    {\n      "word": "flickr",\n      "count": 14\n    },\n    {\n      "word": "flute",\n      "count": 14\n    },\n    {\n      "word": "geisha",\n      "count": 14\n    },\n    {\n      "word": "giants",\n      "count": 14\n    },\n    {\n      "word": "gondola",\n      "count": 14\n    },\n    {\n      "word": "grand",\n      "count": 14\n    },\n    {\n      "word": "grit",\n      "count": 14\n    },\n    {\n      "word": "i",\n      "count": 14\n    },\n    {\n      "word": "ivory",\n      "count": 14\n    },\n    {\n      "word": "jackson",\n      "count": 14\n    },\n    {\n      "word": "kicking",\n      "count": 14\n    },\n    {\n      "word": "launch",\n      "count": 14\n    },\n    {\n      "word": "longboard",\n      "count": 14\n    },\n    {\n      "word": "looker",\n      "count": 14\n    },\n    {\n      "word": "lufthansa",\n      "count": 14\n    },\n    {\n      "word": "mansion",\n      "count": 14\n    },\n    {\n      "word": "mechanic",\n      "count": 14\n    },\n    {\n      "word": "messy",\n      "count": 14\n    },\n    {\n      "word": "milkshake",\n      "count": 14\n    },\n    {\n      "word": "necessity",\n      "count": 14\n    },\n    {\n      "word": "ornate",\n      "count": 14\n    },\n    {\n      "word": "overlook",\n      "count": 14\n    },\n    {\n      "word": "paperback",\n      "count": 14\n    },\n    {\n      "word": "parliament",\n      "count": 14\n    },\n    {\n      "word": "polar",\n      "count": 14\n    },\n    {\n      "word": "pomegranate",\n      "count": 14\n    },\n    {\n      "word": "powdery",\n      "count": 14\n    },\n    {\n      "word": "professional",\n      "count": 14\n    },\n    {\n      "word": "property",\n      "count": 14\n    },\n    {\n      "word": "protector",\n      "count": 14\n    },\n    {\n      "word": "pylon",\n      "count": 14\n    },\n    {\n      "word": "restaraunt",\n      "count": 14\n    },\n    {\n      "word": "rower",\n      "count": 14\n    },\n    {\n      "word": "rowing",\n      "count": 14\n    },\n    {\n      "word": "satchel",\n      "count": 14\n    },\n    {\n      "word": "satin",\n      "count": 14\n    },\n    {\n      "word": "scallop",\n      "count": 14\n    },\n    {\n      "word": "segment",\n      "count": 14\n    },\n    {\n      "word": "shrine",\n      "count": 14\n    },\n    {\n      "word": "skis",\n      "count": 14\n    },\n    {\n      "word": "sleeping",\n      "count": 14\n    },\n    {\n      "word": "slot",\n      "count": 14\n    },\n    {\n      "word": "spanish",\n      "count": 14\n    },\n    {\n      "word": "spiky",\n      "count": 14\n    },\n    {\n      "word": "stake",\n      "count": 14\n    },\n    {\n      "word": "stilt",\n      "count": 14\n    },\n    {\n      "word": "strand",\n      "count": 14\n    },\n    {\n      "word": "suburb",\n      "count": 14\n    },\n    {\n      "word": "surrounded",\n      "count": 14\n    },\n    {\n      "word": "tapestry",\n      "count": 14\n    },\n    {\n      "word": "tart",\n      "count": 14\n    },\n    {\n      "word": "telescope",\n      "count": 14\n    },\n    {\n      "word": "test",\n      "count": 14\n    },\n    {\n      "word": "throwing",\n      "count": 14\n    },\n    {\n      "word": "thru",\n      "count": 14\n    },\n    {\n      "word": "trumpet",\n      "count": 14\n    },\n    {\n      "word": "tupperware",\n      "count": 14\n    },\n    {\n      "word": "vacuum",\n      "count": 14\n    },\n    {\n      "word": "veil",\n      "count": 14\n    },\n    {\n      "word": "vein",\n      "count": 14\n    },\n    {\n      "word": "vitamin",\n      "count": 14\n    },\n    {\n      "word": "volunteer",\n      "count": 14\n    },\n    {\n      "word": "wheelbarrow",\n      "count": 14\n    },\n    {\n      "word": "widow",\n      "count": 14\n    },\n    {\n      "word": "winner",\n      "count": 14\n    },\n    {\n      "word": "witch",\n      "count": 14\n    },\n    {\n      "word": "woods",\n      "count": 14\n    },\n    {\n      "word": "wrestler",\n      "count": 14\n    },\n    {\n      "word": "wtih",\n      "count": 14\n    },\n    {\n      "word": "affection",\n      "count": 13\n    },\n    {\n      "word": "amp",\n      "count": 13\n    },\n    {\n      "word": "arid",\n      "count": 13\n    },\n    {\n      "word": "asian",\n      "count": 13\n    },\n    {\n      "word": "bald",\n      "count": 13\n    },\n    {\n      "word": "baltimore",\n      "count": 13\n    },\n    {\n      "word": "bomb",\n      "count": 13\n    },\n    {\n      "word": "bovine",\n      "count": 13\n    },\n    {\n      "word": "bristle",\n      "count": 13\n    },\n    {\n      "word": "bulldozer",\n      "count": 13\n    },\n    {\n      "word": "bulletin",\n      "count": 13\n    },\n    {\n      "word": "buy",\n      "count": 13\n    },\n    {\n      "word": "charge",\n      "count": 13\n    },\n    {\n      "word": "climber",\n      "count": 13\n    },\n    {\n      "word": "clothesline",\n      "count": 13\n    },\n    {\n      "word": "comfort",\n      "count": 13\n    },\n    {\n      "word": "cranberry",\n      "count": 13\n    },\n    {\n      "word": "cruiser",\n      "count": 13\n    },\n    {\n      "word": "dodge",\n      "count": 13\n    },\n    {\n      "word": "dollop",\n      "count": 13\n    },\n    {\n      "word": "elvis",\n      "count": 13\n    },\n    {\n      "word": "entryway",\n      "count": 13\n    },\n    {\n      "word": "falcon",\n      "count": 13\n    },\n    {\n      "word": "ferris",\n      "count": 13\n    },\n    {\n      "word": "fielder",\n      "count": 13\n    },\n    {\n      "word": "florida",\n      "count": 13\n    },\n    {\n      "word": "footprint",\n      "count": 13\n    },\n    {\n      "word": "fowl",\n      "count": 13\n    },\n    {\n      "word": "foyer",\n      "count": 13\n    },\n    {\n      "word": "frisbie",\n      "count": 13\n    },\n    {\n      "word": "garland",\n      "count": 13\n    },\n    {\n      "word": "gliding",\n      "count": 13\n    },\n    {\n      "word": "go",\n      "count": 13\n    },\n    {\n      "word": "greyhound",\n      "count": 13\n    },\n    {\n      "word": "heating",\n      "count": 13\n    },\n    {\n      "word": "high",\n      "count": 13\n    },\n    {\n      "word": "hike",\n      "count": 13\n    },\n    {\n      "word": "idea",\n      "count": 13\n    },\n    {\n      "word": "identification",\n      "count": 13\n    },\n    {\n      "word": "jetty",\n      "count": 13\n    },\n    {\n      "word": "kept",\n      "count": 13\n    },\n    {\n      "word": "la",\n      "count": 13\n    },\n    {\n      "word": "lining",\n      "count": 13\n    },\n    {\n      "word": "linoleum",\n      "count": 13\n    },\n    {\n      "word": "makeshift",\n      "count": 13\n    },\n    {\n      "word": "mare",\n      "count": 13\n    },\n    {\n      "word": "marquee",\n      "count": 13\n    },\n    {\n      "word": "miniature",\n      "count": 13\n    },\n    {\n      "word": "motorola",\n      "count": 13\n    },\n    {\n      "word": "obelisk",\n      "count": 13\n    },\n    {\n      "word": "observation",\n      "count": 13\n    },\n    {\n      "word": "olympic",\n      "count": 13\n    },\n    {\n      "word": "outcropping",\n      "count": 13\n    },\n    {\n      "word": "paddling",\n      "count": 13\n    },\n    {\n      "word": "panini",\n      "count": 13\n    },\n    {\n      "word": "panty",\n      "count": 13\n    },\n    {\n      "word": "paraphernalia",\n      "count": 13\n    },\n    {\n      "word": "passage",\n      "count": 13\n    },\n    {\n      "word": "paul",\n      "count": 13\n    },\n    {\n      "word": "peopel",\n      "count": 13\n    },\n    {\n      "word": "possession",\n      "count": 13\n    },\n    {\n      "word": "prison",\n      "count": 13\n    },\n    {\n      "word": "quantity",\n      "count": 13\n    },\n    {\n      "word": "quote",\n      "count": 13\n    },\n    {\n      "word": "reclining",\n      "count": 13\n    },\n    {\n      "word": "refreshment",\n      "count": 13\n    },\n    {\n      "word": "remodeling",\n      "count": 13\n    },\n    {\n      "word": "saint",\n      "count": 13\n    },\n    {\n      "word": "science",\n      "count": 13\n    },\n    {\n      "word": "seaside",\n      "count": 13\n    },\n    {\n      "word": "second",\n      "count": 13\n    },\n    {\n      "word": "seminar",\n      "count": 13\n    },\n    {\n      "word": "shampoo",\n      "count": 13\n    },\n    {\n      "word": "share",\n      "count": 13\n    },\n    {\n      "word": "shield",\n      "count": 13\n    },\n    {\n      "word": "showcase",\n      "count": 13\n    },\n    {\n      "word": "south",\n      "count": 13\n    },\n    {\n      "word": "sprint",\n      "count": 13\n    },\n    {\n      "word": "surprise",\n      "count": 13\n    },\n    {\n      "word": "suspension",\n      "count": 13\n    },\n    {\n      "word": "takeout",\n      "count": 13\n    },\n    {\n      "word": "tear",\n      "count": 13\n    },\n    {\n      "word": "technique",\n      "count": 13\n    },\n    {\n      "word": "thai",\n      "count": 13\n    },\n    {\n      "word": "thermo",\n      "count": 13\n    },\n    {\n      "word": "thought",\n      "count": 13\n    },\n    {\n      "word": "times",\n      "count": 13\n    },\n    {\n      "word": "trestle",\n      "count": 13\n    },\n    {\n      "word": "tundra",\n      "count": 13\n    },\n    {\n      "word": "turning",\n      "count": 13\n    },\n    {\n      "word": "twp",\n      "count": 13\n    },\n    {\n      "word": "vicinity",\n      "count": 13\n    },\n    {\n      "word": "wander",\n      "count": 13\n    },\n    {\n      "word": "whisk",\n      "count": 13\n    },\n    {\n      "word": "woodland",\n      "count": 13\n    },\n    {\n      "word": "worm",\n      "count": 13\n    },\n    {\n      "word": "wreck",\n      "count": 13\n    },\n    {\n      "word": "accordion",\n      "count": 12\n    },\n    {\n      "word": "afro",\n      "count": 12\n    },\n    {\n      "word": "al",\n      "count": 12\n    },\n    {\n      "word": "angels",\n      "count": 12\n    },\n    {\n      "word": "artifact",\n      "count": 12\n    },\n    {\n      "word": "ban",\n      "count": 12\n    },\n    {\n      "word": "baord",\n      "count": 12\n    },\n    {\n      "word": "barbie",\n      "count": 12\n    },\n    {\n      "word": "barstool",\n      "count": 12\n    },\n    {\n      "word": "biking",\n      "count": 12\n    },\n    {\n      "word": "buddy",\n      "count": 12\n    },\n    {\n      "word": "build",\n      "count": 12\n    },\n    {\n      "word": "capture",\n      "count": 12\n    },\n    {\n      "word": "carcass",\n      "count": 12\n    },\n    {\n      "word": "casino",\n      "count": 12\n    },\n    {\n      "word": "chat",\n      "count": 12\n    },\n    {\n      "word": "choo",\n      "count": 12\n    },\n    {\n      "word": "chug",\n      "count": 12\n    },\n    {\n      "word": "clapping",\n      "count": 12\n    },\n    {\n      "word": "clean",\n      "count": 12\n    },\n    {\n      "word": "clear",\n      "count": 12\n    },\n    {\n      "word": "clipboard",\n      "count": 12\n    },\n    {\n      "word": "colander",\n      "count": 12\n    },\n    {\n      "word": "colleague",\n      "count": 12\n    },\n    {\n      "word": "contrast",\n      "count": 12\n    },\n    {\n      "word": "danish",\n      "count": 12\n    },\n    {\n      "word": "deal",\n      "count": 12\n    },\n    {\n      "word": "deco",\n      "count": 12\n    },\n    {\n      "word": "demon",\n      "count": 12\n    },\n    {\n      "word": "departure",\n      "count": 12\n    },\n    {\n      "word": "designer",\n      "count": 12\n    },\n    {\n      "word": "dinnerware",\n      "count": 12\n    },\n    {\n      "word": "do",\n      "count": 12\n    },\n    {\n      "word": "dreadlock",\n      "count": 12\n    },\n    {\n      "word": "dug",\n      "count": 12\n    },\n    {\n      "word": "dvds",\n      "count": 12\n    },\n    {\n      "word": "fifth",\n      "count": 12\n    },\n    {\n      "word": "fort",\n      "count": 12\n    },\n    {\n      "word": "gardening",\n      "count": 12\n    },\n    {\n      "word": "grade",\n      "count": 12\n    },\n    {\n      "word": "grater",\n      "count": 12\n    },\n    {\n      "word": "grease",\n      "count": 12\n    },\n    {\n      "word": "greasy",\n      "count": 12\n    },\n    {\n      "word": "grid",\n      "count": 12\n    },\n    {\n      "word": "guardrail",\n      "count": 12\n    },\n    {\n      "word": "hen",\n      "count": 12\n    },\n    {\n      "word": "houseboat",\n      "count": 12\n    },\n    {\n      "word": "houseplant",\n      "count": 12\n    },\n    {\n      "word": "installation",\n      "count": 12\n    },\n    {\n      "word": "international",\n      "count": 12\n    },\n    {\n      "word": "italy",\n      "count": 12\n    },\n    {\n      "word": "knoll",\n      "count": 12\n    },\n    {\n      "word": "koala",\n      "count": 12\n    },\n    {\n      "word": "kraut",\n      "count": 12\n    },\n    {\n      "word": "ladybug",\n      "count": 12\n    },\n    {\n      "word": "lo",\n      "count": 12\n    },\n    {\n      "word": "louis",\n      "count": 12\n    },\n    {\n      "word": "low",\n      "count": 12\n    },\n    {\n      "word": "lucky",\n      "count": 12\n    },\n    {\n      "word": "luncheon",\n      "count": 12\n    },\n    {\n      "word": "mary",\n      "count": 12\n    },\n    {\n      "word": "michigan",\n      "count": 12\n    },\n    {\n      "word": "mouthwash",\n      "count": 12\n    },\n    {\n      "word": "munching",\n      "count": 12\n    },\n    {\n      "word": "non",\n      "count": 12\n    },\n    {\n      "word": "oats",\n      "count": 12\n    },\n    {\n      "word": "observer",\n      "count": 12\n    },\n    {\n      "word": "operator",\n      "count": 12\n    },\n    {\n      "word": "orioles",\n      "count": 12\n    },\n    {\n      "word": "overripe",\n      "count": 12\n    },\n    {\n      "word": "packing",\n      "count": 12\n    },\n    {\n      "word": "pastrami",\n      "count": 12\n    },\n    {\n      "word": "pate",\n      "count": 12\n    },\n    {\n      "word": "peg",\n      "count": 12\n    },\n    {\n      "word": "peson",\n      "count": 12\n    },\n    {\n      "word": "plywood",\n      "count": 12\n    },\n    {\n      "word": "pom",\n      "count": 12\n    },\n    {\n      "word": "poop",\n      "count": 12\n    },\n    {\n      "word": "popcorn",\n      "count": 12\n    },\n    {\n      "word": "prince",\n      "count": 12\n    },\n    {\n      "word": "protein",\n      "count": 12\n    },\n    {\n      "word": "question",\n      "count": 12\n    },\n    {\n      "word": "rafter",\n      "count": 12\n    },\n    {\n      "word": "ranger",\n      "count": 12\n    },\n    {\n      "word": "renaissance",\n      "count": 12\n    },\n    {\n      "word": "rickshaw",\n      "count": 12\n    },\n    {\n      "word": "rode",\n      "count": 12\n    },\n    {\n      "word": "rule",\n      "count": 12\n    },\n    {\n      "word": "salami",\n      "count": 12\n    },\n    {\n      "word": "scanner",\n      "count": 12\n    },\n    {\n      "word": "scarecrow",\n      "count": 12\n    },\n    {\n      "word": "scone",\n      "count": 12\n    },\n    {\n      "word": "scratch",\n      "count": 12\n    },\n    {\n      "word": "scuba",\n      "count": 12\n    },\n    {\n      "word": "siding",\n      "count": 12\n    },\n    {\n      "word": "slate",\n      "count": 12\n    },\n    {\n      "word": "smiling",\n      "count": 12\n    },\n    {\n      "word": "smirk",\n      "count": 12\n    },\n    {\n      "word": "snail",\n      "count": 12\n    },\n    {\n      "word": "snuggle",\n      "count": 12\n    },\n    {\n      "word": "sombrero",\n      "count": 12\n    },\n    {\n      "word": "spoonful",\n      "count": 12\n    },\n    {\n      "word": "sports",\n      "count": 12\n    },\n    {\n      "word": "sprayer",\n      "count": 12\n    },\n    {\n      "word": "sprinkler",\n      "count": 12\n    },\n    {\n      "word": "steet",\n      "count": 12\n    },\n    {\n      "word": "stride",\n      "count": 12\n    },\n    {\n      "word": "tab",\n      "count": 12\n    },\n    {\n      "word": "tandem",\n      "count": 12\n    },\n    {\n      "word": "toiler",\n      "count": 12\n    },\n    {\n      "word": "treed",\n      "count": 12\n    },\n    {\n      "word": "trooper",\n      "count": 12\n    },\n    {\n      "word": "upside",\n      "count": 12\n    },\n    {\n      "word": "valentine",\n      "count": 12\n    },\n    {\n      "word": "vancouver",\n      "count": 12\n    },\n    {\n      "word": "variation",\n      "count": 12\n    },\n    {\n      "word": "vinyl",\n      "count": 12\n    },\n    {\n      "word": "wand",\n      "count": 12\n    },\n    {\n      "word": "wth",\n      "count": 12\n    },\n    {\n      "word": "y",\n      "count": 12\n    },\n    {\n      "word": "young",\n      "count": 12\n    },\n    {\n      "word": "zipper",\n      "count": 12\n    },\n    {\n      "word": "zoom",\n      "count": 12\n    },\n    {\n      "word": "adobe",\n      "count": 11\n    },\n    {\n      "word": "advantage",\n      "count": 11\n    },\n    {\n      "word": "algae",\n      "count": 11\n    },\n    {\n      "word": "ally",\n      "count": 11\n    },\n    {\n      "word": "anchovy",\n      "count": 11\n    },\n    {\n      "word": "apricot",\n      "count": 11\n    },\n    {\n      "word": "arabic",\n      "count": 11\n    },\n    {\n      "word": "arizona",\n      "count": 11\n    },\n    {\n      "word": "armrest",\n      "count": 11\n    },\n    {\n      "word": "arrival",\n      "count": 11\n    },\n    {\n      "word": "arugula",\n      "count": 11\n    },\n    {\n      "word": "ascent",\n      "count": 11\n    },\n    {\n      "word": "atmosphere",\n      "count": 11\n    },\n    {\n      "word": "atrium",\n      "count": 11\n    },\n    {\n      "word": "auction",\n      "count": 11\n    },\n    {\n      "word": "barbwire",\n      "count": 11\n    },\n    {\n      "word": "beast",\n      "count": 11\n    },\n    {\n      "word": "best",\n      "count": 11\n    },\n    {\n      "word": "blacktop",\n      "count": 11\n    },\n    {\n      "word": "blond",\n      "count": 11\n    },\n    {\n      "word": "bookbag",\n      "count": 11\n    },\n    {\n      "word": "books",\n      "count": 11\n    },\n    {\n      "word": "booster",\n      "count": 11\n    },\n    {\n      "word": "bowel",\n      "count": 11\n    },\n    {\n      "word": "braid",\n      "count": 11\n    },\n    {\n      "word": "brow",\n      "count": 11\n    },\n    {\n      "word": "campground",\n      "count": 11\n    },\n    {\n      "word": "campsite",\n      "count": 11\n    },\n    {\n      "word": "cardigan",\n      "count": 11\n    },\n    {\n      "word": "casing",\n      "count": 11\n    },\n    {\n      "word": "catering",\n      "count": 11\n    },\n    {\n      "word": "chairs",\n      "count": 11\n    },\n    {\n      "word": "chasing",\n      "count": 11\n    },\n    {\n      "word": "chuck",\n      "count": 11\n    },\n    {\n      "word": "cilantro",\n      "count": 11\n    },\n    {\n      "word": "clipper",\n      "count": 11\n    },\n    {\n      "word": "clover",\n      "count": 11\n    },\n    {\n      "word": "colgate",\n      "count": 11\n    },\n    {\n      "word": "compute",\n      "count": 11\n    },\n    {\n      "word": "concession",\n      "count": 11\n    },\n    {\n      "word": "convertible",\n      "count": 11\n    },\n    {\n      "word": "cosmetic",\n      "count": 11\n    },\n    {\n      "word": "courthouse",\n      "count": 11\n    },\n    {\n      "word": "cove",\n      "count": 11\n    },\n    {\n      "word": "crotch",\n      "count": 11\n    },\n    {\n      "word": "cubby",\n      "count": 11\n    },\n    {\n      "word": "darth",\n      "count": 11\n    },\n    {\n      "word": "death",\n      "count": 11\n    },\n    {\n      "word": "doughnuts",\n      "count": 11\n    },\n    {\n      "word": "draw",\n      "count": 11\n    },\n    {\n      "word": "dressed",\n      "count": 11\n    },\n    {\n      "word": "drill",\n      "count": 11\n    },\n    {\n      "word": "dwelling",\n      "count": 11\n    },\n    {\n      "word": "effort",\n      "count": 11\n    },\n    {\n      "word": "elizabeth",\n      "count": 11\n    },\n    {\n      "word": "elk",\n      "count": 11\n    },\n    {\n      "word": "embrace",\n      "count": 11\n    },\n    {\n      "word": "extension",\n      "count": 11\n    },\n    {\n      "word": "farmland",\n      "count": 11\n    },\n    {\n      "word": "fin",\n      "count": 11\n    },\n    {\n      "word": "five",\n      "count": 11\n    },\n    {\n      "word": "flannel",\n      "count": 11\n    },\n    {\n      "word": "flask",\n      "count": 11\n    },\n    {\n      "word": "fluid",\n      "count": 11\n    },\n    {\n      "word": "frown",\n      "count": 11\n    },\n    {\n      "word": "frying",\n      "count": 11\n    },\n    {\n      "word": "games",\n      "count": 11\n    },\n    {\n      "word": "geese",\n      "count": 11\n    },\n    {\n      "word": "gooey",\n      "count": 11\n    },\n    {\n      "word": "graham",\n      "count": 11\n    },\n    {\n      "word": "hot",\n      "count": 11\n    },\n    {\n      "word": "icecream",\n      "count": 11\n    },\n    {\n      "word": "ina",\n      "count": 11\n    },\n    {\n      "word": "jones",\n      "count": 11\n    },\n    {\n      "word": "jumbo",\n      "count": 11\n    },\n    {\n      "word": "kickstand",\n      "count": 11\n    },\n    {\n      "word": "leak",\n      "count": 11\n    },\n    {\n      "word": "lick",\n      "count": 11\n    },\n    {\n      "word": "lightning",\n      "count": 11\n    },\n    {\n      "word": "lollipop",\n      "count": 11\n    },\n    {\n      "word": "major",\n      "count": 11\n    },\n    {\n      "word": "mallet",\n      "count": 11\n    },\n    {\n      "word": "mandarin",\n      "count": 11\n    },\n    {\n      "word": "manhole",\n      "count": 11\n    },\n    {\n      "word": "measure",\n      "count": 11\n    },\n    {\n      "word": "meaty",\n      "count": 11\n    },\n    {\n      "word": "meet",\n      "count": 11\n    },\n    {\n      "word": "memory",\n      "count": 11\n    },\n    {\n      "word": "mlb",\n      "count": 11\n    },\n    {\n      "word": "motorcycles",\n      "count": 11\n    },\n    {\n      "word": "multicolor",\n      "count": 11\n    },\n    {\n      "word": "munch",\n      "count": 11\n    },\n    {\n      "word": "nerd",\n      "count": 11\n    },\n    {\n      "word": "netbook",\n      "count": 11\n    },\n    {\n      "word": "noon",\n      "count": 11\n    },\n    {\n      "word": "nozzle",\n      "count": 11\n    },\n    {\n      "word": "oakland",\n      "count": 11\n    },\n    {\n      "word": "occupant",\n      "count": 11\n    },\n    {\n      "word": "opener",\n      "count": 11\n    },\n    {\n      "word": "organization",\n      "count": 11\n    },\n    {\n      "word": "outline",\n      "count": 11\n    },\n    {\n      "word": "pain",\n      "count": 11\n    },\n    {\n      "word": "pasty",\n      "count": 11\n    },\n    {\n      "word": "payphone",\n      "count": 11\n    },\n    {\n      "word": "pee",\n      "count": 11\n    },\n    {\n      "word": "perimeter",\n      "count": 11\n    },\n    {\n      "word": "phrase",\n      "count": 11\n    },\n    {\n      "word": "pictures",\n      "count": 11\n    },\n    {\n      "word": "plater",\n      "count": 11\n    },\n    {\n      "word": "pride",\n      "count": 11\n    },\n    {\n      "word": "prosciutto",\n      "count": 11\n    },\n    {\n      "word": "protestor",\n      "count": 11\n    },\n    {\n      "word": "pup",\n      "count": 11\n    },\n    {\n      "word": "ravioli",\n      "count": 11\n    },\n    {\n      "word": "rent",\n      "count": 11\n    },\n    {\n      "word": "resturant",\n      "count": 11\n    },\n    {\n      "word": "rind",\n      "count": 11\n    },\n    {\n      "word": "riverside",\n      "count": 11\n    },\n    {\n      "word": "rue",\n      "count": 11\n    },\n    {\n      "word": "sandy",\n      "count": 11\n    },\n    {\n      "word": "sanitizer",\n      "count": 11\n    },\n    {\n      "word": "saw",\n      "count": 11\n    },\n    {\n      "word": "score",\n      "count": 11\n    },\n    {\n      "word": "seattle",\n      "count": 11\n    },\n    {\n      "word": "silk",\n      "count": 11\n    },\n    {\n      "word": "slip",\n      "count": 11\n    },\n    {\n      "word": "snowshoe",\n      "count": 11\n    },\n    {\n      "word": "span",\n      "count": 11\n    },\n    {\n      "word": "spaniel",\n      "count": 11\n    },\n    {\n      "word": "special",\n      "count": 11\n    },\n    {\n      "word": "spirit",\n      "count": 11\n    },\n    {\n      "word": "starbuck",\n      "count": 11\n    },\n    {\n      "word": "steaming",\n      "count": 11\n    },\n    {\n      "word": "stoop",\n      "count": 11\n    },\n    {\n      "word": "subject",\n      "count": 11\n    },\n    {\n      "word": "sustenance",\n      "count": 11\n    },\n    {\n      "word": "sweeper",\n      "count": 11\n    },\n    {\n      "word": "tailgate",\n      "count": 11\n    },\n    {\n      "word": "talking",\n      "count": 11\n    },\n    {\n      "word": "taxidermy",\n      "count": 11\n    },\n    {\n      "word": "teal",\n      "count": 11\n    },\n    {\n      "word": "tether",\n      "count": 11\n    },\n    {\n      "word": "ther",\n      "count": 11\n    },\n    {\n      "word": "thigh",\n      "count": 11\n    },\n    {\n      "word": "tint",\n      "count": 11\n    },\n    {\n      "word": "torch",\n      "count": 11\n    },\n    {\n      "word": "touring",\n      "count": 11\n    },\n    {\n      "word": "tp",\n      "count": 11\n    },\n    {\n      "word": "tress",\n      "count": 11\n    },\n    {\n      "word": "trot",\n      "count": 11\n    },\n    {\n      "word": "trouble",\n      "count": 11\n    },\n    {\n      "word": "twist",\n      "count": 11\n    },\n    {\n      "word": "underwater",\n      "count": 11\n    },\n    {\n      "word": "unloading",\n      "count": 11\n    },\n    {\n      "word": "vader",\n      "count": 11\n    },\n    {\n      "word": "vcr",\n      "count": 11\n    },\n    {\n      "word": "venice",\n      "count": 11\n    },\n    {\n      "word": "vineyard",\n      "count": 11\n    },\n    {\n      "word": "vista",\n      "count": 11\n    },\n    {\n      "word": "wakeboard",\n      "count": 11\n    },\n    {\n      "word": "walkie",\n      "count": 11\n    },\n    {\n      "word": "wardrobe",\n      "count": 11\n    },\n    {\n      "word": "warming",\n      "count": 11\n    },\n    {\n      "word": "waves",\n      "count": 11\n    },\n    {\n      "word": "website",\n      "count": 11\n    },\n    {\n      "word": "welcome",\n      "count": 11\n    },\n    {\n      "word": "whiskey",\n      "count": 11\n    },\n    {\n      "word": "whiteboard",\n      "count": 11\n    },\n    {\n      "word": "wipe",\n      "count": 11\n    },\n    {\n      "word": "wizard",\n      "count": 11\n    },\n    {\n      "word": "wrestling",\n      "count": 11\n    },\n    {\n      "word": "x",\n      "count": 11\n    },\n    {\n      "word": "ability",\n      "count": 10\n    },\n    {\n      "word": "afghan",\n      "count": 10\n    },\n    {\n      "word": "airfrance",\n      "count": 10\n    },\n    {\n      "word": "ajar",\n      "count": 10\n    },\n    {\n      "word": "angry",\n      "count": 10\n    },\n    {\n      "word": "animation",\n      "count": 10\n    },\n    {\n      "word": "antler",\n      "count": 10\n    },\n    {\n      "word": "app",\n      "count": 10\n    },\n    {\n      "word": "atlantic",\n      "count": 10\n    },\n    {\n      "word": "ballon",\n      "count": 10\n    },\n    {\n      "word": "bannana",\n      "count": 10\n    },\n    {\n      "word": "beat",\n      "count": 10\n    },\n    {\n      "word": "behind",\n      "count": 10\n    },\n    {\n      "word": "blackboard",\n      "count": 10\n    },\n    {\n      "word": "blt",\n      "count": 10\n    },\n    {\n      "word": "bonnet",\n      "count": 10\n    },\n    {\n      "word": "boom",\n      "count": 10\n    },\n    {\n      "word": "buck",\n      "count": 10\n    },\n    {\n      "word": "bureau",\n      "count": 10\n    },\n    {\n      "word": "burning",\n      "count": 10\n    },\n    {\n      "word": "catsup",\n      "count": 10\n    },\n    {\n      "word": "charging",\n      "count": 10\n    },\n    {\n      "word": "charity",\n      "count": 10\n    },\n    {\n      "word": "checkout",\n      "count": 10\n    },\n    {\n      "word": "chevy",\n      "count": 10\n    },\n    {\n      "word": "chinatown",\n      "count": 10\n    },\n    {\n      "word": "civilian",\n      "count": 10\n    },\n    {\n      "word": "clipping",\n      "count": 10\n    },\n    {\n      "word": "closed",\n      "count": 10\n    },\n    {\n      "word": "cocoa",\n      "count": 10\n    },\n    {\n      "word": "colored",\n      "count": 10\n    },\n    {\n      "word": "coloring",\n      "count": 10\n    },\n    {\n      "word": "convoy",\n      "count": 10\n    },\n    {\n      "word": "coop",\n      "count": 10\n    },\n    {\n      "word": "cozy",\n      "count": 10\n    },\n    {\n      "word": "crayon",\n      "count": 10\n    },\n    {\n      "word": "crescent",\n      "count": 10\n    },\n    {\n      "word": "crock",\n      "count": 10\n    },\n    {\n      "word": "crossbone",\n      "count": 10\n    },\n    {\n      "word": "cutlery",\n      "count": 10\n    },\n    {\n      "word": "dalmation",\n      "count": 10\n    },\n    {\n      "word": "datum",\n      "count": 10\n    },\n    {\n      "word": "delight",\n      "count": 10\n    },\n    {\n      "word": "diagram",\n      "count": 10\n    },\n    {\n      "word": "dice",\n      "count": 10\n    },\n    {\n      "word": "dine",\n      "count": 10\n    },\n    {\n      "word": "dr",\n      "count": 10\n    },\n    {\n      "word": "drag",\n      "count": 10\n    },\n    {\n      "word": "drainage",\n      "count": 10\n    },\n    {\n      "word": "earbud",\n      "count": 10\n    },\n    {\n      "word": "elephants",\n      "count": 10\n    },\n    {\n      "word": "expert",\n      "count": 10\n    },\n    {\n      "word": "eyeball",\n      "count": 10\n    },\n    {\n      "word": "farmhouse",\n      "count": 10\n    },\n    {\n      "word": "fe",\n      "count": 10\n    },\n    {\n      "word": "ferret",\n      "count": 10\n    },\n    {\n      "word": "fetch",\n      "count": 10\n    },\n    {\n      "word": "firehydrant",\n      "count": 10\n    },\n    {\n      "word": "flake",\n      "count": 10\n    },\n    {\n      "word": "foothill",\n      "count": 10\n    },\n    {\n      "word": "footstool",\n      "count": 10\n    },\n    {\n      "word": "future",\n      "count": 10\n    },\n    {\n      "word": "garnishment",\n      "count": 10\n    },\n    {\n      "word": "gay",\n      "count": 10\n    },\n    {\n      "word": "germany",\n      "count": 10\n    },\n    {\n      "word": "giraff",\n      "count": 10\n    },\n    {\n      "word": "grandmother",\n      "count": 10\n    },\n    {\n      "word": "greenhouse",\n      "count": 10\n    },\n    {\n      "word": "grimace",\n      "count": 10\n    },\n    {\n      "word": "grinder",\n      "count": 10\n    },\n    {\n      "word": "hearth",\n      "count": 10\n    },\n    {\n      "word": "hippo",\n      "count": 10\n    },\n    {\n      "word": "homeplate",\n      "count": 10\n    },\n    {\n      "word": "huddle",\n      "count": 10\n    },\n    {\n      "word": "hutch",\n      "count": 10\n    },\n    {\n      "word": "hyena",\n      "count": 10\n    },\n    {\n      "word": "id",\n      "count": 10\n    },\n    {\n      "word": "idle",\n      "count": 10\n    },\n    {\n      "word": "in",\n      "count": 10\n    },\n    {\n      "word": "joe",\n      "count": 10\n    },\n    {\n      "word": "joint",\n      "count": 10\n    },\n    {\n      "word": "joker",\n      "count": 10\n    },\n    {\n      "word": "journey",\n      "count": 10\n    },\n    {\n      "word": "joy",\n      "count": 10\n    },\n    {\n      "word": "jumpsuit",\n      "count": 10\n    },\n    {\n      "word": "kabob",\n      "count": 10\n    },\n    {\n      "word": "kart",\n      "count": 10\n    },\n    {\n      "word": "keychain",\n      "count": 10\n    },\n    {\n      "word": "knick",\n      "count": 10\n    },\n    {\n      "word": "landmark",\n      "count": 10\n    },\n    {\n      "word": "lasso",\n      "count": 10\n    },\n    {\n      "word": "leek",\n      "count": 10\n    },\n    {\n      "word": "lei",\n      "count": 10\n    },\n    {\n      "word": "lilac",\n      "count": 10\n    },\n    {\n      "word": "list",\n      "count": 10\n    },\n    {\n      "word": "loader",\n      "count": 10\n    },\n    {\n      "word": "mache",\n      "count": 10\n    },\n    {\n      "word": "mack",\n      "count": 10\n    },\n    {\n      "word": "maid",\n      "count": 10\n    },\n    {\n      "word": "manager",\n      "count": 10\n    },\n    {\n      "word": "marmalade",\n      "count": 10\n    },\n    {\n      "word": "mash",\n      "count": 10\n    },\n    {\n      "word": "matter",\n      "count": 10\n    },\n    {\n      "word": "mcdonalds",\n      "count": 10\n    },\n    {\n      "word": "mein",\n      "count": 10\n    },\n    {\n      "word": "mets",\n      "count": 10\n    },\n    {\n      "word": "mosquito",\n      "count": 10\n    },\n    {\n      "word": "motocycle",\n      "count": 10\n    },\n    {\n      "word": "motorcylce",\n      "count": 10\n    },\n    {\n      "word": "moustache",\n      "count": 10\n    },\n    {\n      "word": "mower",\n      "count": 10\n    },\n    {\n      "word": "mulch",\n      "count": 10\n    },\n    {\n      "word": "nectarine",\n      "count": 10\n    },\n    {\n      "word": "nutella",\n      "count": 10\n    },\n    {\n      "word": "oat",\n      "count": 10\n    },\n    {\n      "word": "orphanage",\n      "count": 10\n    },\n    {\n      "word": "p",\n      "count": 10\n    },\n    {\n      "word": "participant",\n      "count": 10\n    },\n    {\n      "word": "pastor",\n      "count": 10\n    },\n    {\n      "word": "patrick",\n      "count": 10\n    },\n    {\n      "word": "payer",\n      "count": 10\n    },\n    {\n      "word": "pendulum",\n      "count": 10\n    },\n    {\n      "word": "pinstripe",\n      "count": 10\n    },\n    {\n      "word": "pint",\n      "count": 10\n    },\n    {\n      "word": "plume",\n      "count": 10\n    },\n    {\n      "word": "q",\n      "count": 10\n    },\n    {\n      "word": "racehorse",\n      "count": 10\n    },\n    {\n      "word": "receiver",\n      "count": 10\n    },\n    {\n      "word": "recreation",\n      "count": 10\n    },\n    {\n      "word": "resident",\n      "count": 10\n    },\n    {\n      "word": "rubbish",\n      "count": 10\n    },\n    {\n      "word": "rundown",\n      "count": 10\n    },\n    {\n      "word": "rye",\n      "count": 10\n    },\n    {\n      "word": "sectional",\n      "count": 10\n    },\n    {\n      "word": "shawl",\n      "count": 10\n    },\n    {\n      "word": "shooting",\n      "count": 10\n    },\n    {\n      "word": "simulator",\n      "count": 10\n    },\n    {\n      "word": "skies",\n      "count": 10\n    },\n    {\n      "word": "slicer",\n      "count": 10\n    },\n    {\n      "word": "solo",\n      "count": 10\n    },\n    {\n      "word": "spotless",\n      "count": 10\n    },\n    {\n      "word": "spraying",\n      "count": 10\n    },\n    {\n      "word": "stroke",\n      "count": 10\n    },\n    {\n      "word": "sundaes",\n      "count": 10\n    },\n    {\n      "word": "talbe",\n      "count": 10\n    },\n    {\n      "word": "talkie",\n      "count": 10\n    },\n    {\n      "word": "tangle",\n      "count": 10\n    },\n    {\n      "word": "textbook",\n      "count": 10\n    },\n    {\n      "word": "thousand",\n      "count": 10\n    },\n    {\n      "word": "through",\n      "count": 10\n    },\n    {\n      "word": "title",\n      "count": 10\n    },\n    {\n      "word": "toss",\n      "count": 10\n    },\n    {\n      "word": "triumph",\n      "count": 10\n    },\n    {\n      "word": "udder",\n      "count": 10\n    },\n    {\n      "word": "underbrush",\n      "count": 10\n    },\n    {\n      "word": "ups",\n      "count": 10\n    },\n    {\n      "word": "urine",\n      "count": 10\n    },\n    {\n      "word": "vegan",\n      "count": 10\n    },\n    {\n      "word": "vodka",\n      "count": 10\n    },\n    {\n      "word": "watercraft",\n      "count": 10\n    },\n    {\n      "word": "wharf",\n      "count": 10\n    },\n    {\n      "word": "woody",\n      "count": 10\n    },\n    {\n      "word": "yankees",\n      "count": 10\n    },\n    {\n      "word": "airway",\n      "count": 9\n    },\n    {\n      "word": "alto",\n      "count": 9\n    },\n    {\n      "word": "appearance",\n      "count": 9\n    },\n    {\n      "word": "application",\n      "count": 9\n    },\n    {\n      "word": "ascending",\n      "count": 9\n    },\n    {\n      "word": "backing",\n      "count": 9\n    },\n    {\n      "word": "bannister",\n      "count": 9\n    },\n    {\n      "word": "bare",\n      "count": 9\n    },\n    {\n      "word": "beacon",\n      "count": 9\n    },\n    {\n      "word": "beginner",\n      "count": 9\n    },\n    {\n      "word": "blizzard",\n      "count": 9\n    },\n    {\n      "word": "bowler",\n      "count": 9\n    },\n    {\n      "word": "bracket",\n      "count": 9\n    },\n    {\n      "word": "breath",\n      "count": 9\n    },\n    {\n      "word": "brim",\n      "count": 9\n    },\n    {\n      "word": "britain",\n      "count": 9\n    },\n    {\n      "word": "buses",\n      "count": 9\n    },\n    {\n      "word": "candlelight",\n      "count": 9\n    },\n    {\n      "word": "canon",\n      "count": 9\n    },\n    {\n      "word": "carry",\n      "count": 9\n    },\n    {\n      "word": "centre",\n      "count": 9\n    },\n    {\n      "word": "chance",\n      "count": 9\n    },\n    {\n      "word": "chariot",\n      "count": 9\n    },\n    {\n      "word": "cheddar",\n      "count": 9\n    },\n    {\n      "word": "chow",\n      "count": 9\n    },\n    {\n      "word": "clam",\n      "count": 9\n    },\n    {\n      "word": "co",\n      "count": 9\n    },\n    {\n      "word": "compass",\n      "count": 9\n    },\n    {\n      "word": "condo",\n      "count": 9\n    },\n    {\n      "word": "confection",\n      "count": 9\n    },\n    {\n      "word": "cornbread",\n      "count": 9\n    },\n    {\n      "word": "corona",\n      "count": 9\n    },\n    {\n      "word": "covered",\n      "count": 9\n    },\n    {\n      "word": "cowgirl",\n      "count": 9\n    },\n    {\n      "word": "crook",\n      "count": 9\n    },\n    {\n      "word": "crouton",\n      "count": 9\n    },\n    {\n      "word": "crt",\n      "count": 9\n    },\n    {\n      "word": "cubs",\n      "count": 9\n    },\n    {\n      "word": "cupola",\n      "count": 9\n    },\n    {\n      "word": "dam",\n      "count": 9\n    },\n    {\n      "word": "descent",\n      "count": 9\n    },\n    {\n      "word": "detroit",\n      "count": 9\n    },\n    {\n      "word": "drizzle",\n      "count": 9\n    },\n    {\n      "word": "duster",\n      "count": 9\n    },\n    {\n      "word": "electronics",\n      "count": 9\n    },\n    {\n      "word": "essential",\n      "count": 9\n    },\n    {\n      "word": "excitement",\n      "count": 9\n    },\n    {\n      "word": "experiment",\n      "count": 9\n    },\n    {\n      "word": "f",\n      "count": 9\n    },\n    {\n      "word": "falls",\n      "count": 9\n    },\n    {\n      "word": "fender",\n      "count": 9\n    },\n    {\n      "word": "fifty",\n      "count": 9\n    },\n    {\n      "word": "fig",\n      "count": 9\n    },\n    {\n      "word": "fisheye",\n      "count": 9\n    },\n    {\n      "word": "floater",\n      "count": 9\n    },\n    {\n      "word": "flooding",\n      "count": 9\n    },\n    {\n      "word": "footpath",\n      "count": 9\n    },\n    {\n      "word": "furnace",\n      "count": 9\n    },\n    {\n      "word": "george",\n      "count": 9\n    },\n    {\n      "word": "girlfriend",\n      "count": 9\n    },\n    {\n      "word": "girrafe",\n      "count": 9\n    },\n    {\n      "word": "goo",\n      "count": 9\n    },\n    {\n      "word": "gourd",\n      "count": 9\n    },\n    {\n      "word": "graffitti",\n      "count": 9\n    },\n    {\n      "word": "granny",\n      "count": 9\n    },\n    {\n      "word": "headdress",\n      "count": 9\n    },\n    {\n      "word": "hope",\n      "count": 9\n    },\n    {\n      "word": "hub",\n      "count": 9\n    },\n    {\n      "word": "hunter",\n      "count": 9\n    },\n    {\n      "word": "hyde",\n      "count": 9\n    },\n    {\n      "word": "icicle",\n      "count": 9\n    },\n    {\n      "word": "inlet",\n      "count": 9\n    },\n    {\n      "word": "ith",\n      "count": 9\n    },\n    {\n      "word": "kayaker",\n      "count": 9\n    },\n    {\n      "word": "keg",\n      "count": 9\n    },\n    {\n      "word": "kennel",\n      "count": 9\n    },\n    {\n      "word": "kingdom",\n      "count": 9\n    },\n    {\n      "word": "kong",\n      "count": 9\n    },\n    {\n      "word": "latte",\n      "count": 9\n    },\n    {\n      "word": "lcd",\n      "count": 9\n    },\n    {\n      "word": "leaguer",\n      "count": 9\n    },\n    {\n      "word": "legos",\n      "count": 9\n    },\n    {\n      "word": "lorry",\n      "count": 9\n    },\n    {\n      "word": "macintosh",\n      "count": 9\n    },\n    {\n      "word": "manger",\n      "count": 9\n    },\n    {\n      "word": "marathon",\n      "count": 9\n    },\n    {\n      "word": "medley",\n      "count": 9\n    },\n    {\n      "word": "mexico",\n      "count": 9\n    },\n    {\n      "word": "microsoft",\n      "count": 9\n    },\n    {\n      "word": "mode",\n      "count": 9\n    },\n    {\n      "word": "mogul",\n      "count": 9\n    },\n    {\n      "word": "momma",\n      "count": 9\n    },\n    {\n      "word": "mooring",\n      "count": 9\n    },\n    {\n      "word": "moutain",\n      "count": 9\n    },\n    {\n      "word": "nachos",\n      "count": 9\n    },\n    {\n      "word": "nibble",\n      "count": 9\n    },\n    {\n      "word": "notice",\n      "count": 9\n    },\n    {\n      "word": "ontop",\n      "count": 9\n    },\n    {\n      "word": "organizer",\n      "count": 9\n    },\n    {\n      "word": "outward",\n      "count": 9\n    },\n    {\n      "word": "overcoat",\n      "count": 9\n    },\n    {\n      "word": "oxygen",\n      "count": 9\n    },\n    {\n      "word": "pagoda",\n      "count": 9\n    },\n    {\n      "word": "paintbrush",\n      "count": 9\n    },\n    {\n      "word": "pat",\n      "count": 9\n    },\n    {\n      "word": "peal",\n      "count": 9\n    },\n    {\n      "word": "pebble",\n      "count": 9\n    },\n    {\n      "word": "pecan",\n      "count": 9\n    },\n    {\n      "word": "piggy",\n      "count": 9\n    },\n    {\n      "word": "pipeline",\n      "count": 9\n    },\n    {\n      "word": "plaster",\n      "count": 9\n    },\n    {\n      "word": "playroom",\n      "count": 9\n    },\n    {\n      "word": "playstation",\n      "count": 9\n    },\n    {\n      "word": "polish",\n      "count": 9\n    },\n    {\n      "word": "predator",\n      "count": 9\n    },\n    {\n      "word": "problem",\n      "count": 9\n    },\n    {\n      "word": "processing",\n      "count": 9\n    },\n    {\n      "word": "prom",\n      "count": 9\n    },\n    {\n      "word": "propane",\n      "count": 9\n    },\n    {\n      "word": "queue",\n      "count": 9\n    },\n    {\n      "word": "raman",\n      "count": 9\n    },\n    {\n      "word": "rancher",\n      "count": 9\n    },\n    {\n      "word": "ratchet",\n      "count": 9\n    },\n    {\n      "word": "recycling",\n      "count": 9\n    },\n    {\n      "word": "relief",\n      "count": 9\n    },\n    {\n      "word": "remnant",\n      "count": 9\n    },\n    {\n      "word": "result",\n      "count": 9\n    },\n    {\n      "word": "riverboat",\n      "count": 9\n    },\n    {\n      "word": "roam",\n      "count": 9\n    },\n    {\n      "word": "robin",\n      "count": 9\n    },\n    {\n      "word": "salvation",\n      "count": 9\n    },\n    {\n      "word": "saucepan",\n      "count": 9\n    },\n    {\n      "word": "scaffolding",\n      "count": 9\n    },\n    {\n      "word": "scallion",\n      "count": 9\n    },\n    {\n      "word": "scroll",\n      "count": 9\n    },\n    {\n      "word": "scrubby",\n      "count": 9\n    },\n    {\n      "word": "search",\n      "count": 9\n    },\n    {\n      "word": "seashell",\n      "count": 9\n    },\n    {\n      "word": "settee",\n      "count": 9\n    },\n    {\n      "word": "sewer",\n      "count": 9\n    },\n    {\n      "word": "shallow",\n      "count": 9\n    },\n    {\n      "word": "shine",\n      "count": 9\n    },\n    {\n      "word": "shoelace",\n      "count": 9\n    },\n    {\n      "word": "sized",\n      "count": 9\n    },\n    {\n      "word": "skatboard",\n      "count": 9\n    },\n    {\n      "word": "southern",\n      "count": 9\n    },\n    {\n      "word": "spiral",\n      "count": 9\n    },\n    {\n      "word": "sprig",\n      "count": 9\n    },\n    {\n      "word": "standstill",\n      "count": 9\n    },\n    {\n      "word": "statement",\n      "count": 9\n    },\n    {\n      "word": "streets",\n      "count": 9\n    },\n    {\n      "word": "strut",\n      "count": 9\n    },\n    {\n      "word": "stuffed",\n      "count": 9\n    },\n    {\n      "word": "sundae",\n      "count": 9\n    },\n    {\n      "word": "surgery",\n      "count": 9\n    },\n    {\n      "word": "suzuki",\n      "count": 9\n    },\n    {\n      "word": "swell",\n      "count": 9\n    },\n    {\n      "word": "swinging",\n      "count": 9\n    },\n    {\n      "word": "tavern",\n      "count": 9\n    },\n    {\n      "word": "te",\n      "count": 9\n    },\n    {\n      "word": "technician",\n      "count": 9\n    },\n    {\n      "word": "tranquil",\n      "count": 9\n    },\n    {\n      "word": "trap",\n      "count": 9\n    },\n    {\n      "word": "trotting",\n      "count": 9\n    },\n    {\n      "word": "trucking",\n      "count": 9\n    },\n    {\n      "word": "turntable",\n      "count": 9\n    },\n    {\n      "word": "twine",\n      "count": 9\n    },\n    {\n      "word": "two",\n      "count": 9\n    },\n    {\n      "word": "umbrealla",\n      "count": 9\n    },\n    {\n      "word": "viewpoint",\n      "count": 9\n    },\n    {\n      "word": "volkswagen",\n      "count": 9\n    },\n    {\n      "word": "volvo",\n      "count": 9\n    },\n    {\n      "word": "wade",\n      "count": 9\n    },\n    {\n      "word": "wakeboarder",\n      "count": 9\n    },\n    {\n      "word": "washbasin",\n      "count": 9\n    },\n    {\n      "word": "weapon",\n      "count": 9\n    },\n    {\n      "word": "whisker",\n      "count": 9\n    },\n    {\n      "word": "wi",\n      "count": 9\n    },\n    {\n      "word": "wiring",\n      "count": 9\n    },\n    {\n      "word": "wish",\n      "count": 9\n    },\n    {\n      "word": "wispy",\n      "count": 9\n    },\n    {\n      "word": "wonder",\n      "count": 9\n    },\n    {\n      "word": "xbox",\n      "count": 9\n    },\n    {\n      "word": "addition",\n      "count": 8\n    },\n    {\n      "word": "adidas",\n      "count": 8\n    },\n    {\n      "word": "african",\n      "count": 8\n    },\n    {\n      "word": "ann",\n      "count": 8\n    },\n    {\n      "word": "atlanta",\n      "count": 8\n    },\n    {\n      "word": "attendee",\n      "count": 8\n    },\n    {\n      "word": "avatar",\n      "count": 8\n    },\n    {\n      "word": "baggie",\n      "count": 8\n    },\n    {\n      "word": "ballpark",\n      "count": 8\n    },\n    {\n      "word": "bananna",\n      "count": 8\n    },\n    {\n      "word": "barack",\n      "count": 8\n    },\n    {\n      "word": "barren",\n      "count": 8\n    },\n    {\n      "word": "barrow",\n      "count": 8\n    },\n    {\n      "word": "bass",\n      "count": 8\n    },\n    {\n      "word": "battleship",\n      "count": 8\n    },\n    {\n      "word": "bazaar",\n      "count": 8\n    },\n    {\n      "word": "beaver",\n      "count": 8\n    },\n    {\n      "word": "beds",\n      "count": 8\n    },\n    {\n      "word": "beetle",\n      "count": 8\n    },\n    {\n      "word": "benz",\n      "count": 8\n    },\n    {\n      "word": "blend",\n      "count": 8\n    },\n    {\n      "word": "blonde",\n      "count": 8\n    },\n    {\n      "word": "blower",\n      "count": 8\n    },\n    {\n      "word": "bob",\n      "count": 8\n    },\n    {\n      "word": "boeing",\n      "count": 8\n    },\n    {\n      "word": "bronco",\n      "count": 8\n    },\n    {\n      "word": "bucking",\n      "count": 8\n    },\n    {\n      "word": "buiding",\n      "count": 8\n    },\n    {\n      "word": "burritos",\n      "count": 8\n    },\n    {\n      "word": "busy",\n      "count": 8\n    },\n    {\n      "word": "caddy",\n      "count": 8\n    },\n    {\n      "word": "candlestick",\n      "count": 8\n    },\n    {\n      "word": "carafe",\n      "count": 8\n    },\n    {\n      "word": "caravan",\n      "count": 8\n    },\n    {\n      "word": "cashew",\n      "count": 8\n    },\n    {\n      "word": "catamaran",\n      "count": 8\n    },\n    {\n      "word": "char",\n      "count": 8\n    },\n    {\n      "word": "cheerleader",\n      "count": 8\n    },\n    {\n      "word": "cherub",\n      "count": 8\n    },\n    {\n      "word": "chillin",\n      "count": 8\n    },\n    {\n      "word": "cinder",\n      "count": 8\n    },\n    {\n      "word": "cling",\n      "count": 8\n    },\n    {\n      "word": "clinton",\n      "count": 8\n    },\n    {\n      "word": "coating",\n      "count": 8\n    },\n    {\n      "word": "cobbler",\n      "count": 8\n    },\n    {\n      "word": "cocker",\n      "count": 8\n    },\n    {\n      "word": "code",\n      "count": 8\n    },\n    {\n      "word": "collectible",\n      "count": 8\n    },\n    {\n      "word": "collector",\n      "count": 8\n    },\n    {\n      "word": "compound",\n      "count": 8\n    },\n    {\n      "word": "condom",\n      "count": 8\n    },\n    {\n      "word": "contestant",\n      "count": 8\n    },\n    {\n      "word": "cooktop",\n      "count": 8\n    },\n    {\n      "word": "core",\n      "count": 8\n    },\n    {\n      "word": "courch",\n      "count": 8\n    },\n    {\n      "word": "cricket",\n      "count": 8\n    },\n    {\n      "word": "croquet",\n      "count": 8\n    },\n    {\n      "word": "crucifix",\n      "count": 8\n    },\n    {\n      "word": "development",\n      "count": 8\n    },\n    {\n      "word": "director",\n      "count": 8\n    },\n    {\n      "word": "disposal",\n      "count": 8\n    },\n    {\n      "word": "diver",\n      "count": 8\n    },\n    {\n      "word": "dougnut",\n      "count": 8\n    },\n    {\n      "word": "droplet",\n      "count": 8\n    },\n    {\n      "word": "dry",\n      "count": 8\n    },\n    {\n      "word": "ease",\n      "count": 8\n    },\n    {\n      "word": "eclair",\n      "count": 8\n    },\n    {\n      "word": "ed",\n      "count": 8\n    },\n    {\n      "word": "electricity",\n      "count": 8\n    },\n    {\n      "word": "elf",\n      "count": 8\n    },\n    {\n      "word": "email",\n      "count": 8\n    },\n    {\n      "word": "emirates",\n      "count": 8\n    },\n    {\n      "word": "erase",\n      "count": 8\n    },\n    {\n      "word": "escort",\n      "count": 8\n    },\n    {\n      "word": "exchange",\n      "count": 8\n    },\n    {\n      "word": "executive",\n      "count": 8\n    },\n    {\n      "word": "facebook",\n      "count": 8\n    },\n    {\n      "word": "facet",\n      "count": 8\n    },\n    {\n      "word": "farming",\n      "count": 8\n    },\n    {\n      "word": "fat",\n      "count": 8\n    },\n    {\n      "word": "fenced",\n      "count": 8\n    },\n    {\n      "word": "fine",\n      "count": 8\n    },\n    {\n      "word": "firsbee",\n      "count": 8\n    },\n    {\n      "word": "fishnet",\n      "count": 8\n    },\n    {\n      "word": "flowerpot",\n      "count": 8\n    },\n    {\n      "word": "flurry",\n      "count": 8\n    },\n    {\n      "word": "foilage",\n      "count": 8\n    },\n    {\n      "word": "fold",\n      "count": 8\n    },\n    {\n      "word": "forage",\n      "count": 8\n    },\n    {\n      "word": "fourth",\n      "count": 8\n    },\n    {\n      "word": "free",\n      "count": 8\n    },\n    {\n      "word": "freshener",\n      "count": 8\n    },\n    {\n      "word": "fritter",\n      "count": 8\n    },\n    {\n      "word": "fro",\n      "count": 8\n    },\n    {\n      "word": "fron",\n      "count": 8\n    },\n    {\n      "word": "funeral",\n      "count": 8\n    },\n    {\n      "word": "furry",\n      "count": 8\n    },\n    {\n      "word": "g",\n      "count": 8\n    },\n    {\n      "word": "generator",\n      "count": 8\n    },\n    {\n      "word": "getting",\n      "count": 8\n    },\n    {\n      "word": "gingerbread",\n      "count": 8\n    },\n    {\n      "word": "girls",\n      "count": 8\n    },\n    {\n      "word": "goodbye",\n      "count": 8\n    },\n    {\n      "word": "gothic",\n      "count": 8\n    },\n    {\n      "word": "gra",\n      "count": 8\n    },\n    {\n      "word": "grandma",\n      "count": 8\n    },\n    {\n      "word": "graph",\n      "count": 8\n    },\n    {\n      "word": "grown",\n      "count": 8\n    },\n    {\n      "word": "guinea",\n      "count": 8\n    },\n    {\n      "word": "guinness",\n      "count": 8\n    },\n    {\n      "word": "hamper",\n      "count": 8\n    },\n    {\n      "word": "handkerchief",\n      "count": 8\n    },\n    {\n      "word": "hardwood",\n      "count": 8\n    },\n    {\n      "word": "harper",\n      "count": 8\n    },\n    {\n      "word": "hil",\n      "count": 8\n    },\n    {\n      "word": "hoe",\n      "count": 8\n    },\n    {\n      "word": "hong",\n      "count": 8\n    },\n    {\n      "word": "hor",\n      "count": 8\n    },\n    {\n      "word": "hp",\n      "count": 8\n    },\n    {\n      "word": "hull",\n      "count": 8\n    },\n    {\n      "word": "hyrdrant",\n      "count": 8\n    },\n    {\n      "word": "intent",\n      "count": 8\n    },\n    {\n      "word": "japanese",\n      "count": 8\n    },\n    {\n      "word": "jars",\n      "count": 8\n    },\n    {\n      "word": "jewel",\n      "count": 8\n    },\n    {\n      "word": "july",\n      "count": 8\n    },\n    {\n      "word": "jumble",\n      "count": 8\n    },\n    {\n      "word": "k",\n      "count": 8\n    },\n    {\n      "word": "kickflip",\n      "count": 8\n    },\n    {\n      "word": "klm",\n      "count": 8\n    },\n    {\n      "word": "kneepad",\n      "count": 8\n    },\n    {\n      "word": "lad",\n      "count": 8\n    },\n    {\n      "word": "lakeside",\n      "count": 8\n    },\n    {\n      "word": "lama",\n      "count": 8\n    },\n    {\n      "word": "lanyard",\n      "count": 8\n    },\n    {\n      "word": "las",\n      "count": 8\n    },\n    {\n      "word": "lattice",\n      "count": 8\n    },\n    {\n      "word": "lever",\n      "count": 8\n    },\n    {\n      "word": "long",\n      "count": 8\n    },\n    {\n      "word": "lookign",\n      "count": 8\n    },\n    {\n      "word": "lover",\n      "count": 8\n    },\n    {\n      "word": "magenta",\n      "count": 8\n    },\n    {\n      "word": "manikin",\n      "count": 8\n    },\n    {\n      "word": "manmade",\n      "count": 8\n    },\n    {\n      "word": "marijuana",\n      "count": 8\n    },\n    {\n      "word": "matress",\n      "count": 8\n    },\n    {\n      "word": "measuring",\n      "count": 8\n    },\n    {\n      "word": "missile",\n      "count": 8\n    },\n    {\n      "word": "misty",\n      "count": 8\n    },\n    {\n      "word": "modem",\n      "count": 8\n    },\n    {\n      "word": "nad",\n      "count": 8\n    },\n    {\n      "word": "navel",\n      "count": 8\n    },\n    {\n      "word": "nearby",\n      "count": 8\n    },\n    {\n      "word": "niche",\n      "count": 8\n    },\n    {\n      "word": "nourishment",\n      "count": 8\n    },\n    {\n      "word": "odd",\n      "count": 8\n    },\n    {\n      "word": "ona",\n      "count": 8\n    },\n    {\n      "word": "operating",\n      "count": 8\n    },\n    {\n      "word": "operation",\n      "count": 8\n    },\n    {\n      "word": "oreo",\n      "count": 8\n    },\n    {\n      "word": "oyster",\n      "count": 8\n    },\n    {\n      "word": "painter",\n      "count": 8\n    },\n    {\n      "word": "panting",\n      "count": 8\n    },\n    {\n      "word": "passing",\n      "count": 8\n    },\n    {\n      "word": "peck",\n      "count": 8\n    },\n    {\n      "word": "peeling",\n      "count": 8\n    },\n    {\n      "word": "pennsylvania",\n      "count": 8\n    },\n    {\n      "word": "peter",\n      "count": 8\n    },\n    {\n      "word": "petting",\n      "count": 8\n    },\n    {\n      "word": "phase",\n      "count": 8\n    },\n    {\n      "word": "phones",\n      "count": 8\n    },\n    {\n      "word": "photoshop",\n      "count": 8\n    },\n    {\n      "word": "pigtail",\n      "count": 8\n    },\n    {\n      "word": "pinto",\n      "count": 8\n    },\n    {\n      "word": "piping",\n      "count": 8\n    },\n    {\n      "word": "planet",\n      "count": 8\n    },\n    {\n      "word": "pleasure",\n      "count": 8\n    },\n    {\n      "word": "poem",\n      "count": 8\n    },\n    {\n      "word": "pressure",\n      "count": 8\n    },\n    {\n      "word": "qantas",\n      "count": 8\n    },\n    {\n      "word": "reason",\n      "count": 8\n    },\n    {\n      "word": "reef",\n      "count": 8\n    },\n    {\n      "word": "reindeer",\n      "count": 8\n    },\n    {\n      "word": "rental",\n      "count": 8\n    },\n    {\n      "word": "ripeness",\n      "count": 8\n    },\n    {\n      "word": "rolex",\n      "count": 8\n    },\n    {\n      "word": "rotunda",\n      "count": 8\n    },\n    {\n      "word": "ruck",\n      "count": 8\n    },\n    {\n      "word": "running",\n      "count": 8\n    },\n    {\n      "word": "sash",\n      "count": 8\n    },\n    {\n      "word": "saver",\n      "count": 8\n    },\n    {\n      "word": "scatter",\n      "count": 8\n    },\n    {\n      "word": "scoreboard",\n      "count": 8\n    },\n    {\n      "word": "scraper",\n      "count": 8\n    },\n    {\n      "word": "screwdriver",\n      "count": 8\n    },\n    {\n      "word": "scrubber",\n      "count": 8\n    },\n    {\n      "word": "several",\n      "count": 8\n    },\n    {\n      "word": "shades",\n      "count": 8\n    },\n    {\n      "word": "shamrock",\n      "count": 8\n    },\n    {\n      "word": "sharpener",\n      "count": 8\n    },\n    {\n      "word": "sheer",\n      "count": 8\n    },\n    {\n      "word": "shetland",\n      "count": 8\n    },\n    {\n      "word": "sideways",\n      "count": 8\n    },\n    {\n      "word": "sil",\n      "count": 8\n    },\n    {\n      "word": "singer",\n      "count": 8\n    },\n    {\n      "word": "single",\n      "count": 8\n    },\n    {\n      "word": "situation",\n      "count": 8\n    },\n    {\n      "word": "skim",\n      "count": 8\n    },\n    {\n      "word": "slender",\n      "count": 8\n    },\n    {\n      "word": "slogan",\n      "count": 8\n    },\n    {\n      "word": "slug",\n      "count": 8\n    },\n    {\n      "word": "smith",\n      "count": 8\n    },\n    {\n      "word": "souffle",\n      "count": 8\n    },\n    {\n      "word": "soul",\n      "count": 8\n    },\n    {\n      "word": "spike",\n      "count": 8\n    },\n    {\n      "word": "spin",\n      "count": 8\n    },\n    {\n      "word": "spoke",\n      "count": 8\n    },\n    {\n      "word": "sponsor",\n      "count": 8\n    },\n    {\n      "word": "squid",\n      "count": 8\n    },\n    {\n      "word": "stairs",\n      "count": 8\n    },\n    {\n      "word": "staring",\n      "count": 8\n    },\n    {\n      "word": "steamboat",\n      "count": 8\n    },\n    {\n      "word": "stretching",\n      "count": 8\n    },\n    {\n      "word": "succulent",\n      "count": 8\n    },\n    {\n      "word": "super",\n      "count": 8\n    },\n    {\n      "word": "susan",\n      "count": 8\n    },\n    {\n      "word": "sweaty",\n      "count": 8\n    },\n    {\n      "word": "sydney",\n      "count": 8\n    },\n    {\n      "word": "taht",\n      "count": 8\n    },\n    {\n      "word": "tap",\n      "count": 8\n    },\n    {\n      "word": "tender",\n      "count": 8\n    },\n    {\n      "word": "the",\n      "count": 8\n    },\n    {\n      "word": "tim",\n      "count": 8\n    },\n    {\n      "word": "toga",\n      "count": 8\n    },\n    {\n      "word": "tortoiseshell",\n      "count": 8\n    },\n    {\n      "word": "tummy",\n      "count": 8\n    },\n    {\n      "word": "turret",\n      "count": 8\n    },\n    {\n      "word": "typing",\n      "count": 8\n    },\n    {\n      "word": "unbrella",\n      "count": 8\n    },\n    {\n      "word": "vegas",\n      "count": 8\n    },\n    {\n      "word": "vender",\n      "count": 8\n    },\n    {\n      "word": "veteran",\n      "count": 8\n    },\n    {\n      "word": "victoria",\n      "count": 8\n    },\n    {\n      "word": "warmer",\n      "count": 8\n    },\n    {\n      "word": "wastebasket",\n      "count": 8\n    },\n    {\n      "word": "webcam",\n      "count": 8\n    },\n    {\n      "word": "westminster",\n      "count": 8\n    },\n    {\n      "word": "whirlpool",\n      "count": 8\n    },\n    {\n      "word": "wintry",\n      "count": 8\n    },\n    {\n      "word": "withe",\n      "count": 8\n    },\n    {\n      "word": "wonderland",\n      "count": 8\n    },\n    {\n      "word": "woodpecker",\n      "count": 8\n    },\n    {\n      "word": "wristband",\n      "count": 8\n    },\n    {\n      "word": "adn",\n      "count": 7\n    },\n    {\n      "word": "airborne",\n      "count": 7\n    },\n    {\n      "word": "alfalfa",\n      "count": 7\n    },\n    {\n      "word": "alfredo",\n      "count": 7\n    },\n    {\n      "word": "anime",\n      "count": 7\n    },\n    {\n      "word": "anniversary",\n      "count": 7\n    },\n    {\n      "word": "announcer",\n      "count": 7\n    },\n    {\n      "word": "antennae",\n      "count": 7\n    },\n    {\n      "word": "applesauce",\n      "count": 7\n    },\n    {\n      "word": "ash",\n      "count": 7\n    },\n    {\n      "word": "attack",\n      "count": 7\n    },\n    {\n      "word": "backpacker",\n      "count": 7\n    },\n    {\n      "word": "barbershop",\n      "count": 7\n    },\n    {\n      "word": "bard",\n      "count": 7\n    },\n    {\n      "word": "bareback",\n      "count": 7\n    },\n    {\n      "word": "barnyard",\n      "count": 7\n    },\n    {\n      "word": "baron",\n      "count": 7\n    },\n    {\n      "word": "beg",\n      "count": 7\n    },\n    {\n      "word": "beret",\n      "count": 7\n    },\n    {\n      "word": "berlin",\n      "count": 7\n    },\n    {\n      "word": "bicycler",\n      "count": 7\n    },\n    {\n      "word": "bid",\n      "count": 7\n    },\n    {\n      "word": "birch",\n      "count": 7\n    },\n    {\n      "word": "blast",\n      "count": 7\n    },\n    {\n      "word": "blob",\n      "count": 7\n    },\n    {\n      "word": "bluff",\n      "count": 7\n    },\n    {\n      "word": "boards",\n      "count": 7\n    },\n    {\n      "word": "booklet",\n      "count": 7\n    },\n    {\n      "word": "boys",\n      "count": 7\n    },\n    {\n      "word": "brahma",\n      "count": 7\n    },\n    {\n      "word": "brunch",\n      "count": 7\n    },\n    {\n      "word": "brussels",\n      "count": 7\n    },\n    {\n      "word": "ca",\n      "count": 7\n    },\n    {\n      "word": "cabana",\n      "count": 7\n    },\n    {\n      "word": "calfs",\n      "count": 7\n    },\n    {\n      "word": "cam",\n      "count": 7\n    },\n    {\n      "word": "campfire",\n      "count": 7\n    },\n    {\n      "word": "cannon",\n      "count": 7\n    },\n    {\n      "word": "carrots",\n      "count": 7\n    },\n    {\n      "word": "cause",\n      "count": 7\n    },\n    {\n      "word": "celebratory",\n      "count": 7\n    },\n    {\n      "word": "celebrity",\n      "count": 7\n    },\n    {\n      "word": "chap",\n      "count": 7\n    },\n    {\n      "word": "chapel",\n      "count": 7\n    },\n    {\n      "word": "chart",\n      "count": 7\n    },\n    {\n      "word": "checkpoint",\n      "count": 7\n    },\n    {\n      "word": "cheer",\n      "count": 7\n    },\n    {\n      "word": "cheetah",\n      "count": 7\n    },\n    {\n      "word": "chilli",\n      "count": 7\n    },\n    {\n      "word": "chiquita",\n      "count": 7\n    },\n    {\n      "word": "choir",\n      "count": 7\n    },\n    {\n      "word": "circuit",\n      "count": 7\n    },\n    {\n      "word": "clamp",\n      "count": 7\n    },\n    {\n      "word": "cloudless",\n      "count": 7\n    },\n    {\n      "word": "cloudy",\n      "count": 7\n    },\n    {\n      "word": "confinement",\n      "count": 7\n    },\n    {\n      "word": "congregate",\n      "count": 7\n    },\n    {\n      "word": "corgi",\n      "count": 7\n    },\n    {\n      "word": "courtroom",\n      "count": 7\n    },\n    {\n      "word": "cradle",\n      "count": 7\n    },\n    {\n      "word": "creepy",\n      "count": 7\n    },\n    {\n      "word": "crutch",\n      "count": 7\n    },\n    {\n      "word": "ct",\n      "count": 7\n    },\n    {\n      "word": "curio",\n      "count": 7\n    },\n    {\n      "word": "de",\n      "count": 7\n    },\n    {\n      "word": "dead",\n      "count": 7\n    },\n    {\n      "word": "dentist",\n      "count": 7\n    },\n    {\n      "word": "deodorant",\n      "count": 7\n    },\n    {\n      "word": "derelict",\n      "count": 7\n    },\n    {\n      "word": "dirty",\n      "count": 7\n    },\n    {\n      "word": "disco",\n      "count": 7\n    },\n    {\n      "word": "dole",\n      "count": 7\n    },\n    {\n      "word": "doorstep",\n      "count": 7\n    },\n    {\n      "word": "dressage",\n      "count": 7\n    },\n    {\n      "word": "driftwood",\n      "count": 7\n    },\n    {\n      "word": "duo",\n      "count": 7\n    },\n    {\n      "word": "dusting",\n      "count": 7\n    },\n    {\n      "word": "duvet",\n      "count": 7\n    },\n    {\n      "word": "easter",\n      "count": 7\n    },\n    {\n      "word": "elegant",\n      "count": 7\n    },\n    {\n      "word": "enterprise",\n      "count": 7\n    },\n    {\n      "word": "euro",\n      "count": 7\n    },\n    {\n      "word": "experience",\n      "count": 7\n    },\n    {\n      "word": "fill",\n      "count": 7\n    },\n    {\n      "word": "fingertip",\n      "count": 7\n    },\n    {\n      "word": "fireplug",\n      "count": 7\n    },\n    {\n      "word": "first",\n      "count": 7\n    },\n    {\n      "word": "flagpole",\n      "count": 7\n    },\n    {\n      "word": "flipper",\n      "count": 7\n    },\n    {\n      "word": "floors",\n      "count": 7\n    },\n    {\n      "word": "fluorescent",\n      "count": 7\n    },\n    {\n      "word": "framing",\n      "count": 7\n    },\n    {\n      "word": "friday",\n      "count": 7\n    },\n    {\n      "word": "fried",\n      "count": 7\n    },\n    {\n      "word": "frisbees",\n      "count": 7\n    },\n    {\n      "word": "frito",\n      "count": 7\n    },\n    {\n      "word": "frizbe",\n      "count": 7\n    },\n    {\n      "word": "frontier",\n      "count": 7\n    },\n    {\n      "word": "funnel",\n      "count": 7\n    },\n    {\n      "word": "gatorade",\n      "count": 7\n    },\n    {\n      "word": "gauze",\n      "count": 7\n    },\n    {\n      "word": "geek",\n      "count": 7\n    },\n    {\n      "word": "gelato",\n      "count": 7\n    },\n    {\n      "word": "gerbil",\n      "count": 7\n    },\n    {\n      "word": "german",\n      "count": 7\n    },\n    {\n      "word": "gingham",\n      "count": 7\n    },\n    {\n      "word": "girraffe",\n      "count": 7\n    },\n    {\n      "word": "glimpse",\n      "count": 7\n    },\n    {\n      "word": "glob",\n      "count": 7\n    },\n    {\n      "word": "glory",\n      "count": 7\n    },\n    {\n      "word": "granddaughter",\n      "count": 7\n    },\n    {\n      "word": "grating",\n      "count": 7\n    },\n    {\n      "word": "great",\n      "count": 7\n    },\n    {\n      "word": "greet",\n      "count": 7\n    },\n    {\n      "word": "grizzle",\n      "count": 7\n    },\n    {\n      "word": "handset",\n      "count": 7\n    },\n    {\n      "word": "hashbrown",\n      "count": 7\n    },\n    {\n      "word": "headscarf",\n      "count": 7\n    },\n    {\n      "word": "heap",\n      "count": 7\n    },\n    {\n      "word": "hear",\n      "count": 7\n    },\n    {\n      "word": "heritage",\n      "count": 7\n    },\n    {\n      "word": "hide",\n      "count": 7\n    },\n    {\n      "word": "hitch",\n      "count": 7\n    },\n    {\n      "word": "hoody",\n      "count": 7\n    },\n    {\n      "word": "hummus",\n      "count": 7\n    },\n    {\n      "word": "hunting",\n      "count": 7\n    },\n    {\n      "word": "idiot",\n      "count": 7\n    },\n    {\n      "word": "initial",\n      "count": 7\n    },\n    {\n      "word": "inscription",\n      "count": 7\n    },\n    {\n      "word": "interface",\n      "count": 7\n    },\n    {\n      "word": "iris",\n      "count": 7\n    },\n    {\n      "word": "iv",\n      "count": 7\n    },\n    {\n      "word": "jeff",\n      "count": 7\n    },\n    {\n      "word": "jello",\n      "count": 7\n    },\n    {\n      "word": "kill",\n      "count": 7\n    },\n    {\n      "word": "knickknack",\n      "count": 7\n    },\n    {\n      "word": "lampshade",\n      "count": 7\n    },\n    {\n      "word": "layout",\n      "count": 7\n    },\n    {\n      "word": "learning",\n      "count": 7\n    },\n    {\n      "word": "lined",\n      "count": 7\n    },\n    {\n      "word": "literature",\n      "count": 7\n    },\n    {\n      "word": "longhorn",\n      "count": 7\n    },\n    {\n      "word": "luck",\n      "count": 7\n    },\n    {\n      "word": "madison",\n      "count": 7\n    },\n    {\n      "word": "malaysia",\n      "count": 7\n    },\n    {\n      "word": "manhattan",\n      "count": 7\n    },\n    {\n      "word": "margarita",\n      "count": 7\n    },\n    {\n      "word": "margherita",\n      "count": 7\n    },\n    {\n      "word": "maze",\n      "count": 7\n    },\n    {\n      "word": "mechanism",\n      "count": 7\n    },\n    {\n      "word": "memento",\n      "count": 7\n    },\n    {\n      "word": "metropolitan",\n      "count": 7\n    },\n    {\n      "word": "michael",\n      "count": 7\n    },\n    {\n      "word": "mission",\n      "count": 7\n    },\n    {\n      "word": "mixing",\n      "count": 7\n    },\n    {\n      "word": "monochrome",\n      "count": 7\n    },\n    {\n      "word": "monopoly",\n      "count": 7\n    },\n    {\n      "word": "mortar",\n      "count": 7\n    },\n    {\n      "word": "moth",\n      "count": 7\n    },\n    {\n      "word": "motionless",\n      "count": 7\n    },\n    {\n      "word": "mph",\n      "count": 7\n    },\n    {\n      "word": "nectar",\n      "count": 7\n    },\n    {\n      "word": "network",\n      "count": 7\n    },\n    {\n      "word": "numerals",\n      "count": 7\n    },\n    {\n      "word": "nuzzle",\n      "count": 7\n    },\n    {\n      "word": "nw",\n      "count": 7\n    },\n    {\n      "word": "offering",\n      "count": 7\n    },\n    {\n      "word": "oj",\n      "count": 7\n    },\n    {\n      "word": "op",\n      "count": 7\n    },\n    {\n      "word": "othman",\n      "count": 7\n    },\n    {\n      "word": "overtop",\n      "count": 7\n    },\n    {\n      "word": "parasailing",\n      "count": 7\n    },\n    {\n      "word": "parasailor",\n      "count": 7\n    },\n    {\n      "word": "pawn",\n      "count": 7\n    },\n    {\n      "word": "peddling",\n      "count": 7\n    },\n    {\n      "word": "peep",\n      "count": 7\n    },\n    {\n      "word": "peope",\n      "count": 7\n    },\n    {\n      "word": "peperoni",\n      "count": 7\n    },\n    {\n      "word": "perform",\n      "count": 7\n    },\n    {\n      "word": "persona",\n      "count": 7\n    },\n    {\n      "word": "philadelphia",\n      "count": 7\n    },\n    {\n      "word": "postage",\n      "count": 7\n    },\n    {\n      "word": "posture",\n      "count": 7\n    },\n    {\n      "word": "preserver",\n      "count": 7\n    },\n    {\n      "word": "pretend",\n      "count": 7\n    },\n    {\n      "word": "printing",\n      "count": 7\n    },\n    {\n      "word": "quad",\n      "count": 7\n    },\n    {\n      "word": "r",\n      "count": 7\n    },\n    {\n      "word": "rainy",\n      "count": 7\n    },\n    {\n      "word": "really",\n      "count": 7\n    },\n    {\n      "word": "receptacle",\n      "count": 7\n    },\n    {\n      "word": "recess",\n      "count": 7\n    },\n    {\n      "word": "ref",\n      "count": 7\n    },\n    {\n      "word": "refrigeration",\n      "count": 7\n    },\n    {\n      "word": "relaxing",\n      "count": 7\n    },\n    {\n      "word": "rocker",\n      "count": 7\n    },\n    {\n      "word": "rolls",\n      "count": 7\n    },\n    {\n      "word": "roost",\n      "count": 7\n    },\n    {\n      "word": "runaway",\n      "count": 7\n    },\n    {\n      "word": "safeway",\n      "count": 7\n    },\n    {\n      "word": "sailboard",\n      "count": 7\n    },\n    {\n      "word": "sailer",\n      "count": 7\n    },\n    {\n      "word": "sandwhiche",\n      "count": 7\n    },\n    {\n      "word": "schedule",\n      "count": 7\n    },\n    {\n      "word": "schoolbus",\n      "count": 7\n    },\n    {\n      "word": "screensaver",\n      "count": 7\n    },\n    {\n      "word": "screw",\n      "count": 7\n    },\n    {\n      "word": "seller",\n      "count": 7\n    },\n    {\n      "word": "sharpie",\n      "count": 7\n    },\n    {\n      "word": "shelfs",\n      "count": 7\n    },\n    {\n      "word": "sheriff",\n      "count": 7\n    },\n    {\n      "word": "sideburn",\n      "count": 7\n    },\n    {\n      "word": "sidwalk",\n      "count": 7\n    },\n    {\n      "word": "signing",\n      "count": 7\n    },\n    {\n      "word": "signs",\n      "count": 7\n    },\n    {\n      "word": "simpsons",\n      "count": 7\n    },\n    {\n      "word": "skatebaord",\n      "count": 7\n    },\n    {\n      "word": "smoothy",\n      "count": 7\n    },\n    {\n      "word": "snowball",\n      "count": 7\n    },\n    {\n      "word": "soccor",\n      "count": 7\n    },\n    {\n      "word": "solder",\n      "count": 7\n    },\n    {\n      "word": "souvenir",\n      "count": 7\n    },\n    {\n      "word": "spill",\n      "count": 7\n    },\n    {\n      "word": "squirt",\n      "count": 7\n    },\n    {\n      "word": "stature",\n      "count": 7\n    },\n    {\n      "word": "steve",\n      "count": 7\n    },\n    {\n      "word": "stitch",\n      "count": 7\n    },\n    {\n      "word": "strain",\n      "count": 7\n    },\n    {\n      "word": "stret",\n      "count": 7\n    },\n    {\n      "word": "sum",\n      "count": 7\n    },\n    {\n      "word": "sunday",\n      "count": 7\n    },\n    {\n      "word": "superman",\n      "count": 7\n    },\n    {\n      "word": "taker",\n      "count": 7\n    },\n    {\n      "word": "talbot",\n      "count": 7\n    },\n    {\n      "word": "ten",\n      "count": 7\n    },\n    {\n      "word": "thanksgiving",\n      "count": 7\n    },\n    {\n      "word": "three",\n      "count": 7\n    },\n    {\n      "word": "tigers",\n      "count": 7\n    },\n    {\n      "word": "tomatoes",\n      "count": 7\n    },\n    {\n      "word": "tortoise",\n      "count": 7\n    },\n    {\n      "word": "traffice",\n      "count": 7\n    },\n    {\n      "word": "trampoline",\n      "count": 7\n    },\n    {\n      "word": "trial",\n      "count": 7\n    },\n    {\n      "word": "umbella",\n      "count": 7\n    },\n    {\n      "word": "usage",\n      "count": 7\n    },\n    {\n      "word": "vapor",\n      "count": 7\n    },\n    {\n      "word": "vat",\n      "count": 7\n    },\n    {\n      "word": "vision",\n      "count": 7\n    },\n    {\n      "word": "wafer",\n      "count": 7\n    },\n    {\n      "word": "ward",\n      "count": 7\n    },\n    {\n      "word": "weathervane",\n      "count": 7\n    },\n    {\n      "word": "webpage",\n      "count": 7\n    },\n    {\n      "word": "wink",\n      "count": 7\n    },\n    {\n      "word": "wording",\n      "count": 7\n    },\n    {\n      "word": "workbench",\n      "count": 7\n    },\n    {\n      "word": "zip",\n      "count": 7\n    },\n    {\n      "word": "zodiac",\n      "count": 7\n    },\n    {\n      "word": "zookeeper",\n      "count": 7\n    },\n    {\n      "word": "abandon",\n      "count": 6\n    },\n    {\n      "word": "abraham",\n      "count": 6\n    },\n    {\n      "word": "acrobat",\n      "count": 6\n    },\n    {\n      "word": "advertise",\n      "count": 6\n    },\n    {\n      "word": "aim",\n      "count": 6\n    },\n    {\n      "word": "airplanes",\n      "count": 6\n    },\n    {\n      "word": "allot",\n      "count": 6\n    },\n    {\n      "word": "altar",\n      "count": 6\n    },\n    {\n      "word": "amazon",\n      "count": 6\n    },\n    {\n      "word": "approval",\n      "count": 6\n    },\n    {\n      "word": "arc",\n      "count": 6\n    },\n    {\n      "word": "assistance",\n      "count": 6\n    },\n    {\n      "word": "audio",\n      "count": 6\n    },\n    {\n      "word": "australian",\n      "count": 6\n    },\n    {\n      "word": "babe",\n      "count": 6\n    },\n    {\n      "word": "balancing",\n      "count": 6\n    },\n    {\n      "word": "balck",\n      "count": 6\n    },\n    {\n      "word": "ballroom",\n      "count": 6\n    },\n    {\n      "word": "barley",\n      "count": 6\n    },\n    {\n      "word": "bassinet",\n      "count": 6\n    },\n    {\n      "word": "bater",\n      "count": 6\n    },\n    {\n      "word": "beachfront",\n      "count": 6\n    },\n    {\n      "word": "beanbag",\n      "count": 6\n    },\n    {\n      "word": "benches",\n      "count": 6\n    },\n    {\n      "word": "bending",\n      "count": 6\n    },\n    {\n      "word": "bible",\n      "count": 6\n    },\n    {\n      "word": "bicycles",\n      "count": 6\n    },\n    {\n      "word": "binocular",\n      "count": 6\n    },\n    {\n      "word": "birdbath",\n      "count": 6\n    },\n    {\n      "word": "blaze",\n      "count": 6\n    },\n    {\n      "word": "boad",\n      "count": 6\n    },\n    {\n      "word": "boats",\n      "count": 6\n    },\n    {\n      "word": "bodyboard",\n      "count": 6\n    },\n    {\n      "word": "bog",\n      "count": 6\n    },\n    {\n      "word": "bolt",\n      "count": 6\n    },\n    {\n      "word": "bond",\n      "count": 6\n    },\n    {\n      "word": "boombox",\n      "count": 6\n    },\n    {\n      "word": "bouncy",\n      "count": 6\n    },\n    {\n      "word": "bourbon",\n      "count": 6\n    },\n    {\n      "word": "bout",\n      "count": 6\n    },\n    {\n      "word": "boutique",\n      "count": 6\n    },\n    {\n      "word": "bowls",\n      "count": 6\n    },\n    {\n      "word": "bratwurst",\n      "count": 6\n    },\n    {\n      "word": "braves",\n      "count": 6\n    },\n    {\n      "word": "bridal",\n      "count": 6\n    },\n    {\n      "word": "brocoli",\n      "count": 6\n    },\n    {\n      "word": "bullpen",\n      "count": 6\n    },\n    {\n      "word": "bunkbed",\n      "count": 6\n    },\n    {\n      "word": "buttery",\n      "count": 6\n    },\n    {\n      "word": "cakes",\n      "count": 6\n    },\n    {\n      "word": "cal",\n      "count": 6\n    },\n    {\n      "word": "calorie",\n      "count": 6\n    },\n    {\n      "word": "caricature",\n      "count": 6\n    },\n    {\n      "word": "carport",\n      "count": 6\n    },\n    {\n      "word": "carraige",\n      "count": 6\n    },\n    {\n      "word": "carseat",\n      "count": 6\n    },\n    {\n      "word": "catalog",\n      "count": 6\n    },\n    {\n      "word": "caterpillar",\n      "count": 6\n    },\n    {\n      "word": "chaise",\n      "count": 6\n    },\n    {\n      "word": "chamber",\n      "count": 6\n    },\n    {\n      "word": "cheeto",\n      "count": 6\n    },\n    {\n      "word": "chemical",\n      "count": 6\n    },\n    {\n      "word": "chickpea",\n      "count": 6\n    },\n    {\n      "word": "chilie",\n      "count": 6\n    },\n    {\n      "word": "christ",\n      "count": 6\n    },\n    {\n      "word": "clinic",\n      "count": 6\n    },\n    {\n      "word": "collision",\n      "count": 6\n    },\n    {\n      "word": "combat",\n      "count": 6\n    },\n    {\n      "word": "comparison",\n      "count": 6\n    },\n    {\n      "word": "concoction",\n      "count": 6\n    },\n    {\n      "word": "continental",\n      "count": 6\n    },\n    {\n      "word": "cookout",\n      "count": 6\n    },\n    {\n      "word": "corsage",\n      "count": 6\n    },\n    {\n      "word": "cough",\n      "count": 6\n    },\n    {\n      "word": "coup",\n      "count": 6\n    },\n    {\n      "word": "coupon",\n      "count": 6\n    },\n    {\n      "word": "coverall",\n      "count": 6\n    },\n    {\n      "word": "coworker",\n      "count": 6\n    },\n    {\n      "word": "crater",\n      "count": 6\n    },\n    {\n      "word": "creamer",\n      "count": 6\n    },\n    {\n      "word": "crumble",\n      "count": 6\n    },\n    {\n      "word": "crush",\n      "count": 6\n    },\n    {\n      "word": "cups",\n      "count": 6\n    },\n    {\n      "word": "curly",\n      "count": 6\n    },\n    {\n      "word": "dart",\n      "count": 6\n    },\n    {\n      "word": "del",\n      "count": 6\n    },\n    {\n      "word": "delicious",\n      "count": 6\n    },\n    {\n      "word": "depiction",\n      "count": 6\n    },\n    {\n      "word": "detailing",\n      "count": 6\n    },\n    {\n      "word": "detergent",\n      "count": 6\n    },\n    {\n      "word": "devil",\n      "count": 6\n    },\n    {\n      "word": "dick",\n      "count": 6\n    },\n    {\n      "word": "dig",\n      "count": 6\n    },\n    {\n      "word": "diorama",\n      "count": 6\n    },\n    {\n      "word": "distant",\n      "count": 6\n    },\n    {\n      "word": "division",\n      "count": 6\n    },\n    {\n      "word": "downward",\n      "count": 6\n    },\n    {\n      "word": "drier",\n      "count": 6\n    },\n    {\n      "word": "drip",\n      "count": 6\n    },\n    {\n      "word": "drug",\n      "count": 6\n    },\n    {\n      "word": "duke",\n      "count": 6\n    },\n    {\n      "word": "dwarf",\n      "count": 6\n    },\n    {\n      "word": "easy",\n      "count": 6\n    },\n    {\n      "word": "elder",\n      "count": 6\n    },\n    {\n      "word": "ensemble",\n      "count": 6\n    },\n    {\n      "word": "entertainer",\n      "count": 6\n    },\n    {\n      "word": "error",\n      "count": 6\n    },\n    {\n      "word": "eva",\n      "count": 6\n    },\n    {\n      "word": "expo",\n      "count": 6\n    },\n    {\n      "word": "extra",\n      "count": 6\n    },\n    {\n      "word": "facing",\n      "count": 6\n    },\n    {\n      "word": "fairground",\n      "count": 6\n    },\n    {\n      "word": "feathery",\n      "count": 6\n    },\n    {\n      "word": "fee",\n      "count": 6\n    },\n    {\n      "word": "fillet",\n      "count": 6\n    },\n    {\n      "word": "finch",\n      "count": 6\n    },\n    {\n      "word": "flags",\n      "count": 6\n    },\n    {\n      "word": "flair",\n      "count": 6\n    },\n    {\n      "word": "flap",\n      "count": 6\n    },\n    {\n      "word": "fleece",\n      "count": 6\n    },\n    {\n      "word": "flesh",\n      "count": 6\n    },\n    {\n      "word": "gargoyle",\n      "count": 6\n    },\n    {\n      "word": "gin",\n      "count": 6\n    },\n    {\n      "word": "glance",\n      "count": 6\n    },\n    {\n      "word": "glassy",\n      "count": 6\n    },\n    {\n      "word": "golfer",\n      "count": 6\n    },\n    {\n      "word": "gong",\n      "count": 6\n    },\n    {\n      "word": "grave",\n      "count": 6\n    },\n    {\n      "word": "gril",\n      "count": 6\n    },\n    {\n      "word": "grille",\n      "count": 6\n    },\n    {\n      "word": "grime",\n      "count": 6\n    },\n    {\n      "word": "gummy",\n      "count": 6\n    },\n    {\n      "word": "gump",\n      "count": 6\n    },\n    {\n      "word": "gyro",\n      "count": 6\n    },\n    {\n      "word": "halo",\n      "count": 6\n    },\n    {\n      "word": "harbour",\n      "count": 6\n    },\n    {\n      "word": "hatchback",\n      "count": 6\n    },\n    {\n      "word": "haze",\n      "count": 6\n    },\n    {\n      "word": "he",\n      "count": 6\n    },\n    {\n      "word": "her",\n      "count": 6\n    },\n    {\n      "word": "highlight",\n      "count": 6\n    },\n    {\n      "word": "hilton",\n      "count": 6\n    },\n    {\n      "word": "himself",\n      "count": 6\n    },\n    {\n      "word": "homemade",\n      "count": 6\n    },\n    {\n      "word": "hotplate",\n      "count": 6\n    },\n    {\n      "word": "hugging",\n      "count": 6\n    },\n    {\n      "word": "hunch",\n      "count": 6\n    },\n    {\n      "word": "hyrdant",\n      "count": 6\n    },\n    {\n      "word": "illusion",\n      "count": 6\n    },\n    {\n      "word": "impersonator",\n      "count": 6\n    },\n    {\n      "word": "impression",\n      "count": 6\n    },\n    {\n      "word": "indiana",\n      "count": 6\n    },\n    {\n      "word": "indians",\n      "count": 6\n    },\n    {\n      "word": "interaction",\n      "count": 6\n    },\n    {\n      "word": "issue",\n      "count": 6\n    },\n    {\n      "word": "jay",\n      "count": 6\n    },\n    {\n      "word": "jogger",\n      "count": 6\n    },\n    {\n      "word": "johns",\n      "count": 6\n    },\n    {\n      "word": "jordan",\n      "count": 6\n    },\n    {\n      "word": "juvenile",\n      "count": 6\n    },\n    {\n      "word": "karaoke",\n      "count": 6\n    },\n    {\n      "word": "kawasaki",\n      "count": 6\n    },\n    {\n      "word": "kernel",\n      "count": 6\n    },\n    {\n      "word": "kerry",\n      "count": 6\n    },\n    {\n      "word": "keypad",\n      "count": 6\n    },\n    {\n      "word": "keys",\n      "count": 6\n    },\n    {\n      "word": "kitcehn",\n      "count": 6\n    },\n    {\n      "word": "kitchenware",\n      "count": 6\n    },\n    {\n      "word": "kleenex",\n      "count": 6\n    },\n    {\n      "word": "lack",\n      "count": 6\n    },\n    {\n      "word": "lapt",\n      "count": 6\n    },\n    {\n      "word": "laser",\n      "count": 6\n    },\n    {\n      "word": "leader",\n      "count": 6\n    },\n    {\n      "word": "leave",\n      "count": 6\n    },\n    {\n      "word": "led",\n      "count": 6\n    },\n    {\n      "word": "lg",\n      "count": 6\n    },\n    {\n      "word": "lifejacket",\n      "count": 6\n    },\n    {\n      "word": "lifting",\n      "count": 6\n    },\n    {\n      "word": "limousine",\n      "count": 6\n    },\n    {\n      "word": "loafs",\n      "count": 6\n    },\n    {\n      "word": "loan",\n      "count": 6\n    },\n    {\n      "word": "longboarder",\n      "count": 6\n    },\n    {\n      "word": "lookout",\n      "count": 6\n    },\n    {\n      "word": "lounger",\n      "count": 6\n    },\n    {\n      "word": "lower",\n      "count": 6\n    },\n    {\n      "word": "manchester",\n      "count": 6\n    },\n    {\n      "word": "manicure",\n      "count": 6\n    },\n    {\n      "word": "mardi",\n      "count": 6\n    },\n    {\n      "word": "marilyn",\n      "count": 6\n    },\n    {\n      "word": "martin",\n      "count": 6\n    },\n    {\n      "word": "martini",\n      "count": 6\n    },\n    {\n      "word": "matt",\n      "count": 6\n    },\n    {\n      "word": "mauve",\n      "count": 6\n    },\n    {\n      "word": "measurement",\n      "count": 6\n    },\n    {\n      "word": "meme",\n      "count": 6\n    },\n    {\n      "word": "minnesota",\n      "count": 6\n    },\n    {\n      "word": "minnie",\n      "count": 6\n    },\n    {\n      "word": "missing",\n      "count": 6\n    },\n    {\n      "word": "modern",\n      "count": 6\n    },\n    {\n      "word": "mold",\n      "count": 6\n    },\n    {\n      "word": "month",\n      "count": 6\n    },\n    {\n      "word": "mosaic",\n      "count": 6\n    },\n    {\n      "word": "motorcade",\n      "count": 6\n    },\n    {\n      "word": "motorhome",\n      "count": 6\n    },\n    {\n      "word": "mover",\n      "count": 6\n    },\n    {\n      "word": "mussel",\n      "count": 6\n    },\n    {\n      "word": "myspace",\n      "count": 6\n    },\n    {\n      "word": "mystery",\n      "count": 6\n    },\n    {\n      "word": "nack",\n      "count": 6\n    },\n    {\n      "word": "nativity",\n      "count": 6\n    },\n    {\n      "word": "neighbor",\n      "count": 6\n    },\n    {\n      "word": "nerf",\n      "count": 6\n    },\n    {\n      "word": "none",\n      "count": 6\n    },\n    {\n      "word": "ny",\n      "count": 6\n    },\n    {\n      "word": "nypd",\n      "count": 6\n    },\n    {\n      "word": "otter",\n      "count": 6\n    },\n    {\n      "word": "over",\n      "count": 6\n    },\n    {\n      "word": "padding",\n      "count": 6\n    },\n    {\n      "word": "pakistan",\n      "count": 6\n    },\n    {\n      "word": "pale",\n      "count": 6\n    },\n    {\n      "word": "pancakes",\n      "count": 6\n    },\n    {\n      "word": "parcel",\n      "count": 6\n    },\n    {\n      "word": "passageway",\n      "count": 6\n    },\n    {\n      "word": "paver",\n      "count": 6\n    },\n    {\n      "word": "pda",\n      "count": 6\n    },\n    {\n      "word": "peddle",\n      "count": 6\n    },\n    {\n      "word": "peice",\n      "count": 6\n    },\n    {\n      "word": "pharmacy",\n      "count": 6\n    },\n    {\n      "word": "pinwheel",\n      "count": 6\n    },\n    {\n      "word": "planner",\n      "count": 6\n    },\n    {\n      "word": "pokemon",\n      "count": 6\n    },\n    {\n      "word": "poking",\n      "count": 6\n    },\n    {\n      "word": "poultry",\n      "count": 6\n    },\n    {\n      "word": "procedure",\n      "count": 6\n    },\n    {\n      "word": "production",\n      "count": 6\n    },\n    {\n      "word": "pursuit",\n      "count": 6\n    },\n    {\n      "word": "rad",\n      "count": 6\n    },\n    {\n      "word": "rank",\n      "count": 6\n    },\n    {\n      "word": "re",\n      "count": 6\n    },\n    {\n      "word": "refigerator",\n      "count": 6\n    },\n    {\n      "word": "restuarant",\n      "count": 6\n    },\n    {\n      "word": "rhode",\n      "count": 6\n    },\n    {\n      "word": "rom",\n      "count": 6\n    },\n    {\n      "word": "roman",\n      "count": 6\n    },\n    {\n      "word": "ronald",\n      "count": 6\n    },\n    {\n      "word": "router",\n      "count": 6\n    },\n    {\n      "word": "routine",\n      "count": 6\n    },\n    {\n      "word": "ruby",\n      "count": 6\n    },\n    {\n      "word": "rump",\n      "count": 6\n    },\n    {\n      "word": "russian",\n      "count": 6\n    },\n    {\n      "word": "saddlebag",\n      "count": 6\n    },\n    {\n      "word": "salesman",\n      "count": 6\n    },\n    {\n      "word": "sam",\n      "count": 6\n    },\n    {\n      "word": "seatbelt",\n      "count": 6\n    },\n    {\n      "word": "seeing",\n      "count": 6\n    },\n    {\n      "word": "sell",\n      "count": 6\n    },\n    {\n      "word": "semis",\n      "count": 6\n    },\n    {\n      "word": "senior",\n      "count": 6\n    },\n    {\n      "word": "sensor",\n      "count": 6\n    },\n    {\n      "word": "shock",\n      "count": 6\n    },\n    {\n      "word": "sideboard",\n      "count": 6\n    },\n    {\n      "word": "sideway",\n      "count": 6\n    },\n    {\n      "word": "sighn",\n      "count": 6\n    },\n    {\n      "word": "sin",\n      "count": 6\n    },\n    {\n      "word": "singapore",\n      "count": 6\n    },\n    {\n      "word": "skat",\n      "count": 6\n    },\n    {\n      "word": "skiff",\n      "count": 6\n    },\n    {\n      "word": "slew",\n      "count": 6\n    },\n    {\n      "word": "sliced",\n      "count": 6\n    },\n    {\n      "word": "slow",\n      "count": 6\n    },\n    {\n      "word": "smart",\n      "count": 6\n    },\n    {\n      "word": "smokestack",\n      "count": 6\n    },\n    {\n      "word": "snowfall",\n      "count": 6\n    },\n    {\n      "word": "soaking",\n      "count": 6\n    },\n    {\n      "word": "soccerball",\n      "count": 6\n    },\n    {\n      "word": "spam",\n      "count": 6\n    },\n    {\n      "word": "spandex",\n      "count": 6\n    },\n    {\n      "word": "spigot",\n      "count": 6\n    },\n    {\n      "word": "stagecoach",\n      "count": 6\n    },\n    {\n      "word": "stemware",\n      "count": 6\n    },\n    {\n      "word": "stencil",\n      "count": 6\n    },\n    {\n      "word": "stethoscope",\n      "count": 6\n    },\n    {\n      "word": "streetlamp",\n      "count": 6\n    },\n    {\n      "word": "stud",\n      "count": 6\n    },\n    {\n      "word": "stuffing",\n      "count": 6\n    },\n    {\n      "word": "stylus",\n      "count": 6\n    },\n    {\n      "word": "sundown",\n      "count": 6\n    },\n    {\n      "word": "sunshade",\n      "count": 6\n    },\n    {\n      "word": "surge",\n      "count": 6\n    },\n    {\n      "word": "survey",\n      "count": 6\n    },\n    {\n      "word": "swab",\n      "count": 6\n    },\n    {\n      "word": "swarm",\n      "count": 6\n    },\n    {\n      "word": "sync",\n      "count": 6\n    },\n    {\n      "word": "tagging",\n      "count": 6\n    },\n    {\n      "word": "ted",\n      "count": 6\n    },\n    {\n      "word": "teeshirt",\n      "count": 6\n    },\n    {\n      "word": "th",\n      "count": 6\n    },\n    {\n      "word": "thatch",\n      "count": 6\n    },\n    {\n      "word": "thinking",\n      "count": 6\n    },\n    {\n      "word": "tick",\n      "count": 6\n    },\n    {\n      "word": "toa",\n      "count": 6\n    },\n    {\n      "word": "tom",\n      "count": 6\n    },\n    {\n      "word": "tomatoe",\n      "count": 6\n    },\n    {\n      "word": "too",\n      "count": 6\n    },\n    {\n      "word": "toolbox",\n      "count": 6\n    },\n    {\n      "word": "townhouse",\n      "count": 6\n    },\n    {\n      "word": "treatment",\n      "count": 6\n    },\n    {\n      "word": "treeline",\n      "count": 6\n    },\n    {\n      "word": "troop",\n      "count": 6\n    },\n    {\n      "word": "twinkie",\n      "count": 6\n    },\n    {\n      "word": "upstairs",\n      "count": 6\n    },\n    {\n      "word": "vegitable",\n      "count": 6\n    },\n    {\n      "word": "vehicles",\n      "count": 6\n    },\n    {\n      "word": "ventilation",\n      "count": 6\n    },\n    {\n      "word": "vice",\n      "count": 6\n    },\n    {\n      "word": "villager",\n      "count": 6\n    },\n    {\n      "word": "warthog",\n      "count": 6\n    },\n    {\n      "word": "waterboard",\n      "count": 6\n    },\n    {\n      "word": "waterhole",\n      "count": 6\n    },\n    {\n      "word": "wavy",\n      "count": 6\n    },\n    {\n      "word": "weave",\n      "count": 6\n    },\n    {\n      "word": "whiile",\n      "count": 6\n    },\n    {\n      "word": "whole",\n      "count": 6\n    },\n    {\n      "word": "williams",\n      "count": 6\n    },\n    {\n      "word": "windsock",\n      "count": 6\n    },\n    {\n      "word": "woodwork",\n      "count": 6\n    },\n    {\n      "word": "wrinkle",\n      "count": 6\n    },\n    {\n      "word": "yam",\n      "count": 6\n    },\n    {\n      "word": "yankee",\n      "count": 6\n    },\n    {\n      "word": "yawn",\n      "count": 6\n    },\n    {\n      "word": "zeebra",\n      "count": 6\n    },\n    {\n      "word": "adapter",\n      "count": 5\n    },\n    {\n      "word": "advert",\n      "count": 5\n    },\n    {\n      "word": "aer",\n      "count": 5\n    },\n    {\n      "word": "airy",\n      "count": 5\n    },\n    {\n      "word": "alien",\n      "count": 5\n    },\n    {\n      "word": "alight",\n      "count": 5\n    },\n    {\n      "word": "all",\n      "count": 5\n    },\n    {\n      "word": "alphabet",\n      "count": 5\n    },\n    {\n      "word": "americans",\n      "count": 5\n    },\n    {\n      "word": "amoco",\n      "count": 5\n    },\n    {\n      "word": "animals",\n      "count": 5\n    },\n    {\n      "word": "announcement",\n      "count": 5\n    },\n    {\n      "word": "anther",\n      "count": 5\n    },\n    {\n      "word": "applie",\n      "count": 5\n    },\n    {\n      "word": "appreciation",\n      "count": 5\n    },\n    {\n      "word": "arbor",\n      "count": 5\n    },\n    {\n      "word": "arctic",\n      "count": 5\n    },\n    {\n      "word": "argyle",\n      "count": 5\n    },\n    {\n      "word": "astro",\n      "count": 5\n    },\n    {\n      "word": "astronaut",\n      "count": 5\n    },\n    {\n      "word": "astroturf",\n      "count": 5\n    },\n    {\n      "word": "attic",\n      "count": 5\n    },\n    {\n      "word": "author",\n      "count": 5\n    },\n    {\n      "word": "autism",\n      "count": 5\n    },\n    {\n      "word": "backback",\n      "count": 5\n    },\n    {\n      "word": "backhoe",\n      "count": 5\n    },\n    {\n      "word": "ballgame",\n      "count": 5\n    },\n    {\n      "word": "ballplayer",\n      "count": 5\n    },\n    {\n      "word": "banan",\n      "count": 5\n    },\n    {\n      "word": "barrack",\n      "count": 5\n    },\n    {\n      "word": "barry",\n      "count": 5\n    },\n    {\n      "word": "bathe",\n      "count": 5\n    },\n    {\n      "word": "bathmat",\n      "count": 5\n    },\n    {\n      "word": "beachgoer",\n      "count": 5\n    },\n    {\n      "word": "beater",\n      "count": 5\n    },\n    {\n      "word": "bedsheet",\n      "count": 5\n    },\n    {\n      "word": "benedict",\n      "count": 5\n    },\n    {\n      "word": "beware",\n      "count": 5\n    },\n    {\n      "word": "biathlon",\n      "count": 5\n    },\n    {\n      "word": "birdseed",\n      "count": 5\n    },\n    {\n      "word": "bitch",\n      "count": 5\n    },\n    {\n      "word": "blackbird",\n      "count": 5\n    },\n    {\n      "word": "blindfold",\n      "count": 5\n    },\n    {\n      "word": "bmx",\n      "count": 5\n    },\n    {\n      "word": "bologna",\n      "count": 5\n    },\n    {\n      "word": "bound",\n      "count": 5\n    },\n    {\n      "word": "boyfriend",\n      "count": 5\n    },\n    {\n      "word": "bp",\n      "count": 5\n    },\n    {\n      "word": "brave",\n      "count": 5\n    },\n    {\n      "word": "breathing",\n      "count": 5\n    },\n    {\n      "word": "brocclie",\n      "count": 5\n    },\n    {\n      "word": "broiler",\n      "count": 5\n    },\n    {\n      "word": "brook",\n      "count": 5\n    },\n    {\n      "word": "brooklyn",\n      "count": 5\n    },\n    {\n      "word": "burgundy",\n      "count": 5\n    },\n    {\n      "word": "buying",\n      "count": 5\n    },\n    {\n      "word": "cacti",\n      "count": 5\n    },\n    {\n      "word": "campbell",\n      "count": 5\n    },\n    {\n      "word": "capability",\n      "count": 5\n    },\n    {\n      "word": "caper",\n      "count": 5\n    },\n    {\n      "word": "cask",\n      "count": 5\n    },\n    {\n      "word": "cassette",\n      "count": 5\n    },\n    {\n      "word": "catnip",\n      "count": 5\n    },\n    {\n      "word": "cats",\n      "count": 5\n    },\n    {\n      "word": "cedar",\n      "count": 5\n    },\n    {\n      "word": "cessna",\n      "count": 5\n    },\n    {\n      "word": "charlie",\n      "count": 5\n    },\n    {\n      "word": "chatting",\n      "count": 5\n    },\n    {\n      "word": "chee",\n      "count": 5\n    },\n    {\n      "word": "chief",\n      "count": 5\n    },\n    {\n      "word": "chill",\n      "count": 5\n    },\n    {\n      "word": "cleaver",\n      "count": 5\n    },\n    {\n      "word": "clouds",\n      "count": 5\n    },\n    {\n      "word": "coffeemaker",\n      "count": 5\n    },\n    {\n      "word": "coil",\n      "count": 5\n    },\n    {\n      "word": "collaboration",\n      "count": 5\n    },\n    {\n      "word": "collard",\n      "count": 5\n    },\n    {\n      "word": "cologne",\n      "count": 5\n    },\n    {\n      "word": "colorado",\n      "count": 5\n    },\n    {\n      "word": "commercial",\n      "count": 5\n    },\n    {\n      "word": "compilation",\n      "count": 5\n    },\n    {\n      "word": "concern",\n      "count": 5\n    },\n    {\n      "word": "concourse",\n      "count": 5\n    },\n    {\n      "word": "condensation",\n      "count": 5\n    },\n    {\n      "word": "congress",\n      "count": 5\n    },\n    {\n      "word": "consumer",\n      "count": 5\n    },\n    {\n      "word": "conversing",\n      "count": 5\n    },\n    {\n      "word": "copse",\n      "count": 5\n    },\n    {\n      "word": "cornfield",\n      "count": 5\n    },\n    {\n      "word": "cpu",\n      "count": 5\n    },\n    {\n      "word": "crockery",\n      "count": 5\n    },\n    {\n      "word": "cronut",\n      "count": 5\n    },\n    {\n      "word": "crosstown",\n      "count": 5\n    },\n    {\n      "word": "cuff",\n      "count": 5\n    },\n    {\n      "word": "curiosity",\n      "count": 5\n    },\n    {\n      "word": "decade",\n      "count": 5\n    },\n    {\n      "word": "decline",\n      "count": 5\n    },\n    {\n      "word": "defender",\n      "count": 5\n    },\n    {\n      "word": "defense",\n      "count": 5\n    },\n    {\n      "word": "degree",\n      "count": 5\n    },\n    {\n      "word": "delicacy",\n      "count": 5\n    },\n    {\n      "word": "demo",\n      "count": 5\n    },\n    {\n      "word": "devise",\n      "count": 5\n    },\n    {\n      "word": "dhl",\n      "count": 5\n    },\n    {\n      "word": "die",\n      "count": 5\n    },\n    {\n      "word": "disarray",\n      "count": 5\n    },\n    {\n      "word": "doge",\n      "count": 5\n    },\n    {\n      "word": "don",\n      "count": 5\n    },\n    {\n      "word": "dork",\n      "count": 5\n    },\n    {\n      "word": "dormitory",\n      "count": 5\n    },\n    {\n      "word": "duel",\n      "count": 5\n    },\n    {\n      "word": "dynamite",\n      "count": 5\n    },\n    {\n      "word": "eaten",\n      "count": 5\n    },\n    {\n      "word": "emu",\n      "count": 5\n    },\n    {\n      "word": "enjoyment",\n      "count": 5\n    },\n    {\n      "word": "entourage",\n      "count": 5\n    },\n    {\n      "word": "entre",\n      "count": 5\n    },\n    {\n      "word": "exception",\n      "count": 5\n    },\n    {\n      "word": "exiting",\n      "count": 5\n    },\n    {\n      "word": "expressway",\n      "count": 5\n    },\n    {\n      "word": "extreme",\n      "count": 5\n    },\n    {\n      "word": "eyebrow",\n      "count": 5\n    },\n    {\n      "word": "fancy",\n      "count": 5\n    },\n    {\n      "word": "fantasy",\n      "count": 5\n    },\n    {\n      "word": "fare",\n      "count": 5\n    },\n    {\n      "word": "fast",\n      "count": 5\n    },\n    {\n      "word": "fastball",\n      "count": 5\n    },\n    {\n      "word": "fece",\n      "count": 5\n    },\n    {\n      "word": "federer",\n      "count": 5\n    },\n    {\n      "word": "festivity",\n      "count": 5\n    },\n    {\n      "word": "feta",\n      "count": 5\n    },\n    {\n      "word": "fettuccine",\n      "count": 5\n    },\n    {\n      "word": "fiction",\n      "count": 5\n    },\n    {\n      "word": "filet",\n      "count": 5\n    },\n    {\n      "word": "fixin",\n      "count": 5\n    },\n    {\n      "word": "flare",\n      "count": 5\n    },\n    {\n      "word": "flatware",\n      "count": 5\n    },\n    {\n      "word": "flotation",\n      "count": 5\n    },\n    {\n      "word": "flusher",\n      "count": 5\n    },\n    {\n      "word": "foaming",\n      "count": 5\n    },\n    {\n      "word": "footboard",\n      "count": 5\n    },\n    {\n      "word": "footbridge",\n      "count": 5\n    },\n    {\n      "word": "formica",\n      "count": 5\n    },\n    {\n      "word": "forrest",\n      "count": 5\n    },\n    {\n      "word": "foundation",\n      "count": 5\n    },\n    {\n      "word": "freightliner",\n      "count": 5\n    },\n    {\n      "word": "fresh",\n      "count": 5\n    },\n    {\n      "word": "fribee",\n      "count": 5\n    },\n    {\n      "word": "fringe",\n      "count": 5\n    },\n    {\n      "word": "frittata",\n      "count": 5\n    },\n    {\n      "word": "frond",\n      "count": 5\n    },\n    {\n      "word": "gallop",\n      "count": 5\n    },\n    {\n      "word": "gamer",\n      "count": 5\n    },\n    {\n      "word": "garbanzo",\n      "count": 5\n    },\n    {\n      "word": "gateway",\n      "count": 5\n    },\n    {\n      "word": "gather",\n      "count": 5\n    },\n    {\n      "word": "gel",\n      "count": 5\n    },\n    {\n      "word": "generation",\n      "count": 5\n    },\n    {\n      "word": "girafe",\n      "count": 5\n    },\n    {\n      "word": "gladiator",\n      "count": 5\n    },\n    {\n      "word": "goldfish",\n      "count": 5\n    },\n    {\n      "word": "gordon",\n      "count": 5\n    },\n    {\n      "word": "graffit",\n      "count": 5\n    },\n    {\n      "word": "graffito",\n      "count": 5\n    },\n    {\n      "word": "grandson",\n      "count": 5\n    },\n    {\n      "word": "gras",\n      "count": 5\n    },\n    {\n      "word": "greek",\n      "count": 5\n    },\n    {\n      "word": "greenwich",\n      "count": 5\n    },\n    {\n      "word": "gren",\n      "count": 5\n    },\n    {\n      "word": "griaffe",\n      "count": 5\n    },\n    {\n      "word": "grinning",\n      "count": 5\n    },\n    {\n      "word": "guitarist",\n      "count": 5\n    },\n    {\n      "word": "guys",\n      "count": 5\n    },\n    {\n      "word": "haired",\n      "count": 5\n    },\n    {\n      "word": "hairstyle",\n      "count": 5\n    },\n    {\n      "word": "harvest",\n      "count": 5\n    },\n    {\n      "word": "haunch",\n      "count": 5\n    },\n    {\n      "word": "hd",\n      "count": 5\n    },\n    {\n      "word": "heading",\n      "count": 5\n    },\n    {\n      "word": "headpiece",\n      "count": 5\n    },\n    {\n      "word": "helping",\n      "count": 5\n    },\n    {\n      "word": "henry",\n      "count": 5\n    },\n    {\n      "word": "hippopotamus",\n      "count": 5\n    },\n    {\n      "word": "historic",\n      "count": 5\n    },\n    {\n      "word": "homer",\n      "count": 5\n    },\n    {\n      "word": "homework",\n      "count": 5\n    },\n    {\n      "word": "hop",\n      "count": 5\n    },\n    {\n      "word": "houses",\n      "count": 5\n    },\n    {\n      "word": "hunk",\n      "count": 5\n    },\n    {\n      "word": "hunt",\n      "count": 5\n    },\n    {\n      "word": "hydrangea",\n      "count": 5\n    },\n    {\n      "word": "ibm",\n      "count": 5\n    },\n    {\n      "word": "improvement",\n      "count": 5\n    },\n    {\n      "word": "index",\n      "count": 5\n    },\n    {\n      "word": "indian",\n      "count": 5\n    },\n    {\n      "word": "insignia",\n      "count": 5\n    },\n    {\n      "word": "inspection",\n      "count": 5\n    },\n    {\n      "word": "intel",\n      "count": 5\n    },\n    {\n      "word": "interview",\n      "count": 5\n    },\n    {\n      "word": "ion",\n      "count": 5\n    },\n    {\n      "word": "jal",\n      "count": 5\n    },\n    {\n      "word": "jalapeno",\n      "count": 5\n    },\n    {\n      "word": "james",\n      "count": 5\n    },\n    {\n      "word": "jobs",\n      "count": 5\n    },\n    {\n      "word": "journal",\n      "count": 5\n    },\n    {\n      "word": "kfc",\n      "count": 5\n    },\n    {\n      "word": "khakis",\n      "count": 5\n    },\n    {\n      "word": "kingfisher",\n      "count": 5\n    },\n    {\n      "word": "kitche",\n      "count": 5\n    },\n    {\n      "word": "kneck",\n      "count": 5\n    },\n    {\n      "word": "lacy",\n      "count": 5\n    },\n    {\n      "word": "laden",\n      "count": 5\n    },\n    {\n      "word": "leaving",\n      "count": 5\n    },\n    {\n      "word": "legging",\n      "count": 5\n    },\n    {\n      "word": "lentil",\n      "count": 5\n    },\n    {\n      "word": "let",\n      "count": 5\n    },\n    {\n      "word": "letting",\n      "count": 5\n    },\n    {\n      "word": "liter",\n      "count": 5\n    },\n    {\n      "word": "littel",\n      "count": 5\n    },\n    {\n      "word": "llamas",\n      "count": 5\n    },\n    {\n      "word": "loaded",\n      "count": 5\n    },\n    {\n      "word": "luke",\n      "count": 5\n    },\n    {\n      "word": "luther",\n      "count": 5\n    },\n    {\n      "word": "macaw",\n      "count": 5\n    },\n    {\n      "word": "maine",\n      "count": 5\n    },\n    {\n      "word": "mangos",\n      "count": 5\n    },\n    {\n      "word": "manuever",\n      "count": 5\n    },\n    {\n      "word": "manufacturer",\n      "count": 5\n    },\n    {\n      "word": "manufacturing",\n      "count": 5\n    },\n    {\n      "word": "marching",\n      "count": 5\n    },\n    {\n      "word": "mariners",\n      "count": 5\n    },\n    {\n      "word": "marriage",\n      "count": 5\n    },\n    {\n      "word": "meaning",\n      "count": 5\n    },\n    {\n      "word": "medallion",\n      "count": 5\n    },\n    {\n      "word": "memo",\n      "count": 5\n    },\n    {\n      "word": "mermaid",\n      "count": 5\n    },\n    {\n      "word": "metallic",\n      "count": 5\n    },\n    {\n      "word": "miami",\n      "count": 5\n    },\n    {\n      "word": "microscope",\n      "count": 5\n    },\n    {\n      "word": "midday",\n      "count": 5\n    },\n    {\n      "word": "mike",\n      "count": 5\n    },\n    {\n      "word": "miller",\n      "count": 5\n    },\n    {\n      "word": "mm",\n      "count": 5\n    },\n    {\n      "word": "mmm",\n      "count": 5\n    },\n    {\n      "word": "mommy",\n      "count": 5\n    },\n    {\n      "word": "monica",\n      "count": 5\n    },\n    {\n      "word": "monroe",\n      "count": 5\n    },\n    {\n      "word": "monstrosity",\n      "count": 5\n    },\n    {\n      "word": "moonlight",\n      "count": 5\n    },\n    {\n      "word": "morgan",\n      "count": 5\n    },\n    {\n      "word": "motercycle",\n      "count": 5\n    },\n    {\n      "word": "moto",\n      "count": 5\n    },\n    {\n      "word": "motorized",\n      "count": 5\n    },\n    {\n      "word": "mt",\n      "count": 5\n    },\n    {\n      "word": "naan",\n      "count": 5\n    },\n    {\n      "word": "napping",\n      "count": 5\n    },\n    {\n      "word": "nathan",\n      "count": 5\n    },\n    {\n      "word": "nathans",\n      "count": 5\n    },\n    {\n      "word": "nation",\n      "count": 5\n    },\n    {\n      "word": "nd",\n      "count": 5\n    },\n    {\n      "word": "newborn",\n      "count": 5\n    },\n    {\n      "word": "next",\n      "count": 5\n    },\n    {\n      "word": "noise",\n      "count": 5\n    },\n    {\n      "word": "nyc",\n      "count": 5\n    },\n    {\n      "word": "oasis",\n      "count": 5\n    },\n    {\n      "word": "obstruction",\n      "count": 5\n    },\n    {\n      "word": "onesie",\n      "count": 5\n    },\n    {\n      "word": "onlooking",\n      "count": 5\n    },\n    {\n      "word": "onstage",\n      "count": 5\n    },\n    {\n      "word": "ont",\n      "count": 5\n    },\n    {\n      "word": "opportunity",\n      "count": 5\n    },\n    {\n      "word": "oranges",\n      "count": 5\n    },\n    {\n      "word": "organic",\n      "count": 5\n    },\n    {\n      "word": "outing",\n      "count": 5\n    },\n    {\n      "word": "pac",\n      "count": 5\n    },\n    {\n      "word": "paddington",\n      "count": 5\n    },\n    {\n      "word": "paddleboard",\n      "count": 5\n    },\n    {\n      "word": "palette",\n      "count": 5\n    },\n    {\n      "word": "paraglider",\n      "count": 5\n    },\n    {\n      "word": "parasurfer",\n      "count": 5\n    },\n    {\n      "word": "pasadena",\n      "count": 5\n    },\n    {\n      "word": "pastel",\n      "count": 5\n    },\n    {\n      "word": "payment",\n      "count": 5\n    },\n    {\n      "word": "peepee",\n      "count": 5\n    },\n    {\n      "word": "penalty",\n      "count": 5\n    },\n    {\n      "word": "peopl",\n      "count": 5\n    },\n    {\n      "word": "peple",\n      "count": 5\n    },\n    {\n      "word": "perched",\n      "count": 5\n    },\n    {\n      "word": "personal",\n      "count": 5\n    },\n    {\n      "word": "phillies",\n      "count": 5\n    },\n    {\n      "word": "phillips",\n      "count": 5\n    },\n    {\n      "word": "pickel",\n      "count": 5\n    },\n    {\n      "word": "picutre",\n      "count": 5\n    },\n    {\n      "word": "piercing",\n      "count": 5\n    },\n    {\n      "word": "pikachu",\n      "count": 5\n    },\n    {\n      "word": "placard",\n      "count": 5\n    },\n    {\n      "word": "plateau",\n      "count": 5\n    },\n    {\n      "word": "poeple",\n      "count": 5\n    },\n    {\n      "word": "porridge",\n      "count": 5\n    },\n    {\n      "word": "porter",\n      "count": 5\n    },\n    {\n      "word": "potholder",\n      "count": 5\n    },\n    {\n      "word": "pots",\n      "count": 5\n    },\n    {\n      "word": "prarie",\n      "count": 5\n    },\n    {\n      "word": "presence",\n      "count": 5\n    },\n    {\n      "word": "priority",\n      "count": 5\n    },\n    {\n      "word": "programming",\n      "count": 5\n    },\n    {\n      "word": "pullman",\n      "count": 5\n    },\n    {\n      "word": "pulp",\n      "count": 5\n    },\n    {\n      "word": "queens",\n      "count": 5\n    },\n    {\n      "word": "rainforest",\n      "count": 5\n    },\n    {\n      "word": "ramekin",\n      "count": 5\n    },\n    {\n      "word": "randy",\n      "count": 5\n    },\n    {\n      "word": "raptor",\n      "count": 5\n    },\n    {\n      "word": "ravine",\n      "count": 5\n    },\n    {\n      "word": "recovery",\n      "count": 5\n    },\n    {\n      "word": "reel",\n      "count": 5\n    },\n    {\n      "word": "refueling",\n      "count": 5\n    },\n    {\n      "word": "representation",\n      "count": 5\n    },\n    {\n      "word": "representative",\n      "count": 5\n    },\n    {\n      "word": "reptile",\n      "count": 5\n    },\n    {\n      "word": "reserve",\n      "count": 5\n    },\n    {\n      "word": "reservoir",\n      "count": 5\n    },\n    {\n      "word": "response",\n      "count": 5\n    },\n    {\n      "word": "restraunt",\n      "count": 5\n    },\n    {\n      "word": "review",\n      "count": 5\n    },\n    {\n      "word": "richmond",\n      "count": 5\n    },\n    {\n      "word": "rickety",\n      "count": 5\n    },\n    {\n      "word": "roaster",\n      "count": 5\n    },\n    {\n      "word": "rockefeller",\n      "count": 5\n    },\n    {\n      "word": "roger",\n      "count": 5\n    },\n    {\n      "word": "rollerblade",\n      "count": 5\n    },\n    {\n      "word": "rover",\n      "count": 5\n    },\n    {\n      "word": "ruffle",\n      "count": 5\n    },\n    {\n      "word": "rush",\n      "count": 5\n    },\n    {\n      "word": "saab",\n      "count": 5\n    },\n    {\n      "word": "sac",\n      "count": 5\n    },\n    {\n      "word": "sandwiches",\n      "count": 5\n    },\n    {\n      "word": "sardine",\n      "count": 5\n    },\n    {\n      "word": "sas",\n      "count": 5\n    },\n    {\n      "word": "scaffold",\n      "count": 5\n    },\n    {\n      "word": "scar",\n      "count": 5\n    },\n    {\n      "word": "schoolhouse",\n      "count": 5\n    },\n    {\n      "word": "scotland",\n      "count": 5\n    },\n    {\n      "word": "scott",\n      "count": 5\n    },\n    {\n      "word": "screenshot",\n      "count": 5\n    },\n    {\n      "word": "script",\n      "count": 5\n    },\n    {\n      "word": "segway",\n      "count": 5\n    },\n    {\n      "word": "servicemen",\n      "count": 5\n    },\n    {\n      "word": "settler",\n      "count": 5\n    },\n    {\n      "word": "seuss",\n      "count": 5\n    },\n    {\n      "word": "shakespeare",\n      "count": 5\n    },\n    {\n      "word": "sharing",\n      "count": 5\n    },\n    {\n      "word": "showerhead",\n      "count": 5\n    },\n    {\n      "word": "shut",\n      "count": 5\n    },\n    {\n      "word": "skatebord",\n      "count": 5\n    },\n    {\n      "word": "skiboard",\n      "count": 5\n    },\n    {\n      "word": "skyteam",\n      "count": 5\n    },\n    {\n      "word": "smoker",\n      "count": 5\n    },\n    {\n      "word": "snowflake",\n      "count": 5\n    },\n    {\n      "word": "spaceship",\n      "count": 5\n    },\n    {\n      "word": "sparkle",\n      "count": 5\n    },\n    {\n      "word": "speedometer",\n      "count": 5\n    },\n    {\n      "word": "spending",\n      "count": 5\n    },\n    {\n      "word": "spinning",\n      "count": 5\n    },\n    {\n      "word": "squatting",\n      "count": 5\n    },\n    {\n      "word": "squint",\n      "count": 5\n    },\n    {\n      "word": "standng",\n      "count": 5\n    },\n    {\n      "word": "stapler",\n      "count": 5\n    },\n    {\n      "word": "statuette",\n      "count": 5\n    },\n    {\n      "word": "status",\n      "count": 5\n    },\n    {\n      "word": "stevens",\n      "count": 5\n    },\n    {\n      "word": "sticking",\n      "count": 5\n    },\n    {\n      "word": "stillness",\n      "count": 5\n    },\n    {\n      "word": "stirrer",\n      "count": 5\n    },\n    {\n      "word": "streaming",\n      "count": 5\n    },\n    {\n      "word": "stree",\n      "count": 5\n    },\n    {\n      "word": "strength",\n      "count": 5\n    },\n    {\n      "word": "sucker",\n      "count": 5\n    },\n    {\n      "word": "sunbeam",\n      "count": 5\n    },\n    {\n      "word": "swat",\n      "count": 5\n    },\n    {\n      "word": "sweatpant",\n      "count": 5\n    },\n    {\n      "word": "swiss",\n      "count": 5\n    },\n    {\n      "word": "taillight",\n      "count": 5\n    },\n    {\n      "word": "tampa",\n      "count": 5\n    },\n    {\n      "word": "tarmack",\n      "count": 5\n    },\n    {\n      "word": "tassel",\n      "count": 5\n    },\n    {\n      "word": "tax",\n      "count": 5\n    },\n    {\n      "word": "taxiway",\n      "count": 5\n    },\n    {\n      "word": "teeter",\n      "count": 5\n    },\n    {\n      "word": "telegraph",\n      "count": 5\n    },\n    {\n      "word": "teresa",\n      "count": 5\n    },\n    {\n      "word": "testing",\n      "count": 5\n    },\n    {\n      "word": "texture",\n      "count": 5\n    },\n    {\n      "word": "thailand",\n      "count": 5\n    },\n    {\n      "word": "thrift",\n      "count": 5\n    },\n    {\n      "word": "throat",\n      "count": 5\n    },\n    {\n      "word": "tightrope",\n      "count": 5\n    },\n    {\n      "word": "tilting",\n      "count": 5\n    },\n    {\n      "word": "tinsel",\n      "count": 5\n    },\n    {\n      "word": "tog",\n      "count": 5\n    },\n    {\n      "word": "toilets",\n      "count": 5\n    },\n    {\n      "word": "tombstone",\n      "count": 5\n    },\n    {\n      "word": "tonight",\n      "count": 5\n    },\n    {\n      "word": "topless",\n      "count": 5\n    },\n    {\n      "word": "toronto",\n      "count": 5\n    },\n    {\n      "word": "tortillas",\n      "count": 5\n    },\n    {\n      "word": "trainyard",\n      "count": 5\n    },\n    {\n      "word": "traveling",\n      "count": 5\n    },\n    {\n      "word": "treetop",\n      "count": 5\n    },\n    {\n      "word": "treetops",\n      "count": 5\n    },\n    {\n      "word": "trelli",\n      "count": 5\n    },\n    {\n      "word": "trombone",\n      "count": 5\n    },\n    {\n      "word": "trucker",\n      "count": 5\n    },\n    {\n      "word": "trundle",\n      "count": 5\n    },\n    {\n      "word": "trust",\n      "count": 5\n    },\n    {\n      "word": "tubing",\n      "count": 5\n    },\n    {\n      "word": "tuft",\n      "count": 5\n    },\n    {\n      "word": "tweed",\n      "count": 5\n    },\n    {\n      "word": "tweezer",\n      "count": 5\n    },\n    {\n      "word": "twitter",\n      "count": 5\n    },\n    {\n      "word": "umberella",\n      "count": 5\n    },\n    {\n      "word": "unicorn",\n      "count": 5\n    },\n    {\n      "word": "unicycle",\n      "count": 5\n    },\n    {\n      "word": "uphill",\n      "count": 5\n    },\n    {\n      "word": "urinals",\n      "count": 5\n    },\n    {\n      "word": "valance",\n      "count": 5\n    },\n    {\n      "word": "villa",\n      "count": 5\n    },\n    {\n      "word": "violet",\n      "count": 5\n    },\n    {\n      "word": "virginia",\n      "count": 5\n    },\n    {\n      "word": "vote",\n      "count": 5\n    },\n    {\n      "word": "walsk",\n      "count": 5\n    },\n    {\n      "word": "warplane",\n      "count": 5\n    },\n    {\n      "word": "warrior",\n      "count": 5\n    },\n    {\n      "word": "watcher",\n      "count": 5\n    },\n    {\n      "word": "waterski",\n      "count": 5\n    },\n    {\n      "word": "wayland",\n      "count": 5\n    },\n    {\n      "word": "weenie",\n      "count": 5\n    },\n    {\n      "word": "weiner",\n      "count": 5\n    },\n    {\n      "word": "whle",\n      "count": 5\n    },\n    {\n      "word": "widescreen",\n      "count": 5\n    },\n    {\n      "word": "william",\n      "count": 5\n    },\n    {\n      "word": "wilson",\n      "count": 5\n    },\n    {\n      "word": "wiper",\n      "count": 5\n    },\n    {\n      "word": "woma",\n      "count": 5\n    },\n    {\n      "word": "workman",\n      "count": 5\n    },\n    {\n      "word": "workout",\n      "count": 5\n    },\n    {\n      "word": "wrestle",\n      "count": 5\n    },\n    {\n      "word": "wwii",\n      "count": 5\n    },\n    {\n      "word": "yawning",\n      "count": 5\n    },\n    {\n      "word": "zips",\n      "count": 5\n    }\n  ],\n  "verb": [\n    {\n      "word": "sit",\n      "count": 63068\n    },\n    {\n      "word": "stand",\n      "count": 46558\n    },\n    {\n      "word": "hold",\n      "count": 30649\n    },\n    {\n      "word": "ride",\n      "count": 19368\n    },\n    {\n      "word": "walk",\n      "count": 16701\n    },\n    {\n      "word": "look",\n      "count": 15651\n    },\n    {\n      "word": "play",\n      "count": 15623\n    },\n    {\n      "word": "park",\n      "count": 11651\n    },\n    {\n      "word": "fly",\n      "count": 11430\n    },\n    {\n      "word": "eat",\n      "count": 10783\n    },\n    {\n      "word": "wear",\n      "count": 9908\n    },\n    {\n      "word": "lay",\n      "count": 9902\n    },\n    {\n      "word": "cover",\n      "count": 7111\n    },\n    {\n      "word": "fill",\n      "count": 6371\n    },\n    {\n      "word": "take",\n      "count": 6215\n    },\n    {\n      "word": "show",\n      "count": 4917\n    },\n    {\n      "word": "watch",\n      "count": 4889\n    },\n    {\n      "word": "drive",\n      "count": 4269\n    },\n    {\n      "word": "make",\n      "count": 4206\n    },\n    {\n      "word": "hit",\n      "count": 4156\n    },\n    {\n      "word": "pose",\n      "count": 4082\n    },\n    {\n      "word": "go",\n      "count": 4018\n    },\n    {\n      "word": "hang",\n      "count": 3904\n    },\n    {\n      "word": "swing",\n      "count": 3889\n    },\n    {\n      "word": "get",\n      "count": 3785\n    },\n    {\n      "word": "use",\n      "count": 3718\n    },\n    {\n      "word": "smile",\n      "count": 3622\n    },\n    {\n      "word": "cut",\n      "count": 3616\n    },\n    {\n      "word": "jump",\n      "count": 3520\n    },\n    {\n      "word": "stuff",\n      "count": 3438\n    },\n    {\n      "word": "travel",\n      "count": 3302\n    },\n    {\n      "word": "graze",\n      "count": 3254\n    },\n    {\n      "word": "do",\n      "count": 3249\n    },\n    {\n      "word": "prepare",\n      "count": 3162\n    },\n    {\n      "word": "pull",\n      "count": 3147\n    },\n    {\n      "word": "talk",\n      "count": 3117\n    },\n    {\n      "word": "carry",\n      "count": 2885\n    },\n    {\n      "word": "wait",\n      "count": 2837\n    },\n    {\n      "word": "ski",\n      "count": 2800\n    },\n    {\n      "word": "run",\n      "count": 2791\n    },\n    {\n      "word": "top",\n      "count": 2756\n    },\n    {\n      "word": "lie",\n      "count": 2730\n    },\n    {\n      "word": "throw",\n      "count": 2728\n    },\n    {\n      "word": "catch",\n      "count": 2350\n    },\n    {\n      "word": "set",\n      "count": 2210\n    },\n    {\n      "word": "stop",\n      "count": 2165\n    },\n    {\n      "word": "surround",\n      "count": 2130\n    },\n    {\n      "word": "display",\n      "count": 2096\n    },\n    {\n      "word": "dress",\n      "count": 2095\n    },\n    {\n      "word": "sleep",\n      "count": 2067\n    },\n    {\n      "word": "serve",\n      "count": 2060\n    },\n    {\n      "word": "surf",\n      "count": 1972\n    },\n    {\n      "word": "see",\n      "count": 1967\n    },\n    {\n      "word": "line",\n      "count": 1951\n    },\n    {\n      "word": "lean",\n      "count": 1896\n    },\n    {\n      "word": "cross",\n      "count": 1837\n    },\n    {\n      "word": "come",\n      "count": 1835\n    },\n    {\n      "word": "decorate",\n      "count": 1823\n    },\n    {\n      "word": "rest",\n      "count": 1805\n    },\n    {\n      "word": "pass",\n      "count": 1655\n    },\n    {\n      "word": "paint",\n      "count": 1548\n    },\n    {\n      "word": "read",\n      "count": 1548\n    },\n    {\n      "word": "light",\n      "count": 1520\n    },\n    {\n      "word": "work",\n      "count": 1497\n    },\n    {\n      "word": "try",\n      "count": 1480\n    },\n    {\n      "word": "attach",\n      "count": 1477\n    },\n    {\n      "word": "drink",\n      "count": 1473\n    },\n    {\n      "word": "move",\n      "count": 1459\n    },\n    {\n      "word": "include",\n      "count": 1418\n    },\n    {\n      "word": "contain",\n      "count": 1394\n    },\n    {\n      "word": "gather",\n      "count": 1394\n    },\n    {\n      "word": "tie",\n      "count": 1369\n    },\n    {\n      "word": "perform",\n      "count": 1352\n    },\n    {\n      "word": "cook",\n      "count": 1296\n    },\n    {\n      "word": "place",\n      "count": 1279\n    },\n    {\n      "word": "brush",\n      "count": 1242\n    },\n    {\n      "word": "skateboard",\n      "count": 1227\n    },\n    {\n      "word": "enjoy",\n      "count": 1215\n    },\n    {\n      "word": "perch",\n      "count": 1198\n    },\n    {\n      "word": "put",\n      "count": 1196\n    },\n    {\n      "word": "reach",\n      "count": 1193\n    },\n    {\n      "word": "stick",\n      "count": 1142\n    },\n    {\n      "word": "have",\n      "count": 1137\n    },\n    {\n      "word": "feed",\n      "count": 1040\n    },\n    {\n      "word": "mount",\n      "count": 999\n    },\n    {\n      "word": "tile",\n      "count": 986\n    },\n    {\n      "word": "fence",\n      "count": 903\n    },\n    {\n      "word": "float",\n      "count": 903\n    },\n    {\n      "word": "leave",\n      "count": 856\n    },\n    {\n      "word": "face",\n      "count": 833\n    },\n    {\n      "word": "feature",\n      "count": 826\n    },\n    {\n      "word": "open",\n      "count": 821\n    },\n    {\n      "word": "turn",\n      "count": 813\n    },\n    {\n      "word": "say",\n      "count": 806\n    },\n    {\n      "word": "star",\n      "count": 792\n    },\n    {\n      "word": "shape",\n      "count": 780\n    },\n    {\n      "word": "fall",\n      "count": 714\n    },\n    {\n      "word": "dock",\n      "count": 700\n    },\n    {\n      "word": "lead",\n      "count": 699\n    },\n    {\n      "word": "stack",\n      "count": 699\n    },\n    {\n      "word": "draw",\n      "count": 692\n    },\n    {\n      "word": "seat",\n      "count": 678\n    },\n    {\n      "word": "give",\n      "count": 677\n    },\n    {\n      "word": "appear",\n      "count": 665\n    },\n    {\n      "word": "bear",\n      "count": 647\n    },\n    {\n      "word": "overlook",\n      "count": 627\n    },\n    {\n      "word": "load",\n      "count": 622\n    },\n    {\n      "word": "live",\n      "count": 614\n    },\n    {\n      "word": "write",\n      "count": 613\n    },\n    {\n      "word": "point",\n      "count": 610\n    },\n    {\n      "word": "slice",\n      "count": 600\n    },\n    {\n      "word": "pick",\n      "count": 596\n    },\n    {\n      "word": "arrange",\n      "count": 589\n    },\n    {\n      "word": "woode",\n      "count": 589\n    },\n    {\n      "word": "kick",\n      "count": 579\n    },\n    {\n      "word": "pet",\n      "count": 568\n    },\n    {\n      "word": "build",\n      "count": 553\n    },\n    {\n      "word": "snowboard",\n      "count": 526\n    },\n    {\n      "word": "approach",\n      "count": 517\n    },\n    {\n      "word": "touch",\n      "count": 513\n    },\n    {\n      "word": "wrap",\n      "count": 511\n    },\n    {\n      "word": "attempt",\n      "count": 502\n    },\n    {\n      "word": "color",\n      "count": 502\n    },\n    {\n      "word": "board",\n      "count": 500\n    },\n    {\n      "word": "head",\n      "count": 487\n    },\n    {\n      "word": "kneel",\n      "count": 479\n    },\n    {\n      "word": "relax",\n      "count": 470\n    },\n    {\n      "word": "slide",\n      "count": 470\n    },\n    {\n      "word": "swim",\n      "count": 464\n    },\n    {\n      "word": "grow",\n      "count": 463\n    },\n    {\n      "word": "pitch",\n      "count": 456\n    },\n    {\n      "word": "sink",\n      "count": 451\n    },\n    {\n      "word": "race",\n      "count": 446\n    },\n    {\n      "word": "oven",\n      "count": 445\n    },\n    {\n      "word": "blow",\n      "count": 443\n    },\n    {\n      "word": "bend",\n      "count": 434\n    },\n    {\n      "word": "check",\n      "count": 434\n    },\n    {\n      "word": "push",\n      "count": 433\n    },\n    {\n      "word": "sell",\n      "count": 433\n    },\n    {\n      "word": "sign",\n      "count": 425\n    },\n    {\n      "word": "reflect",\n      "count": 423\n    },\n    {\n      "word": "skate",\n      "count": 421\n    },\n    {\n      "word": "follow",\n      "count": 420\n    },\n    {\n      "word": "roll",\n      "count": 402\n    },\n    {\n      "word": "close",\n      "count": 399\n    },\n    {\n      "word": "pour",\n      "count": 391\n    },\n    {\n      "word": "return",\n      "count": 391\n    },\n    {\n      "word": "practice",\n      "count": 385\n    },\n    {\n      "word": "stretch",\n      "count": 385\n    },\n    {\n      "word": "help",\n      "count": 382\n    },\n    {\n      "word": "post",\n      "count": 375\n    },\n    {\n      "word": "frost",\n      "count": 366\n    },\n    {\n      "word": "miss",\n      "count": 362\n    },\n    {\n      "word": "curl",\n      "count": 359\n    },\n    {\n      "word": "clean",\n      "count": 357\n    },\n    {\n      "word": "bake",\n      "count": 354\n    },\n    {\n      "word": "enclose",\n      "count": 350\n    },\n    {\n      "word": "break",\n      "count": 342\n    },\n    {\n      "word": "shin",\n      "count": 342\n    },\n    {\n      "word": "share",\n      "count": 337\n    },\n    {\n      "word": "climb",\n      "count": 333\n    },\n    {\n      "word": "raise",\n      "count": 333\n    },\n    {\n      "word": "land",\n      "count": 331\n    },\n    {\n      "word": "glaze",\n      "count": 330\n    },\n    {\n      "word": "pile",\n      "count": 329\n    },\n    {\n      "word": "grab",\n      "count": 319\n    },\n    {\n      "word": "bite",\n      "count": 317\n    },\n    {\n      "word": "laugh",\n      "count": 312\n    },\n    {\n      "word": "pack",\n      "count": 309\n    },\n    {\n      "word": "chop",\n      "count": 305\n    },\n    {\n      "word": "balance",\n      "count": 301\n    },\n    {\n      "word": "toss",\n      "count": 299\n    },\n    {\n      "word": "dry",\n      "count": 298\n    },\n    {\n      "word": "strip",\n      "count": 297\n    },\n    {\n      "word": "grill",\n      "count": 291\n    },\n    {\n      "word": "seem",\n      "count": 289\n    },\n    {\n      "word": "bat",\n      "count": 286\n    },\n    {\n      "word": "picture",\n      "count": 282\n    },\n    {\n      "word": "view",\n      "count": 282\n    },\n    {\n      "word": "spread",\n      "count": 281\n    },\n    {\n      "word": "sniff",\n      "count": 274\n    },\n    {\n      "word": "hide",\n      "count": 273\n    },\n    {\n      "word": "shake",\n      "count": 272\n    },\n    {\n      "word": "train",\n      "count": 268\n    },\n    {\n      "word": "fix",\n      "count": 263\n    },\n    {\n      "word": "fold",\n      "count": 257\n    },\n    {\n      "word": "hug",\n      "count": 257\n    },\n    {\n      "word": "keep",\n      "count": 253\n    },\n    {\n      "word": "chase",\n      "count": 252\n    },\n    {\n      "word": "connect",\n      "count": 251\n    },\n    {\n      "word": "lounge",\n      "count": 251\n    },\n    {\n      "word": "lick",\n      "count": 250\n    },\n    {\n      "word": "locate",\n      "count": 248\n    },\n    {\n      "word": "peel",\n      "count": 248\n    },\n    {\n      "word": "fry",\n      "count": 247\n    },\n    {\n      "word": "shower",\n      "count": 245\n    },\n    {\n      "word": "sail",\n      "count": 244\n    },\n    {\n      "word": "tow",\n      "count": 243\n    },\n    {\n      "word": "enter",\n      "count": 241\n    },\n    {\n      "word": "rise",\n      "count": 238\n    },\n    {\n      "word": "extend",\n      "count": 237\n    },\n    {\n      "word": "consist",\n      "count": 235\n    },\n    {\n      "word": "observe",\n      "count": 232\n    },\n    {\n      "word": "indicate",\n      "count": 231\n    },\n    {\n      "word": "depict",\n      "count": 226\n    },\n    {\n      "word": "design",\n      "count": 224\n    },\n    {\n      "word": "prop",\n      "count": 224\n    },\n    {\n      "word": "crouch",\n      "count": 220\n    },\n    {\n      "word": "chew",\n      "count": 217\n    },\n    {\n      "word": "wash",\n      "count": 217\n    },\n    {\n      "word": "block",\n      "count": 216\n    },\n    {\n      "word": "type",\n      "count": 216\n    },\n    {\n      "word": "peek",\n      "count": 213\n    },\n    {\n      "word": "paddle",\n      "count": 212\n    },\n    {\n      "word": "leap",\n      "count": 209\n    },\n    {\n      "word": "ice",\n      "count": 208\n    },\n    {\n      "word": "lift",\n      "count": 208\n    },\n    {\n      "word": "adjust",\n      "count": 203\n    },\n    {\n      "word": "start",\n      "count": 202\n    },\n    {\n      "word": "speak",\n      "count": 198\n    },\n    {\n      "word": "celebrate",\n      "count": 196\n    },\n    {\n      "word": "abandon",\n      "count": 195\n    },\n    {\n      "word": "spray",\n      "count": 194\n    },\n    {\n      "word": "flip",\n      "count": 193\n    },\n    {\n      "word": "fight",\n      "count": 192\n    },\n    {\n      "word": "grind",\n      "count": 192\n    },\n    {\n      "word": "hand",\n      "count": 192\n    },\n    {\n      "word": "checker",\n      "count": 188\n    },\n    {\n      "word": "herd",\n      "count": 188\n    },\n    {\n      "word": "mix",\n      "count": 187\n    },\n    {\n      "word": "compete",\n      "count": 186\n    },\n    {\n      "word": "photograph",\n      "count": 184\n    },\n    {\n      "word": "smoke",\n      "count": 184\n    },\n    {\n      "word": "crowd",\n      "count": 183\n    },\n    {\n      "word": "roam",\n      "count": 182\n    },\n    {\n      "word": "wad",\n      "count": 181\n    },\n    {\n      "word": "tower",\n      "count": 178\n    },\n    {\n      "word": "wind",\n      "count": 178\n    },\n    {\n      "word": "wave",\n      "count": 176\n    },\n    {\n      "word": "multicolore",\n      "count": 174\n    },\n    {\n      "word": "scatter",\n      "count": 174\n    },\n    {\n      "word": "position",\n      "count": 172\n    },\n    {\n      "word": "suspend",\n      "count": 172\n    },\n    {\n      "word": "kiss",\n      "count": 171\n    },\n    {\n      "word": "await",\n      "count": 170\n    },\n    {\n      "word": "begin",\n      "count": 170\n    },\n    {\n      "word": "let",\n      "count": 170\n    },\n    {\n      "word": "learn",\n      "count": 168\n    },\n    {\n      "word": "engage",\n      "count": 167\n    },\n    {\n      "word": "kit",\n      "count": 167\n    },\n    {\n      "word": "adorn",\n      "count": 166\n    },\n    {\n      "word": "operate",\n      "count": 162\n    },\n    {\n      "word": "need",\n      "count": 158\n    },\n    {\n      "word": "match",\n      "count": 154\n    },\n    {\n      "word": "peer",\n      "count": 154\n    },\n    {\n      "word": "strike",\n      "count": 154\n    },\n    {\n      "word": "squat",\n      "count": 152\n    },\n    {\n      "word": "shop",\n      "count": 151\n    },\n    {\n      "word": "step",\n      "count": 151\n    },\n    {\n      "word": "burn",\n      "count": 150\n    },\n    {\n      "word": "haul",\n      "count": 150\n    },\n    {\n      "word": "organize",\n      "count": 147\n    },\n    {\n      "word": "arrive",\n      "count": 146\n    },\n    {\n      "word": "mark",\n      "count": 146\n    },\n    {\n      "word": "row",\n      "count": 146\n    },\n    {\n      "word": "remove",\n      "count": 144\n    },\n    {\n      "word": "direct",\n      "count": 141\n    },\n    {\n      "word": "fork",\n      "count": 140\n    },\n    {\n      "word": "hook",\n      "count": 140\n    },\n    {\n      "word": "stir",\n      "count": 139\n    },\n    {\n      "word": "rub",\n      "count": 136\n    },\n    {\n      "word": "listen",\n      "count": 134\n    },\n    {\n      "word": "crash",\n      "count": 133\n    },\n    {\n      "word": "finish",\n      "count": 133\n    },\n    {\n      "word": "frame",\n      "count": 133\n    },\n    {\n      "word": "poke",\n      "count": 133\n    },\n    {\n      "word": "tell",\n      "count": 132\n    },\n    {\n      "word": "illuminate",\n      "count": 129\n    },\n    {\n      "word": "mirror",\n      "count": 129\n    },\n    {\n      "word": "s",\n      "count": 129\n    },\n    {\n      "word": "chain",\n      "count": 128\n    },\n    {\n      "word": "call",\n      "count": 127\n    },\n    {\n      "word": "examine",\n      "count": 127\n    },\n    {\n      "word": "tag",\n      "count": 127\n    },\n    {\n      "word": "broccoli",\n      "count": 126\n    },\n    {\n      "word": "provide",\n      "count": 126\n    },\n    {\n      "word": "strap",\n      "count": 126\n    },\n    {\n      "word": "flower",\n      "count": 124\n    },\n    {\n      "word": "wander",\n      "count": 123\n    },\n    {\n      "word": "guide",\n      "count": 122\n    },\n    {\n      "word": "theme",\n      "count": 122\n    },\n    {\n      "word": "huddle",\n      "count": 121\n    },\n    {\n      "word": "cool",\n      "count": 119\n    },\n    {\n      "word": "taste",\n      "count": 119\n    },\n    {\n      "word": "unload",\n      "count": 119\n    },\n    {\n      "word": "teach",\n      "count": 118\n    },\n    {\n      "word": "interact",\n      "count": 117\n    },\n    {\n      "word": "cuddle",\n      "count": 116\n    },\n    {\n      "word": "warn",\n      "count": 116\n    },\n    {\n      "word": "furnish",\n      "count": 115\n    },\n    {\n      "word": "advertise",\n      "count": 114\n    },\n    {\n      "word": "carve",\n      "count": 114\n    },\n    {\n      "word": "rock",\n      "count": 114\n    },\n    {\n      "word": "buy",\n      "count": 113\n    },\n    {\n      "word": "separate",\n      "count": 113\n    },\n    {\n      "word": "tend",\n      "count": 113\n    },\n    {\n      "word": "snuggle",\n      "count": 110\n    },\n    {\n      "word": "vase",\n      "count": 110\n    },\n    {\n      "word": "add",\n      "count": 109\n    },\n    {\n      "word": "glow",\n      "count": 108\n    },\n    {\n      "word": "pattern",\n      "count": 108\n    },\n    {\n      "word": "inspect",\n      "count": 107\n    },\n    {\n      "word": "control",\n      "count": 106\n    },\n    {\n      "word": "deliver",\n      "count": 106\n    },\n    {\n      "word": "present",\n      "count": 106\n    },\n    {\n      "word": "find",\n      "count": 105\n    },\n    {\n      "word": "gaze",\n      "count": 104\n    },\n    {\n      "word": "lettuce",\n      "count": 104\n    },\n    {\n      "word": "lock",\n      "count": 104\n    },\n    {\n      "word": "round",\n      "count": 103\n    },\n    {\n      "word": "rust",\n      "count": 103\n    },\n    {\n      "word": "cast",\n      "count": 101\n    },\n    {\n      "word": "drop",\n      "count": 100\n    },\n    {\n      "word": "shave",\n      "count": 100\n    },\n    {\n      "word": "soar",\n      "count": 100\n    },\n    {\n      "word": "forest",\n      "count": 99\n    },\n    {\n      "word": "handle",\n      "count": 99\n    },\n    {\n      "word": "stove",\n      "count": 99\n    },\n    {\n      "word": "greet",\n      "count": 97\n    },\n    {\n      "word": "horn",\n      "count": 97\n    },\n    {\n      "word": "tap",\n      "count": 97\n    },\n    {\n      "word": "transport",\n      "count": 97\n    },\n    {\n      "word": "change",\n      "count": 96\n    },\n    {\n      "word": "sauce",\n      "count": 96\n    },\n    {\n      "word": "shade",\n      "count": 96\n    },\n    {\n      "word": "speed",\n      "count": 96\n    },\n    {\n      "word": "state",\n      "count": 96\n    },\n    {\n      "word": "glide",\n      "count": 95\n    },\n    {\n      "word": "smell",\n      "count": 95\n    },\n    {\n      "word": "sprinkle",\n      "count": 95\n    },\n    {\n      "word": "shoot",\n      "count": 94\n    },\n    {\n      "word": "soak",\n      "count": 94\n    },\n    {\n      "word": "tuck",\n      "count": 94\n    },\n    {\n      "word": "group",\n      "count": 93\n    },\n    {\n      "word": "orange",\n      "count": 93\n    },\n    {\n      "word": "poise",\n      "count": 93\n    },\n    {\n      "word": "clothe",\n      "count": 92\n    },\n    {\n      "word": "plat",\n      "count": 92\n    },\n    {\n      "word": "scratch",\n      "count": 92\n    },\n    {\n      "word": "tiere",\n      "count": 92\n    },\n    {\n      "word": "whip",\n      "count": 92\n    },\n    {\n      "word": "deck",\n      "count": 91\n    },\n    {\n      "word": "hike",\n      "count": 91\n    },\n    {\n      "word": "label",\n      "count": 91\n    },\n    {\n      "word": "moor",\n      "count": 91\n    },\n    {\n      "word": "allow",\n      "count": 90\n    },\n    {\n      "word": "melt",\n      "count": 90\n    },\n    {\n      "word": "bed",\n      "count": 89\n    },\n    {\n      "word": "knit",\n      "count": 89\n    },\n    {\n      "word": "cap",\n      "count": 88\n    },\n    {\n      "word": "equip",\n      "count": 88\n    },\n    {\n      "word": "instal",\n      "count": 88\n    },\n    {\n      "word": "rid",\n      "count": 88\n    },\n    {\n      "word": "bath",\n      "count": 87\n    },\n    {\n      "word": "bow",\n      "count": 87\n    },\n    {\n      "word": "pay",\n      "count": 87\n    },\n    {\n      "word": "press",\n      "count": 87\n    },\n    {\n      "word": "admire",\n      "count": 86\n    },\n    {\n      "word": "attend",\n      "count": 86\n    },\n    {\n      "word": "capture",\n      "count": 86\n    },\n    {\n      "word": "dance",\n      "count": 86\n    },\n    {\n      "word": "plug",\n      "count": 86\n    },\n    {\n      "word": "power",\n      "count": 86\n    },\n    {\n      "word": "measure",\n      "count": 85\n    },\n    {\n      "word": "toast",\n      "count": 85\n    },\n    {\n      "word": "stock",\n      "count": 84\n    },\n    {\n      "word": "anchor",\n      "count": 83\n    },\n    {\n      "word": "mill",\n      "count": 83\n    },\n    {\n      "word": "racquet",\n      "count": 83\n    },\n    {\n      "word": "situate",\n      "count": 83\n    },\n    {\n      "word": "spot",\n      "count": 82\n    },\n    {\n      "word": "tilt",\n      "count": 82\n    },\n    {\n      "word": "know",\n      "count": 81\n    },\n    {\n      "word": "meet",\n      "count": 81\n    },\n    {\n      "word": "milk",\n      "count": 81\n    },\n    {\n      "word": "dig",\n      "count": 80\n    },\n    {\n      "word": "flow",\n      "count": 80\n    },\n    {\n      "word": "parasaile",\n      "count": 80\n    },\n    {\n      "word": "remodel",\n      "count": 80\n    },\n    {\n      "word": "style",\n      "count": 80\n    },\n    {\n      "word": "cage",\n      "count": 79\n    },\n    {\n      "word": "oppose",\n      "count": 79\n    },\n    {\n      "word": "side",\n      "count": 79\n    },\n    {\n      "word": "warm",\n      "count": 79\n    },\n    {\n      "word": "windsurf",\n      "count": 79\n    },\n    {\n      "word": "expose",\n      "count": 78\n    },\n    {\n      "word": "fish",\n      "count": 78\n    },\n    {\n      "word": "reveal",\n      "count": 78\n    },\n    {\n      "word": "texte",\n      "count": 78\n    },\n    {\n      "word": "create",\n      "count": 77\n    },\n    {\n      "word": "describe",\n      "count": 77\n    },\n    {\n      "word": "pretend",\n      "count": 77\n    },\n    {\n      "word": "rain",\n      "count": 77\n    },\n    {\n      "word": "sing",\n      "count": 77\n    },\n    {\n      "word": "vandalize",\n      "count": 77\n    },\n    {\n      "word": "exit",\n      "count": 76\n    },\n    {\n      "word": "order",\n      "count": 76\n    },\n    {\n      "word": "receive",\n      "count": 76\n    },\n    {\n      "word": "stain",\n      "count": 76\n    },\n    {\n      "word": "want",\n      "count": 76\n    },\n    {\n      "word": "rack",\n      "count": 75\n    },\n    {\n      "word": "drape",\n      "count": 74\n    },\n    {\n      "word": "offer",\n      "count": 74\n    },\n    {\n      "word": "darken",\n      "count": 73\n    },\n    {\n      "word": "drag",\n      "count": 73\n    },\n    {\n      "word": "print",\n      "count": 73\n    },\n    {\n      "word": "wipe",\n      "count": 73\n    },\n    {\n      "word": "clutter",\n      "count": 72\n    },\n    {\n      "word": "file",\n      "count": 72\n    },\n    {\n      "word": "film",\n      "count": 72\n    },\n    {\n      "word": "outstretche",\n      "count": 72\n    },\n    {\n      "word": "protect",\n      "count": 72\n    },\n    {\n      "word": "sand",\n      "count": 72\n    },\n    {\n      "word": "back",\n      "count": 71\n    },\n    {\n      "word": "carrot",\n      "count": 70\n    },\n    {\n      "word": "groom",\n      "count": 70\n    },\n    {\n      "word": "launch",\n      "count": 70\n    },\n    {\n      "word": "nap",\n      "count": 70\n    },\n    {\n      "word": "splash",\n      "count": 70\n    },\n    {\n      "word": "strew",\n      "count": 70\n    },\n    {\n      "word": "bring",\n      "count": 69\n    },\n    {\n      "word": "garnish",\n      "count": 69\n    },\n    {\n      "word": "release",\n      "count": 69\n    },\n    {\n      "word": "shred",\n      "count": 69\n    },\n    {\n      "word": "beach",\n      "count": 68\n    },\n    {\n      "word": "carpet",\n      "count": 68\n    },\n    {\n      "word": "descend",\n      "count": 68\n    },\n    {\n      "word": "knock",\n      "count": 68\n    },\n    {\n      "word": "nestle",\n      "count": 68\n    },\n    {\n      "word": "bounce",\n      "count": 67\n    },\n    {\n      "word": "trim",\n      "count": 67\n    },\n    {\n      "word": "focus",\n      "count": 66\n    },\n    {\n      "word": "bench",\n      "count": 65\n    },\n    {\n      "word": "dip",\n      "count": 65\n    },\n    {\n      "word": "embrace",\n      "count": 65\n    },\n    {\n      "word": "mop",\n      "count": 65\n    },\n    {\n      "word": "panel",\n      "count": 65\n    },\n    {\n      "word": "converse",\n      "count": 64\n    },\n    {\n      "word": "gallop",\n      "count": 64\n    },\n    {\n      "word": "rail",\n      "count": 64\n    },\n    {\n      "word": "snow",\n      "count": 64\n    },\n    {\n      "word": "trail",\n      "count": 64\n    },\n    {\n      "word": "mean",\n      "count": 63\n    },\n    {\n      "word": "mound",\n      "count": 63\n    },\n    {\n      "word": "dive",\n      "count": 62\n    },\n    {\n      "word": "fade",\n      "count": 62\n    },\n    {\n      "word": "fit",\n      "count": 62\n    },\n    {\n      "word": "flavor",\n      "count": 62\n    },\n    {\n      "word": "nuzzle",\n      "count": 62\n    },\n    {\n      "word": "submerge",\n      "count": 62\n    },\n    {\n      "word": "accompany",\n      "count": 61\n    },\n    {\n      "word": "dot",\n      "count": 61\n    },\n    {\n      "word": "fashion",\n      "count": 61\n    },\n    {\n      "word": "hover",\n      "count": 61\n    },\n    {\n      "word": "resemble",\n      "count": 61\n    },\n    {\n      "word": "stare",\n      "count": 61\n    },\n    {\n      "word": "store",\n      "count": 61\n    },\n    {\n      "word": "tear",\n      "count": 61\n    },\n    {\n      "word": "ascend",\n      "count": 60\n    },\n    {\n      "word": "litter",\n      "count": 60\n    },\n    {\n      "word": "pause",\n      "count": 60\n    },\n    {\n      "word": "comb",\n      "count": 59\n    },\n    {\n      "word": "gate",\n      "count": 59\n    },\n    {\n      "word": "investigate",\n      "count": 59\n    },\n    {\n      "word": "kiteboarde",\n      "count": 59\n    },\n    {\n      "word": "smother",\n      "count": 59\n    },\n    {\n      "word": "utensil",\n      "count": 59\n    },\n    {\n      "word": "end",\n      "count": 58\n    },\n    {\n      "word": "flatbe",\n      "count": 58\n    },\n    {\n      "word": "navigate",\n      "count": 58\n    },\n    {\n      "word": "spin",\n      "count": 58\n    },\n    {\n      "word": "aim",\n      "count": 57\n    },\n    {\n      "word": "floor",\n      "count": 57\n    },\n    {\n      "word": "m",\n      "count": 57\n    },\n    {\n      "word": "bundle",\n      "count": 56\n    },\n    {\n      "word": "damage",\n      "count": 56\n    },\n    {\n      "word": "name",\n      "count": 56\n    },\n    {\n      "word": "occupy",\n      "count": 56\n    },\n    {\n      "word": "peak",\n      "count": 56\n    },\n    {\n      "word": "repair",\n      "count": 56\n    },\n    {\n      "word": "stay",\n      "count": 56\n    },\n    {\n      "word": "bricke",\n      "count": 55\n    },\n    {\n      "word": "bust",\n      "count": 55\n    },\n    {\n      "word": "crawl",\n      "count": 55\n    },\n    {\n      "word": "lower",\n      "count": 55\n    },\n    {\n      "word": "taxi",\n      "count": 55\n    },\n    {\n      "word": "think",\n      "count": 55\n    },\n    {\n      "word": "wake",\n      "count": 55\n    },\n    {\n      "word": "border",\n      "count": 54\n    },\n    {\n      "word": "lose",\n      "count": 54\n    },\n    {\n      "word": "surfboard",\n      "count": 54\n    },\n    {\n      "word": "vend",\n      "count": 54\n    },\n    {\n      "word": "lunge",\n      "count": 53\n    },\n    {\n      "word": "salad",\n      "count": 53\n    },\n    {\n      "word": "visit",\n      "count": 53\n    },\n    {\n      "word": "depart",\n      "count": 52\n    },\n    {\n      "word": "produce",\n      "count": 52\n    },\n    {\n      "word": "spill",\n      "count": 52\n    },\n    {\n      "word": "study",\n      "count": 52\n    },\n    {\n      "word": "beat",\n      "count": 51\n    },\n    {\n      "word": "complete",\n      "count": 51\n    },\n    {\n      "word": "march",\n      "count": 51\n    },\n    {\n      "word": "microwave",\n      "count": 51\n    },\n    {\n      "word": "squash",\n      "count": 51\n    },\n    {\n      "word": "package",\n      "count": 50\n    },\n    {\n      "word": "trek",\n      "count": 50\n    },\n    {\n      "word": "bloom",\n      "count": 49\n    },\n    {\n      "word": "charge",\n      "count": 49\n    },\n    {\n      "word": "double",\n      "count": 49\n    },\n    {\n      "word": "hunch",\n      "count": 49\n    },\n    {\n      "word": "love",\n      "count": 49\n    },\n    {\n      "word": "plow",\n      "count": 49\n    },\n    {\n      "word": "crack",\n      "count": 48\n    },\n    {\n      "word": "destroy",\n      "count": 48\n    },\n    {\n      "word": "mess",\n      "count": 48\n    },\n    {\n      "word": "monitor",\n      "count": 48\n    },\n    {\n      "word": "recline",\n      "count": 48\n    },\n    {\n      "word": "shelve",\n      "count": 48\n    },\n    {\n      "word": "wale",\n      "count": 48\n    },\n    {\n      "word": "form",\n      "count": 47\n    },\n    {\n      "word": "remain",\n      "count": 47\n    },\n    {\n      "word": "shear",\n      "count": 47\n    },\n    {\n      "word": "sheer",\n      "count": 47\n    },\n    {\n      "word": "wire",\n      "count": 47\n    },\n    {\n      "word": "affix",\n      "count": 46\n    },\n    {\n      "word": "assist",\n      "count": 46\n    },\n    {\n      "word": "boat",\n      "count": 46\n    },\n    {\n      "word": "demonstrate",\n      "count": 46\n    },\n    {\n      "word": "discuss",\n      "count": 46\n    },\n    {\n      "word": "participate",\n      "count": 46\n    },\n    {\n      "word": "wall",\n      "count": 46\n    },\n    {\n      "word": "browse",\n      "count": 45\n    },\n    {\n      "word": "cake",\n      "count": 45\n    },\n    {\n      "word": "marry",\n      "count": 45\n    },\n    {\n      "word": "pin",\n      "count": 45\n    },\n    {\n      "word": "retrieve",\n      "count": 45\n    },\n    {\n      "word": "spoon",\n      "count": 45\n    },\n    {\n      "word": "wheel",\n      "count": 45\n    },\n    {\n      "word": "assemble",\n      "count": 44\n    },\n    {\n      "word": "blend",\n      "count": 44\n    },\n    {\n      "word": "crochet",\n      "count": 44\n    },\n    {\n      "word": "cruise",\n      "count": 44\n    },\n    {\n      "word": "dangle",\n      "count": 44\n    },\n    {\n      "word": "din",\n      "count": 44\n    },\n    {\n      "word": "flank",\n      "count": 44\n    },\n    {\n      "word": "forage",\n      "count": 44\n    },\n    {\n      "word": "rope",\n      "count": 44\n    },\n    {\n      "word": "socialize",\n      "count": 44\n    },\n    {\n      "word": "win",\n      "count": 44\n    },\n    {\n      "word": "bike",\n      "count": 43\n    },\n    {\n      "word": "blur",\n      "count": 43\n    },\n    {\n      "word": "counter",\n      "count": 43\n    },\n    {\n      "word": "divide",\n      "count": 43\n    },\n    {\n      "word": "man",\n      "count": 43\n    },\n    {\n      "word": "steer",\n      "count": 43\n    },\n    {\n      "word": "stroll",\n      "count": 43\n    },\n    {\n      "word": "weather",\n      "count": 43\n    },\n    {\n      "word": "boil",\n      "count": 42\n    },\n    {\n      "word": "bury",\n      "count": 42\n    },\n    {\n      "word": "chat",\n      "count": 42\n    },\n    {\n      "word": "clock",\n      "count": 41\n    },\n    {\n      "word": "construct",\n      "count": 41\n    },\n    {\n      "word": "dice",\n      "count": 41\n    },\n    {\n      "word": "like",\n      "count": 41\n    },\n    {\n      "word": "record",\n      "count": 41\n    },\n    {\n      "word": "spew",\n      "count": 41\n    },\n    {\n      "word": "surfer",\n      "count": 41\n    },\n    {\n      "word": "water",\n      "count": 41\n    },\n    {\n      "word": "flush",\n      "count": 40\n    },\n    {\n      "word": "hear",\n      "count": 40\n    },\n    {\n      "word": "search",\n      "count": 40\n    },\n    {\n      "word": "yellow",\n      "count": 40\n    },\n    {\n      "word": "animate",\n      "count": 39\n    },\n    {\n      "word": "choose",\n      "count": 39\n    },\n    {\n      "word": "loom",\n      "count": 39\n    },\n    {\n      "word": "plant",\n      "count": 39\n    },\n    {\n      "word": "wrestle",\n      "count": 39\n    },\n    {\n      "word": "collect",\n      "count": 38\n    },\n    {\n      "word": "dump",\n      "count": 38\n    },\n    {\n      "word": "frown",\n      "count": 38\n    },\n    {\n      "word": "gear",\n      "count": 38\n    },\n    {\n      "word": "grip",\n      "count": 38\n    },\n    {\n      "word": "hitch",\n      "count": 38\n    },\n    {\n      "word": "knick",\n      "count": 38\n    },\n    {\n      "word": "protrude",\n      "count": 38\n    },\n    {\n      "word": "rare",\n      "count": 38\n    },\n    {\n      "word": "slop",\n      "count": 38\n    },\n    {\n      "word": "tub",\n      "count": 38\n    },\n    {\n      "word": "base",\n      "count": 37\n    },\n    {\n      "word": "guard",\n      "count": 37\n    },\n    {\n      "word": "maintain",\n      "count": 37\n    },\n    {\n      "word": "modify",\n      "count": 37\n    },\n    {\n      "word": "preform",\n      "count": 37\n    },\n    {\n      "word": "test",\n      "count": 37\n    },\n    {\n      "word": "attack",\n      "count": 36\n    },\n    {\n      "word": "avoid",\n      "count": 36\n    },\n    {\n      "word": "billow",\n      "count": 36\n    },\n    {\n      "word": "congregate",\n      "count": 36\n    },\n    {\n      "word": "craft",\n      "count": 36\n    },\n    {\n      "word": "emerge",\n      "count": 36\n    },\n    {\n      "word": "flood",\n      "count": 36\n    },\n    {\n      "word": "iron",\n      "count": 36\n    },\n    {\n      "word": "proceed",\n      "count": 36\n    },\n    {\n      "word": "shine",\n      "count": 36\n    },\n    {\n      "word": "split",\n      "count": 36\n    },\n    {\n      "word": "stylize",\n      "count": 36\n    },\n    {\n      "word": "tip",\n      "count": 36\n    },\n    {\n      "word": "bathtub",\n      "count": 35\n    },\n    {\n      "word": "bunt",\n      "count": 35\n    },\n    {\n      "word": "chair",\n      "count": 35\n    },\n    {\n      "word": "concentrate",\n      "count": 35\n    },\n    {\n      "word": "consume",\n      "count": 35\n    },\n    {\n      "word": "join",\n      "count": 35\n    },\n    {\n      "word": "meat",\n      "count": 35\n    },\n    {\n      "word": "nibble",\n      "count": 35\n    },\n    {\n      "word": "nurse",\n      "count": 35\n    },\n    {\n      "word": "sip",\n      "count": 35\n    },\n    {\n      "word": "steam",\n      "count": 35\n    },\n    {\n      "word": "stem",\n      "count": 35\n    },\n    {\n      "word": "vary",\n      "count": 35\n    },\n    {\n      "word": "cluster",\n      "count": 34\n    },\n    {\n      "word": "clutch",\n      "count": 34\n    },\n    {\n      "word": "combine",\n      "count": 34\n    },\n    {\n      "word": "curve",\n      "count": 34\n    },\n    {\n      "word": "fire",\n      "count": 34\n    },\n    {\n      "word": "obscure",\n      "count": 34\n    },\n    {\n      "word": "rush",\n      "count": 34\n    },\n    {\n      "word": "service",\n      "count": 34\n    },\n    {\n      "word": "can",\n      "count": 33\n    },\n    {\n      "word": "curtain",\n      "count": 33\n    },\n    {\n      "word": "outfit",\n      "count": 33\n    },\n    {\n      "word": "scoop",\n      "count": 33\n    },\n    {\n      "word": "sidewalk",\n      "count": 33\n    },\n    {\n      "word": "spell",\n      "count": 33\n    },\n    {\n      "word": "trot",\n      "count": 33\n    },\n    {\n      "word": "yawn",\n      "count": 33\n    },\n    {\n      "word": "cauliflower",\n      "count": 32\n    },\n    {\n      "word": "coat",\n      "count": 32\n    },\n    {\n      "word": "grasp",\n      "count": 32\n    },\n    {\n      "word": "hole",\n      "count": 32\n    },\n    {\n      "word": "pave",\n      "count": 32\n    },\n    {\n      "word": "peck",\n      "count": 32\n    },\n    {\n      "word": "ready",\n      "count": 32\n    },\n    {\n      "word": "renovate",\n      "count": 32\n    },\n    {\n      "word": "saddle",\n      "count": 32\n    },\n    {\n      "word": "sample",\n      "count": 32\n    },\n    {\n      "word": "span",\n      "count": 32\n    },\n    {\n      "word": "support",\n      "count": 32\n    },\n    {\n      "word": "act",\n      "count": 31\n    },\n    {\n      "word": "cheese",\n      "count": 31\n    },\n    {\n      "word": "involve",\n      "count": 31\n    },\n    {\n      "word": "rip",\n      "count": 31\n    },\n    {\n      "word": "signal",\n      "count": 31\n    },\n    {\n      "word": "sprawl",\n      "count": 31\n    },\n    {\n      "word": "alter",\n      "count": 30\n    },\n    {\n      "word": "bread",\n      "count": 30\n    },\n    {\n      "word": "happen",\n      "count": 30\n    },\n    {\n      "word": "link",\n      "count": 30\n    },\n    {\n      "word": "munch",\n      "count": 30\n    },\n    {\n      "word": "pant",\n      "count": 30\n    },\n    {\n      "word": "protest",\n      "count": 30\n    },\n    {\n      "word": "purchase",\n      "count": 30\n    },\n    {\n      "word": "relate",\n      "count": 30\n    },\n    {\n      "word": "tone",\n      "count": 30\n    },\n    {\n      "word": "angle",\n      "count": 29\n    },\n    {\n      "word": "cheer",\n      "count": 29\n    },\n    {\n      "word": "convert",\n      "count": 29\n    },\n    {\n      "word": "decide",\n      "count": 29\n    },\n    {\n      "word": "discard",\n      "count": 29\n    },\n    {\n      "word": "explain",\n      "count": 29\n    },\n    {\n      "word": "flap",\n      "count": 29\n    },\n    {\n      "word": "hop",\n      "count": 29\n    },\n    {\n      "word": "pierce",\n      "count": 29\n    },\n    {\n      "word": "project",\n      "count": 29\n    },\n    {\n      "word": "react",\n      "count": 29\n    },\n    {\n      "word": "ripen",\n      "count": 29\n    },\n    {\n      "word": "roast",\n      "count": 29\n    },\n    {\n      "word": "slow",\n      "count": 29\n    },\n    {\n      "word": "spend",\n      "count": 29\n    },\n    {\n      "word": "squeeze",\n      "count": 29\n    },\n    {\n      "word": "station",\n      "count": 29\n    },\n    {\n      "word": "bar",\n      "count": 28\n    },\n    {\n      "word": "bicycle",\n      "count": 28\n    },\n    {\n      "word": "care",\n      "count": 28\n    },\n    {\n      "word": "feel",\n      "count": 28\n    },\n    {\n      "word": "house",\n      "count": 28\n    },\n    {\n      "word": "maneuver",\n      "count": 28\n    },\n    {\n      "word": "pancake",\n      "count": 28\n    },\n    {\n      "word": "straddle",\n      "count": 28\n    },\n    {\n      "word": "straighten",\n      "count": 28\n    },\n    {\n      "word": "stream",\n      "count": 28\n    },\n    {\n      "word": "string",\n      "count": 28\n    },\n    {\n      "word": "cry",\n      "count": 27\n    },\n    {\n      "word": "date",\n      "count": 27\n    },\n    {\n      "word": "demolish",\n      "count": 27\n    },\n    {\n      "word": "embed",\n      "count": 27\n    },\n    {\n      "word": "filter",\n      "count": 27\n    },\n    {\n      "word": "flash",\n      "count": 27\n    },\n    {\n      "word": "frolic",\n      "count": 27\n    },\n    {\n      "word": "juggle",\n      "count": 27\n    },\n    {\n      "word": "pepperoni",\n      "count": 27\n    },\n    {\n      "word": "rug",\n      "count": 27\n    },\n    {\n      "word": "smash",\n      "count": 27\n    },\n    {\n      "word": "struggle",\n      "count": 27\n    },\n    {\n      "word": "become",\n      "count": 26\n    },\n    {\n      "word": "belong",\n      "count": 26\n    },\n    {\n      "word": "designate",\n      "count": 26\n    },\n    {\n      "word": "die",\n      "count": 26\n    },\n    {\n      "word": "drizzle",\n      "count": 26\n    },\n    {\n      "word": "glare",\n      "count": 26\n    },\n    {\n      "word": "overturn",\n      "count": 26\n    },\n    {\n      "word": "select",\n      "count": 26\n    },\n    {\n      "word": "send",\n      "count": 26\n    },\n    {\n      "word": "sort",\n      "count": 26\n    },\n    {\n      "word": "sport",\n      "count": 26\n    },\n    {\n      "word": "sweep",\n      "count": 26\n    },\n    {\n      "word": "chip",\n      "count": 25\n    },\n    {\n      "word": "collide",\n      "count": 25\n    },\n    {\n      "word": "entertain",\n      "count": 25\n    },\n    {\n      "word": "exchange",\n      "count": 25\n    },\n    {\n      "word": "fasten",\n      "count": 25\n    },\n    {\n      "word": "glass",\n      "count": 25\n    },\n    {\n      "word": "near",\n      "count": 25\n    },\n    {\n      "word": "restore",\n      "count": 25\n    },\n    {\n      "word": "season",\n      "count": 25\n    },\n    {\n      "word": "silhouette",\n      "count": 25\n    },\n    {\n      "word": "wilt",\n      "count": 25\n    },\n    {\n      "word": "arch",\n      "count": 24\n    },\n    {\n      "word": "beg",\n      "count": 24\n    },\n    {\n      "word": "bent",\n      "count": 24\n    },\n    {\n      "word": "box",\n      "count": 24\n    },\n    {\n      "word": "bus",\n      "count": 24\n    },\n    {\n      "word": "camp",\n      "count": 24\n    },\n    {\n      "word": "catcher",\n      "count": 24\n    },\n    {\n      "word": "clear",\n      "count": 24\n    },\n    {\n      "word": "contemplate",\n      "count": 24\n    },\n    {\n      "word": "dust",\n      "count": 24\n    },\n    {\n      "word": "empty",\n      "count": 24\n    },\n    {\n      "word": "injure",\n      "count": 24\n    },\n    {\n      "word": "instruct",\n      "count": 24\n    },\n    {\n      "word": "mingle",\n      "count": 24\n    },\n    {\n      "word": "paw",\n      "count": 24\n    },\n    {\n      "word": "pop",\n      "count": 24\n    },\n    {\n      "word": "sandwich",\n      "count": 24\n    },\n    {\n      "word": "sle",\n      "count": 24\n    },\n    {\n      "word": "bowl",\n      "count": 23\n    },\n    {\n      "word": "clip",\n      "count": 23\n    },\n    {\n      "word": "compare",\n      "count": 23\n    },\n    {\n      "word": "crumble",\n      "count": 23\n    },\n    {\n      "word": "dye",\n      "count": 23\n    },\n    {\n      "word": "emit",\n      "count": 23\n    },\n    {\n      "word": "grin",\n      "count": 23\n    },\n    {\n      "word": "jog",\n      "count": 23\n    },\n    {\n      "word": "ketchup",\n      "count": 23\n    },\n    {\n      "word": "manicure",\n      "count": 23\n    },\n    {\n      "word": "nail",\n      "count": 23\n    },\n    {\n      "word": "showcase",\n      "count": 23\n    },\n    {\n      "word": "steal",\n      "count": 23\n    },\n    {\n      "word": "zoom",\n      "count": 23\n    },\n    {\n      "word": "apply",\n      "count": 22\n    },\n    {\n      "word": "bedroom",\n      "count": 22\n    },\n    {\n      "word": "center",\n      "count": 22\n    },\n    {\n      "word": "contrast",\n      "count": 22\n    },\n    {\n      "word": "cow",\n      "count": 22\n    },\n    {\n      "word": "cramp",\n      "count": 22\n    },\n    {\n      "word": "hum",\n      "count": 22\n    },\n    {\n      "word": "pan",\n      "count": 22\n    },\n    {\n      "word": "quilt",\n      "count": 22\n    },\n    {\n      "word": "represent",\n      "count": 22\n    },\n    {\n      "word": "shut",\n      "count": 22\n    },\n    {\n      "word": "slat",\n      "count": 22\n    },\n    {\n      "word": "wax",\n      "count": 22\n    },\n    {\n      "word": "expire",\n      "count": 21\n    },\n    {\n      "word": "explore",\n      "count": 21\n    },\n    {\n      "word": "gain",\n      "count": 21\n    },\n    {\n      "word": "generate",\n      "count": 21\n    },\n    {\n      "word": "gesture",\n      "count": 21\n    },\n    {\n      "word": "hat",\n      "count": 21\n    },\n    {\n      "word": "heat",\n      "count": 21\n    },\n    {\n      "word": "laze",\n      "count": 21\n    },\n    {\n      "word": "leak",\n      "count": 21\n    },\n    {\n      "word": "list",\n      "count": 21\n    },\n    {\n      "word": "preppe",\n      "count": 21\n    },\n    {\n      "word": "secure",\n      "count": 21\n    },\n    {\n      "word": "snap",\n      "count": 21\n    },\n    {\n      "word": "track",\n      "count": 21\n    },\n    {\n      "word": "accept",\n      "count": 20\n    },\n    {\n      "word": "align",\n      "count": 20\n    },\n    {\n      "word": "anticipate",\n      "count": 20\n    },\n    {\n      "word": "confine",\n      "count": 20\n    },\n    {\n      "word": "crane",\n      "count": 20\n    },\n    {\n      "word": "crest",\n      "count": 20\n    },\n    {\n      "word": "dribble",\n      "count": 20\n    },\n    {\n      "word": "elephant",\n      "count": 20\n    },\n    {\n      "word": "eye",\n      "count": 20\n    },\n    {\n      "word": "harness",\n      "count": 20\n    },\n    {\n      "word": "highlight",\n      "count": 20\n    },\n    {\n      "word": "model",\n      "count": 20\n    },\n    {\n      "word": "ornate",\n      "count": 20\n    },\n    {\n      "word": "rot",\n      "count": 20\n    },\n    {\n      "word": "size",\n      "count": 20\n    },\n    {\n      "word": "sky",\n      "count": 20\n    },\n    {\n      "word": "stab",\n      "count": 20\n    },\n    {\n      "word": "suck",\n      "count": 20\n    },\n    {\n      "word": "tail",\n      "count": 20\n    },\n    {\n      "word": "towel",\n      "count": 20\n    },\n    {\n      "word": "twist",\n      "count": 20\n    },\n    {\n      "word": "appoint",\n      "count": 19\n    },\n    {\n      "word": "bark",\n      "count": 19\n    },\n    {\n      "word": "deface",\n      "count": 19\n    },\n    {\n      "word": "execute",\n      "count": 19\n    },\n    {\n      "word": "exercise",\n      "count": 19\n    },\n    {\n      "word": "interview",\n      "count": 19\n    },\n    {\n      "word": "kayak",\n      "count": 19\n    },\n    {\n      "word": "overflow",\n      "count": 19\n    },\n    {\n      "word": "parade",\n      "count": 19\n    },\n    {\n      "word": "populate",\n      "count": 19\n    },\n    {\n      "word": "replace",\n      "count": 19\n    },\n    {\n      "word": "rollerblade",\n      "count": 19\n    },\n    {\n      "word": "shove",\n      "count": 19\n    },\n    {\n      "word": "squirt",\n      "count": 19\n    },\n    {\n      "word": "streak",\n      "count": 19\n    },\n    {\n      "word": "title",\n      "count": 19\n    },\n    {\n      "word": "utilize",\n      "count": 19\n    },\n    {\n      "word": "weave",\n      "count": 19\n    },\n    {\n      "word": "accent",\n      "count": 18\n    },\n    {\n      "word": "desk",\n      "count": 18\n    },\n    {\n      "word": "dish",\n      "count": 18\n    },\n    {\n      "word": "elevate",\n      "count": 18\n    },\n    {\n      "word": "leash",\n      "count": 18\n    },\n    {\n      "word": "note",\n      "count": 18\n    },\n    {\n      "word": "patrol",\n      "count": 18\n    },\n    {\n      "word": "pillow",\n      "count": 18\n    },\n    {\n      "word": "pizza",\n      "count": 18\n    },\n    {\n      "word": "pump",\n      "count": 18\n    },\n    {\n      "word": "seal",\n      "count": 18\n    },\n    {\n      "word": "shrimp",\n      "count": 18\n    },\n    {\n      "word": "silverware",\n      "count": 18\n    },\n    {\n      "word": "ske",\n      "count": 18\n    },\n    {\n      "word": "slant",\n      "count": 18\n    },\n    {\n      "word": "strawberry",\n      "count": 18\n    },\n    {\n      "word": "sunglasse",\n      "count": 18\n    },\n    {\n      "word": "tangle",\n      "count": 18\n    },\n    {\n      "word": "wade",\n      "count": 18\n    },\n    {\n      "word": "announce",\n      "count": 17\n    },\n    {\n      "word": "bunch",\n      "count": 17\n    },\n    {\n      "word": "cause",\n      "count": 17\n    },\n    {\n      "word": "commode",\n      "count": 17\n    },\n    {\n      "word": "congratulate",\n      "count": 17\n    },\n    {\n      "word": "cram",\n      "count": 17\n    },\n    {\n      "word": "fail",\n      "count": 17\n    },\n    {\n      "word": "grass",\n      "count": 17\n    },\n    {\n      "word": "insert",\n      "count": 17\n    },\n    {\n      "word": "landscape",\n      "count": 17\n    },\n    {\n      "word": "market",\n      "count": 17\n    },\n    {\n      "word": "merge",\n      "count": 17\n    },\n    {\n      "word": "plan",\n      "count": 17\n    },\n    {\n      "word": "shield",\n      "count": 17\n    },\n    {\n      "word": "squint",\n      "count": 17\n    },\n    {\n      "word": "stoop",\n      "count": 17\n    },\n    {\n      "word": "sugar",\n      "count": 17\n    },\n    {\n      "word": "table",\n      "count": 17\n    },\n    {\n      "word": "traverse",\n      "count": 17\n    },\n    {\n      "word": "undergo",\n      "count": 17\n    },\n    {\n      "word": "update",\n      "count": 17\n    },\n    {\n      "word": "welcome",\n      "count": 17\n    },\n    {\n      "word": "ask",\n      "count": 16\n    },\n    {\n      "word": "at",\n      "count": 16\n    },\n    {\n      "word": "brown",\n      "count": 16\n    },\n    {\n      "word": "chill",\n      "count": 16\n    },\n    {\n      "word": "dunk",\n      "count": 16\n    },\n    {\n      "word": "erect",\n      "count": 16\n    },\n    {\n      "word": "figure",\n      "count": 16\n    },\n    {\n      "word": "hoist",\n      "count": 16\n    },\n    {\n      "word": "inspire",\n      "count": 16\n    },\n    {\n      "word": "powder",\n      "count": 16\n    },\n    {\n      "word": "scaffold",\n      "count": 16\n    },\n    {\n      "word": "soccer",\n      "count": 16\n    },\n    {\n      "word": "suckle",\n      "count": 16\n    },\n    {\n      "word": "texture",\n      "count": 16\n    },\n    {\n      "word": "tour",\n      "count": 16\n    },\n    {\n      "word": "waterskie",\n      "count": 16\n    },\n    {\n      "word": "wedge",\n      "count": 16\n    },\n    {\n      "word": "wonder",\n      "count": 16\n    },\n    {\n      "word": "wrinkle",\n      "count": 16\n    },\n    {\n      "word": "yell",\n      "count": 16\n    },\n    {\n      "word": "attract",\n      "count": 15\n    },\n    {\n      "word": "bump",\n      "count": 15\n    },\n    {\n      "word": "butt",\n      "count": 15\n    },\n    {\n      "word": "clap",\n      "count": 15\n    },\n    {\n      "word": "clasp",\n      "count": 15\n    },\n    {\n      "word": "continue",\n      "count": 15\n    },\n    {\n      "word": "disembark",\n      "count": 15\n    },\n    {\n      "word": "drift",\n      "count": 15\n    },\n    {\n      "word": "gild",\n      "count": 15\n    },\n    {\n      "word": "grey",\n      "count": 15\n    },\n    {\n      "word": "hope",\n      "count": 15\n    },\n    {\n      "word": "pat",\n      "count": 15\n    },\n    {\n      "word": "portray",\n      "count": 15\n    },\n    {\n      "word": "pray",\n      "count": 15\n    },\n    {\n      "word": "require",\n      "count": 15\n    },\n    {\n      "word": "score",\n      "count": 15\n    },\n    {\n      "word": "seperate",\n      "count": 15\n    },\n    {\n      "word": "sew",\n      "count": 15\n    },\n    {\n      "word": "shirt",\n      "count": 15\n    },\n    {\n      "word": "snooze",\n      "count": 15\n    },\n    {\n      "word": "sunbathe",\n      "count": 15\n    },\n    {\n      "word": "surfs",\n      "count": 15\n    },\n    {\n      "word": "switch",\n      "count": 15\n    },\n    {\n      "word": "tackle",\n      "count": 15\n    },\n    {\n      "word": "tint",\n      "count": 15\n    },\n    {\n      "word": "toil",\n      "count": 15\n    },\n    {\n      "word": "wing",\n      "count": 15\n    },\n    {\n      "word": "avocado",\n      "count": 14\n    },\n    {\n      "word": "barbecue",\n      "count": 14\n    },\n    {\n      "word": "beak",\n      "count": 14\n    },\n    {\n      "word": "bind",\n      "count": 14\n    },\n    {\n      "word": "bumper",\n      "count": 14\n    },\n    {\n      "word": "circle",\n      "count": 14\n    },\n    {\n      "word": "click",\n      "count": 14\n    },\n    {\n      "word": "coach",\n      "count": 14\n    },\n    {\n      "word": "cock",\n      "count": 14\n    },\n    {\n      "word": "cub",\n      "count": 14\n    },\n    {\n      "word": "drench",\n      "count": 14\n    },\n    {\n      "word": "edge",\n      "count": 14\n    },\n    {\n      "word": "edit",\n      "count": 14\n    },\n    {\n      "word": "escort",\n      "count": 14\n    },\n    {\n      "word": "feast",\n      "count": 14\n    },\n    {\n      "word": "front",\n      "count": 14\n    },\n    {\n      "word": "grate",\n      "count": 14\n    },\n    {\n      "word": "ground",\n      "count": 14\n    },\n    {\n      "word": "lite",\n      "count": 14\n    },\n    {\n      "word": "manipulate",\n      "count": 14\n    },\n    {\n      "word": "neck",\n      "count": 14\n    },\n    {\n      "word": "outline",\n      "count": 14\n    },\n    {\n      "word": "pair",\n      "count": 14\n    },\n    {\n      "word": "par",\n      "count": 14\n    },\n    {\n      "word": "paste",\n      "count": 14\n    },\n    {\n      "word": "plaster",\n      "count": 14\n    },\n    {\n      "word": "prohibit",\n      "count": 14\n    },\n    {\n      "word": "promote",\n      "count": 14\n    },\n    {\n      "word": "relish",\n      "count": 14\n    },\n    {\n      "word": "rider",\n      "count": 14\n    },\n    {\n      "word": "salute",\n      "count": 14\n    },\n    {\n      "word": "seagull",\n      "count": 14\n    },\n    {\n      "word": "shovel",\n      "count": 14\n    },\n    {\n      "word": "strand",\n      "count": 14\n    },\n    {\n      "word": "tether",\n      "count": 14\n    },\n    {\n      "word": "tug",\n      "count": 14\n    },\n    {\n      "word": "tusk",\n      "count": 14\n    },\n    {\n      "word": "witte",\n      "count": 14\n    },\n    {\n      "word": "zip",\n      "count": 14\n    },\n    {\n      "word": "automate",\n      "count": 13\n    },\n    {\n      "word": "batter",\n      "count": 13\n    },\n    {\n      "word": "believe",\n      "count": 13\n    },\n    {\n      "word": "butter",\n      "count": 13\n    },\n    {\n      "word": "camouflage",\n      "count": 13\n    },\n    {\n      "word": "chocolate",\n      "count": 13\n    },\n    {\n      "word": "claim",\n      "count": 13\n    },\n    {\n      "word": "cream",\n      "count": 13\n    },\n    {\n      "word": "desert",\n      "count": 13\n    },\n    {\n      "word": "develop",\n      "count": 13\n    },\n    {\n      "word": "engrave",\n      "count": 13\n    },\n    {\n      "word": "fuel",\n      "count": 13\n    },\n    {\n      "word": "intersect",\n      "count": 13\n    },\n    {\n      "word": "invite",\n      "count": 13\n    },\n    {\n      "word": "joust",\n      "count": 13\n    },\n    {\n      "word": "motorize",\n      "count": 13\n    },\n    {\n      "word": "parachute",\n      "count": 13\n    },\n    {\n      "word": "parasail",\n      "count": 13\n    },\n    {\n      "word": "peep",\n      "count": 13\n    },\n    {\n      "word": "pen",\n      "count": 13\n    },\n    {\n      "word": "pot",\n      "count": 13\n    },\n    {\n      "word": "scream",\n      "count": 13\n    },\n    {\n      "word": "sharpen",\n      "count": 13\n    },\n    {\n      "word": "sneak",\n      "count": 13\n    },\n    {\n      "word": "stag",\n      "count": 13\n    },\n    {\n      "word": "stroke",\n      "count": 13\n    },\n    {\n      "word": "text",\n      "count": 13\n    },\n    {\n      "word": "trash",\n      "count": 13\n    },\n    {\n      "word": "twirl",\n      "count": 13\n    },\n    {\n      "word": "wakeboarde",\n      "count": 13\n    },\n    {\n      "word": "bag",\n      "count": 12\n    },\n    {\n      "word": "barb",\n      "count": 12\n    },\n    {\n      "word": "bottle",\n      "count": 12\n    },\n    {\n      "word": "bridle",\n      "count": 12\n    },\n    {\n      "word": "buck",\n      "count": 12\n    },\n    {\n      "word": "chrome",\n      "count": 12\n    },\n    {\n      "word": "conduct",\n      "count": 12\n    },\n    {\n      "word": "contend",\n      "count": 12\n    },\n    {\n      "word": "coordinate",\n      "count": 12\n    },\n    {\n      "word": "crush",\n      "count": 12\n    },\n    {\n      "word": "crust",\n      "count": 12\n    },\n    {\n      "word": "deploy",\n      "count": 12\n    },\n    {\n      "word": "down",\n      "count": 12\n    },\n    {\n      "word": "drain",\n      "count": 12\n    },\n    {\n      "word": "ear",\n      "count": 12\n    },\n    {\n      "word": "express",\n      "count": 12\n    },\n    {\n      "word": "field",\n      "count": 12\n    },\n    {\n      "word": "halve",\n      "count": 12\n    },\n    {\n      "word": "hose",\n      "count": 12\n    },\n    {\n      "word": "intend",\n      "count": 12\n    },\n    {\n      "word": "intertwine",\n      "count": 12\n    },\n    {\n      "word": "jut",\n      "count": 12\n    },\n    {\n      "word": "layer",\n      "count": 12\n    },\n    {\n      "word": "log",\n      "count": 12\n    },\n    {\n      "word": "manage",\n      "count": 12\n    },\n    {\n      "word": "meander",\n      "count": 12\n    },\n    {\n      "word": "mimic",\n      "count": 12\n    },\n    {\n      "word": "mitt",\n      "count": 12\n    },\n    {\n      "word": "nose",\n      "count": 12\n    },\n    {\n      "word": "peddle",\n      "count": 12\n    },\n    {\n      "word": "photoshoppe",\n      "count": 12\n    },\n    {\n      "word": "ring",\n      "count": 12\n    },\n    {\n      "word": "save",\n      "count": 12\n    },\n    {\n      "word": "sculpt",\n      "count": 12\n    },\n    {\n      "word": "shadow",\n      "count": 12\n    },\n    {\n      "word": "smear",\n      "count": 12\n    },\n    {\n      "word": "splay",\n      "count": 12\n    },\n    {\n      "word": "sprout",\n      "count": 12\n    },\n    {\n      "word": "strain",\n      "count": 12\n    },\n    {\n      "word": "suit",\n      "count": 12\n    },\n    {\n      "word": "syrup",\n      "count": 12\n    },\n    {\n      "word": "tax",\n      "count": 12\n    },\n    {\n      "word": "vie",\n      "count": 12\n    },\n    {\n      "word": "war",\n      "count": 12\n    },\n    {\n      "word": "white",\n      "count": 12\n    },\n    {\n      "word": "age",\n      "count": 11\n    },\n    {\n      "word": "answer",\n      "count": 11\n    },\n    {\n      "word": "bleed",\n      "count": 11\n    },\n    {\n      "word": "bruise",\n      "count": 11\n    },\n    {\n      "word": "buckle",\n      "count": 11\n    },\n    {\n      "word": "cabinet",\n      "count": 11\n    },\n    {\n      "word": "cling",\n      "count": 11\n    },\n    {\n      "word": "cord",\n      "count": 11\n    },\n    {\n      "word": "dominate",\n      "count": 11\n    },\n    {\n      "word": "droop",\n      "count": 11\n    },\n    {\n      "word": "foot",\n      "count": 11\n    },\n    {\n      "word": "grace",\n      "count": 11\n    },\n    {\n      "word": "harvest",\n      "count": 11\n    },\n    {\n      "word": "heap",\n      "count": 11\n    },\n    {\n      "word": "hue",\n      "count": 11\n    },\n    {\n      "word": "hunt",\n      "count": 11\n    },\n    {\n      "word": "invert",\n      "count": 11\n    },\n    {\n      "word": "judge",\n      "count": 11\n    },\n    {\n      "word": "mustard",\n      "count": 11\n    },\n    {\n      "word": "notice",\n      "count": 11\n    },\n    {\n      "word": "nudge",\n      "count": 11\n    },\n    {\n      "word": "overgrow",\n      "count": 11\n    },\n    {\n      "word": "paper",\n      "count": 11\n    },\n    {\n      "word": "paraglide",\n      "count": 11\n    },\n    {\n      "word": "pedal",\n      "count": 11\n    },\n    {\n      "word": "pee",\n      "count": 11\n    },\n    {\n      "word": "pole",\n      "count": 11\n    },\n    {\n      "word": "propel",\n      "count": 11\n    },\n    {\n      "word": "range",\n      "count": 11\n    },\n    {\n      "word": "re",\n      "count": 11\n    },\n    {\n      "word": "retain",\n      "count": 11\n    },\n    {\n      "word": "rim",\n      "count": 11\n    },\n    {\n      "word": "rotate",\n      "count": 11\n    },\n    {\n      "word": "ruffle",\n      "count": 11\n    },\n    {\n      "word": "snack",\n      "count": 11\n    },\n    {\n      "word": "suggest",\n      "count": 11\n    },\n    {\n      "word": "sun",\n      "count": 11\n    },\n    {\n      "word": "transform",\n      "count": 11\n    },\n    {\n      "word": "tree",\n      "count": 11\n    },\n    {\n      "word": "uncover",\n      "count": 11\n    },\n    {\n      "word": "urinate",\n      "count": 11\n    },\n    {\n      "word": "weigh",\n      "count": 11\n    },\n    {\n      "word": "wrangle",\n      "count": 11\n    },\n    {\n      "word": "wreck",\n      "count": 11\n    },\n    {\n      "word": "address",\n      "count": 10\n    },\n    {\n      "word": "appreciate",\n      "count": 10\n    },\n    {\n      "word": "articulate",\n      "count": 10\n    },\n    {\n      "word": "backhand",\n      "count": 10\n    },\n    {\n      "word": "bask",\n      "count": 10\n    },\n    {\n      "word": "bathroom",\n      "count": 10\n    },\n    {\n      "word": "buidle",\n      "count": 10\n    },\n    {\n      "word": "cheste",\n      "count": 10\n    },\n    {\n      "word": "chicken",\n      "count": 10\n    },\n    {\n      "word": "chug",\n      "count": 10\n    },\n    {\n      "word": "clown",\n      "count": 10\n    },\n    {\n      "word": "compose",\n      "count": 10\n    },\n    {\n      "word": "corn",\n      "count": 10\n    },\n    {\n      "word": "costume",\n      "count": 10\n    },\n    {\n      "word": "countertop",\n      "count": 10\n    },\n    {\n      "word": "crop",\n      "count": 10\n    },\n    {\n      "word": "crow",\n      "count": 10\n    },\n    {\n      "word": "cup",\n      "count": 10\n    },\n    {\n      "word": "defend",\n      "count": 10\n    },\n    {\n      "word": "devour",\n      "count": 10\n    },\n    {\n      "word": "dodge",\n      "count": 10\n    },\n    {\n      "word": "donut",\n      "count": 10\n    },\n    {\n      "word": "encase",\n      "count": 10\n    },\n    {\n      "word": "etch",\n      "count": 10\n    },\n    {\n      "word": "five",\n      "count": 10\n    },\n    {\n      "word": "fle",\n      "count": 10\n    },\n    {\n      "word": "growl",\n      "count": 10\n    },\n    {\n      "word": "gush",\n      "count": 10\n    },\n    {\n      "word": "ignore",\n      "count": 10\n    },\n    {\n      "word": "inform",\n      "count": 10\n    },\n    {\n      "word": "kitesurfe",\n      "count": 10\n    },\n    {\n      "word": "lack",\n      "count": 10\n    },\n    {\n      "word": "lap",\n      "count": 10\n    },\n    {\n      "word": "lighten",\n      "count": 10\n    },\n    {\n      "word": "mat",\n      "count": 10\n    },\n    {\n      "word": "mismatch",\n      "count": 10\n    },\n    {\n      "word": "neglect",\n      "count": 10\n    },\n    {\n      "word": "obstruct",\n      "count": 10\n    },\n    {\n      "word": "overtake",\n      "count": 10\n    },\n    {\n      "word": "pickle",\n      "count": 10\n    },\n    {\n      "word": "pink",\n      "count": 10\n    },\n    {\n      "word": "punch",\n      "count": 10\n    },\n    {\n      "word": "rap",\n      "count": 10\n    },\n    {\n      "word": "root",\n      "count": 10\n    },\n    {\n      "word": "scramble",\n      "count": 10\n    },\n    {\n      "word": "screen",\n      "count": 10\n    },\n    {\n      "word": "scroll",\n      "count": 10\n    },\n    {\n      "word": "seek",\n      "count": 10\n    },\n    {\n      "word": "skim",\n      "count": 10\n    },\n    {\n      "word": "skin",\n      "count": 10\n    },\n    {\n      "word": "sled",\n      "count": 10\n    },\n    {\n      "word": "sow",\n      "count": 10\n    },\n    {\n      "word": "sparkle",\n      "count": 10\n    },\n    {\n      "word": "superimpose",\n      "count": 10\n    },\n    {\n      "word": "survey",\n      "count": 10\n    },\n    {\n      "word": "tot",\n      "count": 10\n    },\n    {\n      "word": "trap",\n      "count": 10\n    },\n    {\n      "word": "unripene",\n      "count": 10\n    },\n    {\n      "word": "yield",\n      "count": 10\n    },\n    {\n      "word": "alert",\n      "count": 9\n    },\n    {\n      "word": "backpack",\n      "count": 9\n    },\n    {\n      "word": "barn",\n      "count": 9\n    },\n    {\n      "word": "bead",\n      "count": 9\n    },\n    {\n      "word": "blaze",\n      "count": 9\n    },\n    {\n      "word": "blindfold",\n      "count": 9\n    },\n    {\n      "word": "brim",\n      "count": 9\n    },\n    {\n      "word": "bud",\n      "count": 9\n    },\n    {\n      "word": "bun",\n      "count": 9\n    },\n    {\n      "word": "bush",\n      "count": 9\n    },\n    {\n      "word": "coast",\n      "count": 9\n    },\n    {\n      "word": "confuse",\n      "count": 9\n    },\n    {\n      "word": "controller",\n      "count": 9\n    },\n    {\n      "word": "cushion",\n      "count": 9\n    },\n    {\n      "word": "dispense",\n      "count": 9\n    },\n    {\n      "word": "dude",\n      "count": 9\n    },\n    {\n      "word": "dwarf",\n      "count": 9\n    },\n    {\n      "word": "eachother",\n      "count": 9\n    },\n    {\n      "word": "fascinate",\n      "count": 9\n    },\n    {\n      "word": "fetch",\n      "count": 9\n    },\n    {\n      "word": "helmet",\n      "count": 9\n    },\n    {\n      "word": "hurt",\n      "count": 9\n    },\n    {\n      "word": "kill",\n      "count": 9\n    },\n    {\n      "word": "knot",\n      "count": 9\n    },\n    {\n      "word": "levitate",\n      "count": 9\n    },\n    {\n      "word": "lodge",\n      "count": 9\n    },\n    {\n      "word": "net",\n      "count": 9\n    },\n    {\n      "word": "ocean",\n      "count": 9\n    },\n    {\n      "word": "plate",\n      "count": 9\n    },\n    {\n      "word": "ply",\n      "count": 9\n    },\n    {\n      "word": "prevent",\n      "count": 9\n    },\n    {\n      "word": "process",\n      "count": 9\n    },\n    {\n      "word": "raft",\n      "count": 9\n    },\n    {\n      "word": "rear",\n      "count": 9\n    },\n    {\n      "word": "regard",\n      "count": 9\n    },\n    {\n      "word": "remote",\n      "count": 9\n    },\n    {\n      "word": "ripple",\n      "count": 9\n    },\n    {\n      "word": "roof",\n      "count": 9\n    },\n    {\n      "word": "section",\n      "count": 9\n    },\n    {\n      "word": "shappe",\n      "count": 9\n    },\n    {\n      "word": "shorn",\n      "count": 9\n    },\n    {\n      "word": "skii",\n      "count": 9\n    },\n    {\n      "word": "sleeve",\n      "count": 9\n    },\n    {\n      "word": "slip",\n      "count": 9\n    },\n    {\n      "word": "smack",\n      "count": 9\n    },\n    {\n      "word": "speckle",\n      "count": 9\n    },\n    {\n      "word": "surprise",\n      "count": 9\n    },\n    {\n      "word": "truck",\n      "count": 9\n    },\n    {\n      "word": "undo",\n      "count": 9\n    },\n    {\n      "word": "unfold",\n      "count": 9\n    },\n    {\n      "word": "advise",\n      "count": 8\n    },\n    {\n      "word": "bathe",\n      "count": 8\n    },\n    {\n      "word": "battle",\n      "count": 8\n    },\n    {\n      "word": "beam",\n      "count": 8\n    },\n    {\n      "word": "bet",\n      "count": 8\n    },\n    {\n      "word": "blanket",\n      "count": 8\n    },\n    {\n      "word": "braid",\n      "count": 8\n    },\n    {\n      "word": "brand",\n      "count": 8\n    },\n    {\n      "word": "breast",\n      "count": 8\n    },\n    {\n      "word": "bubble",\n      "count": 8\n    },\n    {\n      "word": "car",\n      "count": 8\n    },\n    {\n      "word": "challenge",\n      "count": 8\n    },\n    {\n      "word": "clog",\n      "count": 8\n    },\n    {\n      "word": "coffee",\n      "count": 8\n    },\n    {\n      "word": "collapse",\n      "count": 8\n    },\n    {\n      "word": "curb",\n      "count": 8\n    },\n    {\n      "word": "customize",\n      "count": 8\n    },\n    {\n      "word": "cute",\n      "count": 8\n    },\n    {\n      "word": "cycle",\n      "count": 8\n    },\n    {\n      "word": "deckere",\n      "count": 8\n    },\n    {\n      "word": "deflate",\n      "count": 8\n    },\n    {\n      "word": "denote",\n      "count": 8\n    },\n    {\n      "word": "dog",\n      "count": 8\n    },\n    {\n      "word": "doodle",\n      "count": 8\n    },\n    {\n      "word": "embellish",\n      "count": 8\n    },\n    {\n      "word": "embroider",\n      "count": 8\n    },\n    {\n      "word": "entwine",\n      "count": 8\n    },\n    {\n      "word": "fan",\n      "count": 8\n    },\n    {\n      "word": "fast",\n      "count": 8\n    },\n    {\n      "word": "freeze",\n      "count": 8\n    },\n    {\n      "word": "full",\n      "count": 8\n    },\n    {\n      "word": "goof",\n      "count": 8\n    },\n    {\n      "word": "gown",\n      "count": 8\n    },\n    {\n      "word": "graffitie",\n      "count": 8\n    },\n    {\n      "word": "gutte",\n      "count": 8\n    },\n    {\n      "word": "hump",\n      "count": 8\n    },\n    {\n      "word": "hurry",\n      "count": 8\n    },\n    {\n      "word": "interlock",\n      "count": 8\n    },\n    {\n      "word": "lasso",\n      "count": 8\n    },\n    {\n      "word": "migrate",\n      "count": 8\n    },\n    {\n      "word": "murder",\n      "count": 8\n    },\n    {\n      "word": "neat",\n      "count": 8\n    },\n    {\n      "word": "negotiate",\n      "count": 8\n    },\n    {\n      "word": "number",\n      "count": 8\n    },\n    {\n      "word": "oversee",\n      "count": 8\n    },\n    {\n      "word": "own",\n      "count": 8\n    },\n    {\n      "word": "pad",\n      "count": 8\n    },\n    {\n      "word": "perk",\n      "count": 8\n    },\n    {\n      "word": "preen",\n      "count": 8\n    },\n    {\n      "word": "puff",\n      "count": 8\n    },\n    {\n      "word": "rainbow",\n      "count": 8\n    },\n    {\n      "word": "ram",\n      "count": 8\n    },\n    {\n      "word": "recess",\n      "count": 8\n    },\n    {\n      "word": "refuel",\n      "count": 8\n    },\n    {\n      "word": "ruin",\n      "count": 8\n    },\n    {\n      "word": "scale",\n      "count": 8\n    },\n    {\n      "word": "skid",\n      "count": 8\n    },\n    {\n      "word": "skier",\n      "count": 8\n    },\n    {\n      "word": "slam",\n      "count": 8\n    },\n    {\n      "word": "square",\n      "count": 8\n    },\n    {\n      "word": "squish",\n      "count": 8\n    },\n    {\n      "word": "stake",\n      "count": 8\n    },\n    {\n      "word": "stitch",\n      "count": 8\n    },\n    {\n      "word": "stow",\n      "count": 8\n    },\n    {\n      "word": "stride",\n      "count": 8\n    },\n    {\n      "word": "swirl",\n      "count": 8\n    },\n    {\n      "word": "tan",\n      "count": 8\n    },\n    {\n      "word": "unmade",\n      "count": 8\n    },\n    {\n      "word": "unpack",\n      "count": 8\n    },\n    {\n      "word": "venture",\n      "count": 8\n    },\n    {\n      "word": "wag",\n      "count": 8\n    },\n    {\n      "word": "wallpaper",\n      "count": 8\n    },\n    {\n      "word": "whisper",\n      "count": 8\n    },\n    {\n      "word": "wield",\n      "count": 8\n    },\n    {\n      "word": "window",\n      "count": 8\n    },\n    {\n      "word": "accord",\n      "count": 7\n    },\n    {\n      "word": "advertisement",\n      "count": 7\n    },\n    {\n      "word": "array",\n      "count": 7\n    },\n    {\n      "word": "artwork",\n      "count": 7\n    },\n    {\n      "word": "assort",\n      "count": 7\n    },\n    {\n      "word": "ball",\n      "count": 7\n    },\n    {\n      "word": "blacken",\n      "count": 7\n    },\n    {\n      "word": "bolt",\n      "count": 7\n    },\n    {\n      "word": "bound",\n      "count": 7\n    },\n    {\n      "word": "canoe",\n      "count": 7\n    },\n    {\n      "word": "case",\n      "count": 7\n    },\n    {\n      "word": "commute",\n      "count": 7\n    },\n    {\n      "word": "correct",\n      "count": 7\n    },\n    {\n      "word": "count",\n      "count": 7\n    },\n    {\n      "word": "dedicate",\n      "count": 7\n    },\n    {\n      "word": "den",\n      "count": 7\n    },\n    {\n      "word": "dent",\n      "count": 7\n    },\n    {\n      "word": "detail",\n      "count": 7\n    },\n    {\n      "word": "dilapidate",\n      "count": 7\n    },\n    {\n      "word": "disassemble",\n      "count": 7\n    },\n    {\n      "word": "dome",\n      "count": 7\n    },\n    {\n      "word": "drip",\n      "count": 7\n    },\n    {\n      "word": "duck",\n      "count": 7\n    },\n    {\n      "word": "eare",\n      "count": 7\n    },\n    {\n      "word": "eatte",\n      "count": 7\n    },\n    {\n      "word": "enhance",\n      "count": 7\n    },\n    {\n      "word": "fiddle",\n      "count": 7\n    },\n    {\n      "word": "flatten",\n      "count": 7\n    },\n    {\n      "word": "flex",\n      "count": 7\n    },\n    {\n      "word": "glance",\n      "count": 7\n    },\n    {\n      "word": "guy",\n      "count": 7\n    },\n    {\n      "word": "honk",\n      "count": 7\n    },\n    {\n      "word": "hood",\n      "count": 7\n    },\n    {\n      "word": "horse",\n      "count": 7\n    },\n    {\n      "word": "idle",\n      "count": 7\n    },\n    {\n      "word": "int",\n      "count": 7\n    },\n    {\n      "word": "intercept",\n      "count": 7\n    },\n    {\n      "word": "jam",\n      "count": 7\n    },\n    {\n      "word": "jeep",\n      "count": 7\n    },\n    {\n      "word": "kiteboard",\n      "count": 7\n    },\n    {\n      "word": "lapse",\n      "count": 7\n    },\n    {\n      "word": "leaf",\n      "count": 7\n    },\n    {\n      "word": "motor",\n      "count": 7\n    },\n    {\n      "word": "mow",\n      "count": 7\n    },\n    {\n      "word": "nest",\n      "count": 7\n    },\n    {\n      "word": "overhang",\n      "count": 7\n    },\n    {\n      "word": "partake",\n      "count": 7\n    },\n    {\n      "word": "perfect",\n      "count": 7\n    },\n    {\n      "word": "pille",\n      "count": 7\n    },\n    {\n      "word": "prove",\n      "count": 7\n    },\n    {\n      "word": "pumpkin",\n      "count": 7\n    },\n    {\n      "word": "ramp",\n      "count": 7\n    },\n    {\n      "word": "recede",\n      "count": 7\n    },\n    {\n      "word": "reduce",\n      "count": 7\n    },\n    {\n      "word": "refurbish",\n      "count": 7\n    },\n    {\n      "word": "respond",\n      "count": 7\n    },\n    {\n      "word": "rumble",\n      "count": 7\n    },\n    {\n      "word": "scrub",\n      "count": 7\n    },\n    {\n      "word": "sever",\n      "count": 7\n    },\n    {\n      "word": "shelter",\n      "count": 7\n    },\n    {\n      "word": "ship",\n      "count": 7\n    },\n    {\n      "word": "shoe",\n      "count": 7\n    },\n    {\n      "word": "shore",\n      "count": 7\n    },\n    {\n      "word": "shout",\n      "count": 7\n    },\n    {\n      "word": "simulate",\n      "count": 7\n    },\n    {\n      "word": "sittign",\n      "count": 7\n    },\n    {\n      "word": "skewer",\n      "count": 7\n    },\n    {\n      "word": "slouch",\n      "count": 7\n    },\n    {\n      "word": "spit",\n      "count": 7\n    },\n    {\n      "word": "sponsor",\n      "count": 7\n    },\n    {\n      "word": "stalk",\n      "count": 7\n    },\n    {\n      "word": "stamp",\n      "count": 7\n    },\n    {\n      "word": "stane",\n      "count": 7\n    },\n    {\n      "word": "steak",\n      "count": 7\n    },\n    {\n      "word": "stte",\n      "count": 7\n    },\n    {\n      "word": "supply",\n      "count": 7\n    },\n    {\n      "word": "swaddle",\n      "count": 7\n    },\n    {\n      "word": "swipe",\n      "count": 7\n    },\n    {\n      "word": "swish",\n      "count": 7\n    },\n    {\n      "word": "till",\n      "count": 7\n    },\n    {\n      "word": "treat",\n      "count": 7\n    },\n    {\n      "word": "trudge",\n      "count": 7\n    },\n    {\n      "word": "tune",\n      "count": 7\n    },\n    {\n      "word": "vegetate",\n      "count": 7\n    },\n    {\n      "word": "volley",\n      "count": 7\n    },\n    {\n      "word": "whiz",\n      "count": 7\n    },\n    {\n      "word": "adopt",\n      "count": 6\n    },\n    {\n      "word": "advance",\n      "count": 6\n    },\n    {\n      "word": "amuse",\n      "count": 6\n    },\n    {\n      "word": "annoy",\n      "count": 6\n    },\n    {\n      "word": "applaud",\n      "count": 6\n    },\n    {\n      "word": "arm",\n      "count": 6\n    },\n    {\n      "word": "award",\n      "count": 6\n    },\n    {\n      "word": "bellow",\n      "count": 6\n    },\n    {\n      "word": "belly",\n      "count": 6\n    },\n    {\n      "word": "blossom",\n      "count": 6\n    },\n    {\n      "word": "boast",\n      "count": 6\n    },\n    {\n      "word": "brighten",\n      "count": 6\n    },\n    {\n      "word": "candle",\n      "count": 6\n    },\n    {\n      "word": "cart",\n      "count": 6\n    },\n    {\n      "word": "communicate",\n      "count": 6\n    },\n    {\n      "word": "comprise",\n      "count": 6\n    },\n    {\n      "word": "concern",\n      "count": 6\n    },\n    {\n      "word": "consider",\n      "count": 6\n    },\n    {\n      "word": "converge",\n      "count": 6\n    },\n    {\n      "word": "couch",\n      "count": 6\n    },\n    {\n      "word": "cradle",\n      "count": 6\n    },\n    {\n      "word": "craze",\n      "count": 6\n    },\n    {\n      "word": "crisscross",\n      "count": 6\n    },\n    {\n      "word": "crumple",\n      "count": 6\n    },\n    {\n      "word": "cucumber",\n      "count": 6\n    },\n    {\n      "word": "disguise",\n      "count": 6\n    },\n    {\n      "word": "dishevel",\n      "count": 6\n    },\n    {\n      "word": "dismantle",\n      "count": 6\n    },\n    {\n      "word": "distribute",\n      "count": 6\n    },\n    {\n      "word": "doughnut",\n      "count": 6\n    },\n    {\n      "word": "elongate",\n      "count": 6\n    },\n    {\n      "word": "evergreen",\n      "count": 6\n    },\n    {\n      "word": "excite",\n      "count": 6\n    },\n    {\n      "word": "expect",\n      "count": 6\n    },\n    {\n      "word": "feather",\n      "count": 6\n    },\n    {\n      "word": "flame",\n      "count": 6\n    },\n    {\n      "word": "fluff",\n      "count": 6\n    },\n    {\n      "word": "grapefruit",\n      "count": 6\n    },\n    {\n      "word": "hate",\n      "count": 6\n    },\n    {\n      "word": "host",\n      "count": 6\n    },\n    {\n      "word": "identify",\n      "count": 6\n    },\n    {\n      "word": "imitate",\n      "count": 6\n    },\n    {\n      "word": "impose",\n      "count": 6\n    },\n    {\n      "word": "incorporate",\n      "count": 6\n    },\n    {\n      "word": "inscribe",\n      "count": 6\n    },\n    {\n      "word": "install",\n      "count": 6\n    },\n    {\n      "word": "intrigue",\n      "count": 6\n    },\n    {\n      "word": "juice",\n      "count": 6\n    },\n    {\n      "word": "kite",\n      "count": 6\n    },\n    {\n      "word": "knead",\n      "count": 6\n    },\n    {\n      "word": "knee",\n      "count": 6\n    },\n    {\n      "word": "knell",\n      "count": 6\n    },\n    {\n      "word": "lace",\n      "count": 6\n    },\n    {\n      "word": "loiter",\n      "count": 6\n    },\n    {\n      "word": "loll",\n      "count": 6\n    },\n    {\n      "word": "lurch",\n      "count": 6\n    },\n    {\n      "word": "object",\n      "count": 6\n    },\n    {\n      "word": "occur",\n      "count": 6\n    },\n    {\n      "word": "onlooke",\n      "count": 6\n    },\n    {\n      "word": "ooze",\n      "count": 6\n    },\n    {\n      "word": "ostriche",\n      "count": 6\n    },\n    {\n      "word": "overlay",\n      "count": 6\n    },\n    {\n      "word": "overload",\n      "count": 6\n    },\n    {\n      "word": "overshadow",\n      "count": 6\n    },\n    {\n      "word": "pear",\n      "count": 6\n    },\n    {\n      "word": "peruse",\n      "count": 6\n    },\n    {\n      "word": "picnic",\n      "count": 6\n    },\n    {\n      "word": "piece",\n      "count": 6\n    },\n    {\n      "word": "ponder",\n      "count": 6\n    },\n    {\n      "word": "pore",\n      "count": 6\n    },\n    {\n      "word": "pounce",\n      "count": 6\n    },\n    {\n      "word": "prance",\n      "count": 6\n    },\n    {\n      "word": "preserve",\n      "count": 6\n    },\n    {\n      "word": "recycle",\n      "count": 6\n    },\n    {\n      "word": "red",\n      "count": 6\n    },\n    {\n      "word": "rig",\n      "count": 6\n    },\n    {\n      "word": "roar",\n      "count": 6\n    },\n    {\n      "word": "romp",\n      "count": 6\n    },\n    {\n      "word": "rummage",\n      "count": 6\n    },\n    {\n      "word": "screw",\n      "count": 6\n    },\n    {\n      "word": "signify",\n      "count": 6\n    },\n    {\n      "word": "sin",\n      "count": 6\n    },\n    {\n      "word": "sittin",\n      "count": 6\n    },\n    {\n      "word": "slalom",\n      "count": 6\n    },\n    {\n      "word": "slap",\n      "count": 6\n    },\n    {\n      "word": "slather",\n      "count": 6\n    },\n    {\n      "word": "snowshoe",\n      "count": 6\n    },\n    {\n      "word": "spark",\n      "count": 6\n    },\n    {\n      "word": "spear",\n      "count": 6\n    },\n    {\n      "word": "spout",\n      "count": 6\n    },\n    {\n      "word": "stall",\n      "count": 6\n    },\n    {\n      "word": "steady",\n      "count": 6\n    },\n    {\n      "word": "suppose",\n      "count": 6\n    },\n    {\n      "word": "swat",\n      "count": 6\n    },\n    {\n      "word": "tattere",\n      "count": 6\n    },\n    {\n      "word": "thread",\n      "count": 6\n    },\n    {\n      "word": "tray",\n      "count": 6\n    },\n    {\n      "word": "trick",\n      "count": 6\n    },\n    {\n      "word": "trunk",\n      "count": 6\n    },\n    {\n      "word": "unleash",\n      "count": 6\n    },\n    {\n      "word": "untie",\n      "count": 6\n    },\n    {\n      "word": "urge",\n      "count": 6\n    },\n    {\n      "word": "vegetable",\n      "count": 6\n    },\n    {\n      "word": "we",\n      "count": 6\n    },\n    {\n      "word": "whack",\n      "count": 6\n    },\n    {\n      "word": "wink",\n      "count": 6\n    },\n    {\n      "word": "wish",\n      "count": 6\n    },\n    {\n      "word": "worship",\n      "count": 6\n    },\n    {\n      "word": "accelerate",\n      "count": 5\n    },\n    {\n      "word": "achieve",\n      "count": 5\n    },\n    {\n      "word": "acknowledge",\n      "count": 5\n    },\n    {\n      "word": "adult",\n      "count": 5\n    },\n    {\n      "word": "aid",\n      "count": 5\n    },\n    {\n      "word": "air",\n      "count": 5\n    },\n    {\n      "word": "airborne",\n      "count": 5\n    },\n    {\n      "word": "associate",\n      "count": 5\n    },\n    {\n      "word": "assume",\n      "count": 5\n    },\n    {\n      "word": "attempe",\n      "count": 5\n    },\n    {\n      "word": "attire",\n      "count": 5\n    },\n    {\n      "word": "barrel",\n      "count": 5\n    },\n    {\n      "word": "barricade",\n      "count": 5\n    },\n    {\n      "word": "bisect",\n      "count": 5\n    },\n    {\n      "word": "blade",\n      "count": 5\n    },\n    {\n      "word": "breed",\n      "count": 5\n    },\n    {\n      "word": "bulge",\n      "count": 5\n    },\n    {\n      "word": "bustle",\n      "count": 5\n    },\n    {\n      "word": "button",\n      "count": 5\n    },\n    {\n      "word": "cable",\n      "count": 5\n    },\n    {\n      "word": "calm",\n      "count": 5\n    },\n    {\n      "word": "carre",\n      "count": 5\n    },\n    {\n      "word": "carriage",\n      "count": 5\n    },\n    {\n      "word": "choke",\n      "count": 5\n    },\n    {\n      "word": "claw",\n      "count": 5\n    },\n    {\n      "word": "commuter",\n      "count": 5\n    },\n    {\n      "word": "compost",\n      "count": 5\n    },\n    {\n      "word": "conceal",\n      "count": 5\n    },\n    {\n      "word": "consult",\n      "count": 5\n    },\n    {\n      "word": "corral",\n      "count": 5\n    },\n    {\n      "word": "couple",\n      "count": 5\n    },\n    {\n      "word": "creep",\n      "count": 5\n    },\n    {\n      "word": "daybe",\n      "count": 5\n    },\n    {\n      "word": "dessert",\n      "count": 5\n    },\n    {\n      "word": "determine",\n      "count": 5\n    },\n    {\n      "word": "dine",\n      "count": 5\n    },\n    {\n      "word": "doze",\n      "count": 5\n    },\n    {\n      "word": "emblazon",\n      "count": 5\n    },\n    {\n      "word": "encircle",\n      "count": 5\n    },\n    {\n      "word": "encounter",\n      "count": 5\n    },\n    {\n      "word": "engulf",\n      "count": 5\n    },\n    {\n      "word": "entrance",\n      "count": 5\n    },\n    {\n      "word": "ep",\n      "count": 5\n    },\n    {\n      "word": "escape",\n      "count": 5\n    },\n    {\n      "word": "experience",\n      "count": 5\n    },\n    {\n      "word": "fist",\n      "count": 5\n    },\n    {\n      "word": "fixture",\n      "count": 5\n    },\n    {\n      "word": "flock",\n      "count": 5\n    },\n    {\n      "word": "foul",\n      "count": 5\n    },\n    {\n      "word": "freestande",\n      "count": 5\n    },\n    {\n      "word": "gleam",\n      "count": 5\n    },\n    {\n      "word": "glitter",\n      "count": 5\n    },\n    {\n      "word": "glue",\n      "count": 5\n    },\n    {\n      "word": "gnaw",\n      "count": 5\n    },\n    {\n      "word": "gray",\n      "count": 5\n    },\n    {\n      "word": "harbor",\n      "count": 5\n    },\n    {\n      "word": "hiss",\n      "count": 5\n    },\n    {\n      "word": "hitchhike",\n      "count": 5\n    },\n    {\n      "word": "hunker",\n      "count": 5\n    },\n    {\n      "word": "illustrate",\n      "count": 5\n    },\n    {\n      "word": "inbetween",\n      "count": 5\n    },\n    {\n      "word": "jack",\n      "count": 5\n    },\n    {\n      "word": "lend",\n      "count": 5\n    },\n    {\n      "word": "letter",\n      "count": 5\n    },\n    {\n      "word": "lid",\n      "count": 5\n    },\n    {\n      "word": "loop",\n      "count": 5\n    },\n    {\n      "word": "loose",\n      "count": 5\n    },\n    {\n      "word": "mash",\n      "count": 5\n    },\n    {\n      "word": "mince",\n      "count": 5\n    },\n    {\n      "word": "mold",\n      "count": 5\n    },\n    {\n      "word": "mouse",\n      "count": 5\n    },\n    {\n      "word": "notify",\n      "count": 5\n    },\n    {\n      "word": "oin",\n      "count": 5\n    },\n    {\n      "word": "overlap",\n      "count": 5\n    },\n    {\n      "word": "paddleboarde",\n      "count": 5\n    },\n    {\n      "word": "pain",\n      "count": 5\n    },\n    {\n      "word": "para",\n      "count": 5\n    },\n    {\n      "word": "patter",\n      "count": 5\n    },\n    {\n      "word": "peal",\n      "count": 5\n    },\n    {\n      "word": "pier",\n      "count": 5\n    },\n    {\n      "word": "pip",\n      "count": 5\n    },\n    {\n      "word": "pirche",\n      "count": 5\n    },\n    {\n      "word": "polka",\n      "count": 5\n    },\n    {\n      "word": "purse",\n      "count": 5\n    },\n    {\n      "word": "pursue",\n      "count": 5\n    },\n    {\n      "word": "rag",\n      "count": 5\n    },\n    {\n      "word": "rake",\n      "count": 5\n    },\n    {\n      "word": "remind",\n      "count": 5\n    },\n    {\n      "word": "reserve",\n      "count": 5\n    },\n    {\n      "word": "review",\n      "count": 5\n    },\n    {\n      "word": "rinse",\n      "count": 5\n    },\n    {\n      "word": "river",\n      "count": 5\n    },\n    {\n      "word": "room",\n      "count": 5\n    },\n    {\n      "word": "roost",\n      "count": 5\n    },\n    {\n      "word": "sauteed",\n      "count": 5\n    },\n    {\n      "word": "scallop",\n      "count": 5\n    },\n    {\n      "word": "scan",\n      "count": 5\n    },\n    {\n      "word": "scar",\n      "count": 5\n    },\n    {\n      "word": "scarf",\n      "count": 5\n    },\n    {\n      "word": "scowl",\n      "count": 5\n    },\n    {\n      "word": "scrap",\n      "count": 5\n    },\n    {\n      "word": "scribble",\n      "count": 5\n    },\n    {\n      "word": "scrunch",\n      "count": 5\n    },\n    {\n      "word": "server",\n      "count": 5\n    },\n    {\n      "word": "sesame",\n      "count": 5\n    },\n    {\n      "word": "settle",\n      "count": 5\n    },\n    {\n      "word": "short",\n      "count": 5\n    },\n    {\n      "word": "signs",\n      "count": 5\n    },\n    {\n      "word": "sittte",\n      "count": 5\n    },\n    {\n      "word": "skiie",\n      "count": 5\n    },\n    {\n      "word": "slick",\n      "count": 5\n    },\n    {\n      "word": "sling",\n      "count": 5\n    },\n    {\n      "word": "snarl",\n      "count": 5\n    },\n    {\n      "word": "space",\n      "count": 5\n    },\n    {\n      "word": "splatter",\n      "count": 5\n    },\n    {\n      "word": "spotless",\n      "count": 5\n    },\n    {\n      "word": "stade",\n      "count": 5\n    },\n    {\n      "word": "staple",\n      "count": 5\n    },\n    {\n      "word": "stnade",\n      "count": 5\n    },\n    {\n      "word": "stone",\n      "count": 5\n    },\n    {\n      "word": "streamline",\n      "count": 5\n    },\n    {\n      "word": "sunlit",\n      "count": 5\n    },\n    {\n      "word": "swerve",\n      "count": 5\n    },\n    {\n      "word": "swoop",\n      "count": 5\n    },\n    {\n      "word": "teethe",\n      "count": 5\n    },\n    {\n      "word": "thumb",\n      "count": 5\n    },\n    {\n      "word": "toothpaste",\n      "count": 5\n    },\n    {\n      "word": "topple",\n      "count": 5\n    },\n    {\n      "word": "trip",\n      "count": 5\n    },\n    {\n      "word": "umpe",\n      "count": 5\n    },\n    {\n      "word": "understand",\n      "count": 5\n    },\n    {\n      "word": "variegate",\n      "count": 5\n    },\n    {\n      "word": "videotape",\n      "count": 5\n    },\n    {\n      "word": "waive",\n      "count": 5\n    },\n    {\n      "word": "wet",\n      "count": 5\n    }\n  ],\n  "noun_template": "a photo of a {lemma}",\n  "verb_template": "a photo contains the action of {lemma}",\n  "lemma_amendment": {\n    "version": "verb_riding_rides_rid_to_ride_v1",\n    "surface_forms": [\n      "riding",\n      "rides"\n    ],\n    "pos": "VERB",\n    "original_lemma": "rid",\n    "corrected_lemma": "ride",\n    "moved_tokens": 18487,\n    "other_verbs_unchanged": true,\n    "nouns_unchanged": true,\n    "source_bank_sha256": "450fc969e0842ca3270d1f899b7e8fe594639260644ad5815a69430fe48264b1"\n  }\n}\n'}
COHORT={'train': [194707, 456695, 330967, 342159, 430521, 137004, 413533, 67439, 38389, 188941, 1282, 554031, 254795, 26383, 48381, 176505, 381323, 399317, 116134, 27484, 349152, 291290, 486457, 460059, 223031, 283744, 44436, 393097, 17181, 497392, 546928, 576705, 444805, 234084, 400558, 140067, 219488, 231087, 478391, 326845, 7125, 159372, 291140, 164179, 334021, 201839, 472089, 523974, 308549, 266676, 265826, 407002, 556478, 355863, 581181, 44147, 139390, 479378, 114240, 449029, 250916, 344702, 157397, 304126, 395022, 347571, 131612, 430220, 579491, 247136, 39406, 346343, 481386, 305451, 187803, 282239, 429726, 441412, 426031, 568883, 114141, 117286, 143154, 227329, 268682, 479172, 160509, 112798, 568878, 282558, 24414, 56075, 221888, 531481, 58531, 471824, 496116, 519522, 376185, 334135, 81042, 401892, 417144, 256906, 449256, 460092, 162915, 142487, 400851, 322691, 297370, 375400, 76866, 13763, 233843, 557375, 461475, 444087, 500764, 457726, 123071, 331900, 381014, 371829, 298527, 429399, 317149, 115319, 60641, 232287, 206662, 102599, 326913, 540347, 307001, 555366, 466339, 279596, 13729, 504977, 419586, 509985, 347925, 494518, 94518, 488974, 230403, 378821, 157673, 82551, 349918, 216613, 16725, 213135, 76593, 468205, 246688, 39711, 311180, 258132, 529824, 267417, 339634, 304818, 567633, 413154, 66124, 227617, 221247, 492008, 2567, 393630, 62591, 505057, 328144, 402632, 316942, 246940, 297360, 147846, 372583, 567881, 189752, 461063, 72442, 56161, 196393, 578958, 426283, 470618, 163385, 98716, 573085, 296693, 508861, 139007, 335552, 157657, 94185, 413628, 549462, 19786, 277914, 237263, 449316, 512561, 148924, 415843, 402255, 311553, 24107, 413903, 474398, 581437, 261151, 459203, 166768, 374253, 542484, 395627, 460702, 310163, 146533, 337390, 408779, 38487, 450800, 106375, 177022, 261256, 207749, 12795, 120785, 560687, 426459, 442878, 71602, 55728, 106029, 370914, 379535, 391258, 126513, 193407, 162226, 42698, 59494, 244847, 71716, 444893, 115569, 309400, 238641, 332276, 55712, 481829, 529137, 43331, 495568, 371638, 552016, 181850, 419194, 536058, 11694, 196675, 467803, 500281, 143188, 208054, 389921, 350620, 532933, 129940, 229569, 471309, 177927, 391029, 459050, 66351, 328917, 277955, 207236, 87083, 132954, 191805, 271864, 195649, 359825, 43350, 303652, 191456, 286583, 457263, 172248, 465893, 452700, 557067, 494486, 376911, 335336, 46813, 188068, 41079, 205059, 283445, 281500, 407091, 432748, 192620, 222948, 219653, 63075, 433278, 494637, 7566, 222994, 258283, 92746, 562192, 79926, 162041, 153159, 126229, 200404, 225529, 380350, 375528, 437224, 513837, 379936, 153938, 270571, 516116, 150225, 103970, 366109, 137757, 49309, 316477, 70192, 431173, 248016, 434873, 333995, 214349, 386640, 193094, 443204, 260547, 169174, 435777, 419294, 527597, 528712, 516186, 209046, 285529, 536034, 78896, 155904, 342411, 283780, 54182, 453722, 236060, 564745, 236174, 546388, 265153, 532622, 73757, 305919, 47164, 554943, 289417, 171845, 406011, 25625, 567448, 307020, 487880, 534734, 320682, 355075, 308847, 407056, 540831, 70269, 101622, 185530, 181929, 292810, 100199, 271590, 75521, 435334, 209746, 146773, 545549, 15518, 376492, 146470, 482574, 136093, 324894, 404226, 46942, 575395, 117307, 44476, 265233, 362568, 494089, 122208, 294267, 572956, 516553, 313436, 201702, 537667, 259232, 187576, 381139, 337219, 378894, 329847, 166560, 236025, 416811, 503200, 97380, 478537, 565608, 417153, 202656, 122306, 133999, 397860, 157341, 282843, 12464, 180411, 27840, 495122, 203768, 57359, 520781, 389399, 248170, 234535, 1144, 41802, 210794, 394747, 485491, 368727, 359439, 523815, 220376, 302071, 83972, 19927, 389477, 477064, 314597, 162275, 100668, 131300, 274760, 374680, 73039, 337415, 31164, 211185, 379232, 51106, 567518, 111811, 344859, 286036, 67749, 114174, 581189, 20178, 48707, 194067, 174603, 376283, 57339, 192500, 466763, 323936, 358606, 359864, 185663, 256231, 107918, 3386, 407162, 145101, 543754, 530086, 579938, 277807, 529562, 10856, 153016, 405581, 258674, 119365, 324927, 237423, 239037, 16905, 464030, 289972, 338304, 284768, 367771, 170623, 518389, 70211, 61588, 114258, 9409, 62877, 551194, 188375, 202438, 251623, 1029, 105504, 337607, 460052, 212967, 225565, 320039, 261605, 38046, 452178, 188311, 122745, 194277, 300415, 66066, 95899, 420446, 75451, 540783, 212122, 538190, 256577, 75183, 536413, 155998, 82759, 494264, 446651, 236630, 83696, 379315, 176038, 26686, 231748, 551100, 63573, 533019, 51453, 113259, 187240, 232420, 31865, 479809, 88773, 222741, 202723, 75423, 402880, 477331, 468789, 263028, 16556, 335525, 319136, 327601, 393714, 93040, 554182, 291677, 131557, 430489, 580706, 558128, 561962, 418625, 241010, 526104, 333721, 301615, 226424, 475564, 169200, 528084, 24181, 532941, 175619, 507996, 17985, 476889, 42278, 8429, 376208, 125870, 438024, 90862, 303010, 266479, 435787, 28797, 494128, 107758, 217465, 4138, 343769, 441430, 328797, 257910, 487357, 350720, 517855, 478136, 133456, 87113, 541677, 344284, 504023, 350688, 95667, 178553, 306453, 490046, 199602, 43176, 122413, 192153, 291606, 353568, 191661, 201929, 186906, 59910, 3219, 176372, 160337, 100958, 247624, 229754, 131299, 301281, 351539, 186265, 536027, 64802, 52628, 516655, 501635, 269557, 414106, 568927, 539041, 561763, 410791, 14487, 367732, 254567, 565098, 560851, 15257, 433545, 151583, 176463, 443490, 376342, 55731, 63745, 119290, 514914, 261790, 256251, 453584, 358171, 326075, 78667, 85518, 574988, 342969, 286939, 255950, 4736, 294226, 469056, 97010, 520235, 445590, 487228, 14230, 469198, 242074, 250594, 313811, 163771, 386927, 528412, 148206, 564789, 69750, 32992, 474208, 387365, 506187, 270534, 53301, 431980, 207995, 128373, 82802, 147501, 478505, 86989, 144816, 304519, 433971, 578493, 565201, 407118, 65668, 515210, 67952, 516168, 478052, 393422, 538848, 81738, 263008, 442250, 331298, 447151, 300661, 383699, 64605, 114286, 486388, 308645, 353282, 288642, 39438, 22834, 101292, 420110, 194216, 129695, 488075, 324923, 508045, 268356, 170601, 323494, 438504, 101948, 395817, 365207, 222640, 422008, 92700, 114692, 543830, 182852, 172778, 424604, 447726, 283005, 283237, 107436, 179070, 537371, 52066, 248116, 206318, 475055, 260285, 44575, 97180, 427467, 377545, 523262, 46941, 318384, 517670, 279935, 46656, 472065, 466694, 163076, 528890, 413996, 456761, 535753, 24518, 250815, 77924, 183788, 147309, 46106, 111944, 88608, 34014, 75973, 501311, 92028, 148994, 33752, 542694, 113370, 299105, 201217, 243825, 330671, 466664, 256950, 70122, 170371, 64908, 480947, 442174, 171649, 321546, 210988, 362175, 570741, 371245, 419892, 353080, 471136, 36652, 229942, 419008, 383386, 402174, 537558, 34389, 260440, 277431, 445900, 281148, 359868, 432832, 63050, 401963, 142208, 302222, 303519, 173997, 492478, 38355, 69051, 558101, 393710, 258023, 252905, 61137, 382589, 118076, 430688, 166468, 520251, 523098, 324246, 348027, 568062, 577868, 5683, 500888, 23084, 439241, 66798, 111943, 264169, 306232, 417577, 139275, 153067, 105102, 408328, 337842, 148274, 359589, 515508, 270893, 295343, 290114, 535130, 193640, 389884, 158486, 226308, 388894, 290652, 75611, 187200, 175096, 335690, 217919, 246160, 169490, 287312, 519308, 13124, 251135, 28957, 527291, 521968, 186938, 250165, 351715, 436647, 175020, 284932, 198382, 373716, 476119, 380632, 262626, 298562, 344399, 229522, 535950, 401836, 424110, 25145, 370613, 342483, 294051, 188589, 523250, 284542, 242868, 416101, 502364, 312886, 136299, 312144, 197809, 250200, 526608, 168994, 13800, 363433, 122357, 222283, 266990, 297724, 545550, 295196, 70190, 379162, 370583, 536120, 539163, 63974, 57381, 64105, 434511, 276789, 360427, 16497, 357774, 263212, 440340, 273961, 279647, 369736, 409226, 222442, 324971, 567637, 57749, 234236, 515566, 568687, 9851, 415349, 377329, 406807, 387900, 308223, 374946, 466070, 233434, 110792, 227269, 185117, 374924, 527643, 189244, 97882, 368978, 263382, 436161, 125662, 93506, 311997, 24296, 448309, 101239, 435360, 166087, 225463, 521906, 333580, 133819, 403940, 125468, 248007, 222625, 359320, 41014, 227359, 372551, 57362, 230881, 391726, 355168, 517625, 118572, 384612, 280505, 409739, 72984, 548936, 373729, 67849, 8646, 425280, 201685, 456485, 262119, 361462, 279846, 325932, 259616, 260634, 298600, 109766, 148398, 247838, 204938, 239504, 345557, 35774, 233672, 465975, 83516, 67004, 99546, 151016, 302740, 298481, 16188, 65247, 188216, 544933, 90718, 104270, 348290, 248876, 93599, 228206, 1375, 306553, 435240, 225623, 50331, 466288, 114710, 250195, 441385, 38033, 18952, 354888, 500507, 184397, 107495, 173680, 197318, 486193, 406665, 36574, 503059, 210915, 156301, 142287, 311933, 471222, 405196, 306878, 104393, 537066, 23687, 182624, 452532, 120994, 333075, 411184, 478032, 75954, 552810, 131152, 464462, 72793, 308576, 56068, 363308, 574885, 346586, 450774, 330727, 395053, 135513, 531040, 493728, 475779, 68063, 370102, 562248, 241461, 195952, 266026, 461815, 405210, 431921, 564352, 99453, 423919, 562241, 459319, 487139, 52638, 33902, 249025, 251577, 475904, 98717, 121340, 429092, 15109, 352317, 303257, 126983, 553934, 353899, 416862, 473347, 424029, 565116, 23247, 167042, 49858, 171380, 505486, 253483, 240706, 437356, 21387, 70931, 164241, 488997, 512797, 43957, 559840, 339876, 508990, 157199, 449414, 263036, 127785, 77172, 459093, 174605, 343235, 279322, 482879, 2521, 34884, 45058, 2315, 121174, 425010, 328368, 541918, 468846, 536969, 388510, 559159, 187336, 43809, 522240, 252744, 301457, 73554, 139306, 378769, 556782, 337247, 447187, 78000, 264708, 562002, 3982, 405222, 485875, 108557, 414787, 9218, 26803, 297895, 418056, 79460, 27146, 136493, 25123, 171090, 140029, 169486, 386313, 337682, 332852, 455735, 98242, 451104, 179964, 481014, 423231, 529350, 255630, 539632, 269282, 348874, 563356, 561069, 237968, 12810, 516471, 269171, 39531, 468357, 419410, 132746, 221856, 26521, 136533, 134378, 170012, 46353, 141448, 298242, 363163, 348816, 329480, 90712, 173647, 13445, 160586, 286064, 60647, 288115, 504893, 423355, 507776, 248655, 357184, 83770, 568154, 163985, 531137, 125365, 85759, 339462, 76492, 211546, 394840, 229391, 101692, 53359, 83034, 93764, 318355, 226278, 439132, 472836, 67456, 542751, 177977, 581507, 321874, 491856, 55809, 10015, 494179, 463844, 316528, 422939, 189005, 467774, 206501, 195579, 478344, 253141, 400625, 368253, 513895, 147105, 98145, 246629, 525546, 557122, 150877, 485954, 80339, 304812, 175494, 131170, 571635, 139181, 516587, 150726, 424858, 16166, 138517, 275961, 375200, 489744, 242051, 301697, 457217, 142238, 246409, 577976, 196747, 559380, 581392, 299207, 265344, 128387, 274834, 159530, 409173, 277907, 348871, 458048, 217312, 552744, 257164, 337711, 54133, 264888, 428665, 383341, 538349, 203269, 198068, 64610, 312850, 123731, 197273, 372913, 92, 483643, 261757, 545493, 8665, 352565, 405179, 315195, 146570, 186703, 363559, 224596, 54011, 152479, 452380, 42385, 443599, 480776, 530782, 310140, 125435, 296205, 283107, 112295, 56303, 252413, 142689, 283863, 542549, 175210, 136992, 286525, 99997, 243717, 534735, 293271, 493485, 252731, 392152, 525124, 188692, 407220, 287656, 556399, 415475, 277991, 287716, 530958, 351810, 544655, 108859, 86823, 287396, 27377, 396811, 390298, 565954, 220772, 149890, 323583, 349263, 260603, 469919, 482148, 376750, 267588, 482122, 447976, 397582, 219322, 96505, 515066, 111754, 532735, 366734, 339903, 223701, 536265, 419353, 459563, 484986, 487925, 449839, 436284, 548824, 451533, 93070, 5313, 17474, 24776, 341048, 576996, 344520, 447546, 338119, 185646, 482371, 297736, 555473, 220072, 316578, 95924, 304447, 446615, 403134, 482974, 434333, 356878, 569591, 429902, 113914, 106440, 178821, 438691, 319216, 45193, 539238, 467137, 12711, 338419, 109414, 406030, 576480, 157616, 310499, 481402, 46024, 252136, 463522, 519329, 200064, 531948, 150016, 266768, 170730, 307757, 24446, 513449, 162237, 263668, 378401, 107625, 372902, 100539, 369941, 515064, 86836, 128972, 44341, 158602, 192764, 86581, 399567, 264424, 285120, 387349, 44136, 564133, 238140, 222659, 570760, 322059, 280340, 501039, 131894, 68414, 276559, 262006, 121452, 523819, 169690, 238948, 113700, 407521, 104624, 168774, 8599, 412302, 152502, 487781, 568979, 422956, 549784, 144666, 122839, 276951, 67953, 249125, 316476, 491943, 209971, 28257, 504378, 67462, 329827, 121884, 341075, 144065, 394766, 249156, 308947, 179915, 401384, 167675, 268512, 348555, 150599, 461458, 546079, 64537, 30196, 540265, 537280, 381426, 560359, 134770, 240511, 168978, 22564, 548464, 268552, 457292, 20644, 526491, 432873, 20768, 346107, 218853, 495456, 26690, 146449, 283152, 212322, 80433, 194538, 440136, 328661, 77025, 28253, 121172, 186868, 223112, 479331, 577979, 350242, 363243, 148452, 379766, 248163, 284772, 280732, 510613, 211632, 409360, 560898, 491666, 283240, 337728, 460722, 12377, 255124, 124796, 422910, 159472, 161349, 452014, 563816, 6005, 309880, 340047, 98949, 28974, 404785, 45097, 410912, 266655, 269394, 416477, 479815, 23363, 426876, 546140, 127035, 373034, 20104, 417766, 479596, 56850, 346936, 303617, 14740, 353933, 47509, 368205, 340058, 504732, 14297, 26799, 240800, 71970, 504172, 129509, 411492, 158846, 202799, 189038, 473771, 171270, 23100, 56753, 481413, 306798, 59403, 523854, 458205, 372714, 127576, 537372, 82716, 544811, 497348, 102787, 215450, 511522, 282667, 467878, 22842, 362852, 124141, 486154, 313971, 117701, 293855, 550093, 97914, 386633, 375621, 435848, 5215, 425441, 351127, 83568, 411142, 100234, 425175, 344703, 168145, 449330, 123359, 459018, 343496, 23569, 459212, 218528, 212584, 233545, 352006, 161015, 388211, 581118, 108293, 336464, 397278, 455543, 409627, 79024, 38865, 331085, 510756, 190000, 426482, 209394, 317172, 332489, 578294, 7627, 253520, 392201, 458135, 234518, 74825, 308274, 81481, 435686, 279418, 11506, 283945, 480872, 160625, 213790, 278431, 550298, 170908, 358451, 198546, 349083, 335318, 46155, 147270, 350075, 394325, 259022, 48892, 302728, 556789, 275130, 274586, 101588, 532335, 186730, 484474, 44312, 326574, 557187, 6936, 254283, 433156, 232187, 102503, 325950, 79578, 260978, 388527, 220289, 538518, 289807, 572900, 512013, 295419, 573655, 112590, 241265, 114338, 334557, 387809, 578651, 386788, 29345, 86687, 281293, 276124, 393394, 330736, 235799, 248017, 16123, 116126, 364656, 576543, 231148, 202171, 319938, 383580, 273708, 547041, 501197, 568177, 405191, 125746, 223086, 27660, 499004, 186443, 76970, 274655, 493846, 80948, 236388, 461917, 128113, 27540, 544449, 314685, 241986, 78985, 577877, 544746, 198247, 21447, 243796, 468960, 196480, 270497, 546361, 82258, 347210, 94922, 67536, 413404, 130440, 423875, 194528, 536926, 400560, 513240, 544005, 240965, 267852, 85777, 487385, 347989, 139921, 347276, 176263, 306049, 209292, 94859, 369016, 295773, 14854, 296067, 266120, 470704, 565149, 164694, 407974, 97492, 283484, 504115, 408098, 21036, 520486, 254096, 518265, 104701, 552460, 261461, 73118, 372580, 314703, 496662, 412004, 349486, 201261, 394246, 165757, 353311, 455916, 224281, 310928, 37900, 430621, 20528, 27829, 482451, 209533, 296767, 143655, 122405, 377611, 271622, 41739, 396006, 250576, 194022, 107982, 349104, 280858, 367982, 461019, 552563, 402042, 498541, 107596, 224136, 431655, 439868, 168032, 301735, 309517, 317971, 57753, 171429, 27975, 458914, 516812, 206741, 282042, 730, 99238, 39022, 419848, 250066, 104715, 195903, 356483, 163879, 464652, 100095, 368363, 335590, 399512, 565805, 433691, 293061, 216681, 307136, 445295, 300264, 207455, 216924, 511843, 459265, 336764, 556613, 225916, 398616, 357109, 334121, 433826, 553753, 226451, 560388, 546760, 48063, 20818, 537909, 477682, 471273, 327462, 287186, 21776, 295294, 495802, 547280, 554507, 436837, 113413, 490097, 386355, 436481, 387706, 274058, 200597, 193599, 246064, 472968, 356052, 480682, 168954, 23359, 382695, 137178, 57454, 487496, 143637, 479391, 329379, 179159, 493504, 163358, 45693, 122229, 265355, 57750, 62562, 162282, 329077, 111824, 529457, 526933, 347715, 261414, 118966, 478331, 281133, 367763, 1739, 119297, 516410, 100985, 378070, 252272, 544556, 210791, 256743, 343284, 50521, 22658, 72505, 259555, 302497, 475710, 547192, 159175, 552420, 130972, 9479, 378375, 485259, 118811, 288283, 348179, 361359, 370331, 234637, 396741, 245462, 473929, 325237, 408930, 21650, 558863, 556057, 478880, 525732, 137654, 116851, 134849, 506592, 11349, 155912, 560083, 80165, 54759, 167128, 128711, 444757, 477617, 528875, 5554, 187574, 554082, 58880, 530403, 455405, 502317, 548879, 224020, 478349, 18563, 187902, 294499, 515655, 385037, 67049, 20133, 430079, 196365, 84752, 295455, 234572, 233264, 514544, 78977, 87141, 156084, 279881, 401416, 46042, 359201, 169228, 480575, 344408, 464188, 399981, 435456, 142999, 450653, 482443, 537827, 497628, 545788, 316152, 374487, 444583, 84396, 357312, 553719, 331266, 541613, 113570, 578019, 473296, 16195, 183807, 462710, 253421, 561524, 512308, 393743, 559012, 221708, 552870, 157911, 485561, 443941, 143736, 214876, 503460, 355918, 266981, 303024, 255295, 83959, 171806, 109858, 42336, 464537, 441929, 533227, 92219, 181837, 278627, 102529, 198269, 132375, 60480, 177749, 38890, 54889, 243980, 220900, 7830, 303586, 212047, 294962, 473571, 567838, 29406, 236884, 536891, 289805, 358980, 19322, 506441, 37746, 96726, 284765, 562527, 55135, 54399, 464372, 295656, 394036, 387812, 562462, 167728, 37695, 467182, 254845, 251789, 458326, 44751, 571125, 70812, 139617, 237340, 68797, 18501, 386363, 296675, 176702, 557515, 472602, 47981, 479360, 143277, 483496, 310435, 491611, 469135, 5503, 158370, 339781, 161193, 244425, 387148, 259543, 336607, 425123, 93025, 324403, 510898, 431846, 333134, 507908, 541282, 273560, 246666, 215132, 311776, 78663, 189356, 87056, 536425, 118906, 48708, 275625, 54803, 313249, 322263, 568668, 135685, 472788, 307573, 354569, 41276, 487404, 90328, 273732, 473208, 220118, 556162, 363426, 39802, 105096, 339796, 317018, 302094, 252779, 205668, 109363, 173695, 547979, 49384, 100689, 495134, 54354, 191298, 218365, 460761, 271857, 281562, 462071, 301628, 422473, 101211, 104053, 330426, 400845, 122047, 498986, 340425, 323528, 565117, 32289, 382631, 413922, 70508, 46303, 22757, 194372, 580294, 159791, 116502, 168260, 124803, 354241, 498510, 313198, 466544, 449154, 59815, 281005, 135681, 296182, 192798, 572089, 500871, 278347, 201646, 168016, 384346, 302298, 383639, 96538, 93363, 378092, 253740, 97363, 178192, 383762, 348083, 562916, 334484, 210124, 162717, 26263, 476298, 194450, 250205, 34779, 274734, 385916, 318154, 389554, 433563, 285000, 401327, 468190, 105877, 564552, 15987, 180817, 436354, 550797, 219832, 125116, 568001, 234769, 338143, 528886, 264502, 237477, 78478, 94570, 196977, 42553, 169172, 193011, 354202, 341947, 19542, 102634, 290192, 443547, 55344, 106644, 488297, 391388, 531844, 523494, 12014, 410447, 277156, 373730, 229563, 93994, 370805, 376767, 463966, 361885, 556707, 552432, 581354, 485367, 253848, 232586, 364372, 146627, 330133, 484044, 500649, 518974, 243737, 319830, 561958, 51471, 413623, 286794, 108375, 116206, 118734, 176993, 84866, 88192, 258136, 551575, 575961, 185351, 554518, 86877, 216497, 295550, 478665, 119494, 332912, 560272, 471837, 449315, 293668, 324688, 12946, 375645, 487943, 269121, 485529, 451834, 7932, 208357, 334995, 251974, 556437, 454291, 556966, 400744, 7159, 340771, 189307, 133229, 391011, 34818, 304218, 230625, 503640, 566681, 443053, 123970, 441327, 89027, 364720, 297984, 7500, 373305, 314734, 59874, 65957, 289901, 19338, 140432, 493272, 299563, 78499, 109164, 56872, 409791, 556505, 331268, 505863, 532985, 251330, 251758, 490168, 482191, 207443, 494456, 34299, 473144, 551900, 418770, 338168, 274612, 365251, 397322, 161917, 166391, 38134, 576295, 189812, 295068, 80927, 369199, 162114, 184611, 170127, 532126, 73163, 337594, 570866, 514296, 505818, 538776, 494534, 180987, 374106, 515937, 321665, 49633, 228133, 276716, 42862, 93581, 237162, 150088, 501820, 433607, 462767, 269089, 169988, 90343, 444037, 72050, 82150, 562679, 69936, 114464, 407202, 156302, 14990, 449676, 302489, 431879, 404208, 132510, 467863, 427750, 386934, 166551, 186709, 363494, 429679, 168521, 434735, 451386, 35722, 229250, 243135, 439987, 118866, 390347, 264722, 265879, 404805, 451213, 431503, 578523, 167533, 165039, 4360, 331129, 533548, 298540, 225987, 42429, 21213, 486627, 334732, 527520, 395491, 325532, 166524, 559186, 33869, 340213, 542959, 406211, 342150, 77926, 111536, 18317, 56312, 391394, 90479, 185373, 154667, 225164, 272162, 245460, 469237, 537145, 5294, 542488, 445313, 235314, 335243, 165874, 100483, 52068, 253415, 214733, 335518, 516738, 488377, 187765, 136903, 460364, 92167, 320812, 220046, 406242, 548766, 53691, 201659, 174258, 172490, 168338, 56313, 547164, 95752, 411883, 226246, 6178, 216176, 460168, 351930, 46591, 491883, 333687, 535526, 500464, 214888, 490947, 246390, 320222, 386211, 294823, 457777, 221587, 166805, 573018, 505583, 236438, 91675, 310091, 502884, 308512, 494904, 233994, 85349, 312917, 292296, 115157, 436468, 468771, 188847, 389352, 137057, 14041, 214719, 574236, 571036, 527759, 412501, 188911, 469495, 317458, 88895, 414189, 18466, 434297, 12169, 465430, 580104, 147082, 160893, 121226, 138760, 208671, 120780, 329587, 563987, 176734, 125009, 416981, 105432, 346924, 499725, 63835, 214995, 80928, 389152, 483489, 169766, 345644, 29911, 470128, 502978, 477497, 186177, 179245, 32339, 557266, 341299, 559203, 159897, 460339, 542316, 294663, 511594, 382731, 26108, 413297, 329762, 437664, 401897, 526197, 240434, 330175, 423837, 89859, 519707, 530624, 260637, 344016, 329954, 409740, 74100, 209664, 203006, 308486, 281003, 189614, 131030, 354012, 174062, 104565, 385700, 229867, 28318, 539604, 244795, 106513, 354072, 32948, 304115, 130948, 493472, 5424, 580142, 508695, 30038, 215914, 190515, 471691, 389984, 20289, 534456, 66503, 305223, 208506, 33383, 476175, 419116, 410715, 186502, 1586, 228293, 73199, 457560, 195761, 507979, 161996, 365784, 9171, 23406, 87115, 20760, 254161, 506701, 129186, 162088, 511165, 431492, 444033, 158143, 255248, 243156, 152958, 88507, 6894, 413391, 36503, 552453, 456569, 99423, 497847, 521819, 290544, 547504, 144533, 280829, 292312, 338787, 292169, 366455, 62613, 450952, 331426, 468442, 136979, 203704, 237188, 193926, 19698, 40181, 578356, 532571, 337661, 21563, 148343, 110573, 564570, 407434, 544590, 103631, 511117, 260581, 577758, 417528, 166536, 201628, 295103, 451824, 165697, 436073, 137787, 138621, 53975, 305693, 447330, 89870, 85145, 71384, 212082, 496179, 34930, 12107, 438186, 325831, 123620, 163091, 175154, 406199, 171154, 460986, 27471, 487013, 403531, 455435, 195509, 99875, 460901, 259030, 294639, 79357, 61106, 394065, 382118, 108071, 347253, 297425, 7558, 350270, 447795, 75906, 153008, 482433, 126925, 344288, 275791, 402335, 363576, 246137, 376065, 456496, 292133, 105282, 457976, 364470, 401458, 258732, 70322, 196981, 308528, 554900, 274291, 91520, 503101, 45768, 570285, 490851, 439824, 229456, 464105, 205289, 241271, 504991, 248014, 494936, 407734, 406462, 351033, 373109, 330694, 6672, 213536, 271401, 202850, 311926, 514988, 511026, 336192, 102356, 485771, 500057, 281976, 183260, 403535, 243882, 104125, 541536, 414495, 23656, 534093, 224426, 320106, 446899, 10818, 311251, 92342, 148295, 60593, 139987, 127110, 372234, 488564, 57586, 566690, 528840, 448801, 151074, 577835, 309615, 481357, 448431, 145542, 195225, 209089, 67084, 558317, 581065, 373184, 446670, 250540, 108212, 88780, 192520, 268622, 47983, 22815, 399885, 370543, 270583, 216579, 498622, 427618, 55534, 22861, 466054, 352389, 381116, 234993, 566368, 336682, 514531, 40768, 4554, 405246, 244782, 25121, 345476, 535050, 186342, 206349, 234091, 277625, 405931, 430061, 461878, 468487, 148542, 12522, 154220, 318330, 354318, 65955, 243600, 401652, 422959, 570386, 69003, 410966, 469137, 115635, 361071, 210097, 78652, 431708, 95841, 189922, 465784, 533250, 150442, 208730, 21049, 181677, 120038, 58705, 326892, 192886, 517484, 431504, 484355, 319609, 107035, 498061, 306703, 71043, 576593, 15006, 10995, 35552, 379211, 253087, 479361, 310597, 26321, 412690, 220366, 517748, 440511, 432805, 533872, 270135, 316667, 306412, 318683, 167390, 484593, 175185, 235914, 178966, 294834, 11264, 283629, 40414, 561446, 359760, 240596, 105975, 495481, 452585, 442089, 7535, 489995, 230892, 507536, 201417, 314189, 424792, 312539, 535335, 175288, 509492, 474075, 118167, 144161, 447437, 120259, 386695, 180383, 10580, 421804, 529902, 233925, 100703, 391685, 470206, 575958, 266099, 339151, 233395, 209898, 464833, 353602, 567973, 288943, 494182, 93553, 140426, 389692, 347596, 195696, 578026, 421703, 135820, 557901, 125098, 485452, 16292, 427142, 300396, 222587, 227909, 473725, 420916, 516856, 134320, 234700, 264846, 7682, 231088, 475457, 573277, 339051, 506808, 408607, 112362, 562851, 325175, 54532, 398986, 374185, 484026, 336766, 435340, 6197, 267870, 409345, 31980, 187511, 164534, 369202, 436391, 295765, 445684, 250579, 558104, 426151, 61672, 109820, 194903, 180682, 118615, 364174, 283720, 209851, 471351, 63683, 516184, 575882, 5131, 65080, 233614, 144320, 108144, 302588, 455381, 472314, 490259, 293749, 202548, 273029, 457587, 107656, 161843, 477622, 121903, 278172, 171461, 41606, 128286, 97039, 461715, 210810, 277761, 62817, 274570, 511066, 281028, 6364, 92216, 54406, 335009, 373580, 436199, 81766, 176527, 130357, 357728, 356479, 71521, 277403, 297639, 190991, 286645, 543719, 14733, 421059, 54706, 175291, 129379, 208882, 427518, 396660, 79722, 271068, 567407, 240684, 41186, 485236, 358070, 51795, 559830, 122144, 306868, 319345, 194941, 572475, 188623, 167642, 369380, 68283, 119618, 568560, 295329, 328203, 522687, 476967, 365728, 333848, 96027, 84852, 384907, 347612, 473153, 581056, 212627, 195355, 109992, 484432, 546765, 527791, 151646, 260293, 9678, 494066, 213535, 540460, 261945, 207543, 239671, 486655, 95105, 176778, 7023, 211444, 92684, 203545, 328315, 160496, 408255, 144622, 521238, 279241, 6809, 201492, 386838, 260852, 353754, 133369, 510680, 570066, 163095, 317672, 104591, 414610, 39512, 497622, 279122, 236234, 272173, 329034, 208043, 283858, 515025, 89580, 541077, 213036, 44500, 499959, 44831, 435228, 100050, 105249, 375053, 81372, 321140, 44464, 530543, 573142, 77563, 196518, 443240, 423125, 425342, 183256, 477587, 40325, 115590, 160524, 116557, 475313, 176474, 343662, 176403, 534464, 9176, 544719, 451952, 109045, 536519, 253408, 287874, 334751, 469139, 358994, 51674, 200668, 442585, 551246, 238301, 509750, 526706, 424596, 415885, 248203, 262172, 398952, 89829, 64551, 139282, 132495, 2842, 204969, 325871, 352724, 412632, 417216, 155719, 450897, 486947, 69077, 367319, 427232, 516273, 485547, 351979, 549560, 362331, 257796, 298312, 1496, 468100, 307323, 554351, 535993, 342130, 145628, 138265, 191754, 436580, 403736, 246252, 518207, 68459, 264340, 390684, 379607, 351972, 73669, 106624, 262862, 273904, 399097, 16673, 334048, 130392, 260654, 401094, 217496, 219246, 407313, 468005, 488151, 422284, 536433, 358399, 365703, 271542, 307108, 70048, 385423, 419271, 528067, 507436, 140332, 53800, 352709, 385912, 443308, 220843, 285450, 128312, 85247, 411274, 354921, 463430, 136396, 514581, 570060, 101485, 330920, 225406, 104809, 387948, 107548, 474283, 370117, 561678, 59691, 425077, 319961, 131832, 577719, 388468, 432500, 491249, 424308, 565258, 402705, 340471, 490791, 255420, 276164, 575254, 490490, 435688, 568202, 211464, 47516, 168417, 385146, 554805, 517464, 508140, 79021, 339111, 372141, 191869, 25423, 550097, 468043, 474862, 442445, 391765, 21772, 564756, 523370, 306855, 350059, 364602, 567305, 18350, 274800, 510571, 15597, 553346, 158908, 535403, 528299, 18444, 39010, 216319, 59335, 289545, 329354, 51831, 56344, 147301, 450551, 453167, 404877, 481326, 339988, 449176, 302168, 265330, 340220, 538891, 453779, 573750, 283323, 364293, 201419, 454326, 527364, 358189, 525834, 430821, 474256, 549183, 26997, 426422, 302684, 31536, 50244, 38693, 109591, 248270, 71857, 555254, 133995, 327592, 72575, 22268, 69496, 287958, 446880, 296698, 419332, 417023, 27439, 156026, 226751, 325472, 467145, 430581, 67697, 73326, 101552, 418113, 160428, 46473, 513451, 8086, 42731, 527092, 373574, 260951, 291921, 28356, 209995, 25987, 454399, 202835, 214908, 373444, 515638, 4243, 195204, 571909, 442543, 420692, 380453, 64417, 232387, 322567, 99580, 358216, 210710, 87862, 462925, 455473, 15859, 492544, 351808, 532662, 418606, 58325, 61133, 334032, 248559, 174527, 257301, 435224, 330562, 578368, 360487, 507966, 45472, 115378, 268334, 521495, 228677, 302758, 306207, 544261, 403028, 348370, 391392, 27191, 494199, 238502, 527886, 320593, 292990, 145926, 38079, 452806, 110157, 350278, 265139, 378911, 360071, 148780, 149604, 75466, 178533, 251524, 75614, 260802, 23343, 140454, 261774, 355801, 251366, 540652, 460927, 256309, 527908, 293086, 20632, 361687, 268068, 366811, 298138, 400202, 546292, 303650, 550796, 133153, 90572, 360441, 439215, 374575, 565797, 32322, 224819, 48682, 551867, 259414, 385070, 490175, 401509, 487528, 344816, 165077, 475874, 512723, 536842, 494531, 272426, 405959, 1392, 534593, 419834, 46859, 540978, 24706, 144162, 161595, 445726, 306822, 89181, 241490, 192607, 390288, 439364, 325031, 29180, 110906, 115927, 137402, 433240, 50277, 110196, 574144, 239331, 535788, 278224, 466020, 581717, 482062, 281177, 530823, 555299, 286001, 302110, 66483, 21905, 336634, 109145, 174890, 411241, 257065, 230148, 226291, 84015, 168626, 497599, 145178, 80698, 283728, 151761, 57306, 217487, 329654, 116650, 105335, 522069, 182756, 516740, 451407, 95814, 562059, 426705, 362620, 37973, 563028, 512276, 234599, 491279, 509547, 384231, 479629, 352143, 183214, 302547, 334867, 545257, 235642, 108836, 557461, 379914, 30357, 60269, 317507, 270650, 218687, 41848, 399618, 471085, 497386, 212864, 151599, 323119, 86512, 41116, 210208, 460413, 543706, 34752, 253600, 468125, 530076, 190094, 540196, 393685, 480059, 518899, 405541, 120697, 44467, 197427, 483090, 579906, 364031, 489191, 164091, 370739, 429820, 138567, 48267, 88635, 565941, 351287, 82381, 136168, 184184, 97097, 452201, 85568, 383173, 167348, 441598, 571610, 458325, 175198, 534312, 526671, 192652, 109797, 47775, 276971, 154587, 293763, 571825, 189365, 275976, 516239, 251205, 95455, 264791, 222047, 52869, 225609, 105222, 534058, 486543, 231954, 4956, 6091, 110894, 391084, 551494, 278486, 128690, 347455, 440377, 482499, 398177, 335233, 331057, 205086, 426427, 474568, 45999, 127115, 415545, 401816, 537132, 564197, 536795, 209733, 117543, 310079, 121110, 37567, 186599, 91948, 439509, 197004, 413754, 259658, 461688, 22683, 232049, 368702, 14494, 77174, 255137, 97744, 275644, 520707, 225591, 427859, 370638, 79298, 91587, 14006, 388616, 451131, 52322, 554104, 10655, 405648, 41087, 101548, 412442, 113716, 396754, 125350, 472924, 379317, 296330, 342682, 54007, 35594, 504277, 558396, 299346, 467157, 320759, 532543, 526941, 495973, 461805, 389677, 77472, 154912, 113810, 286460, 472172, 199363, 312420, 157188, 415275, 459914, 495592, 189027, 293066, 363415, 53740, 124709, 573928, 506372, 146861, 527562, 92815, 154073, 113521, 241797, 415239, 275685, 21204, 217766, 491965, 454227, 469053, 160421, 459214, 436979, 99372, 67117, 473503, 555738, 405093, 58655, 9425, 306284, 121762, 372723, 323379, 568398, 362699, 413971, 361661, 359059, 360097, 35097, 429144, 558587, 276105, 295325, 473190, 257959, 172498, 188511, 193240, 451120, 448509, 237098, 151375, 491287, 128506, 449433, 64520, 108272, 214022, 199127, 77849, 203948, 316229, 341623, 14804, 235446, 171002, 223353, 475443, 435766, 550206, 49123, 261863, 247493, 140488, 490016, 301890, 430251, 186705, 132935, 270661, 318754, 520264, 507187, 147960, 194200, 568816, 28329, 168349, 170774, 423165, 366430, 74902, 17076, 330348, 344996, 544250, 453313, 568216, 164921, 354410, 329107, 275604, 431901, 300981, 136938, 154068, 24285, 82981, 263101, 53184, 101195, 126995, 146601, 337547, 121989, 125314, 11497, 255912, 381228, 133428, 46737, 207215, 551481, 325015, 491474, 343211, 229010, 23093, 250250, 148841, 332271, 107244, 9353, 309341, 296059, 216468, 192799, 38875, 494099, 109118, 498815, 473678, 45007, 266687, 357787, 267571, 406303, 267586, 496429, 263599, 406224, 446332, 129733, 304493, 390855, 373500, 271500, 494706, 32405, 301317, 508915, 176683, 285978, 349170, 494634, 96488, 251736, 458248, 507329, 388042, 18467, 559614, 477305, 60145, 466597, 6730, 20355, 64507, 404797, 378030, 364464, 277472, 183603, 411934, 511527, 481581, 105781, 53361, 436511, 260435, 530397, 395723, 455902, 19783, 402808, 203876, 319855, 400307, 126825, 435681, 475317, 84591, 59918, 152192, 135566, 318892, 336006, 54533, 31345, 351976, 578103, 365485, 349153, 24849, 28093, 221814, 322049, 135303, 495356, 503275, 108287, 360128, 336313, 511506, 3692, 194276, 528047, 281035, 367484, 228179, 55723, 309467, 19624, 260838, 382435, 441873, 40042, 403107, 289895, 275075, 53608, 110710, 359778, 472506, 437947, 98616, 400104, 236604, 475439, 17401, 469570, 34193, 317015, 379771, 359200, 235278, 10624, 390423, 254079, 421556, 67347, 249309, 400443, 71920, 194542, 384146, 29403, 395144, 136233, 123131, 210604, 201004, 408475, 449089, 122796, 301479, 284274, 369712, 481634, 554996, 176912, 63804, 525600, 66853, 345224, 105666, 34285, 419752, 485083, 468867, 322820, 544264, 189951, 540006, 327490, 497082, 428872, 385084, 58522, 305986, 294751, 257020, 340276, 493814, 550449, 419677, 253177, 64723, 295819, 104796, 285285, 390365, 27950, 2232, 273427, 225599, 97899, 373218, 165859, 397358, 321666, 63973, 322064, 548732, 335013, 517487, 111281, 481002, 18425, 444397, 57480, 408746, 498639, 284623, 92145, 101697, 138518, 287387, 354539, 203171, 78062, 272729, 442877, 102469, 449981, 480487, 534784, 246327, 505242, 463102, 532866, 513533, 384348, 315427, 506534, 171494, 576689, 157653, 376209, 504524, 254938, 426268, 560282, 318157, 267552, 431606, 128460, 532967, 67988, 538748, 237309, 164172, 399973, 522897, 103969, 102671, 34777, 575641, 572467, 324995, 462298, 397813, 4212, 396461, 352243, 260110, 161777, 475380, 554156, 494202, 365363, 32681, 19448, 523664, 282336, 386580, 439669, 344862, 161925, 395703, 420827, 198422, 562037, 114968, 575805, 157198, 566395, 44609, 230263, 186345, 486350, 134346, 71503, 1332, 65301, 384853, 161486, 260685, 551598, 335321, 289741, 546686, 342448, 445573, 475929, 370760, 9460, 290898, 138521, 77020, 227399, 539359, 365309, 462602, 166503, 338169, 337164, 522110, 131415, 489978, 87647, 553232, 495681, 107351, 528176, 565571, 475804, 913, 380661, 93746, 163589, 508748, 514605, 403472, 572754, 542257, 484347, 416197, 321683, 424307, 35976, 89320, 431854, 489385, 53229, 216277, 339705, 518262, 240952, 210471, 441012, 466774, 575648, 335692, 168178, 386332, 61624, 32892, 515850, 93232, 308302, 307359, 246982, 479203, 102168, 433285, 43968, 16403, 376763, 526861, 49143, 362140, 94841, 39115, 189775, 481454, 452459, 579561, 169395, 320282, 246417, 576119, 505020, 528944, 126647, 482297, 110200, 555687, 27704, 23415, 295185, 25797, 443916, 265449, 523869, 306080, 201519, 271987, 421105, 133175, 95533, 117418, 274216, 263192, 127301, 153909, 185125, 500044, 571091, 233697, 278336, 518189, 115942, 244949, 273103, 551062, 15517, 513496, 163255, 415429, 200941, 134958, 446130, 462244, 492154, 532106, 287265, 116466, 574499, 114790, 464536, 242619, 70402, 533599, 245713, 469338, 243909, 93842, 344096, 217078, 59079, 396882, 504987, 209993, 476385, 487112, 406244, 228506, 401778, 376397, 95493, 275988, 455665, 143453, 535685, 77003, 41357, 520696, 82287, 556822, 91406, 357641, 519768, 201151, 280779, 9321, 542452, 438447, 392715, 228901, 441605, 115973, 78656, 517026, 151709, 231958, 201859, 511508, 484405, 115144, 565045, 307480, 292102, 148668, 215718, 336021, 172196, 365724, 292782, 457522, 500324, 415243, 507570, 248834, 101686, 515176, 495957, 231017, 365696, 453366, 80094, 2193, 213274, 347653, 79394, 78016, 297260, 120398, 334755, 402109, 249315, 29711, 307125, 428152, 388893, 249111, 475370, 371489, 467723, 54614, 91160, 338427, 94340, 266041, 273205, 565939, 314556, 46640, 425743, 353653, 369326, 499500, 142824, 302869, 197658, 124620, 433618, 123067, 230559, 211945, 302137, 129571, 325470, 369596, 431168, 421406, 494985, 170348, 554661, 423606, 322945, 209548, 368084, 207250, 87320, 374375, 151299, 166797, 387358, 557686, 436774, 332568, 167295, 254407, 459117, 579781, 67615, 36764, 341341, 335954, 451133, 30267, 364243, 435792, 120509, 578383, 79453, 156643, 275837, 409771, 17171, 423562, 152724, 6901, 133667, 359529, 257537, 382482, 247720, 98937, 489573, 121152, 542638, 168146, 458827, 363957, 224790, 268051, 406004, 216925, 440060, 352618, 326757, 268274, 456602, 481747, 69205, 265135, 248300, 529308, 212685, 390565, 131374, 450281, 493618, 109819, 196870, 318346, 412062, 493384, 138, 18824, 144505, 310021, 419755, 552945, 8042, 30457, 335107, 180197, 41890, 254476, 419193, 571970, 271118, 421908, 152962, 265196, 54355, 223605, 70493, 282176, 468613, 331425, 509366, 406805, 225040, 418282, 50586, 563559, 268892, 139549, 495336, 373309, 208002, 71137, 205451, 184490, 117744, 210749, 401167, 195151, 66041, 184218, 95506, 317765, 476851, 539702, 508354, 332417, 251655, 149755, 560217, 45536, 291827, 142360, 505471, 572282, 371796, 80865, 219987, 181894, 498372, 260500, 434485, 255519, 275969, 362145, 264959, 474347, 367462, 538615, 25388, 402623, 24020, 284527, 46686, 193787, 245206, 534564, 429184, 123410, 342523, 424633, 555843, 32056, 338529, 235469, 499632, 296711, 38213, 516503, 532589, 254266, 74937, 367641, 315808, 423363, 85731, 385085, 368448, 419785, 289621, 154772, 230945, 206653, 478434, 465137, 350712, 354891, 62398, 40746, 98127, 361439, 108254, 98987, 503739, 252215, 277048, 288397, 570598, 187949, 162858, 130553, 126352, 129722, 91056, 11332, 580345, 103252, 546631, 75339, 106748, 282949, 305084, 372113, 422283, 392974, 248601, 169229, 570542, 559442, 59526, 256365, 43921, 191729, 439550, 392260, 248565, 212893, 492383, 71828, 297640, 466710, 517758, 349647, 136922, 532142, 40735, 526747, 87845, 412571, 89707, 125698, 441905, 447146, 172877, 336309, 251360, 463782, 139284, 8688, 556370, 442078, 119904, 529203, 195124, 275198, 5823, 172376, 338866, 393896, 268735, 482720, 355661, 299444, 396082, 248457, 133244, 488127, 47386, 370634, 37327, 472, 493189, 94175, 202143, 525513, 260545, 5994, 261298, 195195, 501067, 342762, 406071, 531151, 397512, 217054, 14412, 458208, 269445, 284548, 186558, 308722, 381875, 313186, 81862, 293466, 389673, 226639, 382418, 198234, 10179, 228209, 277623, 228560, 13336, 372788, 44127, 195180, 501739, 264919, 36041, 352187, 467460, 452652, 328786, 469035, 128154, 153229, 337575, 513849, 162113, 262938, 125635, 247487, 430872, 427745, 93034, 423349, 90306, 277776, 511693, 163047, 411653, 247216, 551299, 283955, 389133, 257544, 250372, 252996, 229419, 372288, 46104, 283524, 314672, 164141, 189617, 494986, 221400, 396581, 493407, 397518, 256806, 465178, 17984, 579699, 86988, 139198, 571653, 40783, 400576, 232875, 546646, 450555, 23230, 67729, 198463, 367011, 453767, 402908, 71646, 392722, 557819, 219158, 73436, 304625, 120235, 155386, 579165, 327805, 257853, 264709, 34113, 481349, 356215, 120519, 80915, 10405, 100523, 387250, 257934, 107013, 77603, 106275, 277530, 7796, 474543, 428707, 192501, 207276, 82705, 314825, 181212, 251578, 317105, 519088, 33733, 182972, 570995, 252091, 290864, 476848, 370802, 8580, 506078, 521117, 567764, 109046, 485131, 35242, 580235, 96615, 330881, 335451, 377008, 15073, 92905, 202423, 78542, 467925, 39685, 366560, 121901, 137304, 560421, 205531, 52440, 6229, 307683, 54586, 173883, 105418, 250225, 70702, 330863, 290896, 224477, 18781, 114326, 407080, 356498, 29096, 567870, 409651, 222006, 32190, 364027, 529649, 491899, 564072, 308780, 365271, 102330, 496782, 86140, 49434, 390168, 435988, 11525, 494162, 313473, 220270, 46981, 418141, 377091, 144907, 273783, 136286, 38396, 5736, 536294, 181249, 143068, 334915, 342709, 367876, 313627, 254033, 261876, 213602, 221927, 397337, 331366, 227898, 107977, 538999, 348631, 320875, 63677, 407139, 515423, 22174, 294900, 146091, 392154, 136624, 45621, 367085, 432624, 269751, 124297, 42694, 187878, 2471, 414822, 162641, 313517, 150421, 57595, 227512, 462770, 241803, 490810, 202827, 238708, 515390, 42320, 231524, 454659, 78699, 129166, 158897, 514685, 241453, 223093, 456391, 367706, 495852, 509490, 104621, 467154, 474709, 4424, 448533, 190992, 24974, 14921, 346703, 177245, 579926, 244026, 472103, 372855, 181156, 87244, 20490, 93600, 124145, 456149, 231126, 511251, 48111, 138081, 261344, 305235, 431917, 88082, 518139, 237568, 579814, 252940, 567721, 426874, 39201, 451228, 322749, 231952, 479929, 142269, 282812, 462379, 263311, 361030, 89285, 255660, 472585, 244578, 470622, 110674, 397782, 497838, 219411, 300273, 298362, 36758, 490615, 170955, 73726, 64750, 98290, 156924, 335148, 26503, 166414, 243964, 235621, 560392, 85814, 536720, 105246, 229622, 239917, 75299, 382078, 26517, 361811, 349912, 56886, 163292, 120524, 338987, 87322, 400082, 517508, 132141, 333335, 291102, 561172, 281461, 315488, 490462, 176757, 115003, 24672, 182571, 545670, 275117, 213309, 436882, 426611, 165014, 110829, 351991, 239398, 536375, 8649, 316019, 564226, 502305, 127134, 189461, 300488, 257729, 192722, 558639, 364251, 452968, 137953, 383737, 456640, 357604, 385555, 205002, 509654, 359959, 56289, 290623, 521649, 128224, 420939, 95526, 357278, 237254, 208605, 373198, 262001, 327970, 377023, 31443, 287245, 63566, 218204, 485761, 328935, 373415, 555108, 24428, 382552, 107620, 485016, 158660, 38241, 153039, 527752, 566027, 30395, 251696, 165803, 265134, 26466, 279058, 70240, 104410, 483667, 326978, 535327, 357228, 541025, 385381, 243950, 515750, 493841, 166230, 173163, 259576, 404997, 501388, 179141, 365164, 148644, 403525, 549789, 430804, 403826, 524848, 65191, 507766, 422211, 179186, 75663, 199539, 180739, 453907, 27777, 175072, 81479, 213547, 244032, 374595, 315555, 544046, 139836, 432626, 572608, 480583, 46211, 189785, 394596, 513704, 310203, 518392, 29136, 305191, 421184, 84551, 159456, 107935, 161234, 281414, 34985, 556783, 543264, 15610, 167264, 490126, 360637, 546703, 168992, 192655, 175912, 339734, 198658, 504488, 39083, 431364, 266275, 200659, 166557, 521314, 339171, 498175, 385066, 570275, 357379, 488287, 200678, 164793, 132453, 341335, 510992, 15963, 413225, 219798, 191078, 325366, 45335, 191850, 385238, 269450, 510493, 206800, 540843, 66929, 297812, 273069, 84984, 412508, 216663, 16567, 407387, 183539, 114340, 101936, 115911, 40905, 221561, 460723, 501022, 516458, 407266, 123968, 308103, 492298, 348941, 282730, 413684, 308621, 421491, 62132, 569801, 468124, 79154, 568089, 328163, 81918, 408221, 62623, 446141, 254197, 396266, 283403, 400124, 13893, 193622, 477112, 48491, 386058, 346265, 456204, 559747, 6010, 210731, 136229, 530097, 63306, 314572, 362599, 320566, 105014, 233954, 67023, 130065, 267797, 257255, 304336, 183836, 303962, 573134, 208166, 274559, 196653, 172805, 100207, 317428, 395187, 43758, 455350, 521587, 245415, 82091, 532397, 371490, 387773, 67867, 193936, 508215, 62510, 522425, 310348, 22983, 373487, 337262, 127393, 477949, 98345, 422215, 578591, 371015, 177954, 404507, 129175, 565444, 306095, 339798, 222127, 50322, 284955, 5690, 551793, 129322, 54065, 228910, 140420, 325821, 408792, 401288, 418854, 532793, 39750, 292845, 307534, 110811, 341218, 28367, 540669, 565938, 265173, 572353, 399582, 182240, 17301, 110052, 580651, 446694, 255797, 13815, 247790, 164290, 13202, 490695, 309261, 245174, 443443, 553278, 304031, 520027, 50360, 7393, 370250, 50403, 504955, 34826, 463309, 511325, 7090, 195699, 223595, 361512, 94499, 263582, 463327, 298893, 35076, 22456, 39858, 352498, 138915, 472648, 227222, 360927, 306342, 20784, 63270, 4187, 26645, 345456, 401821, 549297, 452803, 194777, 354237, 417089, 471591, 322297, 344043, 523373, 284509, 29041, 298269, 293300, 448113, 405226, 6873, 498903, 14159, 404227, 422341, 292920, 454623, 297439, 413685, 349408, 534247, 253789, 390291, 42875, 229866, 406749, 121897, 238049, 98656, 554138, 70714, 49062, 115777, 29214, 336958, 347587, 246848, 407822, 171081, 458879, 374559, 533222, 371302, 407289, 150127, 350425, 58167, 562149, 288578, 257434, 386834, 497311, 384258, 341874, 121952, 80385, 581396, 400525, 458594, 364176, 463690, 356940, 135016, 104953, 40894, 60526, 112114, 4244, 412002, 102835, 133343, 201949, 83725, 207803, 421744, 382839, 242911, 4575, 468029, 59319, 39769, 531949, 80634, 188329, 374130, 429293, 352091, 161367, 2697, 64731, 561911, 476188, 496215, 442019, 115692, 36006, 351975, 396004, 581749, 106497, 54592, 368346, 350358, 263783, 76107, 116503, 59685, 178512, 202573, 144580, 554559, 95124, 548858, 117532, 205283, 555861, 462381, 332273, 280331, 80943, 442104, 244016, 494749, 342874, 380931, 390346, 491063, 319542, 290189, 245817, 362716, 231140, 554783, 385805, 551260, 562136, 561354, 12612, 449888, 20536, 170425, 24674, 90444, 255974, 151575, 150197, 339643, 312574, 295571, 429415, 326966, 260604, 466624, 474571, 267082, 69266, 509319, 564636, 194437, 170072, 381967, 286411, 205564, 285195, 553815, 327713, 531647, 217484, 274386, 12321, 549876, 242261, 218220, 503234, 90732, 438698, 458861, 303992, 514891, 537220, 288506, 411472, 197336, 348499, 543681, 51164, 143367, 103059, 446420, 400936, 424542, 194288, 515376, 500977, 517081, 184155, 206017, 218868, 337207, 513136, 431, 65457, 83288, 135367, 564235, 52603, 362008, 315795, 361153, 334837, 492267, 422240, 89223, 368918, 130313, 87018, 489333, 106363, 231631, 542200, 12181, 33345, 22953, 22488, 277945, 222964, 522751, 581686, 457558, 536504, 105768, 408719, 191686, 435709, 536428, 538380, 143594, 53733, 388503, 531596, 51001, 315568, 429038, 544644, 514464, 46242, 228721, 393267, 459585, 428234, 511729, 185250, 369722, 510684, 347544, 304091, 32655, 465695, 28626, 413811, 158869, 213725, 54037, 322260, 459967, 574790, 276986, 483695, 229935, 378583, 357413, 550784, 563163, 82216, 98115, 178754, 345229, 289852, 522454, 527345, 457766, 331354, 114917, 158445, 391435, 3827, 511346, 455758, 563424, 485928, 320503, 182019, 179242, 67727, 138185, 117454, 201930, 520457, 380524, 530337, 342004, 93641, 379742, 524966, 330660, 240288, 297152, 24931, 502479, 31062, 360517, 466223, 431554, 90738, 440783, 51598, 6465, 398071, 84730, 38266, 55317, 188100, 139334, 122682, 168686, 10720, 570295, 443393, 332968, 318174, 432510, 559543, 71752, 70104, 578714, 86869, 263687, 224916, 342401, 308579, 55417, 493388, 51376, 578671, 507886, 195510, 405721, 421619, 456142, 363581, 466301, 575008, 263094, 244428, 98854, 98280, 39322, 259594, 88572, 430950, 369631, 62831, 435682, 187624, 54744, 60394, 461036, 23594, 368268, 342417, 14709, 544533, 217846, 548351, 264572, 417475, 529572, 551840, 251580, 495348, 77847, 561514, 398097, 13965, 230232, 69532, 581881, 139294, 521052, 457599, 77834, 418297, 138954, 350837, 175565, 62661, 542642, 339186, 278799, 429536, 300815, 534065, 290331, 505104, 26294, 344338, 458664, 524991, 46339, 91326, 16870, 336242, 530064, 124167, 496628, 94427, 222517, 108982, 236268, 426026, 184583, 215679, 445774, 18366, 16037, 366884, 385317, 32390, 149896, 25846, 185025, 187464, 565243, 423720, 331326, 240049, 259655, 410933, 150091, 503637, 7583, 281442, 280551, 262884, 369443, 271122, 348146, 294403, 391976, 181122, 420912, 224949, 22880, 87593, 274148, 208894, 346399, 396966, 149388, 431699, 123800, 281217, 26970, 31286, 392556, 569656, 500495, 401439, 500660, 409821, 295461, 205078, 138601, 465759, 532030, 315174, 461876, 332182, 329141, 334782, 557832, 280804, 470288, 84397, 309618, 267560, 551199, 573956, 180662, 524742, 270821, 94877, 8564, 525155, 132688, 528020, 430238, 343574, 100331, 335717, 206636, 393699, 289245, 550239, 448308, 363336, 208976, 549450, 474391, 83658, 357511, 118723, 161745, 218703, 408177, 16838, 305624, 19002, 335376, 493197, 559902, 404633, 362348, 127495, 518015, 338321, 276267, 95595, 26938, 564958, 229544, 292375, 327610, 290449, 511062, 250451, 383512, 528870, 137666, 488362, 267556, 115633, 26719, 453157, 58658, 366040, 51808, 454619, 431404, 395828, 461156, 477374, 416612, 387731, 144429, 362768, 124072, 474882, 499210, 325684, 458311, 539888, 27395, 230031, 307614, 411068, 394713, 475988, 481222, 173682, 427001, 367424, 346898, 317619, 454969, 22705, 330316, 383406, 372966, 206356, 479666, 180129, 420304, 313694, 404007, 273710, 16898, 511495, 533129, 425943, 263710, 382126, 46418, 405991, 259342, 115638, 482912, 274633, 79380, 475248, 152749, 205093, 501841, 47357, 411189, 30067, 560260, 520533, 87602, 46252, 290130, 549597, 30389, 385931, 421833, 276731, 328075, 258085, 201186, 107846, 460728, 285908, 494895, 178763, 357303, 41392, 305550, 486941, 52324, 318616, 523546, 208629, 389702, 545145, 547115, 407693, 151927, 484701, 500833, 358134, 42805, 16344, 572046, 351253, 143944, 82388, 341843, 118417, 534480, 110108, 501177, 153814, 156833, 454543, 476040, 56716, 203463, 507945, 173754, 315609, 496583, 167827, 375180, 299676, 409754, 110434, 301300, 159763, 489463, 225329, 572749, 280297, 16653, 544307, 153520, 373623, 472904, 284253, 315453, 57138, 446106, 397414, 263503, 520124, 128768, 73280, 9211, 555534, 434689, 287466, 568469, 42752, 578221, 42222, 401440, 505919, 81735, 295282, 265796, 144480, 557246, 465047, 228418, 407614, 445368, 423327, 501381, 302250, 51530, 564118, 10442, 406681, 343165, 234709, 478430, 66320, 11340, 503091, 123664, 387388, 21020, 550979, 383452, 228808, 439465, 86564, 359379, 412362, 440623, 39137, 242404, 478337, 81504, 1518, 478433, 492323, 223451, 355360, 499525, 345137, 151229, 52746, 557507, 576031, 507511, 262710, 195639, 258725, 237357, 367599, 82813, 226018, 310300, 484690, 382256, 561761, 394891, 311524, 388031, 224670, 103358, 192497, 558378, 488403, 509482, 78107, 8445, 404271, 337777, 91262, 15366, 351609, 247223, 396106, 531128, 300913, 171287, 281134, 525607, 297980, 49053, 469639, 468405, 532520, 535506, 510593, 425127, 164005, 26424, 151922, 154377, 445046, 9039, 159900, 483737, 282600, 449865, 117327, 140627, 542458, 371531, 144660, 386497, 72850, 496747, 47944, 136365, 254281, 370637, 346138, 170638, 459706, 149192, 233677, 235984, 60018, 572807, 231315, 417374, 129257, 494320, 307423, 429074, 168822, 372087, 96328, 322315, 457800, 222024, 461983, 528404, 332072, 571858, 336465, 326761, 206394, 540483, 315338, 545316, 294119, 360604, 511977, 462022, 281323, 379678, 260925, 308459, 481667, 442451, 467411, 142349, 474526, 230220, 6760, 398858, 84546, 265743, 223216, 46805, 437119, 374485, 303956, 60979, 53616, 224499, 146224, 517702, 99745, 294100, 111907, 242446, 154654, 522195, 378311, 366115, 60569, 55410, 183029, 297520, 456895, 148034, 338704, 335140, 131245, 350013, 205352, 409856, 576801, 16578, 257894, 350631, 541113, 41843, 560350, 142946, 436558, 535588, 346645, 513541, 170687, 277470, 296816, 341198, 209239, 400204, 43338, 551875, 258112, 321567, 82246, 247668, 492524, 324450, 232309, 285910, 264805, 474920, 169959, 245846, 580735, 488002, 554301, 388761, 441858, 268739, 2451, 503855, 94813, 416818, 425461, 578306, 192191, 18127, 495051, 534001, 168811, 447879, 557783, 314693, 230702, 521003, 351807, 251476, 258205, 350794, 328683, 413295, 425815, 533166, 48428, 230932, 68738, 497350, 471528, 187872, 77136, 428301, 397276, 548768, 65207, 287737, 297851, 167964, 262724, 310061, 562747, 260738, 92549, 188762, 298468, 446324, 246012, 78410, 22718, 117137, 264016, 355717, 342862, 465906, 330650, 287236, 54750, 243625, 100686, 527509, 474616, 157158, 253433, 153764, 476504, 478812, 48993, 486789, 176668, 162551, 246985, 284648, 47293, 546693, 86292, 94163, 343933, 59174, 351146, 552538, 334769, 363423, 88330, 16509, 423116, 252604, 428612, 497001, 560662, 48786, 467966, 435811, 454155, 2056, 124931, 557578, 541288, 530202, 137892, 17236, 110673, 245314, 96436, 84448, 55076, 296802, 255339, 22090, 200735, 175256, 6906, 264855, 581697, 14036, 123916, 183330, 414954, 77785, 32819, 332976, 8385, 35663, 68434, 529929, 34019, 143275, 403657, 454862, 77292, 412516, 252160, 563455, 507459, 75283, 97580, 357402, 185265, 268988, 186547, 524690, 514553, 123360, 166533, 208006, 18194, 359865, 248854, 177950, 98194, 498532, 559956, 237783, 28307, 325008, 140540, 413046, 508025, 276387, 499130, 540372, 449132, 119596, 436453, 377814, 110634, 411700, 245559, 378558, 66519, 495797, 208549, 566234, 545154, 393110, 573374, 31392, 46882, 404249, 389634, 76009, 442365, 186254, 16759, 476882, 91666, 96755, 75434, 576098, 311205, 384449, 115574, 191376, 421539, 443780, 247068, 224045, 74096, 163615, 124979, 222958, 3729, 226383, 387561, 184661, 326028, 273999, 494579, 544114, 44724, 401433, 510735, 359081, 464322, 73680, 323186, 137120, 564349, 178189, 393487, 148299, 94453, 161308, 426303, 520349, 249955, 408556, 53457, 2839, 67537, 117108, 519801, 252614, 554863, 42102, 245651, 32963, 400858, 177167, 392443, 425296, 180463, 176037, 256236, 578972, 576397, 360147, 297066, 276749, 377802, 67715, 322735, 56972, 195852, 177166, 124911, 406490, 32095, 95034, 392959, 323396, 371552, 84550, 217034, 173652, 411330, 410585, 437300, 533368, 20769, 326841, 558836, 362278, 50811, 147246, 30769, 278732, 361375, 117179, 119939, 198960, 221685, 313648, 160818, 68260, 441296, 96490, 152202, 468825, 300811, 556851, 515387, 175655, 21061, 373409, 309916, 389810, 574702, 509350, 250239, 40737, 442040, 542027, 302715, 479621, 470386, 347150, 157639, 537825, 528654, 105208, 228749, 330807, 293057, 360660, 118060, 46195, 467516, 349438, 323240, 318906, 327754, 550545, 554842, 388965, 5169, 257860, 81845, 384883, 385940, 200001, 314558, 354639, 67985, 160812, 478195, 404236, 226357, 106104, 261085, 127615, 403687, 488011, 464593, 335299, 488160, 97999, 411028, 469055, 490908, 74177, 226738, 400728, 530433, 264642, 552272, 246915, 251262, 504900, 541010, 214926, 181443, 184123, 271574, 140179, 26129, 313337, 421206, 242457, 401804, 64694, 126457, 38238, 76487, 7040, 519899, 321852, 266914, 289169, 151741, 204767, 437277, 215546, 157577, 545679, 186095, 181022, 470999, 42570, 192111, 450974, 555118, 97369, 85911, 13769, 203732, 218482, 419445, 12153, 544695, 182541, 178625, 195755, 70863, 72531, 501909, 6631, 306353, 25322, 478833, 84980, 183701, 352481, 443086, 79333, 423537, 113, 77123, 502758, 557314, 532037, 455343, 210408, 119145, 109835, 253981, 342515, 450961, 177832, 530002, 565269, 542814, 235876, 422705, 432239, 180827, 164153, 360082, 358884, 29602, 66190, 68331, 362898, 508398, 514118, 394113, 205850, 135392, 141557, 297997, 313993, 57895, 29509, 325545, 536894, 48679, 456699, 128172, 128181, 377920, 3794, 209256, 408795, 9738, 388900, 319061, 269022, 15957, 120032, 552723, 264233, 330796, 565361, 360723, 336245, 270906, 132494, 58146, 124364, 400658, 275723, 425622, 336414, 538247, 567142, 77874, 54323, 506838, 130875, 131276, 170917, 124718, 394326, 126129, 417355, 8690, 127581, 532555, 336668, 311254, 313434, 419738, 411559, 320943, 4595, 551941, 301714, 300868, 493641, 502370, 423630, 83464, 495728, 442097, 115169, 206121, 496121, 499591, 311015, 575753, 255536, 202128, 139586, 105714, 339336, 567149, 34452, 574856, 527670, 555939, 273537, 299689, 322969, 480729, 528687, 34874, 169648, 223587, 156527, 325524, 76079, 439676, 148502, 137806, 476094, 369776, 304165, 102672, 371598, 486912, 279990, 291784, 513676, 405324, 176477, 461226, 311290, 416733, 144618, 46260, 119849, 399887, 150787, 237988, 221378, 23102, 313278, 415492, 79462, 460643, 534543, 187518, 569103, 564339, 577413, 119555, 114474, 573700, 520528, 504969, 370208, 249046, 149423, 525432, 149212, 517819, 460757, 252074, 351322, 120021, 327304, 101531, 223317, 40351, 116270, 196134, 307759, 208372, 194638, 228914, 352162, 107650, 551550, 108112, 503939, 470871, 571881, 27519, 91023, 221693, 98155, 33005, 314495, 362547, 464265, 372494, 160679, 29712, 497194, 163155, 60046, 560332, 168366, 330327, 114219, 58106, 551622, 469727, 398010, 171017, 173033, 98950, 523318, 17179, 51095, 249397, 12704, 68003, 455340, 543740, 402867, 205075, 184642, 383958, 530211, 548976, 387267, 489785, 145480, 234023, 386389, 180271, 144610, 525183, 524297, 323108, 239845, 351521, 303314, 64941, 435454, 335366, 481598, 543729, 391145, 497807, 363548, 568981, 564631, 15764, 459747, 451400, 263478, 369268, 36755, 65836, 27246, 528811, 10643, 519136, 234032, 50778, 204351, 230851, 488822, 104380, 76015, 518957, 151563, 55891, 374239, 403652, 573954, 316012, 310552, 30932, 557677, 525299, 562377, 560890, 569050, 6819, 76004, 493368, 157321, 517306, 313593, 489062, 154020, 254007, 392153, 500499, 45830, 180488, 424483, 83110, 51203, 45063, 149598, 368062, 479936, 413222, 382274, 96589, 215081, 573549, 3478, 490044, 353494, 421877, 186193, 507427, 406591, 315994, 519398, 131509, 106835, 152293, 446512, 369799, 103280, 534141, 44592, 238808, 564902, 503294, 504766, 72872, 173297, 50462, 487665, 144018, 301954, 53823, 237542, 279288, 338535, 413716, 468087, 199598, 368659, 486536, 355792, 183626, 160901, 304387, 453801, 403190, 282066, 193047, 251790, 14547, 505788, 518359, 250268, 193049, 392776, 321036, 411690, 126047, 328802, 452468, 344410, 553871, 157209, 334187, 262136, 501646, 153421, 207738, 499372, 154965, 159887, 221555, 479334, 316154, 326925, 289415, 125476, 209044, 336276, 523204, 48692, 540133, 36421, 575666, 254703, 247049, 246318, 148077, 29801, 79808, 136715, 406368, 504852, 469671, 356767, 288852, 48442, 561448, 268389, 202447, 374211, 218751, 189542, 205911, 335999, 143824, 482134, 219861, 193001, 459923, 69956, 333461, 85998, 23017, 212276, 291518, 387210, 556246, 203251, 271395, 536609, 499826, 85827, 287167, 544222, 190136, 386778, 150309, 513796, 252311, 400298, 58733, 233477, 495344, 499267, 535133, 527048, 380957, 490010, 580505, 301431, 124174, 141240, 526927, 171453, 41461, 10645, 97503, 3915, 526522, 412134, 255123, 121344, 177974, 259439, 238200, 33732, 269973, 21286, 91837, 91177, 534659, 222681, 546218, 2374, 139039, 303548, 129439, 533494, 537878, 542350, 407460, 126230, 203063, 472552, 539830, 12372, 561979, 205847, 556498, 410155, 128201, 26733, 164552, 313807, 367401, 500680, 302000, 417980, 568854, 413289, 544349, 85353, 202431, 468977, 15823, 516535, 260772, 512830, 52365, 374062, 293293, 281601, 451793, 101146, 202076, 468373, 441608, 543661, 213349, 343053, 73786, 456900, 265708, 234903, 560640, 190981, 147518, 386542, 348723, 387401, 283163, 86556, 14377, 317816, 417802, 329843, 344194, 211498, 154244, 474964, 535860, 372612, 313234, 581719, 502148, 381041, 188236, 239252, 251132, 38622, 409262, 69584, 330923, 268603, 124300, 143392, 83000, 137701, 228664, 34691, 243029, 309168, 547284, 201042, 447439, 290690, 160255, 473843, 562208, 101842, 303476, 403547, 458467, 69325, 403951, 110159, 409443, 508787, 256220, 557694, 514207, 576253, 301746, 444691, 226046, 14870, 14591, 241643, 389777, 83548, 415663, 43829, 393292, 477567, 491242, 106525, 346774, 427046, 484165, 105885, 468713, 326116, 466450, 403579, 220670, 357471, 488463, 67320, 5486, 262967, 419735, 91681, 93732, 536592, 31745, 322327, 450786, 2732, 124840, 124834, 18555, 97209, 564267, 454940, 108739, 245780, 554756, 97895, 93287, 362690, 549276, 121210, 479045, 35411, 96820, 571499, 354662, 547869, 509864, 132964, 429745, 569500, 38488, 283303, 301295, 440522, 341226, 208469, 300192, 470300, 576727, 169192, 230212, 71699, 81206, 554161, 506418, 251442, 287601, 417334, 120127, 424340, 524249, 58738, 532239, 478118, 521981, 361746, 510827, 534789, 571745, 23355, 189684, 571514, 493398, 525636, 370816, 512100, 268032, 94059, 527181, 335529, 169845, 160771, 474769, 313209, 261596, 221840, 565951, 505626, 575980, 120966, 20679, 162596, 140840, 345930, 522534, 470762, 550925, 15301, 375591, 462807, 158514, 515917, 445991, 6327, 112950, 198733, 364247, 294783, 48133, 557344, 552318, 194089, 99753, 294716, 477438, 393174, 60990, 32206, 480734, 577029, 160423, 287794, 133510, 136026, 46538, 46171, 312621, 333463, 296567, 51040, 82031, 141857, 398377, 148263, 39163, 263970, 384167, 56373, 338197, 478817, 197561, 272416, 374800, 207998, 556591, 415067, 562573, 403249, 212803, 294545, 380706, 575, 443940, 424739, 454789, 352182, 447194, 521769, 359253, 573725, 524361, 471373, 391637, 253416, 475218, 261955, 355974, 548296, 421763, 383893, 481635, 470700, 22626, 163496, 67127, 51100, 536782, 402364, 247921, 452428, 385346, 457871, 27977, 69052, 514116, 517326, 196885, 369476, 21083, 360297, 114579, 542450, 152353, 11802, 405709, 216116, 479633, 200952, 452341, 163683, 519836, 574402, 54324, 9138, 376941, 274063, 84879, 353298, 377609, 17585, 40931, 556012, 24132, 337649, 556374, 195098, 220152, 68377, 413602, 303376, 498208, 166692, 540452, 476628, 122314, 461794, 309087, 49120, 450054, 189836, 6336, 93106, 151414, 291476, 464812, 166747, 529532, 580919, 310071, 48285, 212116, 152614, 286119, 165893, 364705, 376559, 362108, 98642, 455374, 273184, 90523, 579576, 197090, 279931, 275830, 323483, 558248, 24289, 220764, 561673, 506540, 142667, 210175, 349097, 146614, 144586, 152529, 43451, 67548, 219216, 246782, 268333, 181217, 152461, 48150, 311149, 63209, 382584, 93140, 490022, 537887, 366643, 468885, 263047, 128106, 581326, 138856, 577949, 118838, 334328, 463593, 397094, 151259, 122597, 450364, 45859, 62446, 154, 2276, 113001, 462579, 231682, 422272, 277449, 101467, 428985, 330741, 359290, 294712, 269879, 241317, 581391, 285877, 124466, 212801, 93487, 193342, 254418, 466730, 140333, 358033, 567789, 84097, 416095, 468141, 573546, 280857, 286524, 431967, 381999, 409105, 509988, 74365, 100111, 70347, 359364, 262845, 130074, 550467, 453933, 124707, 248169, 43266, 128920, 390061, 478445, 93860, 439712, 339793, 126531, 424940, 552061, 184543, 91667, 223571, 300200, 509423, 238589, 388248, 237224, 563938, 519652, 535151, 66862, 100271, 442287, 527163, 205217, 185902, 247217, 328316, 336954, 280068, 347673, 319907, 486362, 485931, 200896, 488149, 351250, 4201, 30049, 343062, 438567, 444390, 244933, 265037, 175785, 61561, 278293, 40051, 328500, 470310, 85400, 314055, 437970, 309424, 397959, 172492, 117079, 106589, 263084, 168627, 141065, 103773, 193162, 465859, 52943, 327000, 482295, 142804, 495171, 451364, 422026, 293319, 438426, 229357, 457924, 334364, 297011, 10714, 358717, 174934, 564936, 212935, 50782, 204407, 394560, 489700, 492382, 196924, 403773, 183884, 110961, 89340, 399832, 19838, 16839, 409009, 387603, 349663, 220109, 572284, 378099, 530781, 11202, 480936, 581196, 258518, 84460, 567486, 533879, 560789, 367795, 231476, 385743, 435805, 520750, 309843, 192307, 492243, 544274, 71313, 576927, 257657, 265915, 459879, 187732, 173500, 78361, 222921, 300368, 412373, 544595, 533568, 452236, 514131, 211735, 335865, 8431, 131572, 526419, 172681, 7414, 348202, 187933, 430885, 506927, 86981, 304587, 222564, 261956, 134818, 289425, 366139, 349892, 538330, 512788, 555390, 73427, 430319, 311784, 147290, 306073, 132991, 161852, 450321, 126642, 475641, 509914, 144199, 418409, 117644, 58268, 78290, 138276, 450940, 425032, 12567, 310989, 397903, 125321, 121591, 36048, 22193, 485387, 563015, 22085, 278714, 442687, 167856, 292051, 543178, 515579, 490992, 534881, 492075, 169076, 35880, 52756, 581921, 301753, 556033, 361643, 76571, 32193, 351002, 437239, 559907, 442859, 186240, 429122, 561565, 253430, 422576, 86452, 408081, 176536, 38525, 352334, 405534, 465969, 407698, 18728, 376196, 201078, 473231, 494351, 159954, 388962, 309386, 451985, 30674, 420922, 30565, 147710, 319096, 299041, 303986, 217571, 195153, 334276, 183690, 209133, 9420, 389466, 14160, 384207, 443764, 319388, 362816, 322638, 211540, 336967, 435910, 294909, 326161, 192160, 288862, 341143, 21632, 504980, 267753, 226243, 347179, 10403, 73159, 260679, 106499, 438753, 225054, 453286, 74127, 369973, 187079, 143416, 100588, 507042, 102928, 472079, 454644, 36406, 477826, 73536, 467929, 529376, 158293, 245075, 270873, 418737, 423010, 229140, 442771, 344896, 355776, 162363, 579567, 534848, 395665, 91933, 307371, 478874, 69578, 405444, 502043, 514295, 218766, 311286, 254522, 305168, 248018, 224693, 255088, 141922, 514674, 182799, 534036, 365295, 417458, 307507, 89602, 372480, 72944, 119805, 22180, 62423, 99067, 353102, 404063, 24434, 547726, 50562, 35971, 303250, 205883, 496198, 220471, 500396, 225480, 450098, 128567, 510663, 389410, 426453, 304314, 214875, 440124, 513237, 285681, 200938, 427135, 120159, 517985, 265574, 109738, 175669, 453096, 59463, 3014, 119966, 319468, 408103, 95029, 555778, 443797, 434328, 381642, 14611, 16980, 148825, 470771, 510383, 160669, 212381, 565684, 537139, 238022, 229295, 466058, 87588, 487286, 333231, 99628, 309695, 248958, 219485, 235740, 184664, 112096, 315450, 1090, 153980, 317878, 573572, 378499, 416469, 403489, 176847, 224366, 62041, 83540, 347981, 534711, 237831, 308444, 282840, 395379, 43376, 194729, 573935, 320706, 469834, 109139, 331307, 254101, 57614, 220548, 345869, 354036, 276539, 383819, 502910, 485800, 489467, 157942, 529344, 161733, 279131, 142263, 443684, 205380, 316631, 391754, 267717, 362131, 473918, 276849, 230246, 26939, 318959, 179011, 223165, 165884, 158952, 170024, 337104, 272255, 94148, 539636, 111819, 157365, 581712, 1877, 119093, 4823, 224805, 412592, 41754, 461110, 561563, 485342, 266032, 358620, 470738, 111553, 570824, 341348, 426064, 308119, 380827, 497103, 147735, 355438, 440389, 74466, 283814, 262538, 45687, 107797, 168557, 332705, 528201, 508565, 561102, 415684, 485447, 403995, 374886, 42972, 550118, 101895, 340152, 216398, 346412, 147492, 365569, 128351, 534923, 318888, 217119, 51222, 361085, 89268, 171753, 252659, 507293, 430617, 465163, 572529, 554800, 34739, 132860, 442686, 232910, 364549, 92631, 115102, 332432, 156958, 358868, 333868, 369299, 40248, 141139, 443667, 431712, 91844, 566276, 5038, 246641, 397716, 327059, 272857, 566099, 142985, 225891, 38123, 115992, 571934, 27482, 230507, 245983, 196758, 71095, 92315, 141608, 208585, 352192, 441445, 324682, 205378, 75643, 12839, 41155, 311062, 506075, 110155, 254632, 249587, 578500, 140736, 137420, 191149, 392674, 64442, 533042, 11680, 520232, 500071, 439560, 54254, 253532, 345263, 507789, 555685, 377760, 330728, 113708, 403427, 61943, 215881, 462151, 210024, 483736, 57284, 389031, 314983, 533825, 383029, 396212, 236515, 4442, 508821, 199390, 463944, 422286, 313454, 448537, 327157, 308509, 486834, 359626, 532009, 71492, 428282, 276418, 356725, 122356, 522946, 486821, 286166, 567416, 24921, 405736, 75359, 550645, 225828, 338167, 121599, 101978, 285629, 279882, 159987, 551148, 132544, 157084, 396287, 489107, 387048, 543600, 60179, 105853, 315381, 398895, 167891, 512968, 134366, 491867, 286349, 386390, 40283, 389426, 150797, 544291, 509416, 3326, 396490, 275603, 555355, 26512, 159888, 150640, 175382, 283818, 324717, 268855, 221615, 227242, 232329, 354517, 96931, 451893, 474600, 482634, 223288, 35864, 418047, 502911, 405430, 179031, 425066, 116186, 553267, 541838, 79802, 46004, 260445, 211913, 222842, 221985, 258927, 215523, 553501, 391912, 535424, 191403, 477390, 236760, 321118, 213171, 149180, 401229, 130613, 338884, 396120, 351319, 388341, 351439, 8498, 107630, 183481, 200138, 540245, 371807, 197528, 310629, 375405, 429602, 260336, 560366, 259665, 272901, 424088, 33489, 116941, 96075, 547768, 224149, 453664, 321802, 171103, 78729, 311377, 523316, 164485, 522611, 163394, 343704, 59870, 238748, 474953, 42876, 544856, 137822, 361799, 286186, 147762, 486081, 331072, 320643, 74583, 401132, 114717, 574915, 377509, 346391, 82737, 274888, 486240, 172467, 167486, 128116, 335433, 519758, 566920, 320904, 19981, 171729, 291202, 241806, 436318, 561809, 96006, 397161, 210134, 507615, 534292, 220244, 123462, 467778, 275111, 353427, 30068, 565543, 294829, 117764, 137106, 548575, 164521, 448985, 293130, 102576, 251690, 356478, 425201, 262715, 227765, 57256, 496214, 201477, 298718, 445004, 458249, 472607, 506945, 505234, 376362, 575257, 68633, 239616, 425540, 228456, 565062, 263407, 118195, 283761, 393442, 250749, 495395, 473896, 553541, 3412, 206833, 181278, 314554, 356948, 523515, 183445, 216254, 424258, 120874, 501959, 426542, 459310, 302405, 475076, 58949, 521386, 168060, 160199, 105150, 247360, 228464, 274688, 178084, 300620, 136683, 312032, 140816, 194310, 139530, 569452, 84165, 178941, 496606, 403666, 98502, 43947, 274089, 98443, 479620, 416279, 22198, 524645, 187833, 398813, 147823, 394756, 107633, 305709, 489031, 288921, 413341, 520197, 350235, 28038, 84162, 83716, 242453, 417015, 574812, 448053, 534845, 341272, 187287, 83914, 182860, 200109, 530497, 146981, 469729, 310695, 153811, 960, 69770, 113721, 239676, 330243, 29285, 396729, 298065, 523400, 545214, 543836, 533504, 298928, 513643, 61046, 453562, 9708, 105921, 530082, 560298, 199881, 255682, 259464, 71376, 37678, 533742, 80941, 337379, 36783, 328574, 518199, 336044, 326156, 330238, 479126, 59108, 86728, 231747, 536990, 332355, 307652, 394970, 195850, 505879, 45276, 115060, 182236, 45678, 337915, 382800, 172690, 372307, 475535, 576827, 521232, 414471, 61892, 370448, 398733, 194440, 227187, 392666, 162892, 89582, 342380, 369689, 545253, 155167, 403935, 282583, 423638, 129716, 88185, 333595, 191047, 183412, 164696, 42808, 575191, 10008, 554665, 380416, 53374, 126653, 83682, 37456, 409382, 316561, 433019, 508881, 438858, 417485, 203521, 572301, 360876, 297540, 423081, 391722, 510644, 176086, 425263, 336577, 342271, 118956, 349794, 498982, 109539, 496151, 164105, 153126, 313541, 550972, 494733, 438835, 16704, 525467, 224285, 443545, 174406, 108701, 397736, 20188, 105688, 44774, 241161, 572565, 399011, 484742, 347570, 57550, 34825, 80371, 431192, 410061, 187990, 567186, 95379, 470458, 316617, 71434, 29909, 563656, 338025, 124311, 357808, 220071, 545958, 409890, 579158, 323820, 68167, 458876, 295599, 558579, 85960, 491556, 54962, 100169, 44801, 401235, 1915, 81383, 89468, 518981, 6380, 312074, 467572, 510414, 124525, 435029, 21088, 364244, 427921, 434316, 324252, 270947, 289444, 16523, 413121, 93141, 241611, 250108, 311162, 120827, 44536, 503416, 450526, 217322, 561631, 354380, 195159, 427252, 78494, 575414, 333095, 443952, 170558, 473723, 206444, 226734, 332582, 163057, 147195, 300864, 54974, 79107, 570768, 174690, 536786, 150100, 402454, 66164, 473607, 463739, 228122, 389669, 386768, 505402, 382969, 560009, 119026, 7307, 181519, 149645, 79472, 187887, 563170, 311066, 155083, 125458, 246345, 103818, 357015, 572276, 143744, 396528, 301046, 123974, 269958, 239089, 46169, 571311, 7936, 279421, 400027, 61459, 43007, 368807, 5703, 411130, 509084, 404, 326601, 38769, 579261, 84183, 536830, 136212, 529545, 91387, 158182, 439107, 381667, 201252, 99678, 546466, 566514, 521838, 545238, 334600, 470551, 200457, 304238, 518628, 568243, 442199, 178040, 47545, 147328, 418373, 42178, 398596, 242222, 423818, 328117, 570077, 107969, 110518, 412704, 7819, 301432, 347168, 194381, 40286, 155262, 168416, 401982, 404766, 568557, 220464, 448409, 141518, 9322, 232116, 99179, 538163, 578796, 260024, 366799, 44960, 113095, 433110, 334162, 544109, 528131, 430493, 469173, 66406, 486854, 503008, 337156, 512658, 343821, 35206, 65942, 461673, 292844, 31250, 88265, 551553, 131534, 571361, 546511, 266963, 359677, 461572, 433196, 545982, 581278, 50844, 89555, 235012, 569960, 129474, 355748, 320685, 535340, 187929, 138248, 340491, 186266, 61919, 183756, 418810, 410969, 63488, 359471, 385667, 329268, 457519, 78880, 29030, 350833, 78404, 64497, 114194, 436735, 372219, 126235, 71092, 547196, 125769, 404687, 458583, 520924, 268992, 548933, 314992, 428585, 125213, 460145, 371628, 473103, 443735, 44454, 73094, 96450, 55860, 382939, 266115, 403265, 332739, 117939, 453799, 100345, 136795, 390483, 407318, 254678, 117679, 190135, 371999, 124516, 346334, 32562, 402243, 430285, 29161, 39403, 502347, 88920, 410082, 166463, 354322, 293121, 71726, 413141, 211867, 417610, 456629, 129980, 154465, 112793, 110404, 118994, 318381, 95336, 435690, 178131, 445257, 495291, 281237, 141477, 396670, 473745, 340228, 229078, 81099, 475978, 581601, 11570, 570418, 496813, 259316, 183913, 454611, 30803, 216026, 241935, 539617, 542070, 146757, 195918, 559235, 385695, 66505, 522834, 218368, 226762, 27412, 472666, 77055, 485844, 567901, 288451, 239600, 518638, 115718, 217240, 86912, 578836, 295308, 109783, 514525, 214856, 4173, 30133, 581913, 61675, 35217, 428740, 190499, 328833, 70288, 77099, 274360, 453409, 524019, 231295, 227230, 229984, 281922, 466262, 458903, 235147, 178592, 97564, 461655, 345185, 258488, 577631, 546690, 188520, 256540, 28480, 74080, 516110, 254129, 64909, 88987, 342092, 381318, 346754, 314082, 74984, 217997, 289515, 103821, 368206, 462908, 568665, 419571, 278175, 27441, 576385, 303069, 410980, 247951, 136983, 239537, 246735, 404243, 375245, 458143, 36017, 477474, 352005, 307197, 577125, 352462, 233187, 121557, 292579, 313556, 531796, 354221, 83639, 292554, 243008, 461788, 459382, 327418, 210849, 576607, 36228, 89207, 92747, 324435, 259443, 101906, 230315, 268147, 15614, 164493, 129100, 20255, 148267, 307825, 129187, 430759, 130733, 290628, 215805, 140073, 27942, 478474, 394190, 518586, 525077, 126135, 220898, 126288, 284920, 408417, 171064, 155344, 238065, 262541, 566264, 218759, 175878, 143439, 94609, 111371, 186720, 399741, 285821, 160103, 499135, 562862, 295016, 479328, 328487, 236762, 285784, 427150, 236961, 485871, 340636, 570705, 535369, 434623, 138239, 347390, 532710, 261077, 148965, 114335, 209380, 92526, 90155, 109338, 294990, 51197, 570978, 308979, 297363, 266908, 385438, 259906, 420629, 367509, 282552, 236336, 107051, 51500, 179397, 14252, 573430, 172989, 356673, 43795, 64148, 296894, 408112, 105732, 205439, 155191, 70087, 557539, 52891, 395324, 92529, 89924, 502366, 513054, 30693, 536514, 318333, 576286, 96859, 443857, 297779, 34423, 426906, 278290, 21147, 563024, 477336, 458281, 463087, 366116, 471009, 528802, 249275, 442988, 246285, 171647, 400710, 117834, 167671, 329242, 32074, 468219, 20765, 503235, 314879, 556801, 545491, 288905, 219445, 538285, 455255, 327845, 25529, 159889, 296192, 121249, 149364, 60812, 179553, 293674, 454894, 177762, 26783, 521094, 327165, 61882, 253522, 66737, 191845, 282197, 217395, 325863, 262495, 488883, 165686, 224336, 16520, 32877, 101090, 445775, 138891, 60986, 473131, 227732, 153283, 276572, 388674, 121107, 473420, 404554, 8922, 381786, 535304, 165351, 220348, 163436, 354019, 248297, 182098, 261236, 42704, 225041, 48797, 355462, 64411, 167695, 80340, 552074, 244411, 412267, 178317, 550853, 443820, 284545, 539897, 65837, 426343, 496939, 62617, 557059, 495288, 2388, 210857, 277722, 57646, 327338, 18171, 478140, 156202, 309152, 560307, 348954, 284732, 154801, 579631, 353787, 486496, 49901, 415413, 214932, 202307, 180821, 401735, 477301, 111749, 88272, 460684, 202321, 406611, 232375, 94092, 26560, 138436, 317134, 62995, 496730, 444082, 12429, 20418, 360870, 233137, 48947, 13949, 277810, 329865, 544140, 355186, 200200, 169143, 299715, 92001, 1737, 269234, 426087, 524642, 142757, 31288, 284064, 283534, 461295, 50655, 23287, 348838, 122355, 532387, 574527, 79710, 520736, 383090, 215910, 535251, 282527, 362141, 571835, 503497, 278395, 577632, 264686, 255479, 93587, 94307, 542000, 425685, 571328, 218179, 205317, 253712, 117203, 50410, 39277, 511700, 244484, 93603, 558705, 563178, 403358, 382211, 88176, 550246, 569550, 431257, 563334, 501971, 446358, 149464, 450151, 214821, 206102, 258397, 90464, 203798, 227040, 461506, 33297, 209728, 162429, 27888, 550652, 119550, 215858, 565771, 182511, 511641, 21079, 146620, 90192, 444262, 401688, 349393, 267837, 138789, 446536, 289689, 65829, 456792, 48653, 276444, 7653, 389159, 186399, 276089, 85790, 120162, 48749, 66632, 54344, 363078, 306094, 540753, 470414, 296191, 292287, 34445, 422193, 91779, 23258, 99701, 66238, 497466, 390345, 557804, 211810, 286972, 486172, 266451, 97989, 117322, 244096, 258330, 355471, 521914, 382742, 2570, 382083, 73730, 579524, 91875, 488928, 205280, 88094, 52565, 118730, 412761, 303797, 347367, 282703, 338889, 267969, 401037, 74964, 111262, 304440, 557188, 65161, 302380, 514622, 276459, 204272, 528466, 297871, 475301, 359115, 51443, 456045, 439445, 257615, 53123, 13148, 95620, 475325, 547944, 189855, 487734, 105866, 204131, 567363, 24081, 574242, 98038, 284701, 117275, 307892, 299714, 354045, 110060, 308405, 281718, 27648, 395462, 250223, 390496, 383605, 13489, 136819, 49886, 325343, 263362, 360173, 106266, 230490, 210082, 161503, 500693, 216841, 541182, 61852, 536280, 216133, 12443, 15085, 101632, 559547, 186699, 130512, 578485, 285826, 516996, 126123, 103427, 476074, 440049, 63421, 63214, 58785, 89882, 393014, 108019, 217154, 217959, 333904, 364797, 94318, 420303, 376982, 107084, 209400, 285046, 251768, 264241, 285799, 286458, 260393, 235124, 218984, 54758, 258346, 509095, 234981, 456065, 326570, 494003, 140387, 315294, 241675, 535688, 165522, 5028, 240378, 22377, 533158, 496513, 475567, 461491, 426754, 446131, 28595, 424560, 572469, 551823, 456322, 395293, 125859, 465295, 287415, 345870, 166995, 64249, 78590, 105115, 506901, 304150, 232165, 32244, 138288, 346672, 174079, 60710, 69866, 228123, 446058, 425722, 151778, 508174, 534070, 536834, 397877, 139817, 379716, 102118, 298081, 149993, 228407, 297868, 317790, 43435, 159196, 140130, 251591, 110442, 535604, 373776, 492642, 188459, 308086, 321424, 292346, 37968, 345590, 225608, 111673, 171484, 18414, 111211, 274012, 502827, 300267, 437785, 238377, 84693, 353409, 175390, 176746, 70943, 325856, 462342, 315696, 548562, 165133, 169885, 36748, 198959, 328959, 23163, 439657, 201640, 454959, 521509, 351489, 383533, 440810, 50412, 216805, 438026, 307272, 459770, 152677, 234671, 516012, 388871, 382315, 466680, 116783, 96215, 571168, 19404, 284084, 459800, 118277, 166711, 258356, 61219, 248459, 62819, 306343, 461898, 203108, 447448, 467091, 249838, 143644, 366162, 557174, 308277, 277821, 139897, 237550, 12101, 353662, 24234, 390935, 553867, 93724, 567421, 283054, 408925, 274116, 256544, 362367, 117527, 490280, 69693, 494817, 93280, 499802, 73790, 349321, 358525, 41516, 441500, 435632, 151840, 6189, 252406, 569525, 392183, 428385, 388873, 259103, 58876, 110142, 152120, 467662, 174012, 469634, 473651, 369872, 313946, 576225, 458984, 229583, 140655, 332908, 274536], 'val': [27789, 244815, 191226, 129520, 64409, 323552, 410934, 395497, 478320, 491377, 50753, 309499, 453926, 437540, 94746, 560227, 345998, 133636, 27805, 321062, 534915, 209864, 475359, 317250, 50165, 125643, 327401, 513096, 86625, 64629, 259136, 314907, 293452, 379156, 534935, 362293, 354278, 261758, 151938, 376416, 558528, 305412, 433822, 377881, 272430, 509020, 577182, 408201, 527022, 48917, 222053, 312191, 94002, 286591, 561311, 169356, 265464, 52936, 260695, 154263, 226170, 337563, 522198, 391837, 456483, 505939, 503724, 416700, 538463, 500062, 136915, 466570, 363286, 135902, 67412, 427386, 382542, 248152, 378561, 546229, 88102, 347819, 131335, 539962, 151130, 216050, 482242, 396608, 252468, 547487, 453626, 92602, 308545, 461410, 357957, 175954, 403919, 217521, 38892, 507361, 360772, 185464, 138196, 320715, 41300, 118405, 23369, 154090, 52871, 140068, 315577, 353295, 174866, 184945, 327701, 554296, 210099, 518937, 483165, 528257, 58350, 289201, 215170, 16161, 362327, 465545, 12670, 285742, 465795, 44263, 436968, 323116, 480652, 939, 441511, 281387, 136501, 218294, 510643, 550073, 109012, 264718, 119402, 574413, 379104, 389738, 183991, 356648, 112608, 402869, 146363, 168367, 417217, 266852, 145378, 18224, 339009, 89668, 524575, 464200, 449950, 458175, 27696, 224119, 256145, 468277, 74430, 436694, 387434, 576886, 206878, 275999, 64822, 379441, 236166, 86602, 566436, 234691, 511662, 380868, 17425, 325666, 370260, 531973, 261535, 433066, 389549, 54204, 417373, 97337, 401123, 227526, 213728, 425898, 551459, 18928, 261870, 317595, 315467, 572499, 349414, 93686, 241517, 526732, 107257, 231984, 120632, 127153, 489745, 206806, 331386, 489550, 201993, 377385, 56865, 485081, 141017, 506137, 575946, 184994, 23584, 448078, 139, 547336, 215278, 444631, 141501, 42790, 395344, 541017, 41373, 31093, 160834, 163666, 350309, 29504, 513793, 422729, 159969, 439623, 275270, 410141, 498368, 462341, 501031, 64136, 264385, 137634, 419828, 238537, 269099, 188891, 377670, 497351, 281448, 228504, 379740, 279165, 532833, 481710, 468525, 324040, 140785, 140391, 38709, 252403, 213445, 92020, 190756, 93276, 560756, 225558, 89062, 70987, 333000, 252929, 475244, 326308, 241238, 438907, 439007, 578210, 269015, 317763, 265719, 291256, 471521, 343198, 451123, 325874, 455610, 189772, 247576, 148968, 75636, 9759, 370596, 527691, 63549, 413600, 317206, 411765, 372983, 180095, 110313, 331282, 260595, 183155, 31113, 393243, 383450, 9908, 178810, 570848, 467500, 467599, 529235, 44022, 97479, 566518, 510230, 564355, 237041, 435387, 154711, 70657, 106235, 455345, 128644, 482585, 88846, 29675, 105537, 214648, 283064, 534918, 128748, 14975, 167131, 412339, 54605, 349868, 451571, 464911, 207108, 425676, 564629, 172271, 149197, 26992, 536073, 173383, 353027, 378859, 64779, 528786, 362352, 494122, 568863, 270009, 282847, 222009, 552761, 77473, 472595, 412586, 114401, 210795, 110211, 562803, 440443, 543402, 314757, 2006, 518249, 392493, 232760, 100196, 551170, 213033, 433044, 560361, 112349, 73668, 303971, 114389, 347965, 178516, 57495, 569543, 92624, 198154, 157847, 317120, 218357, 43939, 246398, 185201, 174420, 213951, 44621, 515444, 78741, 551454, 191981, 494620, 455448, 375317, 189452, 345469, 235902, 19306, 235595, 552148, 47624, 328668, 497488, 300221, 307442, 293505, 402903, 533537, 304341, 296907, 11360, 28359, 540159, 379842, 438728, 120356, 398534, 261824, 5225, 404702, 449156, 389869, 479155, 546203, 31736, 203687, 509116, 491470, 491408, 81505, 544607, 3690, 549754, 68993, 529314, 179214, 411596, 197352, 36481, 335785, 176060, 260910, 438505, 181747, 15303, 499571, 397375, 552656, 304741, 41200, 450993, 406129, 189845, 185335, 10386, 486087, 306667, 455044, 369533, 53450, 376450, 26630, 109860, 430670, 451943, 87356, 494393, 4011, 129139, 428168, 530656, 218844, 424039, 422655, 381519, 330109, 386841, 272080, 160004, 395463, 109313, 354584, 267314, 78061, 494751, 193717, 87581, 82396, 131527, 165543, 391774, 534128, 30851, 530470, 446265, 339512, 206025, 172279, 527960, 138040, 111825, 395831, 412658, 228467, 56433, 109055, 348481, 18073, 501824, 40102, 282150, 178535, 209835, 221183, 492107, 432414, 191546, 324313, 468604, 579468, 384513, 550527, 259253, 408624, 575081, 388927, 279714, 162580, 312263, 122542, 17244, 18982, 13004, 503108, 145604, 185903, 291634, 336645, 571196, 412639, 299349, 489023, 348091, 293333, 10432, 541550, 574665, 311327, 151965, 139858, 402765, 138096, 103499, 37670, 348045, 402233, 5724, 178984, 466921, 533140, 134103, 422432, 446574, 363461, 397734, 226662, 44068, 346707, 412304, 544237, 233638, 270215, 166277, 33683, 328601, 451679, 271785, 41279, 309452, 299000, 155192, 511622, 334542, 456756, 236272, 494427, 454478, 178435, 194940, 107468, 159627, 139097, 229858, 486968, 319706, 360706, 388812, 42737, 539386, 350884, 160025, 364608, 452515, 499281, 249371, 148309, 17328, 514248, 416220, 6527, 523292, 158222, 527379, 290231, 369221, 375415, 550471, 91105, 304924, 549055, 203138, 334148, 90367, 539565, 197199, 59152, 505942, 256470, 33882, 340658, 257448, 110048, 458153, 498666, 183391, 243896, 440763, 535997, 260713, 336431, 129675, 450055, 351133, 61315, 249534, 439290, 440508, 574946, 141671, 337011, 134596, 518914, 139872, 374430, 285235, 230247, 258209, 135486, 214604, 461953, 54671, 561853, 444294, 189213, 415967, 196336, 199977, 561599, 18462, 451373, 568882, 281853, 24744, 186009, 223766, 7386, 430245, 559277, 382411, 548528, 100008, 189168, 472387, 310770, 80163, 414744, 97315, 97748, 338878, 440336, 523517, 297374, 176649, 509891, 271844, 529004, 301595, 480917, 575293, 124002, 1369, 310902, 343229, 116750, 505169, 158887, 263613, 435358, 31566, 223777, 362521, 82468, 330140, 111376, 185157, 250227, 371564, 69213, 477067, 172004, 339470, 21919, 50422, 59532, 22168, 544538, 557633, 121106, 3310, 281331, 360434, 218855, 98054, 26946, 125556, 360328, 560637, 268556, 146568, 201765, 366576, 407403, 291675, 488945, 457848, 444152, 18046, 578655, 302318, 442688, 155145, 553057, 511658, 246843, 460035, 235663, 287366, 202825, 125590, 375823, 535808, 124116, 51339, 148286, 542939, 149568, 515445, 14167, 299493, 545114, 137961, 517822, 352789, 572399, 128939, 400596, 542758, 391415, 456574, 200492, 530905, 11788, 279806, 437049, 398209, 498381, 61503, 539390, 532983, 271681, 457453, 321035, 364636, 2587, 332087, 551633, 83364, 335861, 487438, 342869, 243218, 563885, 176828, 250113, 34869, 273246, 371873, 242615, 22191, 205474, 71145, 492016, 129566, 540176, 23709, 98658, 177714, 169152, 36049, 114972, 102159, 42834, 378502, 25453, 387916, 146659, 343976, 574241, 57760, 528399, 67208, 54955, 157052, 472078, 265584, 503292, 358642, 487720, 38837, 171548, 407826, 199017, 408253, 206596, 516026, 41247, 476491, 491836, 11760, 274939, 285212, 551086, 463633, 327780, 450303, 557672, 37740, 345380, 552573, 308120, 127161, 470658, 392928, 310202, 544519, 322816, 178618, 133251, 290432, 556130, 271712, 154222, 239041, 568972, 485916, 30958, 526143, 477042, 92554, 304305, 54164, 280761, 572861, 348263, 492800, 446461, 68128, 379605, 369998, 70334, 180373, 174898, 143961, 149014, 381615, 537153, 562818, 226408, 455624, 174871, 17655, 495533, 24499, 404792, 378723, 295076, 366748, 506335, 143479, 251920, 393804, 513417, 60704, 500316, 58194, 433154, 549112, 349402, 399164, 267709, 279145, 39670, 145684, 9891, 225456, 461007, 318671, 138639, 325838, 382557, 246968, 297610, 240111, 104691, 536683, 308506, 19592, 211837, 386704, 428218, 164170, 100801, 11169, 403975, 6306, 312341, 230661, 505344, 196280, 522393, 144562, 223857, 314792, 217827, 281397, 52982, 154423, 170346, 267225, 302089, 468858, 548164, 477085, 362434, 151988, 469840, 382758, 26174, 318114, 402855, 170605, 125958, 419624, 5754, 502848, 390087, 70865, 340272, 545363, 287285, 20912, 70471, 243031, 578343, 17282, 190760]}
PINS={'boxes.npy': '43a2cb1e8b6cc7011767daad9853a032aa1956b2bcd78b4be257dabe457fc7de', 'coco_ids.npy': '47fb70688d40e62b9ae08639f4d1d492ee0d28a8e697020cae7df3a9b80144b1', 'region_vectors.npy': '1dfa7c3d99079ee8841f6572302ca2d47e08121621432ea89f654796dad4b144', 'text_id.npy': 'fedb194993958ea46eb61d6fb713d2e99bb02e9d58487eb41b103d7a2ecad10b', 'text_vectors.npy': '1d549de3016c662281b853da695d4e378042e162ca81154826a9d387969884b5', 'valid.npy': '6d35255c2411c189799f5d12a7a55a512f415316cfbb3e8ea92047103a66e176'}
PARTS=[{'part': 0, 'receipt_name': 'scst_cache_receipt.json', 'receipt_sha256': '3a2caeb5950518ee5d82ac399de0462b5b4d07eaca0e380f7a9f8c49d5395510', 'archive_sha256': '64eb7d3c958fca4996d717c300191022176e5d688ddd4ba6ea94a5f1a30d3946', 'train_start': 0, 'train_stop': 40000, 'ids_sha256': 'f1fd1507bb3dc6b5dd4ddc57a83e723e2610ec7a45e7a6b05813fa633e519480', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/82'}, {'part': 1, 'receipt_name': 'scst_train_shard_receipt.json', 'receipt_sha256': '7ba08f77f38ce48fc8a8a023dbe70ffd6b9bed142f28d54711a9c3ebd0b98a12', 'archive_sha256': 'b9cadfb1f55067b767ed1e32012d5b5bfc7f9f0000c9ff8df0c2ed9676ab461c', 'train_start': 40000, 'train_stop': 80000, 'ids_sha256': '570fab90751d77fda57aebe84a17537a7cba57d3244fc06a5cdcb55ff2c1d885', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/83'}, {'part': 2, 'receipt_name': 'scst_train_shard_receipt.json', 'receipt_sha256': '3b8247a0e3fc24aff0a0b2125ce5fbd372976274c5fcf5e1f46315516824ad5c', 'archive_sha256': 'a78982bb44fbb4ce5668d36f9df47de5da9536834ac379f123349a893fd0c9c8', 'train_start': 80000, 'train_stop': 113287, 'ids_sha256': '22d47e2713dbe61777a97a8104ba4bc2490e253c4ff8fd71d0e871d57b2d38fb', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/84'}]
for name,source in MODULES.items():
    (WORK/name).write_text(source,encoding='utf8')
sys.path.insert(0,str(WORK))
import torch
from torch import nn
from torch.nn import functional as F
from csa_patch_model import CSAPatchCaptioner
from semantic_data import record_arrays, fixed_banks
from full_data import encode_tokens
from patch_beam import beam_search,text_from_ids
from saved_quality_data import chair_counts

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU accelerator')
device=torch.device('cuda:0')
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
root=fetch_inputs()
input_arrays={name[:-4]:np.load(checked(root/name,pin),allow_pickle=False,mmap_mode='r')
              for name,pin in PINS.items()}
ids=input_arrays['coco_ids'].tolist()
if ids!=COHORT['train']+COHORT['val'] or len(set(ids))!=11000:
    raise ValueError('Wrong v150 cohort and order')
index={cid:i for i,cid in enumerate(ids)}
if input_arrays['valid'].dtype!=np.bool_ or int(input_arrays['valid'].sum())!=32832:
    raise ValueError('Wrong v150 active slots')
if input_arrays['region_vectors'].shape!=(11000,8,1024) or input_arrays['text_vectors'].shape!=(80,1024):
    raise ValueError('Wrong v150 embedding shape')

clip=np.lib.format.open_memmap(WORK/'clip_subset.npy',mode='w+',dtype=np.float16,shape=(11000,196,768))
seen=set()
with zipfile.ZipFile(PATCH_ROOT/'cache_metadata_zip.bin') as metadata:
    for part in PARTS:
        idx=part['part']
        raw=metadata.read(f"part{idx}/{part['receipt_name']}")
        if hashlib.sha256(raw).hexdigest()!=part['receipt_sha256']:
            raise ValueError('Wrong patch cache receipt')
        pins=json.loads(raw)['files_sha256']
        if idx==0:
            words_raw=metadata.read('part0/vocabulary.json')
            if hashlib.sha256(words_raw).hexdigest()!=pins['vocabulary.json']:
                raise ValueError('Wrong vocabulary')
            words=json.loads(words_raw)
        groups=[('train','pilot_ids.npy','pilot_patches.npy')]
        if idx==0:
            groups.append(('valid','ids.npy','patches.npy'))
        for directory,id_name,feature_name in groups:
            member=f'{directory}/{id_name}'
            data=metadata.read(f'part{idx}/{member}')
            if hashlib.sha256(data).hexdigest()!=pins[member]:
                raise ValueError('Wrong patch image IDs')
            keys=np.load(io.BytesIO(data),allow_pickle=False)
            raw=np.load(PATCH_ROOT/f'part{idx}_{directory}_{feature_name}',allow_pickle=False,mmap_mode='r')
            if raw.shape!=(len(keys),196,768) or raw.dtype!=np.float16:
                raise ValueError('Wrong patch shape/dtype')
            for i,cid in enumerate(keys):
                cid=int(cid)
                if cid in index:
                    j=index[cid]
                    if cid in seen or ((j<10000)!=(directory=='train')):
                        raise ValueError('Duplicate or wrong-split patch row')
                    row=raw[i]
                    if not np.isfinite(row).all():
                        raise ValueError('Nonfinite selected patches')
                    clip[j]=row
                    seen.add(cid)
            print('Staged CLIP patches:',len(seen),flush=True)
if seen!=set(ids):
    raise ValueError('Missing selected CLIP patches')
clip.flush()

links,=PROMPT_ROOT.rglob('prompt_links.jsonl')
checked(links,'e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0')
texts=read(checked(links.parent/'prompt_texts.json','35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2'))
vectors=np.load(checked(links.parent/'prompt_embeddings.npy','f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c'),allow_pickle=False)
prompt_rows={}
with links.open(encoding='utf8') as stream:
    for row in map(json.loads,stream):
        cid=row['coco_id']
        if cid not in index:
            continue
        j=index[cid]
        if cid in prompt_rows or row['group']!=('train' if j<10000 else 'val'):
            raise ValueError('Wrong prompt split')
        packed=record_arrays(row,texts)
        if not np.array_equal(packed['object_valid'],input_arrays['valid'][j]):
            raise ValueError('Wrong region mask')
        count=int(packed['object_valid'].sum())
        if count and not np.array_equal(np.asarray([o['roi_bbox'] for o in row['objects']],np.float32),
                                        input_arrays['boxes'][j,:count]):
            raise ValueError('Wrong object box order')
        prompt_rows[cid]=packed
if set(prompt_rows)!=set(ids):
    raise ValueError('Missing prompts')
prompts={k:np.stack([prompt_rows[cid][k] for cid in ids]) for k in prompt_rows[ids[0]]}
del prompt_rows

lookup={word:i for i,word in enumerate(words)}
dataset=read(checked(COCO_JSON,'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d'))
rows={int(r['cocoid']):r for r in dataset['images'] if int(r['cocoid']) in index}
events=[]
image_index=[]
references={}
for cid in ids:
    j=index[cid]
    row=rows[cid]
    if row['split'] not in (('train','restval') if j<10000 else ('val',)):
        raise ValueError('Wrong split references')
    if j<10000:
        for sentence in row['sentences']:
            events.append(encode_tokens(sentence['tokens'],lookup))
            image_index.append(j)
    else:
        references[cid]=[dict(caption=s['raw']) for s in row['sentences']]
events=np.asarray(events,np.int64)
image_index=np.asarray(image_index,np.int64)
if len(references)!=1000 or len(set(image_index))!=10000:
    raise ValueError('Caption coverage differs')
del dataset,rows

class RegionCaptioner(CSAPatchCaptioner):
    def __init__(self,vocabulary,nouns,verbs):
        super().__init__(vocabulary,nouns,verbs)
        self.region_visual=nn.Linear(1024,512,bias=False)
        self.region_text=nn.Linear(1024,512,bias=False)

    def streams(self,patches,p):
        valid=torch.ones(patches.shape[:2],dtype=torch.bool,device=patches.device)
        vo,vr=self.core.encode(patches,valid)
        vision=p['region_visual']
        text=p['region_text']
        similarity=F.cosine_similarity(vision,text,dim=-1).clamp(-1,1)
        obj=torch.cat((self.region_text(text),self.region_visual(vision)*((1+similarity)/2)[...,None],
                       p['object_box']),dim=-1)
        rel=torch.cat((p['relation_text'],torch.bmm(p['subject_weights'],vo),
                       torch.bmm(p['target_weights'],vo),p['relation_geometry']),dim=-1)
        return (self.object_prompt(vo,obj,p['object_weights'],p['object_valid']),
                self.relation_prompt(vr,rel,p['subject_weights']+p['target_weights'],p['relation_valid']))

def inputs(indices):
    p={k:v[indices] for k,v in prompts.items() if not k.endswith('_text_id')}
    for role in ('object','relation'):
        p[role+'_text']=vectors[prompts[role+'_text_id'][indices]]*p[role+'_valid'][...,None]
    if ARM=='REGIONCLIP':
        p['region_visual']=np.asarray(input_arrays['region_vectors'][indices])
        slot_ids=input_arrays['text_id'][indices]
        p['region_text']=input_arrays['text_vectors'][slot_ids.clip(min=0)]*p['object_valid'][...,None]
    return torch.from_numpy(np.array(clip[indices],dtype=np.float32)).to(device),{
        k:torch.from_numpy(np.ascontiguousarray(v)).to(device) for k,v in p.items()}

banks,_,_=fixed_banks(BANK_ROOT)
torch.manual_seed(42)
model_class=CSAPatchCaptioner if ARM=='CLIP' else RegionCaptioner
model=model_class(len(words),torch.from_numpy(banks['noun']),torch.from_numpy(banks['verb'])).to(device)
torch.manual_seed(4242)
optimizer=torch.optim.Adam(model.parameters(),lr=8e-6)
steps=math.ceil(len(events)/32)
metrics_dir=WORK/'metrics'
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--target',str(metrics_dir),'pycocoevalcap==1.2'],
               check=True,timeout=300)
sys.path.insert(0,str(metrics_dir))
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider
ptb=PTBTokenizer()
refs=ptb.tokenize(references)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--target',str(metrics_dir),
                'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],
               check=True,timeout=300)
metric_env=dict(os.environ,PYTHONPATH=str(metrics_dir)+os.pathsep+os.environ.get('PYTHONPATH',''))
subprocess.run([sys.executable,'-c',"from pattern.en import singularize; assert singularize('umbrellas')=='umbrella'"],
               env=metric_env,check=True,timeout=60)
history=[]
best=None
best_captions=None
update=0
for epoch in range(1,11):
    model.train()
    order=np.random.default_rng(42+epoch).permutation(len(events))
    nll=0.
    count=0
    for start in range(0,len(order),32):
        batch=order[start:start+32]
        tokens=int(np.count_nonzero(events[batch,1:]))
        update+=1
        lr=(8e-6+(1e-4-8e-6)*(update-1)/(steps-1) if update<=steps else
            1e-5+.5*(1e-4-1e-5)*(1+math.cos(math.pi*(update-steps)/(9*steps))))
        optimizer.param_groups[0]['lr']=lr
        optimizer.zero_grad(set_to_none=True)
        batch_nll=0.
        for offset in range(0,len(batch),8):
            micro=batch[offset:offset+8]
            seq=events[micro]
            seq=seq[:,:int((seq!=0).sum(1).max())]
            seq=torch.from_numpy(np.ascontiguousarray(seq)).to(device)
            x,p=inputs(image_index[micro])
            scores=model(x,seq[:,:-1],p)
            loss=F.cross_entropy(scores.flatten(0,1),seq[:,1:].reshape(-1),ignore_index=0,reduction='sum')
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite XE loss')
            (loss/tokens).backward()
            batch_nll+=float(loss.detach())
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)
        optimizer.step()
        nll+=batch_nll
        count+=tokens
        if (start//32+1)%100==0:
            print(ARM,'epoch',epoch,'batch',start//32+1,'/',steps,flush=True)
    model.eval()
    generated=[]
    with torch.inference_mode():
        for start in range(10000,11000,8):
            batch=np.arange(start,min(start+8,11000))
            x,p=inputs(batch)
            sequences,_=beam_search(model,model.encode(x,p),width=5,maximum=20)
            generated.extend(dict(coco_id=ids[i],caption=text_from_ids(row,words))
                             for i,row in zip(batch,sequences[:,0].cpu().tolist(),strict=True))
    if [r['coco_id'] for r in generated]!=ids[10000:]:
        raise ValueError('Wrong VALID coverage')
    hyp=ptb.tokenize({r['coco_id']:[dict(caption=r['caption'])] for r in generated})
    cider,per_image=Cider().compute_score(refs,hyp)
    cider=float(cider)
    point=dict(epoch=epoch,CIDEr=cider,train_nll=nll/count,updates=update)
    history.append(point)
    if best is None or cider>best['CIDEr']:
        best=point
        best_captions=generated
        tmp=OUT/'best.tmp'
        torch.save(dict(model=model.state_dict(),arm=ARM,vocab=words,best=best),tmp)
        tmp.replace(OUT/'best.bin')
        write_json(OUT/'best_captions.json',generated)
        write_json(OUT/'best_cider_per_image.json',{i:float(v) for i,v in zip(refs,per_image,strict=True)})
    write_json(OUT/'progress.json',dict(arm=ARM,best=best,history=history))
    print(point,flush=True)

chair=WORK/'chair'
chair.mkdir()
base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
for name,digest in {'chair.py':'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d',
                    'chair.pkl':'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'}.items():
    download(base+name,chair/name,digest)
import nltk
for resource in ('punkt','punkt_tab','wordnet','averaged_perceptron_tagger_eng'):
    nltk.download(resource,quiet=True,raise_on_error=True)
source=OUT/'chair_input.json'
scored=OUT/'chair_output.json'
write_json(source,[dict(image_id=r['coco_id'],caption=r['caption']) for r in best_captions])
subprocess.run([sys.executable,str(chair/'chair.py'),'--cap_file',str(source),'--cache',str(chair/'chair.pkl'),
                '--save_path',str(scored)],cwd=chair,env=metric_env,check=True,timeout=600)
chair_metrics,_=chair_counts(read(scored),best_captions)
result=dict(status='COMPLETE',arm=ARM,train_images=10000,val_images=1000,train_caption_events=len(events),
            parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
            best=best,history=history,**chair_metrics)
write_json(OUT/'result.json',result)
print(result)
review=OUT.with_name(OUT.name+'_review.zip')
with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as archive:
    for name in ('result.json','progress.json','best_captions.json','best_cider_per_image.json','chair_output.json'):
        archive.write(OUT/name,name)
print('SEND:',review)

